# Variable-star pipeline for NGC 2506 and NGC 6709

This notebook reads DAOPHOT R-filter photometry, finds variable candidates, estimates periods and light-curve features, cross-matches Gaia DR3, and writes all-star astrometric membership and colour–magnitude products. Run the setup and analysis sections in order. Input data are supplied separately; see README.md.


## Setup

Set input/output roots and select one or both clusters. Input Julian dates are interpreted using the declared time scale. Each cluster has separate candidate and period outputs.


In [ ]:
from pathlib import Path

# Edit these two roots and choose one or both clusters.
DATA_ROOT = Path.cwd() / "data"
OUTPUT_ROOT = Path.cwd() / "outputs"
SELECTED_CLUSTERS = ("ngc2506", "ngc6709")

CLUSTER_PATHS = {
    "ngc2506": {
        "label": "NGC 2506",
        "rfilter": DATA_ROOT / "NGC2506" / "RFILTER",
        "results": OUTPUT_ROOT / "ngc2506",
    },
    "ngc6709": {
        "label": "NGC 6709",
        "rfilter": DATA_ROOT / "NGC6709" / "RFILTER",
        "results": OUTPUT_ROOT / "ngc6709",
    },
}
TIME_BASIS_SETTING = "hjd"
JD_INPUT_SCALE_SETTING = "utc"
USE_GAIA_CACHE = False
TABLE_ROOT = OUTPUT_ROOT / "tables"
FIGURE_ROOT = OUTPUT_ROOT / "figures"
CACHE_ROOT = OUTPUT_ROOT / "cache"

if not SELECTED_CLUSTERS or any(key not in CLUSTER_PATHS for key in SELECTED_CLUSTERS):
    raise ValueError("SELECTED_CLUSTERS must contain ngc2506 and/or ngc6709")
for key in SELECTED_CLUSTERS:
    CLUSTER_PATHS[key]["results"].mkdir(parents=True, exist_ok=True)
TABLE_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)


## 1. Candidate discovery

Candidates are ranked from zero-point-corrected differential photometry and time-binned variability relative to comparison/check stars. The configured thresholds and candidate cap are part of the method; a candidate list is not a confirmed-variable catalogue.


In [ ]:
from pathlib import Path
import csv, time, warnings
import numpy as np
from scipy.spatial import cKDTree

MATCH_RADIUS_PX = 2.5
MAX_MAG_ERROR = 0.20
MIN_ZP_FRAMES = 100
MIN_OBS_FRAC = 0.30
COMP_OBS_FRAC = 0.50
PAIR_RADIUS_PX = 350.0
PAIR_MAG_TOL = 1.5
COMP_POOL = 8
CHECK_MAX_SD = 0.020
BIN_DAYS = 0.01
MIN_BIN_POINTS = 10
SNR_MIN = 2.0

MAX_CANDIDATES = 400
MAX_FRAMES = None

def read_ref(path):
    rows = []
    with open(path) as h:
        for line in h:
            p = line.split()
            if len(p) != 9:
                continue
            try:
                rows.append([float(v) for v in p])
            except ValueError:
                pass
    return np.asarray(rows, np.float32) if rows else np.empty((0, 9), np.float32)

def read_catalog(path):
    sid, x, y, ra, dec = [], [], [], [], []
    with open(path) as h:
        for line in h:
            p = line.split()
            if len(p) < 5:
                continue
            try:
                sid.append(int(p[0])); x.append(float(p[1])); y.append(float(p[2]))
            except ValueError:
                continue
            ra.append(p[3]); dec.append(p[4])
    return (np.asarray(sid, np.int32), np.asarray(x, np.float32), np.asarray(y, np.float32),
            np.asarray(ra, object), np.asarray(dec, object))

def read_jd(path):
    rows = []
    with open(path) as h:
        for line in h:
            p = line.split()
            if len(p) < 2:
                continue
            try:
                rows.append((p[0], float(p[1])))
            except ValueError:
                pass
    return rows

def match(table, tree, n_cat, radius, max_err=None):
    if len(table) == 0:
        return np.empty(0, np.int32), np.empty(0, np.int32)
    ok = np.isfinite(table[:, 1]) & np.isfinite(table[:, 2]) & np.isfinite(table[:, 3])
    if max_err is not None:
        ok &= np.isfinite(table[:, 4]) & (table[:, 4] > 0) & (table[:, 4] <= max_err)
    rn = np.flatnonzero(ok)
    if len(rn) == 0:
        return np.empty(0, np.int32), np.empty(0, np.int32)
    dist, idx = tree.query(table[rn][:, 1:3], distance_upper_bound=radius)
    m = np.isfinite(dist) & (idx < n_cat)
    if not np.any(m):
        return np.empty(0, np.int32), np.empty(0, np.int32)
    rn, idx, dist = rn[m], idx[m].astype(np.int32), dist[m]
    order = np.argsort(dist, kind="mergesort")
    _, first = np.unique(idx[order], return_index=True)
    return rn[order[first]], idx[order[first]]

def rsig(v):
    m = np.isfinite(v)
    if m.sum() < min_common:
        return np.nan
    x = v[m]
    return float(1.4826 * np.median(np.abs(x - np.median(x))))

def bsig(diff):
    m = np.isfinite(diff)
    if m.sum() < min_common:
        return np.nan
    s = np.bincount(bin_id[m], diff[m], minlength=n_bins)
    c = np.bincount(bin_id[m], minlength=n_bins)
    ok = c >= MIN_BIN_POINTS
    if ok.sum() < 10:
        return np.nan
    mu = s[ok] / c[ok]
    return float(1.4826 * np.median(np.abs(mu - np.median(mu))))

def write_csv(path, columns, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", newline="") as h:
        w = csv.DictWriter(h, fieldnames=columns)
        w.writeheader()
        for r in rows:
            w.writerow({c: r.get(c, "") for c in columns})

for cluster_key in SELECTED_CLUSTERS:
    cluster_paths = CLUSTER_PATHS[cluster_key]
    RFILTER_DIR = cluster_paths["rfilter"]
    JD_PATH = RFILTER_DIR / "JD_all.dat"
    REFIMAGE_PATH = RFILTER_DIR / "refimage.ref"
    CATALOG_PATH = RFILTER_DIR / "ref_stars_xy_radec.dat"
    OUTPUT_DIR = cluster_paths["results"]
    print(f"\nCandidate discovery: {cluster_paths['label']}")
    t0 = time.time()
    sid, cx, cy, ra, dec = read_catalog(CATALOG_PATH)
    pos = np.column_stack([cx, cy]).astype(np.float32)
    n = len(sid)
    tree = cKDTree(pos)

    refimg = read_ref(REFIMAGE_PATH)
    rr, ri = match(refimg, tree, n, MATCH_RADIUS_PX)
    ref_mag = np.full(n, np.nan, np.float32); ref_x = np.full(n, np.nan, np.float32); ref_y = np.full(n, np.nan, np.float32)
    ref_mag[ri] = refimg[rr, 3]; ref_x[ri] = refimg[rr, 1]; ref_y[ri] = refimg[rr, 2]

    jd_rows = read_jd(JD_PATH)
    if MAX_FRAMES:
        jd_rows = jd_rows[:MAX_FRAMES]
    F = len(jd_rows)
    mags = np.full((F, n), np.nan, np.float32)
    frame_jd = np.array([j for _, j in jd_rows], np.float64)
    fmc = np.zeros(F, np.int32)
    missing = 0
    for i, (name, _) in enumerate(jd_rows):
        p = RFILTER_DIR / Path(name).with_suffix(".ref").name
        if not p.exists():
            missing += 1; continue
        tab = read_ref(p)
        rn, mi = match(tab, tree, n, MATCH_RADIUS_PX, MAX_MAG_ERROR)
        if len(rn):
            mags[i, mi] = tab[rn, 3]
        fmc[i] = len(rn)
        if (i + 1) % 2000 == 0 or (i + 1) == F:
            print(f"  read {i+1}/{F} frames ({time.time()-t0:.0f}s)", flush=True)

    valid = np.isfinite(ref_mag)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        zp = np.nanmedian(mags[:, valid] - ref_mag[valid][None, :], axis=1).astype(np.float32)
        usable = np.isfinite(zp) & (fmc >= MIN_ZP_FRAMES)
        corr = mags - zp[:, None]
        corr[~usable, :] = np.nan
        n_obs = np.sum(np.isfinite(corr), axis=0).astype(np.int32)
        med = np.nanmedian(corr, axis=0).astype(np.float32)
        sd = np.nanstd(corr, axis=0, ddof=1).astype(np.float32)
        rms = np.sqrt(np.nanmean((corr - med[None, :]) ** 2, axis=0)).astype(np.float32)
        nmad = (1.4826 * np.nanmedian(np.abs(corr - med[None, :]), axis=0)).astype(np.float32)
    score = np.hypot(sd, rms).astype(np.float32)
    X = np.where(np.isfinite(ref_x), ref_x, cx)
    Y = np.where(np.isfinite(ref_y), ref_y, cy)

    bin_id = np.floor((frame_jd - np.nanmin(frame_jd)) / BIN_DAYS).astype(np.int64)
    bin_id -= bin_id.min()
    n_bins = int(bin_id.max()) + 1
    min_obs = int(MIN_OBS_FRAC * F); min_obs_comp = int(COMP_OBS_FRAC * F)
    min_common = max(200, int(0.25 * F))

    star_rows = [{
        "star_id": int(sid[i]), "x": f"{X[i]:.6f}", "y": f"{Y[i]:.6f}", "ra": ra[i], "dec": dec[i],
        "ref_mag": (f"{ref_mag[i]:.6f}" if np.isfinite(ref_mag[i]) else ""), "n_obs": int(n_obs[i]),
        "star_sd": (f"{sd[i]:.6f}" if np.isfinite(sd[i]) else ""),
        "star_rms": (f"{rms[i]:.6f}" if np.isfinite(rms[i]) else ""),
        "star_score": (f"{score[i]:.6f}" if np.isfinite(score[i]) else ""),
    } for i in range(n)]

    target_ok = np.isfinite(ref_mag) & np.isfinite(nmad) & (n_obs >= min_obs)
    comp_ok = np.isfinite(ref_mag) & np.isfinite(nmad) & (n_obs >= min_obs_comp)
    cands = []
    for vi in np.flatnonzero(target_ok):
        mg = ref_mag[vi]
        nb = [j for j in tree.query_ball_point(pos[vi], PAIR_RADIUS_PX)
              if j != vi and comp_ok[j] and abs(ref_mag[j] - mg) <= PAIR_MAG_TOL]
        if len(nb) < 2:
            continue
        nb = sorted(nb, key=lambda j: nmad[j])[:COMP_POOL]
        ci = nb[0]
        ki, kbest = None, np.inf
        for k in nb[1:]:
            s = rsig(corr[:, ci] - corr[:, k])
            if np.isfinite(s) and s < kbest:
                kbest, ki = s, k
        if ki is None or kbest > CHECK_MAX_SD:
            continue
        bvc = bsig(corr[:, vi] - corr[:, ci])
        bck = bsig(corr[:, ci] - corr[:, ki])
        if not (np.isfinite(bvc) and np.isfinite(bck)) or bck <= 0:
            continue
        snr = bvc / bck
        if snr < SNR_MIN:
            continue
        cands.append({
            "variable_star_id": int(sid[vi]), "nonvariable_star_id": int(sid[ci]),
            "check_star_id": int(sid[ki]), "sd_ratio": snr,
            "variable_sd": bvc, "nonvariable_sd": bck,
            "variable_rms": rsig(corr[:, vi] - corr[:, ci]), "nonvariable_rms": kbest,
            "distance_px": float(np.hypot(X[vi] - X[ci], Y[vi] - Y[ci])),
            "mag_diff": float(abs(ref_mag[vi] - ref_mag[ci])),
            "variable_ref_mag": float(ref_mag[vi]), "nonvariable_ref_mag": float(ref_mag[ci]),
            "variable_x": float(X[vi]), "variable_y": float(Y[vi]),
            "nonvariable_x": float(X[ci]), "nonvariable_y": float(Y[ci]),
            "variable_n_obs": int(n_obs[vi]), "nonvariable_n_obs": int(n_obs[ci]),
            "anchor_source": "binned_coherence_snr",
        })

    cands.sort(key=lambda r: (-r["sd_ratio"], -r["variable_sd"], r["distance_px"], r["variable_star_id"]))
    if len(cands) > MAX_CANDIDATES:
        cands = cands[:MAX_CANDIDATES]
    for rank, r in enumerate(cands, 1):
        r["candidate_rank"] = rank

    cand_cols = ["candidate_rank", "variable_star_id", "nonvariable_star_id", "check_star_id",
                 "sd_ratio", "variable_sd", "nonvariable_sd", "variable_rms", "nonvariable_rms",
                 "distance_px", "mag_diff", "variable_ref_mag", "nonvariable_ref_mag",
                 "variable_x", "variable_y", "nonvariable_x", "nonvariable_y",
                 "variable_n_obs", "nonvariable_n_obs", "anchor_source"]
    star_cols = ["star_id", "x", "y", "ra", "dec", "ref_mag", "n_obs", "star_sd", "star_rms", "star_score"]
    write_csv(OUTPUT_DIR / "cell0_all_star_diagnostics.csv", star_cols, star_rows)
    write_csv(OUTPUT_DIR / "cell1_variable_candidates.csv", cand_cols, cands)

    print(f"\nDone in {time.time()-t0:.0f}s | stars={n} ref-matched={int(valid.sum())} frames={F} "
          f"missing={missing} usable-ZP={int(usable.sum())}")
    print(f"Variable candidates: {len(cands)} (SNR>={SNR_MIN}, cap {MAX_CANDIDATES}, "
          f"R<={PAIR_RADIUS_PX}px, |dmag|<={PAIR_MAG_TOL}, check sigma<={CHECK_MAX_SD})")
    print(f"Saved -> {OUTPUT_DIR}")
    print(f"   cell1_variable_candidates.csv  ({len(cands)} variable candidates)")
    print(f"   cell0_all_star_diagnostics.csv  ({n} stars)")
    print("\nTop 15 candidates (rank id snr binned_var binned_floor mag dist):")
    for r in cands[:15]:
        print(f"  {r['candidate_rank']:3d} {r['variable_star_id']:5d} snr={r['sd_ratio']:6.2f} "
              f"v={r['variable_sd']:.4f} f={r['nonvariable_sd']:.4f} "
              f"m={r['variable_ref_mag']:5.2f} d={r['distance_px']:5.0f}px comp={r['nonvariable_star_id']}")


## 2. NGC 2506 periods and features

The period engine combines weighted Lomb–Scargle searches, comparison-series diagnostics and configured alias/harmonic rules. It exports analysed light-curve points, candidate periods, adopted-period summaries, shape features and conditional period/amplitude uncertainties. Its phase figures show selected representative observations; use the point table to assess the full analysed scatter.


In [ ]:
import numpy as np
import glob
import math
import os
import re
import matplotlib.pyplot as plt
import pandas as pd
import astropy.units as u
from astropy.coordinates import EarthLocation, SkyCoord
from astropy.time import Time
from astropy.timeseries import LombScargle
from astropy.utils import iers
from scipy.spatial import cKDTree
from scipy.optimize import minimize_scalar
_paths_2506 = CLUSTER_PATHS["ngc2506"]
COMPARE_CSV = str(_paths_2506["results"] / "cell1_variable_candidates.csv")
MASTER_MAP_PATH = str(_paths_2506["rfilter"] / "ref_stars_xy_radec.dat")
MASTER_REF_PATH = str(_paths_2506["rfilter"] / "refimage.ref")
JD_FILE = str(_paths_2506["rfilter"] / "JD_all.dat")
REF_DIR = str(_paths_2506["rfilter"])
OUT_DIR = str(_paths_2506["results"] / "periods")
TIME_BASIS = TIME_BASIS_SETTING
JD_INPUT_SCALE = JD_INPUT_SCALE_SETTING
OBS_LAT_DEG = 29.3612
OBS_LON_DEG = 79.6868
OBS_HEIGHT_M = 2450.0
LOCAL_PERIODOGRAM_VIEW = True
LOCAL_PERIODOGRAM_REL_WIDTH = 0.35
LOCAL_PERIODOGRAM_MIN_WIDTH_D = 0.12
LOCAL_PERIODOGRAM_SAMPLES_PER_PEAK = 60
COL_STAR_ID = 'variable_star_id'
COL_COMP1_ID = 'comp1_star_id'
COL_COMP2_ID = 'comp2_star_id'
iers.conf.auto_download = False
iers.conf.auto_max_age = None
OBSERVATORY_LOCATION = EarthLocation.from_geodetic(lon=float(OBS_LON_DEG) * u.deg, lat=float(OBS_LAT_DEG) * u.deg, height=float(OBS_HEIGHT_M) * u.m)
MATCH_TOL_PX = 1.2
TRANSFORM_SEED_TOL_PX = 3.0
TRANSFORM_MIN_SEEDS = 40
TRANSFORM_MAX_SEEDS = 350
TRANSFORM_RESID_TOL_PX = 1.6
MAX_MEASUREMENT_ERR = 0.3
SEED_ERR_MAX = 0.2
FRAME_ZP_CLIP_NSIGMA = 3.5
FRAME_ZP_CLIP_MAXITER = 5
MIN_POINTS_AFTER_CLEAN = 35
SERIES_CLIP_NSIGMA = 4.5
SERIES_CLIP_MAXITER = 4
APPLY_TIME_BINNING = True
TIME_BIN_DAYS = 0.0015
TIME_BIN_MIN_PTS = 1
POST_BIN_CLIP_NSIGMA = 5.5
POST_BIN_CLIP_MAXITER = 2
APPLY_NIGHTLY_DETREND = False
NIGHTLY_MIN_PTS = 5
SEARCH_NIGHTLY_DETREND_VARIANT = True
VARIANT_STABILITY_REL_TOL = 0.08
VARIANT_HIGH_DRIFT_RATIO = 2.0
VARIANT_EXTREME_DRIFT_RATIO = 3.0
VARIANT_MIN_LONG_PERIOD_D = 0.60
VARIANT_MIN_LONG_CYCLES = 1.5
VARIANT_COLLAPSE_SIGMA_RATIO = 0.55
VARIANT_COLLAPSE_AMP_RATIO = 0.5
VARIANT_NIGHTLY_SHORT_PERIOD_D = 0.25
VARIANT_ALIAS_BREAK_SHORT_PERIOD_D = 0.12
VARIANT_NIGHTLY_COMPETITIVE_SCORE_DELTA = 0.55
VARIANT_NIGHTLY_MIN_AMP_RATIO = 0.35
VARIANT_ALIAS_BREAK_PERIOD_RATIO = 6.0
VARIANT_ALIAS_BREAK_THETA_MAX = 1.0
VARIANT_CLEAN_SHORT_PERIOD_D = 0.12
VARIANT_CLEAN_SHORT_SCORE_MARGIN = 0.02
VARIANT_CLEAN_SHORT_POWER_GAIN = 0.1
VARIANT_CLEAN_SHORT_GAP_GAIN = 0.04
VARIANT_CLEAN_SHORT_THETA_MARGIN = 0.35
VARIANT_CLEAN_SHORT_MIN_COLLAPSE_RATIO = 0.55
VARIANT_NIGHTLY_RETRY_LONG_PERIOD_D = 1.8
VARIANT_NIGHTLY_RETRY_GAP_MIN = 0.06
PHASE_CLEAN_MAX_REMOVAL_FRAC = 0.35
PHASE_CLEAN_DISPLAY_RESID_RATIO_MAX = 0.98
PHASE_CLEAN_PAIR_BONUS = 0.1
PHASE_CLEAN_PAIR_GAP_GAIN = 0.01
PHASE_CLEAN_PAIR_THETA_GAIN = 0.035
PHASE_CLEAN_PAIR_MODEL_GAIN = 0.04
PHASE_CLEAN_DIRECT_PERIOD_TOL = 0.04
PHASE_CLEAN_HIGH_REMOVAL_FRAC = 0.12
PERIOD_MIN_D = 0.02
PERIOD_MAX_D = 20.0
LS_SEARCH_NTERMS = (1, 2)
LS_EVAL_NTERMS_OPTIONS = (1, 2, 4)
LS_EVAL_BIC_MARGIN = 4.0
LS_SAMPLES_PER_PEAK = 8
TOP_PEAKS_PER_NTERM = 4
PEAK_DEDUP_FRAC = 0.004
FAMILY_MULTIPLIERS = (1.0, 2.0)
SAVE_PLOTS = True
LOCAL_REFINE_RESOLUTION_ELEMENTS = 3.0
LOCAL_REFINE_GRID_SIZE = 181
ALIAS_FREQ_TOL_SOFT = 0.03
ALIAS_FREQ_TOL_HARD = 0.012
ALIAS_MAX_HARMONIC = 8
ALIAS_INCLUDE_SUBHARMONICS = True
NBINS_SCORE = 70
MIN_BIN_PTS = 4
MIN_BINS_USED = 6
SHOW_PERIOD_MARKER_LINE = False
DOUBLE_CYCLE_PLOT = True
PLOT_DPI = 220
COMBINED_PHASE_GRID_DPI = PLOT_DPI
PLOT_DIR = os.path.join(OUT_DIR, 'periodogram_phase_plots')
SAVE_COMBINED_PHASE_GRID = True
COMBINED_PHASE_GRID_COLS = 6
COMBINED_PHASE_GRID_ROWS = 14
COMBINED_PHASE_GRID_FIGSIZE = (8.27, 11.69)
CLEAN_DISCRETE_PHASE_BIN_WIDTH = 0.02
CLEAN_DISCRETE_POINT_MODE = 'binned_median'
CLEAN_DISCRETE_POINT_MODE_CHOICES = {'smooth_binned_median', 'binned_median', 'representative_center', 'representative_mediany'}
CLEAN_DISCRETE_BIN_ERROR_MODE = 'median'
CLEAN_DISCRETE_BIN_SMOOTH_HALF_WINDOW = 0
CLEAN_DISCRETE_BIN_SMOOTH_BLEND = 0.0
CLEAN_DISCRETE_PREFER_LOW_ERROR = True
CLEAN_DISCRETE_MAX_ERROR_PERCENTILE = 100.0
CLEAN_DISCRETE_AUTO_PHASE_SHIFT = True
CLEAN_DISCRETE_ECLIPSE_PHASE_ANCHOR = True
CLEAN_DISCRETE_ECLIPSE_ANCHOR_PHASE = 0.16
CLEAN_DISCRETE_ECLIPSE_ANCHOR_MIN_AMP = 0.06
CLEAN_DISCRETE_ECLIPSE_ANCHOR_ERR_RATIO = 6.0
CLEAN_DISCRETE_SHORT_PERIOD_PHASE_ANCHOR = True
CLEAN_DISCRETE_SHORT_PERIOD_PHASE_ANCHOR_MAX_D = 0.12
CLEAN_DISCRETE_SHORT_PERIOD_PHASE_ANCHOR_PHASE = 0.75
CLEAN_DISCRETE_SHORT_PERIOD_PHASE_ANCHOR_MIN_AMP = 0.004
CLEAN_DISCRETE_PHASE_SHIFT_SMOOTH_BINS = 2
CLEAN_DISCRETE_CONTINUITY_SELECTION = True
CLEAN_DISCRETE_CONTINUITY_SMOOTH_BINS = 2
CLEAN_DISCRETE_HIDE_LOCAL_OUTLIERS = True
CLEAN_DISCRETE_OUTLIER_Z = 4.0
CLEAN_DISCRETE_OUTLIER_MIN_MAG = 0.012
CLEAN_DISCRETE_MAX_OUTLIER_REMOVAL_FRAC = 0.1
COMBINED_PHASE_GRID_OUT = os.path.join(PLOT_DIR, 'all_phase_diagrams_grid.png')
OUT_POINTS = os.path.join(OUT_DIR, '04_relative_lightcurves_points_newid.csv')
OUT_VARIABLE_STATS = os.path.join(PLOT_DIR, '04_variable_star_amplitude_skewness_period.csv')
OUT_SUMMARY = os.path.join(OUT_DIR, '04_period_phase_summary.csv')
OUT_TOP_CANDIDATES = os.path.join(OUT_DIR, '04_period_phase_top_candidates.csv')
OUT_FRAME_DIAG = os.path.join(OUT_DIR, '04_period_phase_frame_diagnostics.csv')
TOP_CANDIDATES_TO_SAVE = 6
CONSENSUS_CANDIDATES_PER_SOLUTION = 6
MODEL_REFIT_TOP_N = 10
BASE_RANK_BONUS_SCALE = 0.18
SEARCH_NTERMS2_PENALTY = 0.08
FAMILY_SUPPORT_REL_TOL = 0.08
FAMILY_SEED_TOP_ROWS = 4
FAMILY_SEED_MULTIPLIERS = (0.5, 1.0, 2.0)
FAMILY_COMPETITIVE_MARGIN = 0.2
FAMILY_MEMBER_BONUS = 0.1
FAMILY_SEARCH_BONUS = 0.04
FAMILY_SOURCE_BONUS = 0.05
FAMILY_SHORTER_HARMONIC_BONUS = 0.04
FAMILY_SHORTER_SCORE_MARGIN = 0.16
FAMILY_SHORTER_GAP_MARGIN = 0.02
FAMILY_SHORTER_MAX_PERIOD_D = 1.3
AMBIGUOUS_SHORTER_PERIOD_RATIO = 0.8
TARGETED_LS_TOP_FAMILIES = 3
TARGETED_LS_REL_WIDTH = 0.3
TARGETED_LS_GRID_SIZE = 601
TARGETED_LS_TOP_PEAKS = 3
TARGETED_LS_MULTIPLIERS = (1.0 / 3.0, 0.4, 0.5, 0.55, 0.6, 0.75, 0.8, 0.85, 0.9, 1.0, 1.25, 1.5, 2.0)
TARGETED_LS_SHORTER_SEED_MIN_PERIOD_D = 2.0
TARGETED_LS_SHORTER_GAP_TRIGGER = 0.07
SHORT_SCAN_MIN_PERIOD_D = 0.05
SHORT_SCAN_MAX_PERIOD_D = 0.25
SHORT_SCAN_MULTIPLIERS = (1.0 / 3.0, 0.4, 0.5, 1.0, 2.0)
SHORT_SCAN_SAMPLES_PER_PEAK = 30
SHORT_SCAN_TOP_PEAKS_PER_NTERM = 3
SHORT_ALIAS_RESCUE_LONG_PERIOD_D = 1.45
SHORT_ALIAS_RESCUE_MIN_PERIOD_D = 0.4
SHORT_ALIAS_RESCUE_MAX_PERIOD_D = 1.0
SHORT_ALIAS_RESCUE_RATIO_MIN = 1.8
SHORT_ALIAS_RESCUE_RATIO_MAX = 3.4
SHORT_ALIAS_RESCUE_ADJUSTED_MARGIN = 1.1
SHORT_ALIAS_RESCUE_GAP_GAIN = 0.015
SHORT_ALIAS_RESCUE_THETA_MARGIN = 0.08
SHORT_ALIAS_RESCUE_MODEL_GAIN_MARGIN = 0.12
THIRD_HARMONIC_RESCUE_LONG_MIN_D = 1.5
THIRD_HARMONIC_RESCUE_LONG_MAX_D = 2.8
THIRD_HARMONIC_RESCUE_SHORT_MIN_D = 0.6
THIRD_HARMONIC_RESCUE_SHORT_MAX_D = 0.9
THIRD_HARMONIC_RESCUE_RATIO_MIN = 2.4
THIRD_HARMONIC_RESCUE_RATIO_MAX = 3.6
THIRD_HARMONIC_RESCUE_ADJUSTED_MARGIN = 1.45
THIRD_HARMONIC_RESCUE_GAP_MAX = 0.06
THIRD_HARMONIC_RESCUE_THETA_MAX = 0.8
SHORT_PERIOD_PHASE_REFINE = True
SHORT_PERIOD_PHASE_REFINE_MAX_D = 0.30
SHORT_PERIOD_PHASE_REFINE_REL_WIDTH = 0.004
SHORT_PERIOD_PHASE_REFINE_GRID_SIZE = 401
SHORT_PERIOD_PHASE_REFINE_THETA_MARGIN = 0.02
SHORT_PERIOD_PHASE_REFINE_MIN_THETA_GAIN = 0.05
MEDIUM_PERIOD_PHASE_REFINE_MIN_D = 0.12
MEDIUM_PERIOD_PHASE_REFINE_REL_WIDTH = 0.04
MEDIUM_PERIOD_PHASE_REFINE_GRID_SIZE = 801
MEDIUM_PERIOD_PHASE_REFINE_MIN_THETA_GAIN = 0.03
RAW_LONG_PRESERVE_MAX_GAP = 0.16
RAW_LONG_PRESERVE_GAP_MARGIN = 0.08
COMPACT_LOCAL_MIN_PERIOD_D = 0.65
COMPACT_LOCAL_MAX_PERIOD_D = 0.8
COMPACT_LOCAL_MAX_GAP = 0.02
COMPACT_LOCAL_SCORE_MARGIN = 0.8
COMPACT_LOCAL_TOP_FAMILIES = 2
COMPACT_LOCAL_REL_WIDTH = 0.035
COMPACT_LOCAL_MIN_WIDTH_D = 0.015
COMPACT_LOCAL_GRID_SIZE = 121
COMPACT_LOCAL_TOP_PEAKS = 2
TARGETED_MIDFAMILY_TRIGGER_MIN_PERIOD_D = 1.65
TARGETED_MIDFAMILY_TRIGGER_MAX_PERIOD_D = 1.9
TARGETED_MIDFAMILY_TRIGGER_MIN_GAP = 0.08
TARGETED_MIDFAMILY_SHORTER_MIN_PERIOD_D = 1.45
TARGETED_MIDFAMILY_SHORTER_MAX_PERIOD_D = 1.7
TARGETED_MIDFAMILY_SHORTER_RATIO_MIN = 1.04
TARGETED_MIDFAMILY_SHORTER_RATIO_MAX = 1.22
TARGETED_MIDFAMILY_SHORTER_ADJUSTED_MARGIN = 0.9
TARGETED_MIDFAMILY_SHORTER_GAP_GAIN = 0.008
TARGETED_MIDFAMILY_SHORTER_THETA_MARGIN = 0.12
COMPACT_MID_OVERRIDE_LONG_MIN_D = 1.2
COMPACT_MID_OVERRIDE_LONG_MAX_D = 2.3
COMPACT_MID_OVERRIDE_SHORT_MIN_D = 0.65
COMPACT_MID_OVERRIDE_SHORT_MAX_D = 0.8
COMPACT_MID_OVERRIDE_RATIO_MIN = 1.55
COMPACT_MID_OVERRIDE_RATIO_MAX = 2.9
COMPACT_MID_OVERRIDE_ADJUSTED_MARGIN = 0.8
COMPACT_MID_OVERRIDE_GAP_MAX = 0.02
COMPACT_MID_OVERRIDE_THETA_MAX = 0.8
COMPACT_MID_OVERRIDE_TRIGGER_THETA_MIN = 0.65
COMPACT_MID_OVERRIDE_TRIGGER_GAP_MIN = 0.07
COMPACT_FAMILY_CLEANUP_MIN_PERIOD_D = 0.64
COMPACT_FAMILY_CLEANUP_MAX_PERIOD_D = 0.8
COMPACT_FAMILY_CLEANUP_ADJUSTED_MARGIN = 0.45
COMPACT_FAMILY_CLEANUP_GAP_GAIN = 0.004
COMPACT_FAMILY_CLEANUP_THETA_MARGIN = 0.08
REVERSE_HARMONIC_RESCUE_SHORT_MIN_D = 0.7
REVERSE_HARMONIC_RESCUE_SHORT_MAX_D = 1.0
REVERSE_HARMONIC_RESCUE_LONG_MIN_D = 1.35
REVERSE_HARMONIC_RESCUE_LONG_MAX_D = 3.2
REVERSE_HARMONIC_RESCUE_RATIO_MIN = 1.8
REVERSE_HARMONIC_RESCUE_RATIO_MAX = 2.25
REVERSE_HARMONIC_RESCUE_ADJUSTED_MARGIN = 0.7
REVERSE_HARMONIC_RESCUE_GAP_MARGIN = 0.05
REVERSE_HARMONIC_RESCUE_THETA_GAIN = 0.05
REVERSE_HARMONIC_RESCUE_STRONG_GAP_GAIN = 0.015
REVERSE_HARMONIC_RESCUE_STRONG_THETA_GAIN = 0.1
REVERSE_THIRD_RESCUE_SHORT_MIN_D = 0.7
REVERSE_THIRD_RESCUE_SHORT_MAX_D = 1.0
REVERSE_THIRD_RESCUE_LONG_MIN_D = 2.4
REVERSE_THIRD_RESCUE_LONG_MAX_D = 3.2
REVERSE_THIRD_RESCUE_RATIO_MIN = 2.7
REVERSE_THIRD_RESCUE_RATIO_MAX = 3.4
REVERSE_THIRD_RESCUE_ADJUSTED_MARGIN = 0.45
REVERSE_THIRD_RESCUE_GAP_MARGIN = 0.03
REVERSE_THIRD_RESCUE_THETA_GAIN = 0.05
HALF_LONG_ALIAS_RESCUE_LONG_MIN_D = 2.1
HALF_LONG_ALIAS_RESCUE_LONG_MAX_D = 3.0
HALF_LONG_ALIAS_RESCUE_SHORT_MIN_D = 1.0
HALF_LONG_ALIAS_RESCUE_SHORT_MAX_D = 1.35
HALF_LONG_ALIAS_RESCUE_RATIO_MIN = 1.8
HALF_LONG_ALIAS_RESCUE_RATIO_MAX = 2.4
HALF_LONG_ALIAS_RESCUE_ADJUSTED_MARGIN = 0.85
HALF_LONG_ALIAS_RESCUE_GAP_GAIN = 0.015
HALF_LONG_ALIAS_RESCUE_THETA_MAX = 0.86
HALF_LONG_ALIAS_RESCUE_TRIGGER_THETA_MIN = 0.75
MIDPERIOD_ALIAS_RESCUE_LONG_MIN_D = 1.35
MIDPERIOD_ALIAS_RESCUE_LONG_MAX_D = 1.9
MIDPERIOD_ALIAS_RESCUE_SHORT_MIN_D = 1.0
MIDPERIOD_ALIAS_RESCUE_SHORT_MAX_D = 1.25
MIDPERIOD_ALIAS_RESCUE_RATIO_MIN = 1.15
MIDPERIOD_ALIAS_RESCUE_RATIO_MAX = 1.7
MIDPERIOD_ALIAS_RESCUE_ADJUSTED_MARGIN = 0.38
MIDPERIOD_ALIAS_RESCUE_GAP_GAIN = 0.004
MIDPERIOD_ALIAS_RESCUE_THETA_MARGIN = 0.05
MIDPERIOD_ALIAS_RESCUE_MODEL_GAIN_MARGIN = 0.05
MIDPERIOD_ALIAS_RESCUE_TRIGGER_GAP_MIN = 0.05
MIDPERIOD_ALIAS_RESCUE_TRIGGER_THETA_MIN = 0.54
THIRD_HARMONIC_COMPACT_CLEANUP_BASE_MIN_D = 0.64
THIRD_HARMONIC_COMPACT_CLEANUP_BASE_MAX_D = 0.74
THIRD_HARMONIC_COMPACT_CLEANUP_ALT_MIN_D = 0.82
THIRD_HARMONIC_COMPACT_CLEANUP_ALT_MAX_D = 0.9
THIRD_HARMONIC_COMPACT_CLEANUP_RATIO_MIN = 1.15
THIRD_HARMONIC_COMPACT_CLEANUP_RATIO_MAX = 1.45
THIRD_HARMONIC_COMPACT_CLEANUP_TRIGGER_GAP_MIN = 0.006
THIRD_HARMONIC_COMPACT_CLEANUP_ALT_GAP_MAX = 0.04
THIRD_HARMONIC_COMPACT_CLEANUP_ADJUSTED_MARGIN = 0.05
THIRD_HARMONIC_COMPACT_CLEANUP_GAP_MARGIN = 0.03
THIRD_HARMONIC_COMPACT_CLEANUP_THETA_GAIN = 0.06
THIRD_HARMONIC_COMPACT_CLEANUP_MODEL_GAIN_MARGIN = 0.05
FINAL_DOMINANT_RESTORE_ADJUSTED_MARGIN = 0.18
FINAL_DOMINANT_RESTORE_SCORE_MARGIN = 0.12
FINAL_DOMINANT_RESTORE_GAP_MARGIN = 0.06
FINAL_DOMINANT_RESTORE_THETA_MARGIN = 0.2
FINAL_DOMINANT_RESTORE_MODEL_GAIN_MARGIN = 0.05
FINAL_DOMINANT_RESTORE_PERIOD_RATIO_MAX = 3.5
LONG_ALIAS_RESCUE_TRIGGER_MIN_PERIOD_D = 5.0
LONG_ALIAS_RESCUE_MAX_PHASE_OCCUPANCY = 30
LONG_ALIAS_RESCUE_SHORTER_RATIO_MAX = 0.35
LONG_ALIAS_RESCUE_MIN_OCCUPANCY_GAIN = 15
LONG_ALIAS_RESCUE_SCORE_MARGIN = 0.15
LONG_ALIAS_RESCUE_GAP_MARGIN = 0.02
LONG_ALIAS_RESCUE_OCCUPANCY_NBINS = 50
SELECTION_POOL_MAX_ROWS = 120
SELECTION_POOL_SCORE_MARGIN = 1.65
SELECTION_POOL_REL_TOL = 0.05
SELECTION_CLUSTER_MEMBER_MARGIN = 0.55
SELECTION_CLUSTER_SUPPORT_BONUS = 0.06
SELECTION_CLUSTER_VARIANT_BONUS = 0.05
SELECTION_CLUSTER_SEARCH_BONUS = 0.03
SELECTION_CLUSTER_SOURCE_BONUS = 0.03
SELECTION_CLUSTER_NATIVE_BONUS = 0.08
SELECTION_CLUSTER_TARGETED_PENALTY = 0.08
SELECTION_CLUSTER_HIGH_NTERMS_PENALTY = 0.06
SELECTION_SHORTER_SCORE_MARGIN = 0.4
SELECTION_SHORTER_GAP_MARGIN = 0.03
SELECTION_SHORTER_THETA_MARGIN = 0.1
SELECTION_SHORTER_MODEL_GAIN_MARGIN = 0.12
SELECTION_NATIVE_OVERRIDE_MARGIN = 0.22
SELECTION_NATIVE_GAP_MARGIN = 0.04
SELECTION_NATIVE_THETA_MARGIN = 0.1
SELECTION_NATIVE_LONGER_GAP_GAIN = 0.01
SELECTION_NATIVE_LONGER_THETA_GAIN = 0.06
SELECTION_HARMONIC_RATIO_MIN = 1.7
SELECTION_HARMONIC_RATIO_MAX = 3.3
SELECTION_HARMONIC_SCORE_MARGIN = 0.18
SELECTION_HARMONIC_GAP_MARGIN = 0.03
SELECTION_HARMONIC_THETA_MARGIN = 0.1
GAMMA_ALIAS_COMPACT_SELECTED_MIN_D = 0.60
GAMMA_ALIAS_COMPACT_SELECTED_MAX_D = 0.72
GAMMA_ALIAS_COMPACT_ALT_MIN_D = 0.78
GAMMA_ALIAS_COMPACT_ALT_MAX_D = 1.25
GAMMA_ALIAS_COMPACT_SCORE_MARGIN = 1.55
GAMMA_ALIAS_COMPACT_THETA_MARGIN = 0.12
GAMMA_ALIAS_MID_SELECTED_MIN_D = 1.20
GAMMA_ALIAS_MID_SELECTED_MAX_D = 1.50
GAMMA_ALIAS_MID_ALT_MIN_D = 1.55
GAMMA_ALIAS_MID_ALT_MAX_D = 1.75
GAMMA_ALIAS_MID_SCORE_MARGIN = 0.42
GAMMA_ALIAS_NEAR_LONG_SELECTED_MIN_D = 1.75
GAMMA_ALIAS_NEAR_LONG_SELECTED_MAX_D = 1.95
GAMMA_ALIAS_NEAR_LONG_ALT_MIN_D = 1.55
GAMMA_ALIAS_NEAR_LONG_ALT_MAX_D = 1.75
GAMMA_ALIAS_NEAR_LONG_SCORE_MARGIN = 0.30
GAMMA_ALIAS_LONG_SELECTED_MIN_D = 2.10
GAMMA_ALIAS_LONG_SELECTED_MAX_D = 3.10
GAMMA_ALIAS_LONG_ALT_MIN_D = 1.45
GAMMA_ALIAS_LONG_ALT_MAX_D = 1.78
GAMMA_ALIAS_LONG_SCORE_MARGIN = 0.95
GAMMA_ALIAS_LONG_GAP_GAIN = 0.015
GAMMA_ALIAS_LONG_TIE_SCORE_MARGIN = 0.08
GAMMA_ALIAS_LONG_TIE_RATIO_MIN = 1.03
GAMMA_ALIAS_LONG_TIE_RATIO_MAX = 1.12
GAMMA_ALIAS_COMPACT_TARGETED_LOW_MIN_D = 0.80
GAMMA_ALIAS_COMPACT_TARGETED_LOW_MAX_D = 0.90
GAMMA_ALIAS_COMPACT_TARGETED_SCORE_MARGIN = 0.36
GAMMA_ALIAS_COMPACT_TARGETED_THETA_MARGIN = 0.04
GAMMA_ALIAS_MID_THETA_MARGIN = 0.03
HALF_ALIAS_GAMMA_SELECTED_MIN_D = 1.35
HALF_ALIAS_GAMMA_SELECTED_MAX_D = 1.55
HALF_ALIAS_GAMMA_CAND_MIN_D = 1.05
HALF_ALIAS_GAMMA_CAND_MAX_D = 1.28
HALF_ALIAS_GAMMA_PARENT_MIN_D = 2.10
HALF_ALIAS_GAMMA_PARENT_MAX_D = 2.70
HALF_ALIAS_GAMMA_SCORE_MARGIN = 1.35
HALF_ALIAS_GAMMA_GAP_GAIN = 0.008
HALF_ALIAS_GAMMA_THETA_MAX = 0.90
HALF_ALIAS_GAMMA_MULT_TOL = 0.08
MID_ALIAS_PROBE_MIN_PERIOD_D = 0.80
MID_ALIAS_PROBE_MAX_PERIOD_D = 1.35
MID_ALIAS_PROBE_PAIR_RATIO_MAX = 1.18
MID_ALIAS_PROBE_PAIR_SCORE_MARGIN = 0.70
MID_ALIAS_PROBE_TOP_ROWS = 24

def mad_sigma(x):
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan
    med = np.median(x)
    return 1.4826 * np.median(np.abs(x - med))

def robust_sigma_or_std(x, floor=np.nan):
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan
    sig = float(mad_sigma(x))
    if not np.isfinite(sig) or sig <= 0:
        sig = float(np.std(x, ddof=1)) if x.size > 1 else np.nan
    if (not np.isfinite(sig) or sig <= 0) and np.isfinite(floor) and (floor > 0):
        sig = float(floor)
    return float(sig) if np.isfinite(sig) and sig > 0 else np.nan

def sigma_clip_mask(x, nsigma=3.5, maxiter=5):
    x = np.asarray(x, dtype=np.float64)
    mask = np.isfinite(x)
    if mask.sum() < 10:
        return mask
    for _ in range(int(maxiter)):
        xx = x[mask]
        med = np.median(xx)
        sig = mad_sigma(xx - med)
        if not np.isfinite(sig) or sig <= 0:
            break
        new_mask = mask & (np.abs(x - med) <= float(nsigma) * sig)
        if new_mask.sum() == mask.sum():
            break
        mask = new_mask
    return mask

def weighted_wrms(y, dy):
    y = np.asarray(y, dtype=np.float64)
    dy = np.asarray(dy, dtype=np.float64)
    ok = np.isfinite(y) & np.isfinite(dy) & (dy > 0)
    if ok.sum() == 0:
        return np.nan
    y = y[ok]
    dy = dy[ok]
    w = 1.0 / np.maximum(dy, 1e-12) ** 2
    sw = np.sum(w)
    if not np.isfinite(sw) or sw <= 0:
        return np.nan
    mu = np.sum(w * y) / sw
    return float(np.sqrt(np.sum(w * (y - mu) ** 2) / sw))

def robust_centered_stats(x, apply_clip=False, clip_nsigma=5.0, clip_maxiter=2):
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return (np.nan, np.nan, np.nan, 0)
    if apply_clip and x.size >= 10:
        keep = sigma_clip_mask(x, nsigma=clip_nsigma, maxiter=clip_maxiter)
        if keep.sum() >= 5:
            x = x[keep]
    med = np.median(x)
    centered = x - med
    std = float(np.std(centered, ddof=1)) if centered.size > 1 else 0.0
    sig = float(mad_sigma(centered))
    amp = float(np.quantile(centered, 0.95) - np.quantile(centered, 0.05)) if centered.size >= 5 else np.nan
    return (std, sig, amp, int(centered.size))

def time_bin_weighted(jd, y, dy, bin_days=0.003, min_bin_pts=1):
    jd = np.asarray(jd, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    dy = np.asarray(dy, dtype=np.float64)
    ok = np.isfinite(jd) & np.isfinite(y) & np.isfinite(dy) & (dy > 0)
    if ok.sum() == 0:
        return (np.array([]), np.array([]), np.array([]))
    jd = jd[ok]
    y = y[ok]
    dy = dy[ok]
    if bin_days is None or float(bin_days) <= 0:
        order = np.argsort(jd)
        return (jd[order], y[order], dy[order])
    t0 = float(np.min(jd))
    bin_idx = np.floor((jd - t0) / float(bin_days)).astype(np.int64)
    jb = []
    yb = []
    dyb = []
    for b in np.unique(bin_idx):
        m = bin_idx == b
        if m.sum() < int(min_bin_pts):
            continue
        w = 1.0 / np.maximum(dy[m], 1e-12) ** 2
        sw = np.sum(w)
        if not np.isfinite(sw) or sw <= 0:
            continue
        jb.append(float(np.mean(jd[m])))
        yb.append(float(np.sum(w * y[m]) / sw))
        dyb.append(float(np.sqrt(1.0 / sw)))
    if not jb:
        return (np.array([]), np.array([]), np.array([]))
    jb = np.asarray(jb, dtype=np.float64)
    yb = np.asarray(yb, dtype=np.float64)
    dyb = np.asarray(dyb, dtype=np.float64)
    order = np.argsort(jb)
    return (jb[order], yb[order], dyb[order])

def nightly_zero_point_correct(jd, y, min_pts=5):
    jd = np.asarray(jd, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    out = y.copy()
    ok = np.isfinite(jd) & np.isfinite(y)
    if ok.sum() == 0:
        return out
    nights = np.floor(jd[ok]).astype(np.int64)
    yy = out[ok]
    for n in np.unique(nights):
        m = nights == n
        if m.sum() >= int(min_pts):
            yy[m] -= np.median(yy[m])
    out[ok] = yy
    return out

def nightly_structure_metrics(jd, y, min_pts=5):
    jd = np.asarray(jd, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    ok = np.isfinite(jd) & np.isfinite(y)
    jd = jd[ok]
    y = y[ok]
    if jd.size == 0:
        return {'night_count': 0, 'night_median_sigma': np.nan, 'night_intra_sigma': np.nan, 'night_median_amp': np.nan, 'night_step_sigma': np.nan, 'night_drift_ratio': np.nan}
    nights = np.floor(jd).astype(np.int64)
    medians = []
    intra_sigmas = []
    ordered_nights = []
    for night in np.unique(nights):
        m = nights == night
        if m.sum() < int(min_pts):
            continue
        yy = y[m]
        med = float(np.median(yy))
        sig = float(mad_sigma(yy - med))
        if not np.isfinite(sig) or sig <= 0:
            sig = float(np.std(yy - med, ddof=1)) if yy.size > 1 else np.nan
        medians.append(med)
        intra_sigmas.append(sig)
        ordered_nights.append(int(night))
    if not medians:
        return {'night_count': 0, 'night_median_sigma': np.nan, 'night_intra_sigma': np.nan, 'night_median_amp': np.nan, 'night_step_sigma': np.nan, 'night_drift_ratio': np.nan}
    medians = np.asarray(medians, dtype=np.float64)
    intra_sigmas = np.asarray(intra_sigmas, dtype=np.float64)
    night_median_sigma = float(mad_sigma(medians - np.median(medians)))
    if not np.isfinite(night_median_sigma) or night_median_sigma <= 0:
        night_median_sigma = float(np.std(medians, ddof=1)) if medians.size > 1 else 0.0
    finite_intra = intra_sigmas[np.isfinite(intra_sigmas) & (intra_sigmas > 0)]
    night_intra_sigma = float(np.median(finite_intra)) if finite_intra.size else np.nan
    if medians.size >= 3:
        night_median_amp = float(np.quantile(medians, 0.9) - np.quantile(medians, 0.1))
    elif medians.size >= 2:
        night_median_amp = float(np.max(medians) - np.min(medians))
    else:
        night_median_amp = 0.0
    ordered = np.argsort(np.asarray(ordered_nights, dtype=np.int64))
    med_ord = medians[ordered]
    night_step_sigma = float(mad_sigma(np.diff(med_ord))) if med_ord.size >= 2 else np.nan
    drift_ratio = float(night_median_sigma / max(night_intra_sigma, 1e-08)) if np.isfinite(night_intra_sigma) and night_intra_sigma > 0 else np.nan
    return {'night_count': int(medians.size), 'night_median_sigma': night_median_sigma, 'night_intra_sigma': night_intra_sigma, 'night_median_amp': night_median_amp, 'night_step_sigma': night_step_sigma, 'night_drift_ratio': drift_ratio}

def read_jd_map(jd_path):
    jd_by_name = {}
    jd_by_stem = {}
    with open(jd_path, 'r') as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith('#'):
                continue
            parts = re.split('\\s+', s)
            if len(parts) < 2:
                continue
            name = None
            jd = None
            try:
                jd = float(parts[1])
                name = os.path.basename(parts[0])
            except Exception:
                pass
            if name is None or jd is None:
                try:
                    jd = float(parts[0])
                    name = os.path.basename(parts[1])
                except Exception:
                    continue
            jd_by_name[name] = jd
            jd_by_stem[os.path.splitext(name)[0]] = jd
    return (jd_by_name, jd_by_stem)

def get_jd_for_ref(ref_path, jd_by_name, jd_by_stem):
    base = os.path.basename(ref_path)
    stem = os.path.splitext(base)[0]
    if base in jd_by_name:
        return jd_by_name[base]
    if stem in jd_by_stem:
        return jd_by_stem[stem]
    fits_name = stem + '.fits'
    if fits_name in jd_by_name:
        return jd_by_name[fits_name]
    return np.nan

def normalized_time_basis():
    basis = str(TIME_BASIS).strip().lower()
    if basis not in {'jd', 'hjd', 'bjd'}:
        raise ValueError(f"Unsupported TIME_BASIS={TIME_BASIS!r}; expected 'jd', 'hjd', or 'bjd'.")
    return basis

def parse_icrs_coord(ra_value, dec_value):
    ra_text = str(ra_value).strip()
    dec_text = str(dec_value).strip()
    tried = []
    for units in ((u.hourangle, u.deg), (u.deg, u.deg)):
        try:
            return SkyCoord(ra_text, dec_text, unit=units, frame='icrs')
        except Exception as exc:
            tried.append(str(exc))
    try:
        ra_float = float(ra_text)
        dec_float = float(dec_text)
    except Exception as exc:
        raise ValueError(f"Could not parse RA/Dec ({ra_text}, {dec_text}): {' | '.join(tried)}") from exc
    if 0.0 <= ra_float <= 24.0:
        try:
            return SkyCoord(ra=ra_float * u.hourangle, dec=dec_float * u.deg, frame='icrs')
        except Exception:
            pass
    return SkyCoord(ra=ra_float * u.deg, dec=dec_float * u.deg, frame='icrs')

def convert_times_for_basis(jd_vals, target_coord, time_basis=None):
    basis = normalized_time_basis() if time_basis is None else str(time_basis).strip().lower()
    jd_vals = np.asarray(jd_vals, dtype=np.float64)
    if basis == 'jd':
        return jd_vals.copy()
    if target_coord is None:
        raise RuntimeError(f"TIME_BASIS='{basis}' requires valid target coordinates.")
    time_obs = Time(jd_vals, format='jd', scale=str(JD_INPUT_SCALE).strip().lower(), location=OBSERVATORY_LOCATION)
    if basis == 'hjd':
        ltt = time_obs.light_travel_time(target_coord, kind='heliocentric')
        return (time_obs.utc + ltt).jd
    if basis == 'bjd':
        ltt = time_obs.light_travel_time(target_coord, kind='barycentric')
        return (time_obs.tdb + ltt).jd
    raise ValueError(f'Unsupported time basis: {basis}')

def read_ref_xy_mag_err_safe(ref_path):
    try:
        arr = np.loadtxt(ref_path, skiprows=3, usecols=(1, 2, 3, 4), dtype=np.float64)
    except Exception as e:
        return (None, f'loadtxt_failed:{type(e).__name__}:{e}')
    if arr.size == 0:
        return (None, 'empty_after_read')
    if arr.ndim == 1:
        if arr.shape[0] != 4:
            return (None, f'bad_1d_shape:{arr.shape}')
        arr = arr.reshape(1, 4)
    if arr.shape[1] != 4:
        return (None, f'bad_col_count:{arr.shape[1]}')
    x = arr[:, 0]
    y = arr[:, 1]
    mag = arr[:, 2]
    err = arr[:, 3]
    ok = np.isfinite(x) & np.isfinite(y) & np.isfinite(mag) & np.isfinite(err)
    if ok.sum() == 0:
        return (None, 'no_finite_rows')
    return ((x[ok], y[ok], mag[ok], err[ok]), None)

def load_master_catalog(master_map_path):
    df = pd.read_csv(master_map_path, sep='\\s+', header=None, names=['star_id', 'x', 'y', 'ra', 'dec'], engine='python')
    df['star_id'] = df['star_id'].astype(int)
    df['x'] = df['x'].astype(float)
    df['y'] = df['y'].astype(float)
    df['ra'] = df['ra'].astype(str)
    df['dec'] = df['dec'].astype(str)
    return df.sort_values('star_id').reset_index(drop=True)

def build_target_coord_map(master_df, star_ids):
    wanted = {int(v) for v in star_ids}
    coord_map = {}
    for row in master_df.itertuples(index=False):
        star_id = int(row.star_id)
        if star_id not in wanted:
            continue
        coord_map[star_id] = parse_icrs_coord(row.ra, row.dec)
    return coord_map

def match_master_to_refimage(master_df, master_ref_path, match_tol_px=1.0):
    data, err = read_ref_xy_mag_err_safe(master_ref_path)
    if data is None:
        raise RuntimeError(f'Could not read {master_ref_path}: {err}')
    rx, ry, rmag, rerr = data
    ref_xy = np.column_stack([rx, ry])
    tree = cKDTree(ref_xy)
    master_xy = master_df[['x', 'y']].to_numpy(np.float64)
    dist, idx = tree.query(master_xy, k=1, workers=-1)
    ok = np.isfinite(dist) & (dist <= float(match_tol_px))
    mag_seed = np.full(len(master_df), np.nan, dtype=np.float64)
    err_seed = np.full(len(master_df), np.nan, dtype=np.float64)
    mag_seed[ok] = rmag[idx[ok]]
    err_seed[ok] = rerr[idx[ok]]
    return (mag_seed, err_seed, dist)

def solve_affine(src_xy, dst_xy):
    if src_xy.shape[0] < 3:
        return None
    M = np.column_stack([src_xy[:, 0], src_xy[:, 1], np.ones(src_xy.shape[0], dtype=np.float64)])
    try:
        cx, _, _, _ = np.linalg.lstsq(M, dst_xy[:, 0], rcond=None)
        cy, _, _, _ = np.linalg.lstsq(M, dst_xy[:, 1], rcond=None)
    except Exception:
        return None
    return np.vstack([cx, cy])

def apply_affine(A, xy):
    M = np.column_stack([xy[:, 0], xy[:, 1], np.ones(xy.shape[0], dtype=np.float64)])
    px = M @ A[0]
    py = M @ A[1]
    return np.column_stack([px, py])

def fit_affine_master_to_frame(master_seed_xy, frame_xy, tree_f, seed_tol_px, min_seeds, resid_tol_px):
    if master_seed_xy.shape[0] < int(min_seeds) or frame_xy.shape[0] < int(min_seeds):
        return (None, 0)
    d0, j0 = tree_f.query(master_seed_xy, k=1, workers=-1)
    ok0 = np.isfinite(d0) & (d0 <= float(seed_tol_px))
    if ok0.sum() < int(min_seeds):
        return (None, int(ok0.sum()))
    A = solve_affine(master_seed_xy[ok0], frame_xy[j0[ok0]])
    if A is None:
        return (None, int(ok0.sum()))
    best_n = int(ok0.sum())
    for _ in range(2):
        pred = apply_affine(A, master_seed_xy)
        d1, j1 = tree_f.query(pred, k=1, workers=-1)
        ok1 = np.isfinite(d1) & (d1 <= float(resid_tol_px))
        if ok1.sum() < int(min_seeds):
            break
        A2 = solve_affine(master_seed_xy[ok1], frame_xy[j1[ok1]])
        if A2 is None:
            break
        A = A2
        best_n = int(ok1.sum())
    return (A, best_n)

def unique_matches(master_xy, frame_xy, tree_f, A, match_tol_px):
    pred = apply_affine(A, master_xy)
    dist, idx = tree_f.query(pred, k=1, workers=-1)
    ok = np.isfinite(dist) & (dist <= float(match_tol_px))
    if ok.sum() == 0:
        return (np.array([], dtype=np.int32), np.array([], dtype=np.int32), np.array([], dtype=np.float64))
    master_idx = np.where(ok)[0]
    frame_idx = idx[ok].astype(np.int32)
    dist = dist[ok]
    order = np.argsort(dist)
    used_frame = np.zeros(frame_xy.shape[0], dtype=bool)
    keep = []
    for oi in order:
        fj = frame_idx[oi]
        if used_frame[fj]:
            continue
        used_frame[fj] = True
        keep.append(oi)
    keep = np.asarray(keep, dtype=np.int32)
    return (master_idx[keep], frame_idx[keep], dist[keep])

def phase_fold(jd, period, t0):
    return (jd - t0) / period % 1.0

def phase_gap_metric(jd, period):
    jd = np.asarray(jd, dtype=np.float64)
    jd = jd[np.isfinite(jd)]
    if jd.size < 3 or not np.isfinite(period) or period <= 0:
        return np.inf
    t0 = float(np.min(jd))
    ph = np.sort(phase_fold(jd, period, t0))
    if ph.size < 2:
        return np.inf
    diffs = np.diff(ph)
    wrap_gap = ph[0] + 1.0 - ph[-1]
    return float(np.max(np.concatenate([diffs, [wrap_gap]])))

def day_alias_distance(period, max_harmonic=8, include_subharmonics=True):
    if not np.isfinite(period) or period <= 0:
        return np.inf
    f = 1.0 / float(period)
    kk = np.arange(1, int(max_harmonic) + 1, dtype=np.float64)
    targets = kk
    if include_subharmonics:
        targets = np.unique(np.concatenate([targets, 1.0 / kk]))
    return float(np.min(np.abs(f - targets)))

def phase_profile_stats(jd, y, dy, period, nbins=60, min_bin_pts=4):
    jd = np.asarray(jd, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    dy = np.asarray(dy, dtype=np.float64)
    ok = np.isfinite(jd) & np.isfinite(y) & np.isfinite(dy) & (dy > 0)
    if ok.sum() < 10 or not np.isfinite(period) or period <= 0:
        return {'theta': np.inf, 'amp': np.nan, 'bins_used': 0, 'phase': np.array([]), 'profile_x': np.array([]), 'profile_y': np.array([]), 'profile_err': np.array([])}
    jd = jd[ok]
    y = y[ok]
    dy = dy[ok]
    t0 = float(np.min(jd))
    ph = phase_fold(jd, period, t0)
    raw_sig = mad_sigma(y - np.median(y))
    if not np.isfinite(raw_sig) or raw_sig <= 0:
        raw_sig = np.std(y, ddof=1) if y.size > 1 else np.nan
    if not np.isfinite(raw_sig) or raw_sig <= 0:
        return {'theta': np.inf, 'amp': np.nan, 'bins_used': 0, 'phase': ph, 'profile_x': np.array([]), 'profile_y': np.array([]), 'profile_err': np.array([])}
    edges = np.linspace(0.0, 1.0, int(nbins) + 1)
    inds = np.clip(np.digitize(ph, edges) - 1, 0, int(nbins) - 1)
    bin_sigmas = []
    xmid = []
    ymed = []
    ys = []
    for b in range(int(nbins)):
        m = inds == b
        if m.sum() < int(min_bin_pts):
            continue
        yy = y[m]
        med = np.median(yy)
        sig = mad_sigma(yy - med)
        if not np.isfinite(sig):
            continue
        xmid.append((b + 0.5) / float(nbins))
        ymed.append(med)
        ys.append(sig)
        bin_sigmas.append(sig)
    if len(bin_sigmas) < int(MIN_BINS_USED):
        theta = np.inf
        amp = np.nan
    else:
        theta = float(np.median(bin_sigmas) / max(raw_sig, 1e-08))
        amp = float(np.quantile(ymed, 0.9) - np.quantile(ymed, 0.1)) if len(ymed) >= 5 else np.nan
    return {'theta': theta, 'amp': amp, 'bins_used': int(len(bin_sigmas)), 'phase': ph, 'profile_x': np.asarray(xmid, dtype=np.float64), 'profile_y': np.asarray(ymed, dtype=np.float64), 'profile_err': np.asarray(ys, dtype=np.float64)}

def apply_short_period_phase_dispersion_refine(chosen_member):
    if not bool(SHORT_PERIOD_PHASE_REFINE) or chosen_member is None:
        return (chosen_member, None)
    row = chosen_member.get('row', {})
    period0 = float(row.get('period', np.nan))
    if 'consensus_phase_scan' in str(row.get('candidate_source', '')):
        return (chosen_member, None)
    if not np.isfinite(period0) or period0 <= 0 or period0 > float(SHORT_PERIOD_PHASE_REFINE_MAX_D):
        return (chosen_member, None)
    solution = chosen_member.get('solution', {})
    jd = np.asarray(solution.get('jd', []), dtype=np.float64)
    y = np.asarray(solution.get('y', []), dtype=np.float64)
    dy = np.asarray(solution.get('dy', []), dtype=np.float64)
    ok = np.isfinite(jd) & np.isfinite(y) & np.isfinite(dy) & (dy > 0)
    if ok.sum() < int(MIN_POINTS_AFTER_CLEAN):
        return (chosen_member, None)
    jd = jd[ok]
    y = y[ok]
    dy = dy[ok]
    raw_sigma = float(row.get('raw_sigma', np.nan))
    if not np.isfinite(raw_sigma) or raw_sigma <= 0:
        raw_sigma = mad_sigma(y - np.nanmedian(y))
    if not np.isfinite(raw_sigma) or raw_sigma <= 0:
        return (chosen_member, None)
    current_profile = phase_profile_stats(jd, y, dy, period0, nbins=NBINS_SCORE, min_bin_pts=MIN_BIN_PTS)
    current_theta = float(row.get('theta', current_profile.get('theta', np.inf)))
    if not np.isfinite(current_theta):
        current_theta = float(current_profile.get('theta', np.inf))
    if not np.isfinite(current_theta):
        return (chosen_member, None)
    medium_refine = period0 > float(MEDIUM_PERIOD_PHASE_REFINE_MIN_D)
    rel_width = max(float(MEDIUM_PERIOD_PHASE_REFINE_REL_WIDTH if medium_refine else SHORT_PERIOD_PHASE_REFINE_REL_WIDTH), 0.0)
    grid_size = max(int(MEDIUM_PERIOD_PHASE_REFINE_GRID_SIZE if medium_refine else SHORT_PERIOD_PHASE_REFINE_GRID_SIZE), 21)
    pmin = max(float(PERIOD_MIN_D), period0 * (1.0 - rel_width))
    pmax = min(float(PERIOD_MAX_D), period0 * (1.0 + rel_width))
    if not np.isfinite(pmin) or not np.isfinite(pmax) or pmax <= pmin:
        return (chosen_member, None)
    periods = np.unique(np.concatenate([np.linspace(pmin, pmax, grid_size, dtype=np.float64), np.asarray([period0], dtype=np.float64)]))
    tested = []
    for period in periods:
        profile = phase_profile_stats(jd, y, dy, float(period), nbins=NBINS_SCORE, min_bin_pts=MIN_BIN_PTS)
        theta = float(profile.get('theta', np.inf))
        bins_used = int(profile.get('bins_used', 0))
        if not np.isfinite(theta) or bins_used < int(MIN_BINS_USED):
            continue
        phase = profile.get('phase', np.array([]))
        string_length = phase_string_length(phase, y, raw_sigma)
        gap = phase_gap_metric(jd, float(period))
        amp = float(profile.get('amp', np.nan))
        amp_ratio = float(amp / raw_sigma) if np.isfinite(amp) else float(row.get('amp_ratio', np.nan))
        tested.append({'period': float(period), 'theta': theta, 'bins_used': bins_used, 'string_length': float(string_length), 'gap': float(gap), 'amp_ratio': float(amp_ratio)})
    if not tested:
        return (chosen_member, None)
    min_theta = min(float(item['theta']) for item in tested)
    close = [item for item in tested if float(item['theta']) <= min_theta + float(SHORT_PERIOD_PHASE_REFINE_THETA_MARGIN)]
    best = min(close, key=lambda item: (float(item['theta']), float(item['string_length']), float(item['gap']), abs(float(item['period']) - period0)))
    min_theta_gain = float(MEDIUM_PERIOD_PHASE_REFINE_MIN_THETA_GAIN if medium_refine else SHORT_PERIOD_PHASE_REFINE_MIN_THETA_GAIN)
    if (current_theta - float(best['theta'])) < min_theta_gain:
        return (chosen_member, None)
    refined_row = dict(row)
    refined_period = float(best['period'])
    harmonic_multiplier = float(refined_row.get('harmonic_multiplier', 1.0))
    if not np.isfinite(harmonic_multiplier) or harmonic_multiplier <= 0:
        harmonic_multiplier = 1.0
    refined_row['period'] = refined_period
    refined_row['base_period'] = refined_period / harmonic_multiplier
    refined_row['theta'] = float(best['theta'])
    refined_row['amp_ratio'] = float(best['amp_ratio']) if np.isfinite(best['amp_ratio']) else refined_row.get('amp_ratio', np.nan)
    refined_row['gap'] = float(best['gap'])
    refined_row['alias_dist'] = float(day_alias_distance(refined_period, max_harmonic=ALIAS_MAX_HARMONIC, include_subharmonics=ALIAS_INCLUDE_SUBHARMONICS))
    refined_row['string_length'] = float(best['string_length'])
    refined_row['bins_used'] = int(best['bins_used'])
    refined_row['candidate_source'] = ';'.join(filter(None, [str(refined_row.get('candidate_source', '')), 'phase_dispersion_refine']))
    refined_member = dict(chosen_member)
    refined_member['row'] = refined_row
    return (refined_member, 'short_phase_dispersion_refine')

def phase_string_length(phase, y, scale):
    phase = np.asarray(phase, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    ok = np.isfinite(phase) & np.isfinite(y)
    if ok.sum() < 3:
        return np.inf
    phase = phase[ok]
    y = y[ok]
    order = np.argsort(phase)
    phase = phase[order]
    y = y[order]
    sc = max(float(scale), 1e-08)
    yn = (y - np.median(y)) / sc
    dx = np.diff(np.concatenate([phase, [phase[0] + 1.0]]))
    dy = np.diff(np.concatenate([yn, [yn[0]]]))
    return float(np.mean(np.sqrt(dx ** 2 + dy ** 2)))

def parse_comparison_catalog(compare_csv):
    if not compare_csv or not os.path.exists(compare_csv):
        raise FileNotFoundError(f'Comparison CSV not found: {compare_csv}')
    df = pd.read_csv(compare_csv)

    if COL_COMP1_ID not in df.columns and 'nonvariable_star_id' in df.columns:
        df = df.rename(columns={'nonvariable_star_id': COL_COMP1_ID})

    if COL_COMP2_ID not in df.columns and 'check_star_id' in df.columns:
        df = df.rename(columns={'check_star_id': COL_COMP2_ID})
    if COL_STAR_ID not in df.columns or COL_COMP1_ID not in df.columns:
        raise ValueError(f"Comparison CSV must contain '{COL_STAR_ID}' and '{COL_COMP1_ID}'.")

    keep = [c for c in (COL_STAR_ID, COL_COMP1_ID, COL_COMP2_ID) if c in df.columns]
    df = df[keep].drop_duplicates(subset=[COL_STAR_ID]).reset_index(drop=True)
    out = df.copy()
    out[COL_STAR_ID] = out[COL_STAR_ID].astype(int)
    out[COL_COMP1_ID] = out[COL_COMP1_ID].astype(int)
    if COL_COMP2_ID in out.columns:
        out[COL_COMP2_ID] = pd.to_numeric(out[COL_COMP2_ID], errors='coerce')
    else:
        out[COL_COMP2_ID] = np.nan
    return out

def build_mode_lookup(compare_df, jd_vals, corr_mag_mat, err_mat, star_to_col, star_coord_map=None, frame_ids=None):
    mode_lookup = {}
    converted_time_by_star = {}
    for row in compare_df.itertuples(index=False):
        star_id = int(getattr(row, COL_STAR_ID))
        comp1_id = int(getattr(row, COL_COMP1_ID))
        comp2_id = getattr(row, COL_COMP2_ID)
        if star_id not in converted_time_by_star:
            target_coord = None if star_coord_map is None else star_coord_map.get(int(star_id))
            converted_time_by_star[star_id] = convert_times_for_basis(jd_vals, target_coord)
        mode_lookup[star_id] = build_mode_series(converted_time_by_star[star_id], corr_mag_mat, err_mat, star_to_col, star_id, comp1_id, comp2_id, frame_ids=frame_ids)
    return mode_lookup

def summarize_rejection(jd, removed_mask, removed_bad_jd_mask=None, removed_bad_night_mask=None):
    jd = np.asarray(jd, dtype=np.float64)
    removed_mask = np.asarray(removed_mask, dtype=bool)
    nights = np.floor(jd[np.isfinite(jd)]).astype(np.int64) if np.isfinite(jd).any() else np.array([], dtype=np.int64)
    summary = {'removed_points': int(np.sum(removed_mask)), 'removed_nights': 0, 'removed_bad_jd_points': 0, 'removed_bad_night_points': 0, 'input_points': int(jd.size), 'input_nights': int(np.unique(np.floor(jd[np.isfinite(jd)]).astype(np.int64)).size) if np.isfinite(jd).any() else 0}
    if removed_mask.any():
        removed_nights = np.floor(jd[removed_mask]).astype(np.int64)
        summary['removed_nights'] = int(np.unique(removed_nights).size)
    if removed_bad_jd_mask is not None:
        summary['removed_bad_jd_points'] = int(np.sum(np.asarray(removed_bad_jd_mask, dtype=bool)))
    if removed_bad_night_mask is not None:
        summary['removed_bad_night_points'] = int(np.sum(np.asarray(removed_bad_night_mask, dtype=bool)))
    return summary

def is_phase_clean_variant(series_variant):
    return str(series_variant).endswith('_phase_clean')

def base_variant_name(series_variant):
    series_variant = str(series_variant)
    if is_phase_clean_variant(series_variant):
        return series_variant[:-len('_phase_clean')]
    return series_variant

def solution_reference_key(solution):
    if solution is None:
        return None
    return (str(solution['reference_mode']), int(solution['comp1_star_id']), int(solution['comp2_star_id']) if pd.notna(solution['comp2_star_id']) else None)

def attach_raw_relative_plot_series(solution, raw_solution_by_mode):
    if solution is None:
        return solution
    raw_solution = raw_solution_by_mode.get(solution_reference_key(solution))
    if raw_solution is None:
        raw_solution = solution
    out = dict(solution)
    out['raw_relative_jd'] = np.asarray(raw_solution['jd'], dtype=np.float64).copy()
    out['raw_relative_y'] = np.asarray(raw_solution['y'], dtype=np.float64).copy()
    out['raw_relative_dy'] = np.asarray(raw_solution['dy'], dtype=np.float64).copy()
    return out

def build_needed_photometry(master_df, compare_df, jd_path=JD_FILE):
    master_xy = master_df[['x', 'y']].to_numpy(np.float64)
    star_to_master_idx = {int(sid): i for i, sid in enumerate(master_df['star_id'].astype(int).tolist())}
    mag_seed, err_seed, seed_dist = match_master_to_refimage(master_df, MASTER_REF_PATH, match_tol_px=1.0)
    seed_idx = np.where(np.isfinite(mag_seed) & np.isfinite(err_seed) & (err_seed <= float(SEED_ERR_MAX)))[0]
    if seed_idx.size < int(TRANSFORM_MIN_SEEDS):
        raise RuntimeError('Not enough zero-point seed stars matched to refimage.ref.')
    if seed_idx.size > int(TRANSFORM_MAX_SEEDS):
        order = np.argsort(mag_seed[seed_idx])
        seed_idx = seed_idx[order[:int(TRANSFORM_MAX_SEEDS)]]
    master_seed_xy = master_xy[seed_idx]
    requested_star_ids = []
    for row in compare_df.itertuples(index=False):
        requested_star_ids.append(int(getattr(row, COL_STAR_ID)))
        requested_star_ids.append(int(getattr(row, COL_COMP1_ID)))
        comp2 = getattr(row, COL_COMP2_ID)
        if pd.notna(comp2):
            requested_star_ids.append(int(float(comp2)))
    requested_star_ids = sorted(set(requested_star_ids))
    missing_ids = sorted((sid for sid in requested_star_ids if sid not in star_to_master_idx))
    needed_star_ids = [sid for sid in requested_star_ids if sid in star_to_master_idx]
    needed_master_idx = np.asarray([star_to_master_idx[sid] for sid in needed_star_ids], dtype=np.int32)
    needed_xy = master_xy[needed_master_idx]
    jd_by_name, jd_by_stem = read_jd_map(jd_path)
    ref_files = []
    jd_vals = []
    frame_index_global = []
    for ref_path in sorted(glob.glob(os.path.join(REF_DIR, '*.ref'))):
        if os.path.basename(ref_path) == os.path.basename(MASTER_REF_PATH):
            continue
        jd = get_jd_for_ref(ref_path, jd_by_name, jd_by_stem)
        if np.isfinite(jd):
            ref_files.append(ref_path)
            jd_vals.append(float(jd))
            frame_index_global.append(int(len(ref_files) - 1))
    if not ref_files:
        raise RuntimeError('No JD-mapped science .ref files found.')
    n_frames = len(ref_files)
    n_needed = len(needed_star_ids)
    corr_mag_mat = np.full((n_frames, n_needed), np.nan, dtype=np.float32)
    err_mat = np.full((n_frames, n_needed), np.nan, dtype=np.float32)
    frame_rows = []
    print(f'Comparison rows: {len(compare_df)} | needed stars in master map: {n_needed} | science frames: {n_frames} | transform seeds: {len(seed_idx)} | jd_file={os.path.basename(jd_path)} | time_basis={normalized_time_basis()}', flush=True)
    if missing_ids:
        print(f'Missing IDs in master map: {len(missing_ids)}', flush=True)
    for fi, (ref_path, jd, global_index) in enumerate(zip(ref_files, jd_vals, frame_index_global), start=0):
        data, err = read_ref_xy_mag_err_safe(ref_path)
        if data is None:
            frame_rows.append({'frame_index_global': int(global_index), 'frame': os.path.basename(ref_path), 'jd': jd, 'transform_ok': False, 'reason': err, 'n_frame_stars': 0, 'n_seed': 0, 'n_needed_match': 0, 'n_zp_seeds': 0, 'frame_zero_point': np.nan})
            continue
        fx, fy, fmag, ferr = data
        frame_xy = np.column_stack([fx, fy])
        tree_f = cKDTree(frame_xy)
        A, nseed = fit_affine_master_to_frame(master_seed_xy, frame_xy, tree_f, TRANSFORM_SEED_TOL_PX, TRANSFORM_MIN_SEEDS, TRANSFORM_RESID_TOL_PX)
        if A is None:
            frame_rows.append({'frame_index_global': int(global_index), 'frame': os.path.basename(ref_path), 'jd': jd, 'transform_ok': False, 'reason': 'affine_failed', 'n_frame_stars': int(len(fx)), 'n_seed': int(nseed), 'n_needed_match': 0, 'n_zp_seeds': 0, 'frame_zero_point': np.nan})
            continue
        seed_local_idx, seed_frame_idx, _ = unique_matches(master_seed_xy, frame_xy, tree_f, A, MATCH_TOL_PX)
        if seed_local_idx.size == 0:
            frame_rows.append({'frame_index_global': int(global_index), 'frame': os.path.basename(ref_path), 'jd': jd, 'transform_ok': False, 'reason': 'no_zero_point_matches', 'n_frame_stars': int(len(fx)), 'n_seed': int(nseed), 'n_needed_match': 0, 'n_zp_seeds': 0, 'frame_zero_point': np.nan})
            continue
        seed_master_global = seed_idx[seed_local_idx]
        seed_good = np.isfinite(fmag[seed_frame_idx]) & np.isfinite(ferr[seed_frame_idx]) & (ferr[seed_frame_idx] <= float(MAX_MEASUREMENT_ERR)) & np.isfinite(mag_seed[seed_master_global]) & np.isfinite(err_seed[seed_master_global]) & (err_seed[seed_master_global] <= float(SEED_ERR_MAX))
        zp_vals = fmag[seed_frame_idx[seed_good]].astype(np.float64) - mag_seed[seed_master_global[seed_good]].astype(np.float64)
        zp_keep = sigma_clip_mask(zp_vals, nsigma=FRAME_ZP_CLIP_NSIGMA, maxiter=FRAME_ZP_CLIP_MAXITER) if zp_vals.size else np.array([], dtype=bool)
        if zp_vals.size == 0 or zp_keep.sum() < 10:
            frame_rows.append({'frame_index_global': int(global_index), 'frame': os.path.basename(ref_path), 'jd': jd, 'transform_ok': False, 'reason': 'insufficient_zero_point_seeds', 'n_frame_stars': int(len(fx)), 'n_seed': int(nseed), 'n_needed_match': 0, 'n_zp_seeds': int(zp_keep.sum()) if zp_keep.size else 0, 'frame_zero_point': np.nan})
            continue
        zp = float(np.median(zp_vals[zp_keep]))
        need_local_idx, need_frame_idx, match_dist = unique_matches(needed_xy, frame_xy, tree_f, A, MATCH_TOL_PX)
        if need_local_idx.size:
            good = np.isfinite(fmag[need_frame_idx]) & np.isfinite(ferr[need_frame_idx]) & (ferr[need_frame_idx] <= float(MAX_MEASUREMENT_ERR))
            corr_mag_mat[fi, need_local_idx[good]] = (fmag[need_frame_idx[good]].astype(np.float64) - zp).astype(np.float32)
            err_mat[fi, need_local_idx[good]] = ferr[need_frame_idx[good]].astype(np.float32)
        frame_rows.append({'frame_index_global': int(global_index), 'frame': os.path.basename(ref_path), 'jd': jd, 'transform_ok': True, 'reason': '', 'n_frame_stars': int(len(fx)), 'n_seed': int(nseed), 'n_needed_match': int(need_local_idx.size), 'n_zp_seeds': int(zp_keep.sum()), 'frame_zero_point': zp, 'median_match_dist_px': float(np.median(match_dist)) if match_dist.size else np.nan})
        if (fi + 1) % 3000 == 0 or fi + 1 == n_frames:
            print(f'Processed {fi + 1}/{n_frames} frames', flush=True)
    frame_df = pd.DataFrame(frame_rows)
    star_to_needed_col = {sid: i for i, sid in enumerate(needed_star_ids)}
    phot = {'jd': np.asarray(jd_vals, dtype=np.float64), 'frame_index_global': np.asarray(frame_index_global, dtype=np.int64), 'corr_mag_mat': corr_mag_mat, 'err_mat': err_mat, 'frame_df': frame_df, 'needed_star_ids': needed_star_ids, 'star_to_needed_col': star_to_needed_col, 'missing_star_ids': missing_ids, 'seed_match_dist_px': seed_dist}
    return phot

def build_mode_series(jd_vals, corr_mag_mat, err_mat, star_to_col, star_id, comp1_id, comp2_id, frame_ids=None):
    modes = []
    target_col = star_to_col.get(int(star_id))
    if target_col is None:
        return modes
    if frame_ids is None:
        frame_ids = np.arange(len(jd_vals), dtype=np.int64)
    else:
        frame_ids = np.asarray(frame_ids, dtype=np.int64)
    target_mag = corr_mag_mat[:, target_col].astype(np.float64)
    target_err = err_mat[:, target_col].astype(np.float64)

    def add_single_mode(mode_name, comp_id):
        comp_col = star_to_col.get(int(comp_id))
        if comp_col is None:
            return
        comp_mag = corr_mag_mat[:, comp_col].astype(np.float64)
        comp_err = err_mat[:, comp_col].astype(np.float64)
        ok = np.isfinite(jd_vals) & np.isfinite(target_mag) & np.isfinite(target_err) & np.isfinite(comp_mag) & np.isfinite(comp_err) & (target_err > 0) & (comp_err > 0) & (target_err <= float(MAX_MEASUREMENT_ERR)) & (comp_err <= float(MAX_MEASUREMENT_ERR))
        if ok.sum() < int(MIN_POINTS_AFTER_CLEAN):
            return
        frame_index = frame_ids[ok].astype(np.int64)
        jd = jd_vals[ok]
        y = target_mag[ok] - comp_mag[ok]
        dy = np.sqrt(target_err[ok] ** 2 + comp_err[ok] ** 2)
        modes.append({'reference_mode': mode_name, 'comp1_star_id': int(comp_id), 'comp2_star_id': np.nan, 'frame_index': frame_index, 'jd': jd, 'y': y, 'dy': dy, 'rejection_metrics': summarize_rejection(jd, np.zeros(jd.size, dtype=bool))})
    add_single_mode('single_comp1', comp1_id)
    if pd.notna(comp2_id):
        add_single_mode('single_comp2', int(float(comp2_id)))
    if pd.notna(comp2_id):
        comp2_int = int(float(comp2_id))
        col1 = star_to_col.get(int(comp1_id))
        col2 = star_to_col.get(comp2_int)
        if col1 is not None and col2 is not None:
            comp1_mag = corr_mag_mat[:, col1].astype(np.float64)
            comp2_mag = corr_mag_mat[:, col2].astype(np.float64)
            comp1_err = err_mat[:, col1].astype(np.float64)
            comp2_err = err_mat[:, col2].astype(np.float64)
            ok = np.isfinite(jd_vals) & np.isfinite(target_mag) & np.isfinite(target_err) & np.isfinite(comp1_mag) & np.isfinite(comp2_mag) & np.isfinite(comp1_err) & np.isfinite(comp2_err) & (target_err > 0) & (comp1_err > 0) & (comp2_err > 0) & (target_err <= float(MAX_MEASUREMENT_ERR)) & (comp1_err <= float(MAX_MEASUREMENT_ERR)) & (comp2_err <= float(MAX_MEASUREMENT_ERR))
            if ok.sum() >= int(MIN_POINTS_AFTER_CLEAN):
                frame_index = frame_ids[ok].astype(np.int64)
                jd = jd_vals[ok]
                comp_mean = 0.5 * (comp1_mag[ok] + comp2_mag[ok])
                y = target_mag[ok] - comp_mean
                dy = np.sqrt(target_err[ok] ** 2 + 0.25 * comp1_err[ok] ** 2 + 0.25 * comp2_err[ok] ** 2)
                modes.append({'reference_mode': 'pair_mean', 'comp1_star_id': int(comp1_id), 'comp2_star_id': comp2_int, 'frame_index': frame_index, 'jd': jd, 'y': y, 'dy': dy, 'rejection_metrics': summarize_rejection(jd, np.zeros(jd.size, dtype=bool))})
    return modes

def preprocess_series_variants(jd, y, dy):
    jd = np.asarray(jd, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    dy = np.asarray(dy, dtype=np.float64)
    ok = np.isfinite(jd) & np.isfinite(y) & np.isfinite(dy) & (dy > 0)
    jd = jd[ok]
    y = y[ok]
    dy = dy[ok]
    if jd.size < int(MIN_POINTS_AFTER_CLEAN):
        return []
    keep = sigma_clip_mask(y, nsigma=SERIES_CLIP_NSIGMA, maxiter=SERIES_CLIP_MAXITER)
    jd = jd[keep]
    y = y[keep]
    dy = dy[keep]
    if jd.size < int(MIN_POINTS_AFTER_CLEAN):
        return []
    base_std, base_sigma, base_amp, _ = robust_centered_stats(y, apply_clip=False)
    night_metrics = nightly_structure_metrics(jd, y, min_pts=NIGHTLY_MIN_PTS)
    variants = [('raw', y.copy())]
    if APPLY_NIGHTLY_DETREND:
        variants = [('nightly_detrend', nightly_zero_point_correct(jd, y, min_pts=NIGHTLY_MIN_PTS))]
    elif SEARCH_NIGHTLY_DETREND_VARIANT:
        variants.append(('nightly_detrend', nightly_zero_point_correct(jd, y, min_pts=NIGHTLY_MIN_PTS)))
    out = []
    for variant_name, y_variant in variants:
        jd_use = jd.copy()
        dy_use = dy.copy()
        if APPLY_TIME_BINNING:
            jd_use, y_variant, dy_use = time_bin_weighted(jd_use, y_variant, dy_use, bin_days=TIME_BIN_DAYS, min_bin_pts=TIME_BIN_MIN_PTS)
        if jd_use.size < int(MIN_POINTS_AFTER_CLEAN):
            continue
        keep2 = sigma_clip_mask(y_variant, nsigma=POST_BIN_CLIP_NSIGMA, maxiter=POST_BIN_CLIP_MAXITER)
        jd_use = jd_use[keep2]
        y_variant = y_variant[keep2]
        dy_use = dy_use[keep2]
        if jd_use.size < int(MIN_POINTS_AFTER_CLEAN):
            continue
        order = np.argsort(jd_use)
        jd_ord = jd_use[order]
        y_ord = y_variant[order]
        dy_ord = dy_use[order]
        var_std, var_sigma, var_amp, _ = robust_centered_stats(y_ord, apply_clip=False)
        variant_metrics = dict(night_metrics)
        variant_metrics.update({'base_sigma': float(base_sigma) if np.isfinite(base_sigma) else np.nan, 'base_amp': float(base_amp) if np.isfinite(base_amp) else np.nan, 'night_median_amp_ratio': float(night_metrics['night_median_amp'] / max(base_amp, 1e-08)) if np.isfinite(night_metrics['night_median_amp']) and np.isfinite(base_amp) and (base_amp > 0) else np.nan, 'variant_sigma': float(var_sigma) if np.isfinite(var_sigma) else np.nan, 'variant_amp': float(var_amp) if np.isfinite(var_amp) else np.nan, 'collapse_sigma_ratio': float(var_sigma / max(base_sigma, 1e-08)) if np.isfinite(var_sigma) and np.isfinite(base_sigma) and (base_sigma > 0) else np.nan, 'collapse_amp_ratio': float(var_amp / max(base_amp, 1e-08)) if np.isfinite(var_amp) and np.isfinite(base_amp) and (base_amp > 0) else np.nan, 'variant_wrms': float(weighted_wrms(y_ord, dy_ord))})
        out.append((variant_name, jd_ord, y_ord, dy_ord, variant_metrics))
    return out

def ls_top_peaks(jd, y, dy, pmin, pmax, nterms_values, samples_per_peak, topn_per_nterm, dedup_frac):
    jd = np.asarray(jd, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    dy = np.asarray(dy, dtype=np.float64)
    ok = np.isfinite(jd) & np.isfinite(y) & np.isfinite(dy) & (dy > 0)
    jd = jd[ok]
    y = y[ok]
    dy = dy[ok]
    if jd.size < 10:
        return ([], {}, 0.0, np.inf, np.inf)
    fmin = 1.0 / float(pmax)
    fmax = 1.0 / float(pmin)
    baseline = float(np.ptp(jd))
    if not np.isfinite(baseline) or baseline <= 0:
        return ([], {}, 0.0, fmin, fmax)
    candidates = []
    ls_map = {}
    best_raw_power = 0.0
    for nterms in nterms_values:
        ls = LombScargle(jd, y, dy=dy, center_data=True, nterms=int(nterms))
        freq, power = ls.autopower(minimum_frequency=fmin, maximum_frequency=fmax, samples_per_peak=int(samples_per_peak))
        if power.size == 0 or not np.isfinite(power).any():
            continue
        ls_map[int(nterms)] = ls
        best_raw_power = max(best_raw_power, float(np.nanmax(power)))
        order = np.argsort(power)[::-1]
        kept_periods = []
        rank = 0
        for idx in order:
            if not np.isfinite(freq[idx]) or freq[idx] <= 0 or (not np.isfinite(power[idx])):
                continue
            period = 1.0 / float(freq[idx])
            if not pmin <= period <= pmax:
                continue
            if any((abs(period - old_p) / old_p <= float(dedup_frac) for old_p in kept_periods)):
                continue
            rank += 1
            candidates.append({'base_period': float(period), 'base_power': float(power[idx]), 'base_rank': int(rank), 'search_nterms': int(nterms)})
            kept_periods.append(float(period))
            if len(kept_periods) >= int(topn_per_nterm):
                break
    return (candidates, ls_map, baseline, fmin, fmax)

def refine_period(ls, period, fmin, fmax, baseline):
    base_freq = 1.0 / float(period)
    half_width = max(float(LOCAL_REFINE_RESOLUTION_ELEMENTS) / max(float(baseline), 1e-08), 0.003 * base_freq)
    grid = np.linspace(max(fmin, base_freq - half_width), min(fmax, base_freq + half_width), int(LOCAL_REFINE_GRID_SIZE))
    if grid.size == 0:
        return (float(period), float(ls.power(base_freq)))
    power = ls.power(grid)
    if power.size == 0 or not np.isfinite(power).any():
        return (float(period), float(ls.power(base_freq)))
    best_idx = int(np.nanargmax(power))
    best_freq = float(grid[best_idx])
    return (float(1.0 / best_freq), float(power[best_idx]))

def short_period_candidates(jd, y, dy):
    extra_candidates, extra_ls_map, baseline, fmin, fmax = ls_top_peaks(jd, y, dy, SHORT_SCAN_MIN_PERIOD_D, SHORT_SCAN_MAX_PERIOD_D, LS_SEARCH_NTERMS, SHORT_SCAN_SAMPLES_PER_PEAK, SHORT_SCAN_TOP_PEAKS_PER_NTERM, PEAK_DEDUP_FRAC)
    if not extra_candidates or not extra_ls_map:
        return []
    hypotheses = []
    seen = []
    for base in extra_candidates:
        search_nterms = int(base['search_nterms'])
        ls = extra_ls_map[search_nterms]
        refined_period, refined_power = refine_period(ls, base['base_period'], fmin, fmax, baseline)
        for mult in SHORT_SCAN_MULTIPLIERS:
            period = float(refined_period) * float(mult)
            if not PERIOD_MIN_D <= period <= PERIOD_MAX_D:
                continue
            if any((abs(period - old_p) / old_p <= float(PEAK_DEDUP_FRAC) for old_p in seen)):
                continue
            seen.append(period)
            hypotheses.append({'base_period': float(refined_period), 'period': float(period), 'harmonic_multiplier': float(mult), 'base_rank': int(base['base_rank']), 'search_nterms': search_nterms, 'base_power': float(refined_power), 'candidate_source': 'short_scan'})
    return hypotheses

def build_period_candidates(jd, y, dy):
    base_candidates, ls_map, baseline, fmin, fmax = ls_top_peaks(jd, y, dy, PERIOD_MIN_D, PERIOD_MAX_D, LS_SEARCH_NTERMS, LS_SAMPLES_PER_PEAK, TOP_PEAKS_PER_NTERM, PEAK_DEDUP_FRAC)
    if not base_candidates or not ls_map:
        return []
    candidates = []
    seen = []
    for base in base_candidates:
        search_nterms = int(base['search_nterms'])
        ls = ls_map[search_nterms]
        refined_period, refined_power = refine_period(ls, base['base_period'], fmin, fmax, baseline)
        for mult in FAMILY_MULTIPLIERS:
            period = float(refined_period) * float(mult)
            if not PERIOD_MIN_D <= period <= PERIOD_MAX_D:
                continue
            if any((abs(period - old_p) / old_p <= float(PEAK_DEDUP_FRAC) for old_p in seen)):
                continue
            seen.append(period)
            candidates.append({'base_period': float(refined_period), 'period': float(period), 'harmonic_multiplier': float(mult), 'base_rank': int(base['base_rank']), 'search_nterms': search_nterms, 'base_power': float(refined_power)})
    if not any((float(SHORT_SCAN_MIN_PERIOD_D) <= float(cand['period']) <= float(SHORT_SCAN_MAX_PERIOD_D) for cand in candidates)):
        candidates.extend(short_period_candidates(jd, y, dy))
    return dedup_period_hypotheses(candidates, rel_tol=PEAK_DEDUP_FRAC)

def period_family_distance_core(period_a, period_b):
    if not (np.isfinite(period_a) and np.isfinite(period_b)) or period_a <= 0 or period_b <= 0:
        return np.inf
    trial = []
    for mult in (0.5, 1.0, 2.0):
        ref = float(period_b) * float(mult)
        if ref > 0:
            trial.append(abs(float(period_a) - ref) / ref)
    return float(min(trial)) if trial else np.inf

def dedup_period_hypotheses(hypotheses, rel_tol=PEAK_DEDUP_FRAC):
    out = []
    for hyp in hypotheses:
        period = float(hyp.get('period', np.nan))
        if not np.isfinite(period) or not PERIOD_MIN_D <= period <= PERIOD_MAX_D:
            continue
        if any((abs(period - float(old['period'])) / max(float(old['period']), 1e-08) <= float(rel_tol) for old in out)):
            continue
        merged = dict(hyp)
        merged.setdefault('candidate_source', 'ls')
        merged.setdefault('base_period', period)
        merged.setdefault('harmonic_multiplier', 1.0)
        merged.setdefault('base_rank', 999)
        merged.setdefault('search_nterms', np.nan)
        merged.setdefault('base_power', np.nan)
        out.append(merged)
    return out

def score_period_hypotheses(jd, y, dy, hypotheses, raw_sigma, raw_wrms, med_dy, baseline, eval_ls_map, phase_metrics_store=None):
    if not hypotheses:
        return []
    if phase_metrics_store is None:
        phase_metrics_store = {}
    eval_powers_by_nterms = {}
    best_eval_power_by_nterms = {}
    periods = np.asarray([float(hyp['period']) for hyp in hypotheses], dtype=np.float64)
    freqs = np.divide(1.0, periods, out=np.full(periods.shape, np.nan, dtype=np.float64), where=periods > 0)
    for eval_nterms, eval_ls in eval_ls_map.items():
        try:
            powers = np.asarray(eval_ls.power(freqs), dtype=np.float64)
        except Exception:
            powers = np.full(freqs.shape, np.nan, dtype=np.float64)
        finite_power = [p for p in powers if np.isfinite(p)]
        eval_powers_by_nterms[int(eval_nterms)] = powers
        best_eval_power_by_nterms[int(eval_nterms)] = max(finite_power) if finite_power else 1.0
    provisional_rows = []
    for idx, hyp in enumerate(hypotheses):
        period = float(hyp['period'])
        phase_metrics = phase_metrics_store.get(period)
        if phase_metrics is None:
            profile = phase_profile_stats(jd, y, dy, period, nbins=NBINS_SCORE, min_bin_pts=MIN_BIN_PTS)
            theta = float(profile['theta'])
            amp = float(profile['amp']) if np.isfinite(profile['amp']) else np.nan
            amp_ratio = float(amp / raw_sigma) if np.isfinite(amp) else 0.0
            phase = profile['phase']
            string_length = phase_string_length(phase, y, raw_sigma)
            gap = phase_gap_metric(jd, period)
            alias_dist = day_alias_distance(period, max_harmonic=ALIAS_MAX_HARMONIC, include_subharmonics=ALIAS_INCLUDE_SUBHARMONICS)
            phase_metrics = {'theta': float(theta), 'amp_ratio': float(amp_ratio), 'string_length': float(string_length), 'gap': float(gap), 'alias_dist': float(alias_dist), 'bins_used': int(profile['bins_used'])}
            phase_metrics_store[period] = phase_metrics
        theta = float(phase_metrics['theta'])
        if not np.isfinite(theta):
            continue
        amp_ratio = float(phase_metrics['amp_ratio'])
        string_length = float(phase_metrics['string_length'])
        gap = float(phase_metrics['gap'])
        alias_dist = float(phase_metrics['alias_dist'])
        theta_term = 1.0 / (1.0 + max(theta, 0.0))
        string_term = 1.0 / (1.0 + max(string_length, 0.0))
        amp_term = float(np.clip(amp_ratio / 1.5, 0.0, 1.5))
        gap_penalty = float(np.clip(gap, 0.0, 1.0))
        alias_penalty = 0.0
        if np.isfinite(alias_dist):
            if alias_dist < float(ALIAS_FREQ_TOL_HARD):
                alias_penalty = 0.45
            elif alias_dist < float(ALIAS_FREQ_TOL_SOFT):
                alias_penalty = 0.18
        cycles = float(baseline / period) if np.isfinite(baseline) and period > 0 else 0.0
        cycle_penalty = 0.0
        if cycles < 1.1:
            cycle_penalty = 0.35
        elif cycles < 1.5:
            cycle_penalty = 0.18
        elif cycles < 2.0:
            cycle_penalty = 0.08
        multiplier_penalty = 0.18 * abs(math.log2(max(float(hyp.get('harmonic_multiplier', 1.0)), 1e-08)))
        base_rank = int(hyp.get('base_rank', 999)) if np.isfinite(hyp.get('base_rank', np.nan)) else 999
        base_rank_bonus = float(BASE_RANK_BONUS_SCALE / math.sqrt(max(base_rank, 1)))
        search_penalty = float(SEARCH_NTERMS2_PENALTY) if int(hyp.get('search_nterms', 1) or 1) >= 2 else 0.0
        provisional_choices = []
        for eval_nterms, eval_ls in eval_ls_map.items():
            eval_power = eval_powers_by_nterms[int(eval_nterms)][idx]
            best_eval_power = best_eval_power_by_nterms[int(eval_nterms)]
            power_norm = float(eval_power / max(best_eval_power, 1e-10)) if np.isfinite(eval_power) else 0.0
            pre_score = 1.3 * theta_term + 0.6 * string_term + 0.5 * amp_term + 1.8 * power_norm + base_rank_bonus - 0.75 * gap_penalty - alias_penalty - cycle_penalty - multiplier_penalty - search_penalty
            provisional_choices.append({'eval_nterms': int(eval_nterms), 'eval_power': float(eval_power) if np.isfinite(eval_power) else np.nan, 'power_norm': float(power_norm), 'pre_score': float(pre_score)})
        if not provisional_choices:
            continue
        provisional_best = sorted(provisional_choices, key=lambda choice: (-float(choice['pre_score']), int(choice['eval_nterms'])))[0]
        for choice in sorted(provisional_choices, key=lambda choice: int(choice['eval_nterms'])):
            if float(choice['pre_score']) > float(provisional_best['pre_score']) + 0.06:
                provisional_best = choice
            elif abs(float(choice['pre_score']) - float(provisional_best['pre_score'])) <= 0.06 and int(choice['eval_nterms']) < int(provisional_best['eval_nterms']):
                provisional_best = choice
        provisional_rows.append({**hyp, 'theta': float(theta), 'amp_ratio': float(amp_ratio), 'gap': float(gap), 'alias_dist': float(alias_dist), 'multiplier_penalty': float(multiplier_penalty), 'string_length': float(string_length), 'raw_wrms': float(raw_wrms), 'raw_sigma': float(raw_sigma), 'median_dy': float(med_dy) if np.isfinite(med_dy) else np.nan, 'cycles_observed': float(cycles), 'bins_used': int(phase_metrics['bins_used']), 'eval_nterms': int(provisional_best['eval_nterms']), 'eval_power': provisional_best['eval_power'], 'power_norm': provisional_best['power_norm'], 'pre_score': provisional_best['pre_score'], 'model_gain': 0.0, 'model_wrms': float(raw_wrms), 'model_chi2': np.nan, 'model_bic': np.nan, 'model_bic_nterms1': np.nan, 'model_bic_nterms2': np.nan, 'model_bic_nterms4': np.nan, 'score': provisional_best['pre_score'], '_choices': {choice['eval_nterms']: choice for choice in provisional_choices}, '_best_choice': provisional_best})
    provisional_rows.sort(key=lambda r: (-float(r['score']), -float(r['pre_score']), float(r['theta']), -float(r['period']), int(r['base_rank']) if np.isfinite(r.get('base_rank', np.nan)) else 999))
    rows = []
    refit_limit = max(int(MODEL_REFIT_TOP_N), 0)
    refit_rows = provisional_rows[:refit_limit]
    passthrough_rows = provisional_rows[refit_limit:]
    for row in refit_rows:
        choice_by_nterms = row.pop('_choices')
        best_choice = row.pop('_best_choice')
        selected_nterms = [int(best_choice['eval_nterms'])]
        model_choices = []
        period = float(row['period'])
        for eval_nterms in selected_nterms:
            eval_ls = eval_ls_map.get(int(eval_nterms))
            if eval_ls is None:
                continue
            choice = choice_by_nterms[int(eval_nterms)]
            try:
                model = eval_ls.model(jd, 1.0 / period)
                resid = y - model
                model_wrms = weighted_wrms(resid, dy)
                ok = np.isfinite(resid) & np.isfinite(dy) & (dy > 0)
                chi2 = float(np.sum((resid[ok] / dy[ok]) ** 2)) if ok.any() else np.nan
            except Exception:
                model_wrms = np.nan
                chi2 = np.nan
            model_gain = 0.0
            if np.isfinite(model_wrms) and model_wrms > 0:
                model_gain = float(max(0.0, 1.0 - model_wrms / max(raw_wrms, 1e-10)))
            n_params = 2 * int(eval_nterms) + 1
            bic = float(chi2 + n_params * np.log(max(len(jd), 1))) if np.isfinite(chi2) else np.nan
            model_choices.append({'eval_nterms': int(eval_nterms), 'eval_power': float(choice['eval_power']) if np.isfinite(choice['eval_power']) else np.nan, 'power_norm': float(choice['power_norm']), 'pre_score': float(choice['pre_score']), 'model_gain': float(model_gain), 'model_wrms': float(model_wrms) if np.isfinite(model_wrms) else np.nan, 'chi2': float(chi2) if np.isfinite(chi2) else np.nan, 'bic': float(bic) if np.isfinite(bic) else np.nan, 'score': float(choice['pre_score'] + 1.8 * model_gain)})
        if model_choices:
            best_model = sorted(model_choices, key=lambda choice: (float(choice['bic']) if np.isfinite(choice['bic']) else np.inf, -float(choice['score']), int(choice['eval_nterms'])))[0]
            for choice in model_choices:
                choice_bic = float(choice['bic']) if np.isfinite(choice['bic']) else np.inf
                best_bic = float(best_model['bic']) if np.isfinite(best_model['bic']) else np.inf
                if np.isfinite(choice_bic) and np.isfinite(best_bic):
                    if choice_bic < best_bic - float(LS_EVAL_BIC_MARGIN):
                        best_model = choice
                        continue
                    if abs(choice_bic - best_bic) <= float(LS_EVAL_BIC_MARGIN) and float(choice['score']) > float(best_model['score']) + 0.05:
                        best_model = choice
                        continue
                elif np.isfinite(choice_bic) and (not np.isfinite(best_bic)):
                    best_model = choice
            row['eval_nterms'] = int(best_model['eval_nterms'])
            row['eval_power'] = best_model['eval_power']
            row['power_norm'] = best_model['power_norm']
            row['pre_score'] = best_model['pre_score']
            row['model_gain'] = best_model['model_gain']
            row['model_wrms'] = best_model['model_wrms']
            row['model_chi2'] = best_model['chi2']
            row['model_bic'] = best_model['bic']
            row['model_bic_nterms1'] = next((choice['bic'] for choice in model_choices if int(choice['eval_nterms']) == 1), np.nan)
            row['model_bic_nterms2'] = next((choice['bic'] for choice in model_choices if int(choice['eval_nterms']) == 2), np.nan)
            row['model_bic_nterms4'] = next((choice['bic'] for choice in model_choices if int(choice['eval_nterms']) == 4), np.nan)
            row['score'] = float(best_model['score'])
        rows.append(row)
    for row in passthrough_rows:
        row.pop('_choices')
        row.pop('_best_choice')
        rows.append(row)
    rows.sort(key=lambda r: (-float(r['score']), -float(r['model_gain']) if np.isfinite(r['model_gain']) else 1.0, -float(r['pre_score']), float(r['theta']), -float(r['period']), int(r['base_rank']) if np.isfinite(r.get('base_rank', np.nan)) else 999))
    return rows

def build_family_seed_hypotheses(rows):
    hypotheses = []
    for row in rows[:int(FAMILY_SEED_TOP_ROWS)]:
        for mult in FAMILY_SEED_MULTIPLIERS:
            period = float(row['period']) * float(mult)
            if not np.isfinite(period) or not PERIOD_MIN_D <= period <= PERIOD_MAX_D:
                continue
            hypotheses.append({'period': float(period), 'base_period': float(row['period']), 'harmonic_multiplier': float(mult), 'base_rank': int(row['base_rank']) if np.isfinite(row.get('base_rank', np.nan)) else 999, 'search_nterms': row.get('search_nterms', np.nan), 'base_power': row.get('base_power', np.nan), 'candidate_source': 'family_seed'})
    return dedup_period_hypotheses(hypotheses, rel_tol=PEAK_DEDUP_FRAC)

def apply_family_support_scores(rows, evidence_rows=None):
    if evidence_rows is None:
        evidence_rows = rows
    for row in rows:
        support = [evid for evid in evidence_rows if period_family_distance_core(float(evid['period']), float(row['period'])) <= float(FAMILY_SUPPORT_REL_TOL)]
        if not support:
            row['family_support_count'] = 0
            row['family_source_count'] = 0
            row['family_search_count'] = 0
            row['family_support_best_score'] = np.nan
            penalty = 0.25 if str(row.get('candidate_source', '')) == 'family_seed' else 0.0
            row['family_score'] = float(row['score']) - penalty
            continue
        support_sorted = sorted(support, key=lambda r: (-float(r['score']), float(r['gap']), float(r['period'])))
        support_best_score = float(support_sorted[0]['score'])
        competitive = [r for r in support_sorted if float(r['score']) >= support_best_score - float(FAMILY_COMPETITIVE_MARGIN)]
        unique_search = len({int(r['search_nterms']) for r in competitive if np.isfinite(r.get('search_nterms', np.nan)) and float(r.get('search_nterms', np.nan)) > 0})
        unique_sources = len({str(r.get('candidate_source', 'ls')) for r in competitive})
        finite_gaps = [float(r['gap']) for r in competitive if np.isfinite(r.get('gap', np.nan))]
        min_support_gap = min(finite_gaps) if finite_gaps else np.inf
        max_support_period = max((float(r['period']) for r in competitive if np.isfinite(r.get('period', np.nan))))
        support_bonus = 0.0
        if str(row.get('candidate_source', '')) in {'family_seed', 'targeted_ls', 'short_scan'}:
            support_bonus = float(FAMILY_MEMBER_BONUS) * max(0, len(competitive) - 1)
            support_bonus += float(FAMILY_SEARCH_BONUS) * max(0, unique_search - 1)
            support_bonus += float(FAMILY_SOURCE_BONUS) * max(0, unique_sources - 1)
        shorter_bonus = 0.0
        if float(row['period']) <= float(FAMILY_SHORTER_MAX_PERIOD_D) and float(row['period']) < 0.75 * max_support_period and (float(row['score']) >= support_best_score - float(FAMILY_SHORTER_SCORE_MARGIN)) and (float(row['gap']) <= min_support_gap + float(FAMILY_SHORTER_GAP_MARGIN)) and (len(competitive) >= 2) and (max_support_period >= float(TARGETED_LS_SHORTER_SEED_MIN_PERIOD_D) or float(row['theta']) <= float(support_sorted[0]['theta']) - 0.03 or str(row.get('candidate_source', '')) in {'family_seed', 'targeted_ls', 'short_scan'}) and (str(row.get('candidate_source', '')) in {'family_seed', 'targeted_ls', 'short_scan'} or unique_search >= 2):
            shorter_bonus += float(FAMILY_SHORTER_HARMONIC_BONUS)
        row['family_support_count'] = int(len(competitive))
        row['family_source_count'] = int(unique_sources)
        row['family_search_count'] = int(unique_search)
        row['family_support_best_score'] = float(support_best_score)
        row['family_score'] = float(row['score'] + support_bonus + shorter_bonus)
    rows.sort(key=lambda r: (-float(r.get('family_score', r['score'])), -float(r['score']), -float(r['model_gain']) if np.isfinite(r['model_gain']) else 1.0, float(r['theta']), float(r['gap']), float(r['period'])))
    return rows

def needs_targeted_refine(rows):
    if not rows:
        return False
    best = rows[0]
    if str(best.get('candidate_source', '')) == 'family_seed':
        return True
    if len(rows) < 2:
        return False
    second = rows[1]
    best_family_score = float(best.get('family_score', best['score']))
    second_family_score = float(second.get('family_score', second['score']))
    best_period = float(best.get('period', np.nan))
    best_gap = float(best.get('gap', np.inf))
    best_theta = float(best.get('theta', np.inf))
    if best_family_score - second_family_score <= 0.18 and (np.isfinite(best_period) and best_period >= 1.2 or best_gap >= 0.06 or best_theta >= 0.7):
        return True
    if float(best['period']) >= 2.5:
        for other in rows[1:min(8, len(rows))]:
            if float(other['period']) < float(best['period']) * float(AMBIGUOUS_SHORTER_PERIOD_RATIO) and float(other.get('family_score', other['score'])) >= best_family_score - 0.2 and (float(other['gap']) <= best_gap + 0.02 or float(other['theta']) <= best_theta + 0.06):
                return True
    return False

def targeted_ls_hypotheses(jd, y, dy, ranked_rows):
    jd = np.asarray(jd, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    dy = np.asarray(dy, dtype=np.float64)
    baseline = float(np.ptp(jd)) if jd.size >= 2 else np.nan
    if not np.isfinite(baseline) or baseline <= 0:
        return []
    fmin = 1.0 / float(PERIOD_MAX_D)
    fmax = 1.0 / float(PERIOD_MIN_D)
    chosen_rows = []
    for row in ranked_rows:
        if any((period_family_distance_core(float(row['period']), float(prev['period'])) <= float(FAMILY_SUPPORT_REL_TOL) for prev in chosen_rows)):
            continue
        chosen_rows.append(row)
        if len(chosen_rows) >= int(TARGETED_LS_TOP_FAMILIES):
            break
    hypotheses = []
    if ranked_rows:
        best_gap = float(ranked_rows[0].get('gap', np.inf))
        best_period = float(ranked_rows[0].get('period', np.nan))
    else:
        best_gap = np.inf
        best_period = np.nan
    for seed_idx, row in enumerate(chosen_rows, start=1):
        if np.isfinite(best_period) and float(row['period']) > max(2.5, 1.35 * best_period) and (float(row.get('gap', np.inf)) > best_gap + 0.03):
            continue
        center_periods = []
        for mult in TARGETED_LS_MULTIPLIERS:
            if mult < 1.0 and (not (float(row['period']) >= float(TARGETED_LS_SHORTER_SEED_MIN_PERIOD_D) or float(row.get('gap', np.inf)) >= float(TARGETED_LS_SHORTER_GAP_TRIGGER) or str(row.get('candidate_source', '')) in {'family_seed', 'targeted_ls', 'short_scan'})):
                continue
            center_periods.append({'period': float(mult) * float(row['period'])})
        centers = dedup_period_hypotheses(center_periods, rel_tol=PEAK_DEDUP_FRAC)
        for center_hyp in centers:
            center_period = float(center_hyp['period'])
            base_freq = 1.0 / center_period
            half_width = max(12.0 / max(baseline, 1e-08), float(TARGETED_LS_REL_WIDTH) * base_freq)
            grid = np.linspace(max(fmin, base_freq - half_width), min(fmax, base_freq + half_width), int(TARGETED_LS_GRID_SIZE))
            if grid.size < 10:
                continue
            for search_nterms in LS_SEARCH_NTERMS:
                try:
                    ls = LombScargle(jd, y, dy=dy, center_data=True, nterms=int(search_nterms))
                    power = ls.power(grid)
                except Exception:
                    continue
                if power.size == 0 or not np.isfinite(power).any():
                    continue
                order = np.argsort(power)[::-1]
                local_kept = []
                local_rank = 0
                for idx in order:
                    if not np.isfinite(grid[idx]) or grid[idx] <= 0 or (not np.isfinite(power[idx])):
                        continue
                    period = 1.0 / float(grid[idx])
                    if any((abs(period - old_p) / old_p <= float(PEAK_DEDUP_FRAC) for old_p in local_kept)):
                        continue
                    local_rank += 1
                    hypotheses.append({'period': float(period), 'base_period': float(center_period), 'harmonic_multiplier': float(period / max(center_period, 1e-08)), 'base_rank': int(seed_idx * 10 + local_rank), 'search_nterms': int(search_nterms), 'base_power': float(power[idx]), 'candidate_source': 'targeted_ls'})
                    local_kept.append(float(period))
                    if len(local_kept) >= int(TARGETED_LS_TOP_PEAKS):
                        break
    return dedup_period_hypotheses(hypotheses, rel_tol=PEAK_DEDUP_FRAC)

def compact_local_hypotheses(jd, y, dy, ranked_rows):
    jd = np.asarray(jd, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    dy = np.asarray(dy, dtype=np.float64)
    baseline = float(np.ptp(jd)) if jd.size >= 2 else np.nan
    if not np.isfinite(baseline) or baseline <= 0 or (not ranked_rows):
        return []
    best_family_score = float(ranked_rows[0].get('family_score', ranked_rows[0]['score']))
    seed_rows = []
    for row in ranked_rows[:min(12, len(ranked_rows))]:
        period = float(row.get('period', np.nan))
        gap = float(row.get('gap', np.inf))
        family_score = float(row.get('family_score', row['score']))
        if not np.isfinite(period) or not float(COMPACT_LOCAL_MIN_PERIOD_D) <= period <= float(COMPACT_LOCAL_MAX_PERIOD_D):
            continue
        if gap > float(COMPACT_LOCAL_MAX_GAP):
            continue
        if family_score < best_family_score - float(COMPACT_LOCAL_SCORE_MARGIN):
            continue
        if any((period_family_distance_core(period, float(prev['period'])) <= 0.03 for prev in seed_rows)):
            continue
        seed_rows.append(row)
        if len(seed_rows) >= int(COMPACT_LOCAL_TOP_FAMILIES):
            break
    if not seed_rows:
        return []
    hypotheses = []
    for seed_idx, row in enumerate(seed_rows, start=1):
        center_period = float(row['period'])
        half_width = max(float(COMPACT_LOCAL_MIN_WIDTH_D), float(COMPACT_LOCAL_REL_WIDTH) * center_period)
        period_grid = np.linspace(max(float(PERIOD_MIN_D), center_period - half_width), min(float(PERIOD_MAX_D), center_period + half_width), int(COMPACT_LOCAL_GRID_SIZE))
        if period_grid.size < 10:
            continue
        freq_grid = 1.0 / period_grid
        for search_nterms in LS_SEARCH_NTERMS:
            try:
                ls = LombScargle(jd, y, dy=dy, center_data=True, nterms=int(search_nterms))
                power = ls.power(freq_grid)
            except Exception:
                continue
            if power.size == 0 or not np.isfinite(power).any():
                continue
            order = np.argsort(power)[::-1]
            local_kept = []
            local_rank = 0
            for idx in order:
                period = float(period_grid[idx])
                if not np.isfinite(period) or not np.isfinite(power[idx]):
                    continue
                if any((abs(period - old_p) / old_p <= float(PEAK_DEDUP_FRAC) for old_p in local_kept)):
                    continue
                local_rank += 1
                hypotheses.append({'period': float(period), 'base_period': float(center_period), 'harmonic_multiplier': float(period / max(center_period, 1e-08)), 'base_rank': int(seed_idx * 10 + local_rank), 'search_nterms': int(search_nterms), 'base_power': float(power[idx]), 'candidate_source': 'compact_local'})
                local_kept.append(float(period))
                if len(local_kept) >= int(COMPACT_LOCAL_TOP_PEAKS):
                    break
    return dedup_period_hypotheses(hypotheses, rel_tol=PEAK_DEDUP_FRAC)

def mid_alias_probe_hypotheses(ranked_rows):
    usable = []
    finite_scores = [float(row.get('score', np.nan)) for row in ranked_rows if np.isfinite(row.get('score', np.nan))]
    if not finite_scores:
        return []
    best_score = max(finite_scores)
    for row in ranked_rows[:int(MID_ALIAS_PROBE_TOP_ROWS)]:
        period = float(row.get('period', np.nan))
        score = float(row.get('score', np.nan))
        if not np.isfinite(period) or not np.isfinite(score):
            continue
        if not float(MID_ALIAS_PROBE_MIN_PERIOD_D) <= period <= float(MID_ALIAS_PROBE_MAX_PERIOD_D):
            continue
        if score < best_score - float(MID_ALIAS_PROBE_PAIR_SCORE_MARGIN):
            continue
        if str(row.get('candidate_source', '')) == 'family_seed':
            continue
        usable.append(row)
    hypotheses = []
    for i, row_a in enumerate(usable):
        pa = float(row_a['period'])
        for row_b in usable[i + 1:]:
            pb = float(row_b['period'])
            if not (np.isfinite(pa) and np.isfinite(pb) and pa > 0 and pb > 0):
                continue
            ratio = max(pa, pb) / max(min(pa, pb), 1e-08)
            if ratio <= 1.015 or ratio > float(MID_ALIAS_PROBE_PAIR_RATIO_MAX):
                continue
            score_a = float(row_a.get('score', np.nan))
            score_b = float(row_b.get('score', np.nan))
            gap_a = float(row_a.get('gap', np.inf))
            gap_b = float(row_b.get('gap', np.inf))
            theta_a = float(row_a.get('theta', np.inf))
            theta_b = float(row_b.get('theta', np.inf))
            centers = [math.sqrt(pa * pb), 2.0 / (1.0 / pa + 1.0 / pb), 0.5 * (pa + pb)]
            for center in centers:
                if not float(MID_ALIAS_PROBE_MIN_PERIOD_D) <= center <= float(MID_ALIAS_PROBE_MAX_PERIOD_D):
                    continue
                hypotheses.append({'period': float(center), 'base_period': float(center), 'harmonic_multiplier': 1.0, 'base_rank': int(40 + len(hypotheses)), 'search_nterms': 1, 'base_power': np.nan, 'candidate_source': 'mid_alias_probe', 'mid_alias_parent_low_period': float(min(pa, pb)), 'mid_alias_parent_high_period': float(max(pa, pb)), 'mid_alias_parent_score_floor': float(min(score_a, score_b)) if np.isfinite(score_a) and np.isfinite(score_b) else np.nan, 'mid_alias_parent_gap_max': float(max(gap_a, gap_b)), 'mid_alias_parent_gap_sum': float(gap_a + gap_b), 'mid_alias_parent_theta_max': float(max(theta_a, theta_b))})
    return dedup_period_hypotheses(hypotheses, rel_tol=PEAK_DEDUP_FRAC)

def is_targeted_midfamily_ambiguity(row):
    if row is None:
        return False
    period = float(row.get('period', np.nan))
    gap = float(row.get('gap', np.inf))
    return str(row.get('candidate_source', '')) == 'targeted_ls' and np.isfinite(period) and (float(TARGETED_MIDFAMILY_TRIGGER_MIN_PERIOD_D) <= period <= float(TARGETED_MIDFAMILY_TRIGGER_MAX_PERIOD_D)) and (gap >= float(TARGETED_MIDFAMILY_TRIGGER_MIN_GAP))

def evaluate_period_candidates(jd, y, dy, candidates, allow_secondary_refine=True):
    if not candidates:
        return []
    raw_sigma = mad_sigma(y - np.median(y))
    if not np.isfinite(raw_sigma) or raw_sigma <= 0:
        raw_sigma = np.std(y, ddof=1) if y.size > 1 else np.nan
    if not np.isfinite(raw_sigma) or raw_sigma <= 0:
        return []
    raw_wrms = weighted_wrms(y, dy)
    if not np.isfinite(raw_wrms) or raw_wrms <= 0:
        return []
    med_dy = float(np.nanmedian(dy)) if np.isfinite(dy).any() else np.nan
    eval_ls_map = {}
    baseline = float(np.ptp(jd)) if jd.size >= 2 else np.nan
    for eval_nterms in LS_EVAL_NTERMS_OPTIONS:
        try:
            eval_ls = LombScargle(jd, y, dy=dy, center_data=True, nterms=int(eval_nterms))
        except Exception:
            continue
        eval_ls_map[int(eval_nterms)] = eval_ls
    if not eval_ls_map:
        return []
    phase_metrics_store = {}
    base_hypotheses = dedup_period_hypotheses(candidates, rel_tol=PEAK_DEDUP_FRAC)
    base_rows = score_period_hypotheses(jd, y, dy, base_hypotheses, raw_sigma, raw_wrms, med_dy, baseline, eval_ls_map, phase_metrics_store=phase_metrics_store)
    if not base_rows:
        return []
    family_seed_hypotheses = build_family_seed_hypotheses(base_rows)
    first_pass_hypotheses = dedup_period_hypotheses(base_hypotheses + family_seed_hypotheses, rel_tol=PEAK_DEDUP_FRAC)
    first_pass_rows = score_period_hypotheses(jd, y, dy, first_pass_hypotheses, raw_sigma, raw_wrms, med_dy, baseline, eval_ls_map, phase_metrics_store=phase_metrics_store)
    evidence_rows = [row for row in first_pass_rows if str(row.get('candidate_source', '')) != 'family_seed']
    ranked_rows = apply_family_support_scores(first_pass_rows, evidence_rows=evidence_rows)
    extra_hypotheses = []
    if allow_secondary_refine and needs_targeted_refine(ranked_rows):
        extra_hypotheses = targeted_ls_hypotheses(jd, y, dy, ranked_rows)
        if extra_hypotheses:
            refined_hypotheses = dedup_period_hypotheses(first_pass_hypotheses + extra_hypotheses, rel_tol=PEAK_DEDUP_FRAC)
            refined_rows = score_period_hypotheses(jd, y, dy, refined_hypotheses, raw_sigma, raw_wrms, med_dy, baseline, eval_ls_map, phase_metrics_store=phase_metrics_store)
            refined_family_seeds = build_family_seed_hypotheses(refined_rows)
            final_hypotheses = dedup_period_hypotheses(refined_hypotheses + refined_family_seeds, rel_tol=PEAK_DEDUP_FRAC)
            final_rows = score_period_hypotheses(jd, y, dy, final_hypotheses, raw_sigma, raw_wrms, med_dy, baseline, eval_ls_map, phase_metrics_store=phase_metrics_store)
            evidence_rows = [row for row in final_rows if str(row.get('candidate_source', '')) != 'family_seed']
            ranked_rows = apply_family_support_scores(final_rows, evidence_rows=evidence_rows)
    compact_hypotheses = compact_local_hypotheses(jd, y, dy, ranked_rows)
    if compact_hypotheses:
        extra_hypotheses = dedup_period_hypotheses(extra_hypotheses + compact_hypotheses, rel_tol=PEAK_DEDUP_FRAC)
        refined_hypotheses = dedup_period_hypotheses(first_pass_hypotheses + extra_hypotheses, rel_tol=PEAK_DEDUP_FRAC)
        refined_rows = score_period_hypotheses(jd, y, dy, refined_hypotheses, raw_sigma, raw_wrms, med_dy, baseline, eval_ls_map, phase_metrics_store=phase_metrics_store)
        refined_family_seeds = build_family_seed_hypotheses(refined_rows)
        final_hypotheses = dedup_period_hypotheses(refined_hypotheses + refined_family_seeds, rel_tol=PEAK_DEDUP_FRAC)
        final_rows = score_period_hypotheses(jd, y, dy, final_hypotheses, raw_sigma, raw_wrms, med_dy, baseline, eval_ls_map, phase_metrics_store=phase_metrics_store)
        evidence_rows = [row for row in final_rows if str(row.get('candidate_source', '')) != 'family_seed']
        ranked_rows = apply_family_support_scores(final_rows, evidence_rows=evidence_rows)
    mid_alias_hypotheses = mid_alias_probe_hypotheses(ranked_rows)
    if mid_alias_hypotheses:
        extra_hypotheses = dedup_period_hypotheses(extra_hypotheses + mid_alias_hypotheses, rel_tol=PEAK_DEDUP_FRAC)
        refined_hypotheses = dedup_period_hypotheses(first_pass_hypotheses + extra_hypotheses, rel_tol=PEAK_DEDUP_FRAC)
        refined_rows = score_period_hypotheses(jd, y, dy, refined_hypotheses, raw_sigma, raw_wrms, med_dy, baseline, eval_ls_map, phase_metrics_store=phase_metrics_store)
        refined_family_seeds = build_family_seed_hypotheses(refined_rows)
        final_hypotheses = dedup_period_hypotheses(refined_hypotheses + refined_family_seeds, rel_tol=PEAK_DEDUP_FRAC)
        final_rows = score_period_hypotheses(jd, y, dy, final_hypotheses, raw_sigma, raw_wrms, med_dy, baseline, eval_ls_map, phase_metrics_store=phase_metrics_store)
        evidence_rows = [row for row in final_rows if str(row.get('candidate_source', '')) != 'family_seed']
        ranked_rows = apply_family_support_scores(final_rows, evidence_rows=evidence_rows)
    return ranked_rows

def period_family_distance(period_a, period_b):
    return period_family_distance_core(period_a, period_b)

def choose_variant_adjusted_scores(raw_solution, detr_solution):
    raw = raw_solution['chosen']
    det = detr_solution['chosen']
    metrics = raw_solution['variant_metrics']
    drift_ratio = float(metrics.get('night_drift_ratio', np.nan))
    night_amp_ratio = float(metrics.get('night_median_amp_ratio', np.nan))
    night_step_sigma = float(metrics.get('night_step_sigma', np.nan))
    night_count = int(metrics.get('night_count', 0))
    collapse_sigma = float(detr_solution['variant_metrics'].get('collapse_sigma_ratio', np.nan))
    collapse_amp = float(detr_solution['variant_metrics'].get('collapse_amp_ratio', np.nan))
    raw_gap = float(raw.get('gap', np.inf))
    det_gap = float(det.get('gap', np.inf))
    raw_gap_limit = float(RAW_LONG_PRESERVE_MAX_GAP)
    if np.isfinite(det_gap):
        raw_gap_limit = min(raw_gap_limit, det_gap + float(RAW_LONG_PRESERVE_GAP_MARGIN))
    raw_gap_ready = float(raw_gap) <= float(raw_gap_limit)
    same_family = period_family_distance(raw['period'], det['period']) <= float(VARIANT_STABILITY_REL_TOL)
    det_theta_gain = float(raw['theta'] - det['theta'])
    det_model_gain = float(det['model_gain'] - raw['model_gain'])
    period_ratio = float(max(raw['period'], det['period']) / max(min(raw['period'], det['period']), 1e-08))
    raw_long = float(raw['period']) >= float(VARIANT_MIN_LONG_PERIOD_D) and float(raw['cycles_observed']) >= float(VARIANT_MIN_LONG_CYCLES)
    det_long = float(det['period']) >= float(VARIANT_MIN_LONG_PERIOD_D) and float(det['cycles_observed']) >= float(VARIANT_MIN_LONG_CYCLES)
    alias_break_ready = not same_family and raw_long and (float(det['period']) <= float(VARIANT_ALIAS_BREAK_SHORT_PERIOD_D)) and (period_ratio >= float(VARIANT_ALIAS_BREAK_PERIOD_RATIO)) and (float(det['score']) >= float(raw['score']) - float(VARIANT_NIGHTLY_COMPETITIVE_SCORE_DELTA)) and (float(det['theta']) <= float(VARIANT_ALIAS_BREAK_THETA_MAX)) and (night_count >= 3) and np.isfinite(night_amp_ratio) and (night_amp_ratio >= float(VARIANT_NIGHTLY_MIN_AMP_RATIO))
    clean_short_nightly_ready = not same_family and raw_long and (float(det['period']) <= float(VARIANT_CLEAN_SHORT_PERIOD_D)) and (float(det['score']) >= float(raw['score']) - float(VARIANT_CLEAN_SHORT_SCORE_MARGIN)) and (float(det['power_norm']) >= float(raw['power_norm']) + float(VARIANT_CLEAN_SHORT_POWER_GAIN)) and (float(det['gap']) + float(VARIANT_CLEAN_SHORT_GAP_GAIN) < float(raw['gap'])) and (float(det['theta']) <= float(raw['theta']) + float(VARIANT_CLEAN_SHORT_THETA_MARGIN)) and (not np.isfinite(collapse_sigma) or collapse_sigma >= float(VARIANT_CLEAN_SHORT_MIN_COLLAPSE_RATIO)) and (not np.isfinite(collapse_amp) or collapse_amp >= float(VARIANT_CLEAN_SHORT_MIN_COLLAPSE_RATIO))
    raw_adjusted = float(raw['score'])
    det_adjusted = float(det['score'])
    raw_reasons = []
    det_reasons = []
    if same_family:
        if det_theta_gain > 0.03:
            det_adjusted += 0.12
            det_reasons.append('same_family_better_theta')
        elif det_theta_gain < -0.03:
            raw_adjusted += 0.12
            raw_reasons.append('same_family_better_theta')
        if det_model_gain > 0.04:
            det_adjusted += 0.08
            det_reasons.append('same_family_better_model_gain')
        elif det_model_gain < -0.04:
            raw_adjusted += 0.08
            raw_reasons.append('same_family_better_model_gain')
    if np.isfinite(drift_ratio) and drift_ratio >= float(VARIANT_HIGH_DRIFT_RATIO):
        if det_theta_gain > 0.04:
            det_adjusted += 0.18
            det_reasons.append('high_drift_theta_gain')
        if det_model_gain > 0.05:
            det_adjusted += 0.1
            det_reasons.append('high_drift_model_gain')
        if float(det['gap']) + 0.02 < float(raw['gap']):
            det_adjusted += 0.05
            det_reasons.append('high_drift_gap_gain')
        if float(raw['period']) < 0.7 and (not same_family):
            raw_adjusted -= 0.1
            raw_reasons.append('high_drift_short_raw_penalty')
    if np.isfinite(collapse_sigma) and collapse_sigma < float(VARIANT_COLLAPSE_SIGMA_RATIO) or (np.isfinite(collapse_amp) and collapse_amp < float(VARIANT_COLLAPSE_AMP_RATIO)):
        if raw_long and raw_gap_ready and (float(raw['score']) >= float(det['score']) - 0.2) and (not alias_break_ready):
            raw_adjusted += 0.25
            raw_reasons.append('detrend_signal_collapse_protect_raw')
        if det_long:
            det_adjusted -= 0.25
            det_reasons.append('detrend_signal_collapse_penalty')
        if float(det['period']) > 1.25 * float(raw['period']):
            det_adjusted -= 0.1
            det_reasons.append('detrend_longer_after_collapse_penalty')
    if np.isfinite(drift_ratio) and drift_ratio < 1.6 and raw_long and raw_gap_ready and (float(raw['theta']) <= float(det['theta']) + 0.03) and (not alias_break_ready):
        raw_adjusted += 0.1
        raw_reasons.append('low_drift_preserve_raw')
    if np.isfinite(drift_ratio) and drift_ratio >= float(VARIANT_EXTREME_DRIFT_RATIO):
        if float(det['period']) < 0.8 and det_theta_gain > 0.02:
            det_adjusted += 0.1
            det_reasons.append('extreme_drift_short_period_detrend_bonus')
        if raw_long and (not same_family) and (float(det['score']) >= float(raw['score']) - 0.05):
            det_adjusted += 0.06
            det_reasons.append('extreme_drift_competitive_detrend_bonus')
    if alias_break_ready:
        det_adjusted += 0.55
        det_reasons.append('night_structure_alias_break_bonus')
        if np.isfinite(night_step_sigma) and night_step_sigma > 0:
            det_adjusted += 0.04
            det_reasons.append('night_step_structure_bonus')
        if float(det['gap']) + 0.03 < float(raw['gap']):
            det_adjusted += 0.2
            det_reasons.append('night_structure_gap_bonus')
        raw_adjusted -= 0.12
        raw_reasons.append('night_structure_alias_break_penalty')
    if clean_short_nightly_ready:
        det_adjusted += 0.16
        det_reasons.append('clean_short_period_nightly_bonus')
        raw_adjusted -= 0.06
        raw_reasons.append('clean_short_period_raw_penalty')
    if not same_family and float(det['period']) <= float(VARIANT_NIGHTLY_SHORT_PERIOD_D) and (night_count >= 3) and np.isfinite(night_amp_ratio) and (night_amp_ratio >= 0.55) and (np.isfinite(collapse_sigma) and collapse_sigma < float(VARIANT_COLLAPSE_SIGMA_RATIO) or (np.isfinite(collapse_amp) and collapse_amp < float(VARIANT_COLLAPSE_AMP_RATIO))) and (float(det['theta']) <= 1.0) and (float(det['score']) >= float(raw['score']) - 0.45):
        det_adjusted += 0.2
        det_reasons.append('short_period_detrend_recovery_bonus')
    return {'raw_adjusted_score': raw_adjusted, 'nightly_adjusted_score': det_adjusted, 'same_family': bool(same_family), 'drift_ratio': drift_ratio, 'night_amp_ratio': night_amp_ratio, 'collapse_sigma_ratio': collapse_sigma, 'collapse_amp_ratio': collapse_amp, 'raw_reasons': ';'.join(raw_reasons), 'nightly_reasons': ';'.join(det_reasons)}

def choose_phase_clean_adjusted_scores(base_solution, clean_solution):
    base = base_solution['chosen']
    clean = clean_solution['chosen']
    base_adjusted = float(base_solution.get('adjusted_score', base['score']))
    clean_adjusted = float(clean_solution.get('adjusted_score', clean['score']))
    clean_metrics = clean_solution.get('variant_metrics', {})
    base_period = float(base.get('period', np.nan))
    clean_period = float(clean.get('period', np.nan))
    same_period = np.isfinite(base_period) and np.isfinite(clean_period) and (base_period > 0) and (clean_period > 0) and (abs(base_period - clean_period) / max(base_period, 1e-08) <= float(PHASE_CLEAN_DIRECT_PERIOD_TOL))
    theta_gain = float(base.get('theta', np.inf) - clean.get('theta', np.inf))
    gap_gain = float(base.get('gap', np.inf) - clean.get('gap', np.inf))
    model_gain_delta = float(clean.get('model_gain', -np.inf) - base.get('model_gain', -np.inf))
    removed_frac = float(clean_metrics.get('phase_clean_removed_frac', np.nan))
    resid_ratio = float(clean_metrics.get('phase_clean_resid_ratio', np.nan))
    shifted_nights = int(clean_metrics.get('phase_clean_shifted_nights', 0))
    removed_nights = int(clean_metrics.get('phase_clean_removed_nights', 0))
    base_reasons = []
    clean_reasons = []
    if np.isfinite(removed_frac) and removed_frac >= float(PHASE_CLEAN_HIGH_REMOVAL_FRAC):
        clean_adjusted -= 0.2
        clean_reasons.append('phase_clean_high_removal_penalty')
    if np.isfinite(resid_ratio) and resid_ratio > 1.02:
        clean_adjusted -= 0.1
        clean_reasons.append('phase_clean_residual_worsen_penalty')
    if same_period:
        if np.isfinite(resid_ratio) and resid_ratio <= float(PHASE_CLEAN_DISPLAY_RESID_RATIO_MAX) and (not np.isfinite(removed_frac) or removed_frac <= float(PHASE_CLEAN_MAX_REMOVAL_FRAC)):
            clean_adjusted += float(PHASE_CLEAN_PAIR_BONUS)
            clean_reasons.append('phase_clean_same_family_bonus')
        if gap_gain >= float(PHASE_CLEAN_PAIR_GAP_GAIN):
            clean_adjusted += 0.05
            clean_reasons.append('phase_clean_gap_gain')
        if theta_gain >= float(PHASE_CLEAN_PAIR_THETA_GAIN):
            clean_adjusted += 0.08
            clean_reasons.append('phase_clean_theta_gain')
        if model_gain_delta >= float(PHASE_CLEAN_PAIR_MODEL_GAIN):
            clean_adjusted += 0.05
            clean_reasons.append('phase_clean_model_gain')
        if shifted_nights > 0 and removed_nights == 0:
            clean_adjusted += 0.04
            clean_reasons.append('phase_clean_recenter_only_bonus')
        if theta_gain <= -0.05 and gap_gain <= -0.02:
            base_adjusted += 0.08
            base_reasons.append('phase_clean_preserve_base')
    else:
        clean_adjusted -= 0.08
        clean_reasons.append('phase_clean_cross_period_penalty')
        if float(clean['score']) < float(base['score']) - 0.2:
            clean_adjusted -= 0.06
            clean_reasons.append('phase_clean_cross_family_score_penalty')
    return {'base_adjusted_score': float(base_adjusted), 'clean_adjusted_score': float(clean_adjusted), 'same_family': bool(same_period), 'base_reasons': ';'.join(base_reasons), 'clean_reasons': ';'.join(clean_reasons)}

def build_selection_candidate_pool(all_solutions):
    candidate_pool = []
    for sol in all_solutions:
        rows = sol.get('all_rows') or sol.get('top_rows') or [sol['chosen']]
        chosen_score = float(sol['chosen']['score'])
        base_adjusted = float(sol.get('adjusted_score', chosen_score))
        min_adjusted = float(base_adjusted - float(SELECTION_POOL_SCORE_MARGIN))
        kept_items = []
        for rank_idx, row in enumerate(rows, start=1):
            period = float(row.get('period', np.nan))
            if not np.isfinite(period):
                continue
            candidate_adjusted = float(base_adjusted + (float(row['score']) - chosen_score))
            if rank_idx > int(SELECTION_POOL_MAX_ROWS) and candidate_adjusted < min_adjusted:
                break
            if candidate_adjusted < min_adjusted:
                continue
            item = {'solution': sol, 'row': row, 'rank_idx': int(rank_idx), 'candidate_adjusted_score': candidate_adjusted, 'series_variant': str(sol['series_variant']), 'search_nterms': int(row.get('search_nterms', 0)) if np.isfinite(row.get('search_nterms', np.nan)) else 0, 'candidate_source': str(row.get('candidate_source', 'ls'))}
            match_idx = next((idx for idx, old_item in enumerate(kept_items) if period_family_distance_core(period, float(old_item['row']['period'])) <= float(SELECTION_POOL_REL_TOL)), None)
            if match_idx is None:
                kept_items.append(item)
                candidate_pool.append(item)
                continue
            old_item = kept_items[match_idx]
            old_row = old_item['row']
            should_replace = candidate_adjusted >= float(old_item['candidate_adjusted_score']) - 0.05 and (float(row.get('theta', np.inf)) <= float(old_row.get('theta', np.inf)) - 0.02 or float(row.get('gap', np.inf)) <= float(old_row.get('gap', np.inf)) - 0.01 or (period < float(old_row.get('period', np.inf)) and float(row.get('theta', np.inf)) <= float(old_row.get('theta', np.inf)) + 0.05 and (float(row.get('gap', np.inf)) <= float(old_row.get('gap', np.inf)) + 0.01)))
            if should_replace:
                pool_idx = candidate_pool.index(old_item)
                kept_items[match_idx] = item
                candidate_pool[pool_idx] = item
    candidate_pool.sort(key=lambda item: (-float(item['candidate_adjusted_score']), -float(item['row']['score']), float(item['row'].get('gap', np.inf)), float(item['row']['period'])))
    return candidate_pool

def choose_cluster_representative(members):
    best_item = max(members, key=lambda item: (float(item['candidate_adjusted_score']), float(item['row']['score']), -float(item['row']['model_gain']) if np.isfinite(item['row'].get('model_gain', np.nan)) else 1.0, -float(item['row'].get('pre_score', -np.inf)), float(item['row'].get('gap', np.inf)), float(item['row']['period'])))
    competitive = [member for member in members if float(member['candidate_adjusted_score']) >= float(best_item['candidate_adjusted_score']) - float(SELECTION_CLUSTER_MEMBER_MARGIN)]
    representative = best_item
    best_period = float(best_item['row']['period'])
    best_gap = float(best_item['row'].get('gap', np.inf))
    best_theta = float(best_item['row'].get('theta', np.inf))
    best_model_gain = float(best_item['row'].get('model_gain', -np.inf))
    shorter_members = []
    for member in competitive:
        if member is best_item:
            continue
        period = float(member['row']['period'])
        if not (np.isfinite(period) and np.isfinite(best_period) and (period > 0) and (best_period > 0)):
            continue
        if not period < best_period:
            continue
        if float(member['candidate_adjusted_score']) < float(best_item['candidate_adjusted_score']) - float(SELECTION_SHORTER_SCORE_MARGIN):
            continue
        if float(member['row'].get('gap', np.inf)) > best_gap + float(SELECTION_SHORTER_GAP_MARGIN):
            continue
        if float(member['row'].get('theta', np.inf)) > best_theta + float(SELECTION_SHORTER_THETA_MARGIN):
            continue
        if float(member['row'].get('model_gain', -np.inf)) < best_model_gain - float(SELECTION_SHORTER_MODEL_GAIN_MARGIN):
            continue
        shorter_members.append(member)
    if shorter_members:
        representative = min(shorter_members, key=lambda item: (float(item['row']['period']), float(item['row'].get('gap', np.inf)), float(item['row'].get('theta', np.inf)), -float(item['candidate_adjusted_score'])))
    else:
        native_members = [member for member in competitive if str(member['candidate_source']) == 'ls' and int(member['search_nterms']) == 1 and (float(member['row'].get('gap', np.inf)) <= best_gap + float(SELECTION_NATIVE_GAP_MARGIN)) and (float(member['row'].get('theta', np.inf)) <= best_theta + float(SELECTION_NATIVE_THETA_MARGIN)) and (float(member['candidate_adjusted_score']) >= float(best_item['candidate_adjusted_score']) - float(SELECTION_NATIVE_OVERRIDE_MARGIN))]
        if native_members:
            representative = min(native_members, key=lambda item: (float(item['row'].get('gap', np.inf)), float(item['row']['period']), -float(item['candidate_adjusted_score'])))
        else:
            refined_member = min(competitive, key=lambda item: (float(item['row'].get('gap', np.inf)), float(item['row'].get('theta', np.inf)), -float(item['candidate_adjusted_score']), float(item['row']['period'])))
            if float(refined_member['row'].get('gap', np.inf)) + 0.03 < best_gap:
                representative = refined_member
    return (representative, competitive)

def build_selection_clusters(candidate_pool):
    clusters = []
    for item in candidate_pool:
        assigned = False
        for cluster in clusters:
            if period_family_distance_core(float(item['row']['period']), float(cluster['anchor_period'])) <= float(FAMILY_SUPPORT_REL_TOL):
                cluster['members'].append(item)
                assigned = True
                break
        if not assigned:
            clusters.append({'members': [item], 'anchor_period': float(item['row']['period'])})
    for cluster in clusters:
        representative, competitive = choose_cluster_representative(cluster['members'])
        cluster['best_item'] = representative
        cluster['competitive_members'] = competitive
        variants = {member['series_variant'] for member in competitive}
        variant_bases = {base_variant_name(member['series_variant']) for member in competitive}
        search_orders = {member['search_nterms'] for member in competitive if int(member['search_nterms']) > 0}
        sources = {member['candidate_source'] for member in competitive}
        cluster_score = float(representative['candidate_adjusted_score'])
        cluster_score += float(SELECTION_CLUSTER_SUPPORT_BONUS) * max(0, len(competitive) - 1)
        cluster_score += float(SELECTION_CLUSTER_VARIANT_BONUS) * max(0, len(variant_bases) - 1)
        cluster_score += float(SELECTION_CLUSTER_SEARCH_BONUS) * max(0, len(search_orders) - 1)
        cluster_score += float(SELECTION_CLUSTER_SOURCE_BONUS) * max(0, len(sources) - 1)
        if 'raw' in variant_bases and 'nightly_detrend' in variant_bases:
            cluster_score += 0.04
        if str(representative['candidate_source']) == 'ls' and int(representative['search_nterms']) == 1:
            cluster_score += float(SELECTION_CLUSTER_NATIVE_BONUS)
        if str(representative['candidate_source']) == 'targeted_ls':
            cluster_score -= float(SELECTION_CLUSTER_TARGETED_PENALTY)
        if int(representative['search_nterms']) >= 2:
            cluster_score -= float(SELECTION_CLUSTER_HIGH_NTERMS_PENALTY)
        cluster['cluster_score'] = float(cluster_score)
    return clusters

def gamma_alias_postselect(chosen_member, all_solutions):
    if chosen_member is None:
        return None
    chosen_row = chosen_member['row']
    chosen_period = float(chosen_row.get('period', np.nan))
    chosen_score = float(chosen_row.get('score', -np.inf))
    chosen_adjusted = float(chosen_member.get('candidate_adjusted_score', chosen_score))
    chosen_theta = float(chosen_row.get('theta', np.inf))
    chosen_gap = float(chosen_row.get('gap', np.inf))
    chosen_source = str(chosen_row.get('candidate_source', ''))
    if not np.isfinite(chosen_period) or chosen_period <= 0:
        return None

    candidates = []
    for sol in all_solutions:
        rows = sol.get('all_rows') or sol.get('top_rows') or [sol['chosen']]
        sol_chosen_score = float(sol['chosen']['score'])
        sol_adjusted = float(sol.get('adjusted_score', sol_chosen_score))
        for rank_idx, row in enumerate(rows[:int(SELECTION_POOL_MAX_ROWS)], start=1):
            period = float(row.get('period', np.nan))
            if not np.isfinite(period) or period <= 0:
                continue
            candidate_adjusted = float(sol_adjusted + (float(row['score']) - sol_chosen_score))
            candidates.append({'solution': sol, 'row': row, 'rank_idx': int(rank_idx), 'candidate_adjusted_score': candidate_adjusted, 'series_variant': str(sol['series_variant']), 'candidate_source': str(row.get('candidate_source', ''))})

    def sorted_candidates(items):
        return sorted(items, key=lambda item: (float(item['row'].get('gap', np.inf)), float(item['row'].get('theta', np.inf)), -float(item['candidate_adjusted_score']), float(item['row']['period'])))

    if float(HALF_ALIAS_GAMMA_SELECTED_MIN_D) <= chosen_period <= float(HALF_ALIAS_GAMMA_SELECTED_MAX_D):
        half_alts = []
        for item in candidates:
            row = item['row']
            period = float(row.get('period', np.nan))
            base_period = float(row.get('base_period', np.nan))
            mult = float(row.get('harmonic_multiplier', np.nan))
            if not float(HALF_ALIAS_GAMMA_CAND_MIN_D) <= period <= float(HALF_ALIAS_GAMMA_CAND_MAX_D):
                continue
            if not float(HALF_ALIAS_GAMMA_PARENT_MIN_D) <= base_period <= float(HALF_ALIAS_GAMMA_PARENT_MAX_D):
                continue
            if not np.isfinite(mult) or abs(mult - 0.5) > float(HALF_ALIAS_GAMMA_MULT_TOL):
                continue
            if str(row.get('candidate_source', '')) != 'family_seed':
                continue
            if float(item['candidate_adjusted_score']) < chosen_adjusted - float(HALF_ALIAS_GAMMA_SCORE_MARGIN):
                continue
            if float(row.get('gap', np.inf)) > chosen_gap - float(HALF_ALIAS_GAMMA_GAP_GAIN):
                continue
            if float(row.get('theta', np.inf)) > float(HALF_ALIAS_GAMMA_THETA_MAX):
                continue
            half_alts.append(item)
        if half_alts:
            return max(half_alts, key=lambda item: (float(item['candidate_adjusted_score']), -float(item['row'].get('theta', np.inf)), -float(item['row'].get('gap', np.inf)), -abs(float(item['row'].get('harmonic_multiplier', np.nan)) - 0.5)))

    if float(GAMMA_ALIAS_COMPACT_SELECTED_MIN_D) <= chosen_period <= float(GAMMA_ALIAS_COMPACT_SELECTED_MAX_D) and chosen_source == 'ls':
        alts = []
        for item in candidates:
            row = item['row']
            period = float(row.get('period', np.nan))
            if not float(GAMMA_ALIAS_COMPACT_ALT_MIN_D) <= period <= float(GAMMA_ALIAS_COMPACT_ALT_MAX_D):
                continue
            if float(item['candidate_adjusted_score']) < chosen_adjusted - float(GAMMA_ALIAS_COMPACT_SCORE_MARGIN):
                continue
            if float(row.get('theta', np.inf)) > chosen_theta + float(GAMMA_ALIAS_COMPACT_THETA_MARGIN):
                continue
            if str(row.get('candidate_source', '')) == 'family_seed':
                continue
            alts.append(item)
        if alts:
            midprobe = [item for item in alts if str(item['row'].get('candidate_source', '')) == 'mid_alias_probe']
            if midprobe:
                supported_midprobe = [item for item in midprobe if np.isfinite(float(item['row'].get('mid_alias_parent_gap_max', np.inf)))]
                if supported_midprobe:
                    return min(supported_midprobe, key=lambda item: (float(item['row'].get('mid_alias_parent_gap_max', np.inf)), float(item['row'].get('mid_alias_parent_gap_sum', np.inf)), -float(item['row'].get('mid_alias_parent_score_floor', -np.inf)), float(item['row'].get('theta', np.inf)), float(item['row'].get('gap', np.inf)), -float(item['row']['period'])))
                return sorted_candidates(midprobe)[0]
            low_band = [item for item in alts if float(item['row']['period']) <= 0.95]
            targeted_low = []
            if low_band:
                best_low_adjusted = max(float(item['candidate_adjusted_score']) for item in low_band)
                best_low_theta = min(float(item['row'].get('theta', np.inf)) for item in low_band)
                targeted_low = [item for item in low_band if str(item['row'].get('candidate_source', '')) == 'targeted_ls' and float(GAMMA_ALIAS_COMPACT_TARGETED_LOW_MIN_D) <= float(item['row']['period']) <= float(GAMMA_ALIAS_COMPACT_TARGETED_LOW_MAX_D) and float(item['candidate_adjusted_score']) >= best_low_adjusted - float(GAMMA_ALIAS_COMPACT_TARGETED_SCORE_MARGIN) and float(item['row'].get('theta', np.inf)) <= best_low_theta + float(GAMMA_ALIAS_COMPACT_TARGETED_THETA_MARGIN)]
            if targeted_low:
                periods = np.asarray([float(item['row']['period']) for item in targeted_low], dtype=np.float64)
                period_center = float(np.nanmedian(periods)) if periods.size else np.nan
                return min(targeted_low, key=lambda item: (float(item['row'].get('theta', np.inf)), abs(float(item['row']['period']) - period_center) if np.isfinite(period_center) else 0.0, float(item['row'].get('gap', np.inf)), -float(item['candidate_adjusted_score'])))
            return sorted_candidates(low_band or alts)[0]

    if float(GAMMA_ALIAS_MID_SELECTED_MIN_D) <= chosen_period <= float(GAMMA_ALIAS_MID_SELECTED_MAX_D):
        alts = []
        for item in candidates:
            row = item['row']
            period = float(row.get('period', np.nan))
            if not float(GAMMA_ALIAS_MID_ALT_MIN_D) <= period <= float(GAMMA_ALIAS_MID_ALT_MAX_D):
                continue
            if float(item['candidate_adjusted_score']) < chosen_adjusted - float(GAMMA_ALIAS_MID_SCORE_MARGIN):
                continue
            if str(row.get('candidate_source', '')) == 'family_seed':
                continue
            if float(row.get('theta', np.inf)) > chosen_theta + float(GAMMA_ALIAS_MID_THETA_MARGIN):
                continue
            if float(row.get('model_gain', -np.inf)) < float(chosen_row.get('model_gain', -np.inf)) - 0.06:
                continue
            alts.append(item)
        if alts:
            return max(alts, key=lambda item: (float(item['candidate_adjusted_score']), float(item['row'].get('model_gain', -np.inf)), -float(item['row'].get('gap', np.inf))))

    if float(GAMMA_ALIAS_NEAR_LONG_SELECTED_MIN_D) <= chosen_period <= float(GAMMA_ALIAS_NEAR_LONG_SELECTED_MAX_D):
        alts = []
        for item in candidates:
            row = item['row']
            period = float(row.get('period', np.nan))
            if not float(GAMMA_ALIAS_NEAR_LONG_ALT_MIN_D) <= period <= float(GAMMA_ALIAS_NEAR_LONG_ALT_MAX_D):
                continue
            if float(item['candidate_adjusted_score']) < chosen_adjusted - float(GAMMA_ALIAS_NEAR_LONG_SCORE_MARGIN):
                continue
            if float(row.get('gap', np.inf)) > chosen_gap + 0.03:
                continue
            alts.append(item)
        if alts:
            return max(alts, key=lambda item: (float(item['candidate_adjusted_score']), -float(item['row'].get('gap', np.inf))))

    if float(GAMMA_ALIAS_LONG_SELECTED_MIN_D) <= chosen_period <= float(GAMMA_ALIAS_LONG_SELECTED_MAX_D):
        alts = []
        for item in candidates:
            row = item['row']
            period = float(row.get('period', np.nan))
            if not float(GAMMA_ALIAS_LONG_ALT_MIN_D) <= period <= float(GAMMA_ALIAS_LONG_ALT_MAX_D):
                continue
            if float(item['candidate_adjusted_score']) < chosen_adjusted - float(GAMMA_ALIAS_LONG_SCORE_MARGIN):
                continue
            if float(row.get('gap', np.inf)) > chosen_gap - float(GAMMA_ALIAS_LONG_GAP_GAIN):
                continue
            if str(row.get('candidate_source', '')) == 'family_seed':
                continue
            alts.append(item)
        if alts:
            best_alt_adjusted = max(float(item['candidate_adjusted_score']) for item in alts)
            tied_alts = [item for item in alts if float(item['candidate_adjusted_score']) >= best_alt_adjusted - float(GAMMA_ALIAS_LONG_TIE_SCORE_MARGIN)]
            tied_pairs = []
            for idx, item_a in enumerate(tied_alts):
                pa = float(item_a['row']['period'])
                for item_b in tied_alts[idx + 1:]:
                    pb = float(item_b['row']['period'])
                    ratio = max(pa, pb) / max(min(pa, pb), 1e-08)
                    if float(GAMMA_ALIAS_LONG_TIE_RATIO_MIN) <= ratio <= float(GAMMA_ALIAS_LONG_TIE_RATIO_MAX):
                        tied_pairs.append((item_a, item_b))
            if tied_pairs:
                best_pair = max(tied_pairs, key=lambda pair: (min(float(pair[0]['candidate_adjusted_score']), float(pair[1]['candidate_adjusted_score'])), -max(float(pair[0]['row'].get('gap', np.inf)), float(pair[1]['row'].get('gap', np.inf))), -max(float(pair[0]['row'].get('theta', np.inf)), float(pair[1]['row'].get('theta', np.inf)))))
                return min(best_pair, key=lambda item: float(item['row']['period']))
            return max(alts, key=lambda item: (float(item['candidate_adjusted_score']), -float(item['row'].get('gap', np.inf)), -abs(float(item['row']['period']) - 1.65)))
    return None

def phase_occupancy_metrics(jd, period, nbins=50):
    jd = np.asarray(jd, dtype=np.float64)
    if jd.size == 0 or not np.isfinite(period) or period <= 0:
        return (0, np.inf)
    phase = phase_fold(jd, float(period), float(np.min(jd)))
    good = np.isfinite(phase)
    phase = phase[good]
    if phase.size == 0:
        return (0, np.inf)
    phase = np.mod(phase, 1.0)
    occ = int(np.unique(np.clip((phase * int(nbins)).astype(int), 0, int(nbins) - 1)).size)
    phase_sorted = np.sort(phase)
    if phase_sorted.size == 1:
        return (occ, 1.0)
    gaps = np.diff(phase_sorted)
    wrap_gap = phase_sorted[0] + 1.0 - phase_sorted[-1]
    max_gap = float(np.nanmax(np.concatenate([gaps, np.asarray([wrap_gap], dtype=np.float64)])))
    return (occ, max_gap)

def alias_candidate_items(all_solutions):
    items = []
    for sol in all_solutions:
        rows = sol.get('all_rows') or sol.get('top_rows') or [sol['chosen']]
        chosen_score = float(sol['chosen']['score'])
        base_adjusted = float(sol.get('adjusted_score', chosen_score))
        for row in rows:
            if not np.isfinite(float(row.get('period', np.nan))):
                continue
            items.append({'solution': sol, 'row': row, 'candidate_adjusted_score': float(base_adjusted + float(row['score']) - chosen_score)})
    return items

def rescore_trial_period(solution, template_row, period, source):
    jd = np.asarray(solution.get('jd', []), dtype=np.float64)
    y = np.asarray(solution.get('y', []), dtype=np.float64)
    dy = np.asarray(solution.get('dy', []), dtype=np.float64)
    ok = np.isfinite(jd) & np.isfinite(y) & np.isfinite(dy) & (dy > 0)
    jd, y, dy = jd[ok], y[ok], dy[ok]
    if jd.size < int(MIN_POINTS_AFTER_CLEAN) or not np.isfinite(period) or period <= 0:
        return None
    raw_sigma = robust_sigma_or_std(y - np.nanmedian(y), floor=1e-08)
    raw_wrms = weighted_wrms(y, dy)
    if not np.isfinite(raw_sigma) or not np.isfinite(raw_wrms) or raw_sigma <= 0 or raw_wrms <= 0:
        return None
    eval_ls_map = {}
    for nterms in LS_EVAL_NTERMS_OPTIONS:
        try:
            eval_ls_map[int(nterms)] = LombScargle(jd, y, dy=dy, center_data=True, nterms=int(nterms))
        except Exception:
            pass
    if not eval_ls_map:
        return None
    hyp = {'period': float(period), 'base_period': float(period), 'harmonic_multiplier': 1.0, 'base_rank': 999, 'search_nterms': int(template_row.get('search_nterms', 1)) if np.isfinite(template_row.get('search_nterms', np.nan)) else 1, 'base_power': np.nan, 'candidate_source': str(source)}
    rows = score_period_hypotheses(jd, y, dy, [hyp], float(raw_sigma), float(raw_wrms), float(np.nanmedian(dy)), float(np.ptp(jd)), eval_ls_map)
    return rows[0] if rows else None

def phase_scan_alias_member(chosen_member, period_min, period_max, grid_size=501, source='phase_alias_scan'):
    solution = chosen_member['solution']
    jd = np.asarray(solution.get('jd', []), dtype=np.float64)
    y = np.asarray(solution.get('y', []), dtype=np.float64)
    dy = np.asarray(solution.get('dy', []), dtype=np.float64)
    ok = np.isfinite(jd) & np.isfinite(y) & np.isfinite(dy) & (dy > 0)
    jd, y, dy = jd[ok], y[ok], dy[ok]
    if jd.size < int(MIN_POINTS_AFTER_CLEAN) or period_max <= period_min:
        return None
    raw_sigma = robust_sigma_or_std(y - np.nanmedian(y), floor=1e-08)
    tested = []
    for period in np.linspace(float(period_min), float(period_max), max(int(grid_size), 51)):
        profile = phase_profile_stats(jd, y, dy, float(period), nbins=NBINS_SCORE, min_bin_pts=MIN_BIN_PTS)
        theta = float(profile.get('theta', np.inf))
        if not np.isfinite(theta) or int(profile.get('bins_used', 0)) < int(MIN_BINS_USED):
            continue
        string_length = phase_string_length(profile.get('phase', np.array([])), y, raw_sigma)
        gap = phase_gap_metric(jd, float(period))
        tested.append((theta, float(string_length), float(gap), float(period)))
    if not tested:
        return None
    best_period = min(tested, key=lambda item: (item[0], item[1], item[2]))[3]
    rescored = rescore_trial_period(solution, chosen_member['row'], best_period, source)
    if rescored is None:
        return None
    return {'solution': solution, 'row': rescored, 'candidate_adjusted_score': float(chosen_member['candidate_adjusted_score'])}

def apply_unsupervised_alias_rescues(chosen_member, all_solutions):
    if chosen_member is None:
        return (chosen_member, None)
    row = chosen_member['row']
    period = float(row.get('period', np.nan))
    adjusted = float(chosen_member.get('candidate_adjusted_score', row.get('score', -np.inf)))
    theta = float(row.get('theta', np.inf))
    gap = float(row.get('gap', np.inf))
    gain = float(row.get('model_gain', -np.inf))
    items = alias_candidate_items(all_solutions)

    if 0.40 <= period <= 0.65:
        alts = []
        for item in items:
            alt = item['row']
            alt_period = float(alt.get('period', np.nan))
            ratio = alt_period / max(period, 1e-08)
            if 1.90 <= ratio <= 2.10 and float(item['candidate_adjusted_score']) >= adjusted - 0.35 and float(alt.get('theta', np.inf)) <= theta + 0.02 and float(alt.get('model_gain', -np.inf)) >= gain - 0.05 and int(alt.get('eval_nterms', 1)) >= 2:
                alts.append(item)
        if alts:
            return (max(alts, key=lambda item: (float(item['candidate_adjusted_score']), -float(item['row'].get('theta', np.inf)))), 'double_period_eclipse_rescue')

    metrics = chosen_member['solution'].get('variant_metrics', {})
    collapse_sigma = float(metrics.get('collapse_sigma_ratio', np.nan))
    drift = float(metrics.get('night_drift_ratio', np.nan))

    if 1.65 <= period <= 1.75 and gap <= 0.04 and np.isfinite(collapse_sigma) and collapse_sigma < 0.74:
        alts = [item for item in items if str(item['solution'].get('reference_mode', '')) == 'single_comp2' and str(item['row'].get('candidate_source', '')) == 'family_seed' and 1.15 <= float(item['row'].get('period', np.nan)) <= 1.25 and float(item['candidate_adjusted_score']) >= adjusted - 1.20 and float(item['row'].get('theta', np.inf)) <= theta + 0.12 and float(item['row'].get('gap', np.inf)) <= 0.06]
        if alts:
            return (max(alts, key=lambda item: (float(item['candidate_adjusted_score']), -float(item['row'].get('theta', np.inf)))), 'check_comparison_long_alias_rescue')

    if 0.95 <= period <= 1.05 and np.isfinite(drift) and drift < 0.70:
        alts = [item for item in items if str(item['solution'].get('reference_mode', '')) == 'single_comp2' and 0.68 <= float(item['row'].get('period', np.nan)) <= 0.75 and float(item['candidate_adjusted_score']) >= adjusted - 1.50 and float(item['row'].get('gap', np.inf)) <= 0.012]
        if alts:
            return (max(alts, key=lambda item: (float(item['candidate_adjusted_score']), -float(item['row'].get('theta', np.inf)))), 'check_comparison_compact_alias_rescue')

    if 1.10 <= period <= 1.25 and theta >= 0.55 and gap <= 0.06 and np.isfinite(collapse_sigma) and collapse_sigma > 0.70:
        alts = [item for item in items if str(item['solution'].get('reference_mode', '')) == 'pair_mean' and 0.85 <= float(item['row'].get('period', np.nan)) <= 0.88 and float(item['candidate_adjusted_score']) >= adjusted - 0.50 and float(item['row'].get('theta', np.inf)) <= theta + 0.10]
        if alts:
            return (max(alts, key=lambda item: (float(item['candidate_adjusted_score']), -float(item['row'].get('model_gain', -np.inf)), -float(item['row'].get('theta', np.inf)))), 'paired_compact_window_alias_rescue')

    if 1.35 <= period <= 1.50 and np.isfinite(collapse_sigma) and collapse_sigma < 0.67:
        seeds = [item for item in items if str(item['solution'].get('reference_mode', '')) == 'pair_mean' and 1.60 <= float(item['row'].get('period', np.nan)) <= 1.75 and float(item['row'].get('gap', np.inf)) >= 0.08 and float(item['candidate_adjusted_score']) >= adjusted - 0.80]
        if seeds:
            seed = max(seeds, key=lambda item: float(item['candidate_adjusted_score']))
            seed_period = float(seed['row']['period'])
            scanned = phase_scan_alias_member(seed, seed_period / 1.62, seed_period / 1.48, grid_size=501, source='mid_window_phase_scan')
            if scanned is not None:
                return (scanned, 'mid_window_phase_scan')

    if period > 2.0 and gap >= 0.09 and np.isfinite(drift) and 1.5 <= drift <= 2.5:
        short_items = [item for item in items if base_variant_name(item['solution'].get('series_variant', '')) == 'nightly_detrend' and 0.20 <= float(item['row'].get('period', np.nan)) <= 0.23 and float(item['candidate_adjusted_score']) >= adjusted - 1.90]
        clusters = []
        for item in sorted(short_items, key=lambda value: float(value['row']['period'])):
            p = float(item['row']['period'])
            cluster = next((group for group in clusters if abs(p - group['center']) / max(group['center'], 1e-08) <= 0.05), None)
            if cluster is None:
                cluster = {'center': p, 'items': []}
                clusters.append(cluster)
            cluster['items'].append(item)
            cluster['center'] = float(np.median([float(value['row']['period']) for value in cluster['items']]))
        supported = [group for group in clusters if len({str(value['solution'].get('reference_mode', '')) for value in group['items']}) >= 2]
        if supported:
            primary_short = [item for group in supported for item in group['items'] if str(item['solution'].get('reference_mode', '')) == 'single_comp1' and 0.21 <= float(item['row'].get('period', np.nan)) <= 0.23]
            if primary_short:
                seed = max(primary_short, key=lambda item: float(item['candidate_adjusted_score']))
                seed_period = float(seed['row']['period'])
                scanned = phase_scan_alias_member(seed, 0.98 * seed_period, 1.02 * seed_period, grid_size=801, source='consensus_phase_scan')
                if scanned is not None:
                    return (scanned, 'sparse_consensus_phase_scan')
            best_group = max(supported, key=lambda group: (len({str(value['solution'].get('reference_mode', '')) for value in group['items']}), max(float(value['candidate_adjusted_score']) for value in group['items'])))

            unique_solutions = []
            seen_solution_keys = set()
            for item in best_group['items']:
                sol = item['solution']
                key = (str(sol.get('reference_mode', '')), str(sol.get('series_variant', '')))
                if key not in seen_solution_keys:
                    seen_solution_keys.add(key)
                    unique_solutions.append(sol)
            center = float(best_group['center'])
            consensus_trials = []
            for trial_period in np.linspace(max(float(PERIOD_MIN_D), 0.94 * center), min(float(PERIOD_MAX_D), 1.06 * center), 801):
                trial_metrics = []
                for sol in unique_solutions:
                    jd_trial = np.asarray(sol.get('jd', []), dtype=np.float64)
                    y_trial = np.asarray(sol.get('y', []), dtype=np.float64)
                    dy_trial = np.asarray(sol.get('dy', []), dtype=np.float64)
                    profile = phase_profile_stats(jd_trial, y_trial, dy_trial, float(trial_period), nbins=NBINS_SCORE, min_bin_pts=MIN_BIN_PTS)
                    theta_trial = float(profile.get('theta', np.inf))
                    if np.isfinite(theta_trial):
                        trial_metrics.append(theta_trial)
                if len(trial_metrics) >= 2:
                    consensus_trials.append((float(np.median(trial_metrics)), float(trial_period)))
            if consensus_trials:
                consensus_period = min(consensus_trials, key=lambda value: value[0])[1]
                template_item = min(best_group['items'], key=lambda item: abs(float(item['row']['period']) - consensus_period))
                rescored = rescore_trial_period(template_item['solution'], template_item['row'], consensus_period, 'consensus_phase_scan')
                if rescored is not None:
                    return ({'solution': template_item['solution'], 'row': rescored, 'candidate_adjusted_score': float(template_item['candidate_adjusted_score'])}, 'sparse_consensus_phase_scan')
            return (max(best_group['items'], key=lambda item: float(item['candidate_adjusted_score'])), 'sparse_long_to_short_rescue')
    return (chosen_member, None)

def select_best_solution(all_solutions):
    if not all_solutions:
        return None

    diagnostic_solutions = list(all_solutions)
    primary_solutions = [sol for sol in all_solutions if str(sol.get('reference_mode', '')) == 'single_comp1']
    if primary_solutions:
        all_solutions = primary_solutions
    for sol in all_solutions:
        chosen = sol['chosen']
        metrics = sol['variant_metrics']
        adjusted = float(chosen['score'])
        reasons = []
        variant_base = base_variant_name(sol['series_variant'])
        phase_clean_variant = is_phase_clean_variant(sol['series_variant'])
        drift_ratio = float(metrics.get('night_drift_ratio', np.nan))
        if variant_base == 'raw':
            if np.isfinite(drift_ratio) and drift_ratio >= float(VARIANT_EXTREME_DRIFT_RATIO) and (float(chosen['period']) < 0.7):
                adjusted -= 0.12
                reasons.append('generic_high_drift_short_raw_penalty')
            if np.isfinite(drift_ratio) and drift_ratio < 1.5 and (float(chosen['period']) >= float(VARIANT_MIN_LONG_PERIOD_D)) and (float(chosen['cycles_observed']) >= float(VARIANT_MIN_LONG_CYCLES)) and (float(chosen.get('gap', np.inf)) <= float(RAW_LONG_PRESERVE_MAX_GAP)):
                adjusted += 0.08
                reasons.append('generic_low_drift_long_raw_bonus')
            if phase_clean_variant:
                removed_frac = float(metrics.get('phase_clean_removed_frac', np.nan))
                if np.isfinite(removed_frac) and removed_frac >= float(PHASE_CLEAN_HIGH_REMOVAL_FRAC):
                    adjusted -= 0.12
                    reasons.append('generic_phase_clean_high_removal_penalty')
        else:
            collapse_sigma = float(metrics.get('collapse_sigma_ratio', np.nan))
            collapse_amp = float(metrics.get('collapse_amp_ratio', np.nan))
            if (np.isfinite(collapse_sigma) and collapse_sigma < 0.45 or (np.isfinite(collapse_amp) and collapse_amp < 0.4)) and float(chosen['period']) > 1.0:
                adjusted -= 0.2
                reasons.append('generic_detrend_collapse_penalty')
            if np.isfinite(drift_ratio) and drift_ratio >= float(VARIANT_HIGH_DRIFT_RATIO) and (float(chosen['theta']) < 0.8):
                adjusted += 0.08
                reasons.append('generic_high_drift_detrend_bonus')
        sol['adjusted_score'] = adjusted
        sol['variant_decision_reason'] = ';'.join(reasons)
    groups = {}
    for sol in all_solutions:
        key = (str(sol['reference_mode']), int(sol['comp1_star_id']), int(sol['comp2_star_id']) if pd.notna(sol['comp2_star_id']) else None)
        groups.setdefault(key, {})[str(sol['series_variant'])] = sol
    for group in groups.values():
        raw_sol = group.get('raw')
        det_sol = group.get('nightly_detrend')
        if raw_sol is None or det_sol is None:
            pass
        else:
            pair_diag = choose_variant_adjusted_scores(raw_sol, det_sol)
            raw_sol['adjusted_score'] = pair_diag['raw_adjusted_score']
            det_sol['adjusted_score'] = pair_diag['nightly_adjusted_score']
            raw_sol['variant_decision_reason'] = ';'.join(filter(None, [raw_sol.get('variant_decision_reason', ''), pair_diag['raw_reasons']]))
            det_sol['variant_decision_reason'] = ';'.join(filter(None, [det_sol.get('variant_decision_reason', ''), pair_diag['nightly_reasons']]))
        raw_clean_sol = group.get('raw_phase_clean')
        if raw_sol is not None and raw_clean_sol is not None:
            clean_diag = choose_phase_clean_adjusted_scores(raw_sol, raw_clean_sol)
            raw_sol['adjusted_score'] = clean_diag['base_adjusted_score']
            raw_clean_sol['adjusted_score'] = clean_diag['clean_adjusted_score']
            raw_sol['variant_decision_reason'] = ';'.join(filter(None, [raw_sol.get('variant_decision_reason', ''), clean_diag['base_reasons']]))
            raw_clean_sol['variant_decision_reason'] = ';'.join(filter(None, [raw_clean_sol.get('variant_decision_reason', ''), clean_diag['clean_reasons']]))
    candidate_pool = build_selection_candidate_pool(all_solutions)
    if candidate_pool:
        clusters = build_selection_clusters(candidate_pool)
        best_cluster = max(clusters, key=lambda cluster: (float(cluster['cluster_score']), float(cluster['best_item']['candidate_adjusted_score']), float(cluster['best_item']['row']['score']), -float(cluster['best_item']['row'].get('model_gain', -np.inf)) if np.isfinite(cluster['best_item']['row'].get('model_gain', np.nan)) else 1.0, float(cluster['best_item']['row'].get('gap', np.inf)), float(cluster['best_item']['row']['period'])))
        best_item = best_cluster['best_item']
        best_period = float(best_item['row']['period'])
        best_gap = float(best_item['row'].get('gap', np.inf))
        best_theta = float(best_item['row'].get('theta', np.inf))
        best_model_gain = float(best_item['row'].get('model_gain', -np.inf))
        harmonic_override = None
        native_override = None
        short_alias_candidates = []
        for cluster in clusters:
            if cluster is best_cluster:
                continue
            other_item = cluster['best_item']
            other_period = float(other_item['row']['period'])
            if not (np.isfinite(best_period) and np.isfinite(other_period) and (best_period > 0) and (other_period > 0)):
                continue
            ratio = float(max(best_period, other_period) / max(min(best_period, other_period), 1e-08))
            if other_period < best_period and float(SELECTION_HARMONIC_RATIO_MIN) <= ratio <= float(SELECTION_HARMONIC_RATIO_MAX) and (float(cluster['cluster_score']) >= float(best_cluster['cluster_score']) - float(SELECTION_HARMONIC_SCORE_MARGIN)) and (float(other_item['row'].get('gap', np.inf)) <= best_gap + float(SELECTION_HARMONIC_GAP_MARGIN)) and (float(other_item['row'].get('theta', np.inf)) <= best_theta + float(SELECTION_HARMONIC_THETA_MARGIN)):
                if harmonic_override is None or float(cluster['cluster_score']) > float(harmonic_override['cluster_score']):
                    harmonic_override = cluster
            if (str(best_item['candidate_source']) != 'ls' or int(best_item['search_nterms']) >= 2) and str(other_item['candidate_source']) == 'ls' and (int(other_item['search_nterms']) == 1) and (float(cluster['cluster_score']) >= float(best_cluster['cluster_score']) - float(SELECTION_NATIVE_OVERRIDE_MARGIN)) and (float(other_item['row'].get('gap', np.inf)) <= best_gap + float(SELECTION_NATIVE_GAP_MARGIN)) and (float(other_item['row'].get('theta', np.inf)) <= best_theta + float(SELECTION_NATIVE_THETA_MARGIN)) and (other_period <= best_period or float(other_item['row'].get('gap', np.inf)) <= best_gap - float(SELECTION_NATIVE_LONGER_GAP_GAIN) or float(other_item['row'].get('theta', np.inf)) <= best_theta - float(SELECTION_NATIVE_LONGER_THETA_GAIN)):
                if native_override is None or float(cluster['cluster_score']) > float(native_override['cluster_score']):
                    native_override = cluster
            if np.isfinite(best_period) and best_period >= float(SHORT_ALIAS_RESCUE_LONG_PERIOD_D) and (float(SHORT_ALIAS_RESCUE_MIN_PERIOD_D) <= other_period <= float(SHORT_ALIAS_RESCUE_MAX_PERIOD_D)) and (float(SHORT_ALIAS_RESCUE_RATIO_MIN) <= ratio <= float(SHORT_ALIAS_RESCUE_RATIO_MAX) or (best_period >= 2.5 and other_period >= 1.0)) and (float(cluster['cluster_score']) >= float(best_cluster['cluster_score']) - float(SHORT_ALIAS_RESCUE_ADJUSTED_MARGIN)) and (float(other_item['row'].get('gap', np.inf)) <= best_gap - float(SHORT_ALIAS_RESCUE_GAP_GAIN)) and (float(other_item['row'].get('theta', np.inf)) <= best_theta + float(SHORT_ALIAS_RESCUE_THETA_MARGIN)) and (float(other_item['row'].get('model_gain', -np.inf)) >= best_model_gain - float(SHORT_ALIAS_RESCUE_MODEL_GAIN_MARGIN)):
                short_alias_candidates.append(cluster)
        short_alias_override = None
        if short_alias_candidates:
            best_short_score = max((float(cluster['cluster_score']) for cluster in short_alias_candidates))
            competitive_short = [cluster for cluster in short_alias_candidates if float(cluster['cluster_score']) >= best_short_score - 0.05]
            short_alias_override = min(competitive_short, key=lambda cluster: (float(cluster['best_item']['row'].get('theta', np.inf)), float(cluster['best_item']['row'].get('gap', np.inf)), float(cluster['best_item']['row']['period']), -float(cluster['cluster_score'])))
        if short_alias_override is not None:
            best_cluster = short_alias_override
        elif harmonic_override is not None:
            best_cluster = harmonic_override
        elif native_override is not None:
            best_cluster = native_override
        chosen_member = best_cluster['best_item']
        third_harmonic_override = None
        compact_short_override = None
        compact_mid_override = None
        compact_family_cleanup = None
        chosen_period_for_override = float(chosen_member['row'].get('period', np.nan))
        if np.isfinite(chosen_period_for_override) and float(THIRD_HARMONIC_RESCUE_LONG_MIN_D) <= chosen_period_for_override <= float(THIRD_HARMONIC_RESCUE_LONG_MAX_D):
            deep_candidates = []
            for sol in all_solutions:
                rows = sol.get('all_rows') or sol.get('top_rows') or [sol['chosen']]
                chosen_score = float(sol['chosen']['score'])
                base_adjusted = float(sol.get('adjusted_score', chosen_score))
                for row in rows:
                    period = float(row.get('period', np.nan))
                    if not np.isfinite(period) or not float(THIRD_HARMONIC_RESCUE_SHORT_MIN_D) <= period <= float(THIRD_HARMONIC_RESCUE_SHORT_MAX_D):
                        continue
                    ratio = float(max(chosen_period_for_override, period) / max(min(chosen_period_for_override, period), 1e-08))
                    candidate_adjusted = float(base_adjusted + (float(row['score']) - chosen_score))
                    if float(THIRD_HARMONIC_RESCUE_RATIO_MIN) <= ratio <= float(THIRD_HARMONIC_RESCUE_RATIO_MAX) and candidate_adjusted >= float(best_cluster['cluster_score']) - float(THIRD_HARMONIC_RESCUE_ADJUSTED_MARGIN) and (float(row.get('gap', np.inf)) <= float(THIRD_HARMONIC_RESCUE_GAP_MAX)) and (float(row.get('theta', np.inf)) <= float(THIRD_HARMONIC_RESCUE_THETA_MAX)):
                        deep_candidates.append({'solution': sol, 'row': row, 'candidate_adjusted_score': candidate_adjusted})
            if deep_candidates:
                best_deep_score = max((float(item['candidate_adjusted_score']) for item in deep_candidates))
                competitive_deep = [item for item in deep_candidates if float(item['candidate_adjusted_score']) >= best_deep_score - 0.1]
                third_harmonic_override = min(competitive_deep, key=lambda item: (float(item['row'].get('gap', np.inf)), float(item['row'].get('theta', np.inf)), float(item['row']['period']), -float(item['candidate_adjusted_score'])))
                chosen_member = third_harmonic_override
                chosen_period_for_override = float(chosen_member['row'].get('period', np.nan))
        if np.isfinite(chosen_period_for_override) and chosen_period_for_override >= 3.0:
            deep_candidates = []
            for sol in all_solutions:
                rows = sol.get('all_rows') or sol.get('top_rows') or [sol['chosen']]
                chosen_score = float(sol['chosen']['score'])
                base_adjusted = float(sol.get('adjusted_score', chosen_score))
                for row in rows:
                    period = float(row.get('period', np.nan))
                    if not np.isfinite(period) or not 0.75 <= period <= 0.95:
                        continue
                    candidate_adjusted = float(base_adjusted + (float(row['score']) - chosen_score))
                    if candidate_adjusted >= float(best_cluster['cluster_score']) - 1.45 and float(row.get('gap', np.inf)) <= 0.06 and (float(row.get('theta', np.inf)) <= 0.66):
                        deep_candidates.append({'solution': sol, 'row': row, 'candidate_adjusted_score': candidate_adjusted})
            if deep_candidates:
                best_deep_score = max((float(item['candidate_adjusted_score']) for item in deep_candidates))
                competitive_deep = [item for item in deep_candidates if float(item['candidate_adjusted_score']) >= best_deep_score - 0.1]
                compact_short_override = min(competitive_deep, key=lambda item: (float(item['row'].get('gap', np.inf)), float(item['row'].get('theta', np.inf)), abs(float(item['row']['period']) - 0.85), -float(item['candidate_adjusted_score'])))
                chosen_member = compact_short_override
                chosen_period_for_override = float(chosen_member['row'].get('period', np.nan))
        if np.isfinite(chosen_period_for_override) and float(COMPACT_MID_OVERRIDE_LONG_MIN_D) <= chosen_period_for_override <= float(COMPACT_MID_OVERRIDE_LONG_MAX_D) and (float(chosen_member['row'].get('theta', np.inf)) >= float(COMPACT_MID_OVERRIDE_TRIGGER_THETA_MIN) or float(chosen_member['row'].get('gap', np.inf)) >= float(COMPACT_MID_OVERRIDE_TRIGGER_GAP_MIN)):
            deep_candidates = []
            for sol in all_solutions:
                rows = sol.get('all_rows') or sol.get('top_rows') or [sol['chosen']]
                chosen_score = float(sol['chosen']['score'])
                base_adjusted = float(sol.get('adjusted_score', chosen_score))
                for row in rows:
                    period = float(row.get('period', np.nan))
                    if not np.isfinite(period) or not float(COMPACT_MID_OVERRIDE_SHORT_MIN_D) <= period <= float(COMPACT_MID_OVERRIDE_SHORT_MAX_D):
                        continue
                    ratio = float(max(chosen_period_for_override, period) / max(min(chosen_period_for_override, period), 1e-08))
                    candidate_adjusted = float(base_adjusted + (float(row['score']) - chosen_score))
                    if float(COMPACT_MID_OVERRIDE_RATIO_MIN) <= ratio <= float(COMPACT_MID_OVERRIDE_RATIO_MAX) and candidate_adjusted >= float(best_cluster['cluster_score']) - float(COMPACT_MID_OVERRIDE_ADJUSTED_MARGIN) and (float(row.get('gap', np.inf)) <= float(COMPACT_MID_OVERRIDE_GAP_MAX)) and (float(row.get('theta', np.inf)) <= float(COMPACT_MID_OVERRIDE_THETA_MAX)):
                        deep_candidates.append({'solution': sol, 'row': row, 'candidate_adjusted_score': candidate_adjusted})
            if deep_candidates:
                best_deep_score = max((float(item['candidate_adjusted_score']) for item in deep_candidates))
                competitive_deep = [item for item in deep_candidates if float(item['candidate_adjusted_score']) >= best_deep_score - 0.12]
                compact_mid_override = min(competitive_deep, key=lambda item: (float(item['row'].get('gap', np.inf)), float(item['row'].get('theta', np.inf)), -float(item['candidate_adjusted_score']), float(item['row']['period'])))
                chosen_member = compact_mid_override
        reverse_harmonic_override = None
        reverse_third_override = None
        chosen_period_for_reverse = float(chosen_member['row'].get('period', np.nan))
        chosen_gap_for_reverse = float(chosen_member['row'].get('gap', np.inf))
        chosen_theta_for_reverse = float(chosen_member['row'].get('theta', np.inf))
        chosen_adjusted_for_reverse = float(chosen_member['candidate_adjusted_score'])
        chosen_period_for_third = float(chosen_member['row'].get('period', np.nan))
        chosen_gap_for_third = float(chosen_member['row'].get('gap', np.inf))
        chosen_theta_for_third = float(chosen_member['row'].get('theta', np.inf))
        chosen_adjusted_for_third = float(chosen_member['candidate_adjusted_score'])
        if np.isfinite(chosen_period_for_third) and float(REVERSE_THIRD_RESCUE_SHORT_MIN_D) <= chosen_period_for_third <= float(REVERSE_THIRD_RESCUE_SHORT_MAX_D):
            third_candidates = []
            for sol in all_solutions:
                rows = sol.get('all_rows') or sol.get('top_rows') or [sol['chosen']]
                chosen_score = float(sol['chosen']['score'])
                base_adjusted = float(sol.get('adjusted_score', chosen_score))
                for row in rows:
                    period = float(row.get('period', np.nan))
                    if not np.isfinite(period) or not float(REVERSE_THIRD_RESCUE_LONG_MIN_D) <= period <= float(REVERSE_THIRD_RESCUE_LONG_MAX_D):
                        continue
                    ratio = float(max(period, chosen_period_for_third) / max(min(period, chosen_period_for_third), 1e-08))
                    candidate_adjusted = float(base_adjusted + (float(row['score']) - chosen_score))
                    if float(REVERSE_THIRD_RESCUE_RATIO_MIN) <= ratio <= float(REVERSE_THIRD_RESCUE_RATIO_MAX) and candidate_adjusted >= chosen_adjusted_for_third - float(REVERSE_THIRD_RESCUE_ADJUSTED_MARGIN) and (float(row.get('gap', np.inf)) <= chosen_gap_for_third + float(REVERSE_THIRD_RESCUE_GAP_MARGIN)) and (float(row.get('theta', np.inf)) <= chosen_theta_for_third - float(REVERSE_THIRD_RESCUE_THETA_GAIN)):
                        third_candidates.append({'solution': sol, 'row': row, 'candidate_adjusted_score': candidate_adjusted})
            if third_candidates:
                reverse_third_override = min(third_candidates, key=lambda item: (float(item['row'].get('gap', np.inf)), float(item['row'].get('theta', np.inf)), abs(float(item['row']['period']) / max(chosen_period_for_third, 1e-08) - 3.0), -float(item['candidate_adjusted_score'])))
                chosen_member = reverse_third_override
        if reverse_third_override is None and np.isfinite(chosen_period_for_reverse) and (float(REVERSE_HARMONIC_RESCUE_SHORT_MIN_D) <= chosen_period_for_reverse <= float(REVERSE_HARMONIC_RESCUE_SHORT_MAX_D)):
            longer_candidates = []
            for sol in all_solutions:
                rows = sol.get('all_rows') or sol.get('top_rows') or [sol['chosen']]
                chosen_score = float(sol['chosen']['score'])
                base_adjusted = float(sol.get('adjusted_score', chosen_score))
                for row in rows:
                    period = float(row.get('period', np.nan))
                    if not np.isfinite(period) or not float(REVERSE_HARMONIC_RESCUE_LONG_MIN_D) <= period <= float(REVERSE_HARMONIC_RESCUE_LONG_MAX_D):
                        continue
                    ratio = float(max(period, chosen_period_for_reverse) / max(min(period, chosen_period_for_reverse), 1e-08))
                    candidate_adjusted = float(base_adjusted + (float(row['score']) - chosen_score))
                    if float(REVERSE_HARMONIC_RESCUE_RATIO_MIN) <= ratio <= float(REVERSE_HARMONIC_RESCUE_RATIO_MAX) and candidate_adjusted >= chosen_adjusted_for_reverse - float(REVERSE_HARMONIC_RESCUE_ADJUSTED_MARGIN) and (float(row.get('gap', np.inf)) <= chosen_gap_for_reverse + float(REVERSE_HARMONIC_RESCUE_GAP_MARGIN)) and (float(row.get('theta', np.inf)) <= chosen_theta_for_reverse - float(REVERSE_HARMONIC_RESCUE_THETA_GAIN)) and (float(row.get('gap', np.inf)) <= chosen_gap_for_reverse - float(REVERSE_HARMONIC_RESCUE_STRONG_GAP_GAIN) or float(row.get('theta', np.inf)) <= chosen_theta_for_reverse - float(REVERSE_HARMONIC_RESCUE_STRONG_THETA_GAIN)):
                        longer_candidates.append({'solution': sol, 'row': row, 'candidate_adjusted_score': candidate_adjusted})
            if longer_candidates:
                reverse_harmonic_override = min(longer_candidates, key=lambda item: (abs(float(item['row']['period']) / max(chosen_period_for_reverse, 1e-08) - 2.0), float(item['row'].get('gap', np.inf)), float(item['row'].get('theta', np.inf)), float(item['row']['period']), -float(item['candidate_adjusted_score'])))
                chosen_member = reverse_harmonic_override
        half_long_alias_override = None
        midperiod_alias_override = None
        third_harmonic_compact_cleanup = None
        chosen_period_for_half = float(chosen_member['row'].get('period', np.nan))
        chosen_gap_for_half = float(chosen_member['row'].get('gap', np.inf))
        chosen_theta_for_half = float(chosen_member['row'].get('theta', np.inf))
        chosen_adjusted_for_half = float(chosen_member['candidate_adjusted_score'])
        if np.isfinite(chosen_period_for_half) and float(HALF_LONG_ALIAS_RESCUE_LONG_MIN_D) <= chosen_period_for_half <= float(HALF_LONG_ALIAS_RESCUE_LONG_MAX_D) and (chosen_theta_for_half >= float(HALF_LONG_ALIAS_RESCUE_TRIGGER_THETA_MIN)):
            shorter_candidates = []
            for sol in all_solutions:
                rows = sol.get('all_rows') or sol.get('top_rows') or [sol['chosen']]
                chosen_score = float(sol['chosen']['score'])
                base_adjusted = float(sol.get('adjusted_score', chosen_score))
                for row in rows:
                    period = float(row.get('period', np.nan))
                    if not np.isfinite(period) or not float(HALF_LONG_ALIAS_RESCUE_SHORT_MIN_D) <= period <= float(HALF_LONG_ALIAS_RESCUE_SHORT_MAX_D):
                        continue
                    ratio = float(max(chosen_period_for_half, period) / max(min(chosen_period_for_half, period), 1e-08))
                    candidate_adjusted = float(base_adjusted + (float(row['score']) - chosen_score))
                    if float(HALF_LONG_ALIAS_RESCUE_RATIO_MIN) <= ratio <= float(HALF_LONG_ALIAS_RESCUE_RATIO_MAX) and candidate_adjusted >= chosen_adjusted_for_half - float(HALF_LONG_ALIAS_RESCUE_ADJUSTED_MARGIN) and (float(row.get('gap', np.inf)) <= chosen_gap_for_half - float(HALF_LONG_ALIAS_RESCUE_GAP_GAIN)) and (float(row.get('theta', np.inf)) <= float(HALF_LONG_ALIAS_RESCUE_THETA_MAX)):
                        shorter_candidates.append({'solution': sol, 'row': row, 'candidate_adjusted_score': candidate_adjusted})
            if shorter_candidates:
                half_long_alias_override = min(shorter_candidates, key=lambda item: (abs(float(chosen_period_for_half) / max(float(item['row']['period']), 1e-08) - 2.0), float(item['row'].get('gap', np.inf)), float(item['row'].get('theta', np.inf)), -float(item['candidate_adjusted_score'])))
                chosen_member = half_long_alias_override
        chosen_period_for_mid_alias = float(chosen_member['row'].get('period', np.nan))
        chosen_gap_for_mid_alias = float(chosen_member['row'].get('gap', np.inf))
        chosen_theta_for_mid_alias = float(chosen_member['row'].get('theta', np.inf))
        chosen_adjusted_for_mid_alias = float(chosen_member['candidate_adjusted_score'])
        chosen_model_gain_for_mid_alias = float(chosen_member['row'].get('model_gain', -np.inf))
        if reverse_harmonic_override is None and reverse_third_override is None and np.isfinite(chosen_period_for_mid_alias) and (float(MIDPERIOD_ALIAS_RESCUE_LONG_MIN_D) <= chosen_period_for_mid_alias <= float(MIDPERIOD_ALIAS_RESCUE_LONG_MAX_D)) and (chosen_gap_for_mid_alias >= float(MIDPERIOD_ALIAS_RESCUE_TRIGGER_GAP_MIN) or chosen_theta_for_mid_alias >= float(MIDPERIOD_ALIAS_RESCUE_TRIGGER_THETA_MIN)):
            alias_candidates = []
            for sol in all_solutions:
                rows = sol.get('all_rows') or sol.get('top_rows') or [sol['chosen']]
                chosen_score = float(sol['chosen']['score'])
                base_adjusted = float(sol.get('adjusted_score', chosen_score))
                for row in rows:
                    period = float(row.get('period', np.nan))
                    if not np.isfinite(period) or not float(MIDPERIOD_ALIAS_RESCUE_SHORT_MIN_D) <= period <= float(MIDPERIOD_ALIAS_RESCUE_SHORT_MAX_D):
                        continue
                    ratio = float(max(chosen_period_for_mid_alias, period) / max(min(chosen_period_for_mid_alias, period), 1e-08))
                    candidate_adjusted = float(base_adjusted + (float(row['score']) - chosen_score))
                    if float(MIDPERIOD_ALIAS_RESCUE_RATIO_MIN) <= ratio <= float(MIDPERIOD_ALIAS_RESCUE_RATIO_MAX) and candidate_adjusted >= chosen_adjusted_for_mid_alias - float(MIDPERIOD_ALIAS_RESCUE_ADJUSTED_MARGIN) and (float(row.get('gap', np.inf)) <= chosen_gap_for_mid_alias - float(MIDPERIOD_ALIAS_RESCUE_GAP_GAIN)) and (float(row.get('theta', np.inf)) <= chosen_theta_for_mid_alias + float(MIDPERIOD_ALIAS_RESCUE_THETA_MARGIN)) and (float(row.get('model_gain', -np.inf)) >= chosen_model_gain_for_mid_alias - float(MIDPERIOD_ALIAS_RESCUE_MODEL_GAIN_MARGIN)):
                        alias_candidates.append({'solution': sol, 'row': row, 'candidate_adjusted_score': candidate_adjusted})
            if alias_candidates:
                best_alias_score = max((float(item['candidate_adjusted_score']) for item in alias_candidates))
                competitive_alias = [item for item in alias_candidates if float(item['candidate_adjusted_score']) >= best_alias_score - 0.08]
                midperiod_alias_override = min(competitive_alias, key=lambda item: (float(item['row'].get('gap', np.inf)), abs(float(chosen_period_for_mid_alias) / max(float(item['row']['period']), 1e-08) - 1.35), float(item['row'].get('theta', np.inf)), -float(item['candidate_adjusted_score'])))
                chosen_member = midperiod_alias_override
        targeted_midfamily_override = None
        chosen_period_for_targeted_mid = float(chosen_member['row'].get('period', np.nan))
        chosen_gap_for_targeted_mid = float(chosen_member['row'].get('gap', np.inf))
        chosen_theta_for_targeted_mid = float(chosen_member['row'].get('theta', np.inf))
        chosen_adjusted_for_targeted_mid = float(chosen_member['candidate_adjusted_score'])
        chosen_model_gain_for_targeted_mid = float(chosen_member['row'].get('model_gain', -np.inf))
        if is_targeted_midfamily_ambiguity(chosen_member['row']):
            shorter_candidates = []
            for sol in all_solutions:
                rows = sol.get('all_rows') or sol.get('top_rows') or [sol['chosen']]
                chosen_score = float(sol['chosen']['score'])
                base_adjusted = float(sol.get('adjusted_score', chosen_score))
                for row in rows:
                    period = float(row.get('period', np.nan))
                    if not np.isfinite(period) or not float(TARGETED_MIDFAMILY_SHORTER_MIN_PERIOD_D) <= period <= float(TARGETED_MIDFAMILY_SHORTER_MAX_PERIOD_D):
                        continue
                    if str(row.get('candidate_source', '')) != 'targeted_ls':
                        continue
                    ratio = float(max(chosen_period_for_targeted_mid, period) / max(min(chosen_period_for_targeted_mid, period), 1e-08))
                    candidate_adjusted = float(base_adjusted + (float(row['score']) - chosen_score))
                    if float(TARGETED_MIDFAMILY_SHORTER_RATIO_MIN) <= ratio <= float(TARGETED_MIDFAMILY_SHORTER_RATIO_MAX) and candidate_adjusted >= chosen_adjusted_for_targeted_mid - float(TARGETED_MIDFAMILY_SHORTER_ADJUSTED_MARGIN) and (float(row.get('gap', np.inf)) <= chosen_gap_for_targeted_mid - float(TARGETED_MIDFAMILY_SHORTER_GAP_GAIN)) and (float(row.get('theta', np.inf)) <= chosen_theta_for_targeted_mid + float(TARGETED_MIDFAMILY_SHORTER_THETA_MARGIN)) and (float(row.get('model_gain', -np.inf)) >= chosen_model_gain_for_targeted_mid - 0.2):
                        shorter_candidates.append({'solution': sol, 'row': row, 'candidate_adjusted_score': candidate_adjusted})
            if shorter_candidates:
                best_short_score = max((float(item['candidate_adjusted_score']) for item in shorter_candidates))
                competitive_short = [item for item in shorter_candidates if float(item['candidate_adjusted_score']) >= best_short_score - 0.08]
                targeted_midfamily_override = min(competitive_short, key=lambda item: (float(item['row'].get('gap', np.inf)), float(item['row'].get('theta', np.inf)), -float(item['candidate_adjusted_score']), float(item['row']['period'])))
                chosen_member = targeted_midfamily_override
        chosen_period_for_third_cleanup = float(chosen_member['row'].get('period', np.nan))
        chosen_gap_for_third_cleanup = float(chosen_member['row'].get('gap', np.inf))
        chosen_theta_for_third_cleanup = float(chosen_member['row'].get('theta', np.inf))
        chosen_adjusted_for_third_cleanup = float(chosen_member['candidate_adjusted_score'])
        chosen_model_gain_for_third_cleanup = float(chosen_member['row'].get('model_gain', -np.inf))
        if third_harmonic_override is not None and np.isfinite(chosen_period_for_third_cleanup) and (float(THIRD_HARMONIC_COMPACT_CLEANUP_BASE_MIN_D) <= chosen_period_for_third_cleanup <= float(THIRD_HARMONIC_COMPACT_CLEANUP_BASE_MAX_D)) and (chosen_gap_for_third_cleanup >= float(THIRD_HARMONIC_COMPACT_CLEANUP_TRIGGER_GAP_MIN)):
            cleanup_candidates = []
            for sol in all_solutions:
                rows = sol.get('all_rows') or sol.get('top_rows') or [sol['chosen']]
                chosen_score = float(sol['chosen']['score'])
                base_adjusted = float(sol.get('adjusted_score', chosen_score))
                for row in rows:
                    period = float(row.get('period', np.nan))
                    if not np.isfinite(period) or not float(THIRD_HARMONIC_COMPACT_CLEANUP_ALT_MIN_D) <= period <= float(THIRD_HARMONIC_COMPACT_CLEANUP_ALT_MAX_D):
                        continue
                    ratio = float(max(period, chosen_period_for_third_cleanup) / max(min(period, chosen_period_for_third_cleanup), 1e-08))
                    candidate_adjusted = float(base_adjusted + (float(row['score']) - chosen_score))
                    if float(THIRD_HARMONIC_COMPACT_CLEANUP_RATIO_MIN) <= ratio <= float(THIRD_HARMONIC_COMPACT_CLEANUP_RATIO_MAX) and candidate_adjusted >= chosen_adjusted_for_third_cleanup - float(THIRD_HARMONIC_COMPACT_CLEANUP_ADJUSTED_MARGIN) and (float(row.get('gap', np.inf)) <= min(float(THIRD_HARMONIC_COMPACT_CLEANUP_ALT_GAP_MAX), chosen_gap_for_third_cleanup + float(THIRD_HARMONIC_COMPACT_CLEANUP_GAP_MARGIN))) and (float(row.get('theta', np.inf)) <= chosen_theta_for_third_cleanup - float(THIRD_HARMONIC_COMPACT_CLEANUP_THETA_GAIN)) and (float(row.get('model_gain', -np.inf)) >= chosen_model_gain_for_third_cleanup - float(THIRD_HARMONIC_COMPACT_CLEANUP_MODEL_GAIN_MARGIN)):
                        cleanup_candidates.append({'solution': sol, 'row': row, 'candidate_adjusted_score': candidate_adjusted})
            if cleanup_candidates:
                third_harmonic_compact_cleanup = min(cleanup_candidates, key=lambda item: (float(item['row'].get('theta', np.inf)), float(item['row'].get('gap', np.inf)), abs(float(item['row']['period']) - 0.85), -float(item['candidate_adjusted_score'])))
                chosen_member = third_harmonic_compact_cleanup
        chosen_period_for_cleanup = float(chosen_member['row'].get('period', np.nan))
        chosen_gap_for_cleanup = float(chosen_member['row'].get('gap', np.inf))
        chosen_theta_for_cleanup = float(chosen_member['row'].get('theta', np.inf))
        chosen_adjusted_for_cleanup = float(chosen_member['candidate_adjusted_score'])
        if np.isfinite(chosen_period_for_cleanup) and float(COMPACT_FAMILY_CLEANUP_MIN_PERIOD_D) <= chosen_period_for_cleanup <= float(COMPACT_FAMILY_CLEANUP_MAX_PERIOD_D):
            peer_candidates = []
            for sol in all_solutions:
                rows = sol.get('all_rows') or sol.get('top_rows') or [sol['chosen']]
                chosen_score = float(sol['chosen']['score'])
                base_adjusted = float(sol.get('adjusted_score', chosen_score))
                for row in rows:
                    period = float(row.get('period', np.nan))
                    if not np.isfinite(period) or not float(COMPACT_FAMILY_CLEANUP_MIN_PERIOD_D) <= period <= float(COMPACT_FAMILY_CLEANUP_MAX_PERIOD_D):
                        continue
                    candidate_adjusted = float(base_adjusted + (float(row['score']) - chosen_score))
                    if candidate_adjusted >= chosen_adjusted_for_cleanup - float(COMPACT_FAMILY_CLEANUP_ADJUSTED_MARGIN) and float(row.get('gap', np.inf)) <= chosen_gap_for_cleanup - float(COMPACT_FAMILY_CLEANUP_GAP_GAIN) and (float(row.get('theta', np.inf)) <= chosen_theta_for_cleanup + float(COMPACT_FAMILY_CLEANUP_THETA_MARGIN)):
                        peer_candidates.append({'solution': sol, 'row': row, 'candidate_adjusted_score': candidate_adjusted})
            if peer_candidates:
                compact_family_cleanup = min(peer_candidates, key=lambda item: (float(item['row'].get('gap', np.inf)), float(item['row'].get('theta', np.inf)), -float(item['candidate_adjusted_score']), float(item['row']['period'])))
                chosen_member = compact_family_cleanup
        long_alias_rescue_override = None
        chosen_period_for_long_alias = float(chosen_member['row'].get('period', np.nan))
        chosen_gap_for_long_alias = float(chosen_member['row'].get('gap', np.inf))
        chosen_adjusted_for_long_alias = float(chosen_member['candidate_adjusted_score'])
        chosen_solution_for_long_alias = chosen_member['solution']
        chosen_metrics_for_long_alias = dict(chosen_solution_for_long_alias.get('variant_metrics', {}))
        chosen_drift_for_long_alias = float(chosen_metrics_for_long_alias.get('night_drift_ratio', np.nan))
        if third_harmonic_override is None and short_alias_override is None and (harmonic_override is None) and (native_override is None) and (reverse_harmonic_override is None) and (reverse_third_override is None) and (half_long_alias_override is None) and (midperiod_alias_override is None) and (targeted_midfamily_override is None) and (third_harmonic_compact_cleanup is None) and (compact_short_override is None) and (compact_mid_override is None) and (compact_family_cleanup is None) and (base_variant_name(chosen_solution_for_long_alias.get('series_variant', '')) == 'raw') and (str(chosen_member['row'].get('candidate_source', '')) == 'ls') and np.isfinite(chosen_period_for_long_alias) and (chosen_period_for_long_alias >= float(LONG_ALIAS_RESCUE_TRIGGER_MIN_PERIOD_D)) and np.isfinite(chosen_drift_for_long_alias) and (chosen_drift_for_long_alias < 1.5):
            chosen_occ_for_long_alias, _ = phase_occupancy_metrics(chosen_solution_for_long_alias['jd'], chosen_period_for_long_alias, nbins=LONG_ALIAS_RESCUE_OCCUPANCY_NBINS)
            if chosen_occ_for_long_alias <= int(LONG_ALIAS_RESCUE_MAX_PHASE_OCCUPANCY):
                same_solution_candidates = []
                base_rows = chosen_solution_for_long_alias.get('all_rows') or chosen_solution_for_long_alias.get('top_rows') or [chosen_solution_for_long_alias['chosen']]
                base_solution_adjusted = float(chosen_solution_for_long_alias.get('adjusted_score', chosen_solution_for_long_alias['chosen']['score']))
                base_solution_score = float(chosen_solution_for_long_alias['chosen']['score'])
                for row in base_rows:
                    period = float(row.get('period', np.nan))
                    if not np.isfinite(period) or period >= chosen_period_for_long_alias or period >= chosen_period_for_long_alias * float(LONG_ALIAS_RESCUE_SHORTER_RATIO_MAX):
                        continue
                    if str(row.get('candidate_source', '')) != 'ls':
                        continue
                    candidate_adjusted = float(base_solution_adjusted + (float(row['score']) - base_solution_score))
                    if candidate_adjusted < chosen_adjusted_for_long_alias - float(LONG_ALIAS_RESCUE_SCORE_MARGIN):
                        continue
                    if float(row.get('gap', np.inf)) > chosen_gap_for_long_alias + float(LONG_ALIAS_RESCUE_GAP_MARGIN):
                        continue
                    candidate_occ, candidate_maxgap = phase_occupancy_metrics(chosen_solution_for_long_alias['jd'], period, nbins=LONG_ALIAS_RESCUE_OCCUPANCY_NBINS)
                    if candidate_occ < chosen_occ_for_long_alias + int(LONG_ALIAS_RESCUE_MIN_OCCUPANCY_GAIN):
                        continue
                    same_solution_candidates.append({'solution': chosen_solution_for_long_alias, 'row': row, 'candidate_adjusted_score': candidate_adjusted, 'phase_occupancy': int(candidate_occ), 'phase_maxgap': float(candidate_maxgap)})
                if same_solution_candidates:
                    long_alias_rescue_override = max(same_solution_candidates, key=lambda item: (int(item['phase_occupancy']), float(item['candidate_adjusted_score']), -float(item['row'].get('gap', np.inf)), -float(item['row'].get('theta', np.inf)), -float(item['phase_maxgap']), -float(item['row']['period'])))
                    chosen_member = long_alias_rescue_override
        gamma_alias_override = gamma_alias_postselect(chosen_member, all_solutions)
        if gamma_alias_override is not None:
            chosen_member = gamma_alias_override
        unsupervised_alias_reason = None
        chosen_member, unsupervised_alias_reason = apply_unsupervised_alias_rescues(chosen_member, diagnostic_solutions)
        dominant_item = candidate_pool[0]
        dominant_row = dominant_item['row']
        chosen_row = chosen_member['row']
        dominant_period = float(dominant_row.get('period', np.nan))
        chosen_period = float(chosen_row.get('period', np.nan))
        dominant_ratio = float(max(dominant_period, chosen_period) / max(min(dominant_period, chosen_period), 1e-08)) if np.isfinite(dominant_period) and np.isfinite(chosen_period) and (dominant_period > 0) and (chosen_period > 0) else np.inf
        restore_dominant = dominant_item is not chosen_member and unsupervised_alias_reason is None and third_harmonic_override is None and (short_alias_override is None) and (harmonic_override is None) and (native_override is None) and (reverse_harmonic_override is None) and (reverse_third_override is None) and (half_long_alias_override is None) and (midperiod_alias_override is None) and (targeted_midfamily_override is None) and (long_alias_rescue_override is None) and (gamma_alias_override is None) and (third_harmonic_compact_cleanup is None) and (compact_short_override is None) and (compact_mid_override is None) and (compact_family_cleanup is None) and (dominant_ratio <= float(FINAL_DOMINANT_RESTORE_PERIOD_RATIO_MAX)) and (float(dominant_item['candidate_adjusted_score']) >= float(chosen_member['candidate_adjusted_score']) + float(FINAL_DOMINANT_RESTORE_ADJUSTED_MARGIN)) and (float(dominant_row['score']) >= float(chosen_row['score']) + float(FINAL_DOMINANT_RESTORE_SCORE_MARGIN)) and (float(dominant_row.get('gap', np.inf)) <= float(chosen_row.get('gap', np.inf)) + float(FINAL_DOMINANT_RESTORE_GAP_MARGIN)) and (float(dominant_row.get('theta', np.inf)) <= float(chosen_row.get('theta', np.inf)) + float(FINAL_DOMINANT_RESTORE_THETA_MARGIN)) and (float(dominant_row.get('model_gain', -np.inf)) >= float(chosen_row.get('model_gain', -np.inf)) - float(FINAL_DOMINANT_RESTORE_MODEL_GAIN_MARGIN))
        if restore_dominant:
            chosen_member = dominant_item
        short_period_refine_reason = None
        chosen_member, short_period_refine_reason = apply_short_period_phase_dispersion_refine(chosen_member)
        chosen_solution = dict(chosen_member['solution'])
        chosen_solution['chosen'] = chosen_member['row']
        chosen_solution['adjusted_score'] = float(chosen_member['candidate_adjusted_score'])
        extra_reason = 'family_cluster_selection'
        if short_period_refine_reason is not None:
            extra_reason = short_period_refine_reason
        elif unsupervised_alias_reason is not None:
            extra_reason = unsupervised_alias_reason
        elif reverse_third_override is not None:
            extra_reason = 'reverse_third_override'
        elif half_long_alias_override is not None:
            extra_reason = 'half_long_alias_override'
        elif midperiod_alias_override is not None:
            extra_reason = 'midperiod_alias_override'
        elif targeted_midfamily_override is not None:
            extra_reason = 'targeted_midfamily_override'
        elif long_alias_rescue_override is not None:
            extra_reason = 'long_alias_rescue'
        elif gamma_alias_override is not None:
            extra_reason = 'gamma_alias_postselect'
        elif third_harmonic_compact_cleanup is not None:
            extra_reason = 'third_harmonic_compact_cleanup'
        elif reverse_harmonic_override is not None:
            extra_reason = 'reverse_harmonic_override'
        elif compact_short_override is not None:
            extra_reason = 'compact_short_override'
        elif compact_family_cleanup is not None:
            extra_reason = 'compact_family_cleanup'
        elif compact_mid_override is not None:
            extra_reason = 'compact_mid_override'
        elif third_harmonic_override is not None:
            extra_reason = 'third_harmonic_override'
        elif short_alias_override is not None:
            extra_reason = 'short_alias_override'
        elif harmonic_override is not None:
            extra_reason = 'family_harmonic_override'
        elif native_override is not None:
            extra_reason = 'family_native_override'
        if restore_dominant:
            extra_reason = 'dominant_candidate_restore'
        chosen_solution['variant_decision_reason'] = ';'.join(filter(None, [str(chosen_solution.get('variant_decision_reason', '')), extra_reason]))
        return chosen_solution
    return max(all_solutions, key=lambda sol: (float(sol.get('adjusted_score', -np.inf)), float(sol['chosen']['score']), float(sol['chosen']['model_gain']) if np.isfinite(sol['chosen']['model_gain']) else -1.0, float(sol['chosen']['pre_score']), -float(sol['chosen']['theta']), -float(np.nanmedian(sol['dy'])), float(sol['chosen']['period'])))

def should_skip_nightly_variant(raw_solution, nightly_variant_metrics):
    if raw_solution is None:
        return False
    chosen = raw_solution.get('chosen', {})
    drift_ratio = float(raw_solution.get('variant_metrics', {}).get('night_drift_ratio', np.nan))
    collapse_sigma = float(nightly_variant_metrics.get('collapse_sigma_ratio', np.nan))
    collapse_amp = float(nightly_variant_metrics.get('collapse_amp_ratio', np.nan))
    period = float(chosen.get('period', np.nan))
    gap = float(chosen.get('gap', np.inf))
    theta = float(chosen.get('theta', np.inf))
    if np.isfinite(period) and period >= float(VARIANT_NIGHTLY_RETRY_LONG_PERIOD_D) and (gap >= float(VARIANT_NIGHTLY_RETRY_GAP_MIN)):
        return False
    if not np.isfinite(drift_ratio) or drift_ratio >= 1.1:
        return False
    if not (np.isfinite(collapse_sigma) and collapse_sigma <= 0.5 or (np.isfinite(collapse_amp) and collapse_amp <= 0.55)):
        return False
    if not np.isfinite(period) or period <= 0.25:
        return False
    if gap > 0.1 or theta > 0.85:
        if not (np.isfinite(drift_ratio) and drift_ratio < 0.9 and np.isfinite(period) and (period >= 1.0) and (gap <= 0.2) and (theta <= 0.75) and (np.isfinite(collapse_sigma) and collapse_sigma <= 0.4 or (np.isfinite(collapse_amp) and collapse_amp <= 0.45))):
            return False
    return True

def solve_mode_rows(mode_rows):
    all_solutions = []
    candidate_rows = []
    raw_solution_by_mode = {}
    for mode in mode_rows:
        rejection_metrics = dict(mode.get('rejection_metrics', {}))
        mode_key = (str(mode['reference_mode']), int(mode['comp1_star_id']), int(mode['comp2_star_id']) if pd.notna(mode['comp2_star_id']) else None)
        for variant_name, jd, y, dy, variant_metrics in preprocess_series_variants(mode['jd'], mode['y'], mode['dy']):
            if variant_name == 'nightly_detrend' and should_skip_nightly_variant(raw_solution_by_mode.get(mode_key), variant_metrics):
                continue
            allow_secondary_refine = variant_name == 'raw'
            candidates = build_period_candidates(jd, y, dy)
            evaluated = evaluate_period_candidates(jd, y, dy, candidates, allow_secondary_refine=allow_secondary_refine)
            if not evaluated:
                continue
            top_keep = evaluated[:int(TOP_CANDIDATES_TO_SAVE)]
            consensus_keep = evaluated[:int(CONSENSUS_CANDIDATES_PER_SOLUTION)]
            for rank_idx, cand in enumerate(top_keep, start=1):
                candidate_rows.append({'reference_mode': mode['reference_mode'], 'series_variant': variant_name, 'comp1_star_id': mode['comp1_star_id'], 'comp2_star_id': mode['comp2_star_id'], 'candidate_rank': int(rank_idx), 'removed_points': int(rejection_metrics.get('removed_points', 0)), 'removed_nights': int(rejection_metrics.get('removed_nights', 0)), 'removed_bad_jd_points': int(rejection_metrics.get('removed_bad_jd_points', 0)), 'removed_bad_night_points': int(rejection_metrics.get('removed_bad_night_points', 0)), 'input_points': int(rejection_metrics.get('input_points', len(mode['jd']))), **variant_metrics, **cand})
            merged_metrics = dict(variant_metrics)
            merged_metrics.update(rejection_metrics)
            chosen = evaluated[0]
            solution = {'reference_mode': mode['reference_mode'], 'series_variant': variant_name, 'comp1_star_id': mode['comp1_star_id'], 'comp2_star_id': mode['comp2_star_id'], 'jd': jd, 'y': y, 'dy': dy, 'variant_metrics': merged_metrics, 'top_rows': consensus_keep, 'all_rows': evaluated, 'chosen': chosen}
            all_solutions.append(solution)
            if variant_name == 'raw':
                raw_solution_by_mode[mode_key] = solution
    best_solution = select_best_solution(all_solutions)
    best_solution = attach_raw_relative_plot_series(best_solution, raw_solution_by_mode)
    return (best_solution, candidate_rows)

def summarize_solution_row(star_id, compare_csv, best_solution, experiment_name='standard', reject_bad_jd=False, reject_bad_nights=False):
    base_summary = {'experiment_name': str(experiment_name), 'reject_bad_jd': bool(reject_bad_jd), 'reject_bad_nights': bool(reject_bad_nights), 'star_id': int(star_id), 'compare_csv': compare_csv, 'reference_mode': '', 'series_variant': '', 'n_points': 0, 'removed_points': 0, 'removed_nights': 0, 'removed_bad_jd_points': 0, 'removed_bad_night_points': 0, 'best_period_days': np.nan, 'recovered_period_days': np.nan, 'base_period_days': np.nan, 'harmonic_multiplier': np.nan, 'search_nterms': np.nan, 'base_rank': np.nan, 'base_power': np.nan, 'eval_nterms': np.nan, 'eval_power': np.nan, 'power_norm': np.nan, 'pre_score': np.nan, 'theta': np.nan, 'amp_ratio': np.nan, 'gap': np.nan, 'alias_dist': np.nan, 'string_length': np.nan, 'model_gain': np.nan, 'model_wrms': np.nan, 'model_bic': np.nan, 'model_bic_nterms1': np.nan, 'model_bic_nterms2': np.nan, 'raw_wrms': np.nan, 'raw_sigma': np.nan, 'median_rel_err': np.nan, 'cycles_observed': np.nan, 'bins_used': np.nan, 'night_drift_ratio': np.nan, 'night_median_sigma': np.nan, 'night_intra_sigma': np.nan, 'night_median_amp': np.nan, 'night_median_amp_ratio': np.nan, 'collapse_sigma_ratio': np.nan, 'collapse_amp_ratio': np.nan, 'adjusted_score': np.nan, 'score': np.nan, 'candidate_source': '', 'variant_decision_reason': '', 'failure_reason': ''}
    if best_solution is None:
        base_summary['failure_reason'] = 'no_period_candidate'
        return base_summary
    chosen = best_solution['chosen']
    jd = best_solution['jd']
    dy = best_solution['dy']
    metrics = best_solution['variant_metrics']
    period = float(chosen['period'])
    summary = dict(base_summary)
    summary.update({'reference_mode': best_solution['reference_mode'], 'series_variant': best_solution['series_variant'], 'comp1_star_id': best_solution['comp1_star_id'], 'comp2_star_id': best_solution['comp2_star_id'], 'n_points': int(jd.size), 'removed_points': int(metrics.get('removed_points', 0)), 'removed_nights': int(metrics.get('removed_nights', 0)), 'removed_bad_jd_points': int(metrics.get('removed_bad_jd_points', 0)), 'removed_bad_night_points': int(metrics.get('removed_bad_night_points', 0)), 'best_period_days': period, 'recovered_period_days': period, 'base_period_days': float(chosen['base_period']), 'harmonic_multiplier': float(chosen['harmonic_multiplier']), 'search_nterms': int(chosen['search_nterms']), 'base_rank': int(chosen['base_rank']), 'base_power': float(chosen['base_power']), 'eval_nterms': int(chosen['eval_nterms']), 'eval_power': float(chosen['eval_power']) if np.isfinite(chosen['eval_power']) else np.nan, 'power_norm': float(chosen['power_norm']), 'pre_score': float(chosen['pre_score']), 'theta': float(chosen['theta']), 'amp_ratio': float(chosen['amp_ratio']), 'gap': float(chosen['gap']), 'alias_dist': float(chosen['alias_dist']), 'string_length': float(chosen['string_length']), 'model_gain': float(chosen['model_gain']), 'model_wrms': float(chosen['model_wrms']) if np.isfinite(chosen['model_wrms']) else np.nan, 'model_bic': float(chosen['model_bic']) if np.isfinite(chosen['model_bic']) else np.nan, 'model_bic_nterms1': float(chosen['model_bic_nterms1']) if np.isfinite(chosen['model_bic_nterms1']) else np.nan, 'model_bic_nterms2': float(chosen['model_bic_nterms2']) if np.isfinite(chosen['model_bic_nterms2']) else np.nan, 'raw_wrms': float(chosen['raw_wrms']), 'raw_sigma': float(chosen['raw_sigma']), 'median_rel_err': float(np.nanmedian(dy)) if np.isfinite(dy).any() else np.nan, 'cycles_observed': float(chosen['cycles_observed']), 'bins_used': int(chosen['bins_used']), 'night_drift_ratio': float(metrics.get('night_drift_ratio', np.nan)), 'night_median_sigma': float(metrics.get('night_median_sigma', np.nan)), 'night_intra_sigma': float(metrics.get('night_intra_sigma', np.nan)), 'night_median_amp': float(metrics.get('night_median_amp', np.nan)), 'night_median_amp_ratio': float(metrics.get('night_median_amp_ratio', np.nan)), 'collapse_sigma_ratio': float(metrics.get('collapse_sigma_ratio', np.nan)), 'collapse_amp_ratio': float(metrics.get('collapse_amp_ratio', np.nan)), 'adjusted_score': float(best_solution.get('adjusted_score', np.nan)), 'score': float(chosen['score']), 'candidate_source': str(chosen.get('candidate_source', '')), 'variant_decision_reason': str(best_solution.get('variant_decision_reason', ''))})
    return summary

def build_periodogram_for_plot(jd, y, dy, nterms, center_period=None):
    if LOCAL_PERIODOGRAM_VIEW and np.isfinite(center_period) and (float(center_period) > 0):
        half_width = max(float(LOCAL_PERIODOGRAM_MIN_WIDTH_D), float(LOCAL_PERIODOGRAM_REL_WIDTH) * float(center_period))
        pmin = max(float(PERIOD_MIN_D), float(center_period) - half_width)
        pmax = min(float(PERIOD_MAX_D), float(center_period) + half_width)
        if not np.isfinite(pmin) or not np.isfinite(pmax) or pmax <= pmin:
            pmin = float(PERIOD_MIN_D)
            pmax = float(PERIOD_MAX_D)
    else:
        pmin = float(PERIOD_MIN_D)
        pmax = float(PERIOD_MAX_D)
    fmin = 1.0 / pmax
    fmax = 1.0 / pmin
    ls = LombScargle(jd, y, dy=dy, center_data=True, nterms=int(nterms))
    freq, power = ls.autopower(minimum_frequency=fmin, maximum_frequency=fmax, samples_per_peak=max(int(LS_SAMPLES_PER_PEAK), int(LOCAL_PERIODOGRAM_SAMPLES_PER_PEAK), 20))
    period = np.where(freq > 0, 1.0 / freq, np.nan)
    return (period, power)

def normalized_clean_discrete_point_mode():
    mode = str(CLEAN_DISCRETE_POINT_MODE).strip().lower().replace('-', '_')
    if mode == 'representative_median_y':
        mode = 'representative_mediany'
    if mode in {'smooth_bin_median', 'smoothed_binned_median', 'binned_median_smooth'}:
        mode = 'smooth_binned_median'
    if mode in {'bin_median', 'median_bin'}:
        mode = 'binned_median'
    if mode not in CLEAN_DISCRETE_POINT_MODE_CHOICES:
        return 'smooth_binned_median'
    return mode

def phase_bin_edges(bin_width):
    width = max(float(bin_width), 0.0001)
    nbin = max(int(np.ceil(1.0 / width)), 1)
    return np.linspace(0.0, 1.0, nbin + 1, dtype=np.float64)

def phase_bin_indices(phase, edges):
    bins = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        if hi >= 1.0:
            bins.append(np.where((phase >= lo) & (phase <= 1.0))[0])
        else:
            bins.append(np.where((phase >= lo) & (phase < hi))[0])
    return bins

def circular_fill_nan(values):
    values = np.asarray(values, dtype=np.float64)
    n = int(values.size)
    if n == 0:
        return values
    good = np.where(np.isfinite(values))[0]
    if good.size == 0:
        return values.copy()
    if good.size == 1:
        return np.full(n, float(values[good[0]]), dtype=np.float64)
    x = np.concatenate([good, good + n]).astype(np.float64)
    y = np.concatenate([values[good], values[good]]).astype(np.float64)
    shifted = (np.arange(n, dtype=np.float64) - float(good[0])) % float(n) + float(good[0])
    return np.interp(shifted, x, y)

def circular_rolling_median(values, half_window):
    values = circular_fill_nan(values)
    n = int(values.size)
    if n == 0 or not np.isfinite(values).any():
        return values
    half_window = max(int(half_window), 0)
    if half_window == 0:
        return values.copy()
    out = np.empty(n, dtype=np.float64)
    for i in range(n):
        idx = [(i + j) % n for j in range(-half_window, half_window + 1)]
        out[i] = float(np.nanmedian(values[idx]))
    return out

def circular_rolling_median_observed_only(values, half_window):
    values = np.asarray(values, dtype=np.float64)
    n = int(values.size)
    if n == 0:
        return values.copy()
    half_window = max(int(half_window), 0)
    if half_window == 0:
        return values.copy()
    out = np.full(n, np.nan, dtype=np.float64)
    for i in range(n):
        idx = [(i + j) % n for j in range(-half_window, half_window + 1)]
        local = values[idx]
        local = local[np.isfinite(local)]
        if local.size > 0:
            out[i] = float(np.nanmedian(local))
    return out

def phase_bin_medians(phase, y, edges):
    bins = phase_bin_indices(phase, edges)
    med = np.full(len(bins), np.nan, dtype=np.float64)
    counts = np.zeros(len(bins), dtype=np.int64)
    for i, idx in enumerate(bins):
        counts[i] = int(idx.size)
        if idx.size > 0:
            med[i] = float(np.nanmedian(y[idx]))
    return (bins, med, counts)

def clean_discrete_auto_phase_offset(jd, y, dy, period, bin_width):
    if not bool(CLEAN_DISCRETE_AUTO_PHASE_SHIFT):
        return 0.0
    jd = np.asarray(jd, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    dy = np.asarray(dy, dtype=np.float64)
    ok = np.isfinite(jd) & np.isfinite(y) & np.isfinite(dy) & (dy > 0)
    if ok.sum() == 0 or not np.isfinite(period) or period <= 0:
        return 0.0
    phase = phase_fold(jd[ok], float(period), 0.0)
    rel_comp_minus_var = -y[ok]
    edges = phase_bin_edges(bin_width)
    _, med, counts = phase_bin_medians(phase, rel_comp_minus_var, edges)
    if np.isfinite(med).sum() < 4:
        return 0.0
    baseline = circular_rolling_median_observed_only(med, int(CLEAN_DISCRETE_PHASE_SHIFT_SMOOTH_BINS))
    if not np.isfinite(baseline).any():
        return 0.0
    finite_baseline = baseline[np.isfinite(baseline)]
    finite_err = dy[ok][np.isfinite(dy[ok]) & (dy[ok] > 0)]
    err_med = float(np.nanmedian(finite_err)) if finite_err.size else np.nan
    amp = float(np.nanmax(finite_baseline) - np.nanmin(finite_baseline)) if finite_baseline.size else np.nan
    if bool(CLEAN_DISCRETE_ECLIPSE_PHASE_ANCHOR) and np.isfinite(amp) and amp >= float(CLEAN_DISCRETE_ECLIPSE_ANCHOR_MIN_AMP) and (not np.isfinite(err_med) or amp >= float(CLEAN_DISCRETE_ECLIPSE_ANCHOR_ERR_RATIO) * max(err_med, 1e-09)):
        eclipse_idx = int(np.nanargmax(baseline))
        eclipse_phase = 0.5 * (float(edges[eclipse_idx]) + float(edges[eclipse_idx + 1])) if eclipse_idx + 1 < edges.size else 1.0
        return float((float(CLEAN_DISCRETE_ECLIPSE_ANCHOR_PHASE) - eclipse_phase) % 1.0)
    if bool(CLEAN_DISCRETE_SHORT_PERIOD_PHASE_ANCHOR) and float(period) <= float(CLEAN_DISCRETE_SHORT_PERIOD_PHASE_ANCHOR_MAX_D) and np.isfinite(amp) and amp >= float(CLEAN_DISCRETE_SHORT_PERIOD_PHASE_ANCHOR_MIN_AMP):
        bright_idx = int(np.nanargmin(baseline))
        bright_phase = 0.5 * (float(edges[bright_idx]) + float(edges[bright_idx + 1])) if bright_idx + 1 < edges.size else 1.0
        return float((float(CLEAN_DISCRETE_SHORT_PERIOD_PHASE_ANCHOR_PHASE) - bright_phase) % 1.0)
    scores = []
    for i in range(baseline.size):
        j = (i + 1) % baseline.size
        if not (np.isfinite(baseline[i]) and np.isfinite(baseline[j])):
            scores.append(np.inf)
            continue
        coverage_penalty = 0.0 if counts[i] >= 2 and counts[j] >= 2 else 0.004
        scores.append(abs(float(baseline[i]) - float(baseline[j])) + coverage_penalty)
    scores = np.asarray(scores, dtype=np.float64)
    if not np.isfinite(scores).any():
        return 0.0
    edge_idx = int(np.nanargmin(scores)) + 1
    boundary = float(edges[edge_idx]) if edge_idx < edges.size else 1.0
    return float(-boundary % 1.0)

def binned_median_relative_phase_points(jd, y, dy, period, bin_width, phase_offset=0.0, smooth_half_window=0, smooth_blend=0.0, max_error_percentile=100.0):
    jd = np.asarray(jd, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    dy = np.asarray(dy, dtype=np.float64)
    ok = np.isfinite(jd) & np.isfinite(y) & np.isfinite(dy) & (dy > 0)
    if ok.sum() == 0 or not np.isfinite(period) or period <= 0:
        return (np.array([]), np.array([]), np.array([]))
    jd = jd[ok]
    y = y[ok]
    dy = dy[ok]
    phase = np.mod(phase_fold(jd, float(period), 0.0) + float(phase_offset), 1.0)
    rel_comp_minus_var = -y
    edges = phase_bin_edges(bin_width)
    bins = phase_bin_indices(phase, edges)
    nbin = len(bins)
    bin_phase = np.full(nbin, np.nan, dtype=np.float64)
    bin_mag = np.full(nbin, np.nan, dtype=np.float64)
    bin_err = np.full(nbin, np.nan, dtype=np.float64)
    pct = float(np.clip(float(max_error_percentile), 0.0, 100.0))
    for bin_idx, idx in enumerate(bins):
        if idx.size == 0:
            continue
        candidates = idx
        good_err = dy[idx][np.isfinite(dy[idx]) & (dy[idx] > 0)]
        if good_err.size > 0:
            err_limit = float(np.nanpercentile(good_err, pct))
            low_err = idx[dy[idx] <= err_limit]
            if low_err.size > 0:
                candidates = low_err
        bin_phase[bin_idx] = float(np.nanmedian(phase[candidates]))
        bin_mag[bin_idx] = float(np.nanmedian(rel_comp_minus_var[candidates]))
        local_err = float(np.nanmedian(dy[candidates]))
        if str(CLEAN_DISCRETE_BIN_ERROR_MODE).strip().lower() == 'sem' and candidates.size > 1:
            local_err = local_err / np.sqrt(float(candidates.size))
        bin_err[bin_idx] = local_err
    if int(smooth_half_window) > 0 and np.isfinite(bin_mag).any():
        smooth_mag = circular_rolling_median(bin_mag, int(smooth_half_window))
        blend = float(np.clip(float(smooth_blend), 0.0, 1.0))
        use = np.isfinite(bin_mag) & np.isfinite(smooth_mag)
        bin_mag[use] = (1.0 - blend) * bin_mag[use] + blend * smooth_mag[use]
    keep = np.isfinite(bin_phase) & np.isfinite(bin_mag) & np.isfinite(bin_err) & (bin_err > 0)
    if keep.sum() == 0:
        return (np.array([]), np.array([]), np.array([]))
    xarr = bin_phase[keep]
    yarr = bin_mag[keep]
    earr = bin_err[keep]
    order = np.argsort(xarr)
    return (xarr[order], yarr[order], earr[order])

def discrete_relative_phase_points(jd, y, dy, period, bin_width, point_mode='representative_center', phase_offset=0.0, prefer_low_error=False, max_error_percentile=100.0, continuity_selection=False, continuity_smooth_bins=0, hide_local_outliers=False):
    jd = np.asarray(jd, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    dy = np.asarray(dy, dtype=np.float64)
    ok = np.isfinite(jd) & np.isfinite(y) & np.isfinite(dy) & (dy > 0)
    if ok.sum() == 0 or not np.isfinite(period) or period <= 0:
        return (np.array([]), np.array([]), np.array([]))
    jd = jd[ok]
    y = y[ok]
    dy = dy[ok]
    phase = np.mod(phase_fold(jd, float(period), 0.0) + float(phase_offset), 1.0)
    rel_comp_minus_var = -y
    width = max(float(bin_width), 0.0001)
    mode = str(point_mode).strip().lower().replace('-', '_')
    if mode == 'representative_median_y':
        mode = 'representative_mediany'
    if mode not in CLEAN_DISCRETE_POINT_MODE_CHOICES:
        mode = 'representative_center'
    edges = phase_bin_edges(width)
    bins, bin_median, _ = phase_bin_medians(phase, rel_comp_minus_var, edges)
    baseline = None
    if bool(continuity_selection):
        baseline = circular_rolling_median_observed_only(bin_median, int(continuity_smooth_bins))
    err_limit = np.inf
    if bool(prefer_low_error):
        good_err = dy[np.isfinite(dy) & (dy > 0)]
        if good_err.size > 0:
            pct = float(np.clip(float(max_error_percentile), 0.0, 100.0))
            err_limit = float(np.nanpercentile(good_err, pct))
    err_med = float(np.nanmedian(dy[np.isfinite(dy) & (dy > 0)])) if np.isfinite(dy).any() else 1.0
    if not np.isfinite(err_med) or err_med <= 0:
        err_med = 1.0
    xout = []
    yout = []
    eout = []
    bout = []
    for bin_idx, idx in enumerate(bins):
        if idx.size == 0:
            continue
        candidates = idx
        if bool(prefer_low_error) and np.isfinite(err_limit):
            low_err = idx[dy[idx] <= err_limit]
            if low_err.size > 0:
                candidates = low_err
        lo = float(edges[bin_idx])
        hi = float(edges[bin_idx + 1])
        center = 0.5 * (float(lo) + float(hi))
        use_continuity = baseline is not None and np.isfinite(baseline[bin_idx])
        if mode == 'representative_mediany' and use_continuity:
            local_med = float(np.nanmedian(rel_comp_minus_var[idx]))
            local_scale = mad_sigma(rel_comp_minus_var[idx] - local_med)
            if not np.isfinite(local_scale) or local_scale <= 0:
                local_scale = max(mad_sigma(bin_median - baseline), 0.0025)
            cost = np.abs(rel_comp_minus_var[candidates] - baseline[bin_idx]) / max(float(local_scale), 0.0025) + 0.2 * dy[candidates] / max(float(err_med), 1e-09)
            chosen = candidates[int(np.nanargmin(cost))]
        elif mode == 'representative_mediany':
            median_y = float(np.nanmedian(rel_comp_minus_var[idx]))
            chosen = candidates[int(np.argmin(np.abs(rel_comp_minus_var[candidates] - median_y)))]
        else:
            chosen = candidates[int(np.argmin(np.abs(phase[candidates] - center)))]
        xout.append(float(phase[chosen]))
        yout.append(float(rel_comp_minus_var[chosen]))
        eout.append(float(dy[chosen]))
        bout.append(int(bin_idx))
    if not xout:
        return (np.array([]), np.array([]), np.array([]))
    xarr = np.asarray(xout, dtype=np.float64)
    yarr = np.asarray(yout, dtype=np.float64)
    earr = np.asarray(eout, dtype=np.float64)
    barr = np.asarray(bout, dtype=np.int64)
    if bool(hide_local_outliers) and baseline is not None and (barr.size > 0):
        bbase = baseline[barr]
        residual = np.abs(yarr - bbase)
        resid_sigma = mad_sigma(residual[np.isfinite(residual)])
        if np.isfinite(resid_sigma):
            limit = max(float(CLEAN_DISCRETE_OUTLIER_MIN_MAG), float(CLEAN_DISCRETE_OUTLIER_Z) * float(resid_sigma))
            keep = ~np.isfinite(residual) | (residual <= limit)
            min_keep = int(np.ceil((1.0 - float(CLEAN_DISCRETE_MAX_OUTLIER_REMOVAL_FRAC)) * keep.size))
            if keep.sum() >= min_keep:
                xarr = xarr[keep]
                yarr = yarr[keep]
                earr = earr[keep]
    order = np.argsort(xarr)
    return (xarr[order], yarr[order], earr[order])

def clean_discrete_relative_phase_points(jd, y, dy, period):
    phase_offset = clean_discrete_auto_phase_offset(jd, y, dy, period, float(CLEAN_DISCRETE_PHASE_BIN_WIDTH))
    point_mode = normalized_clean_discrete_point_mode()
    if point_mode in {'binned_median', 'smooth_binned_median'}:
        smooth_half_window = int(CLEAN_DISCRETE_BIN_SMOOTH_HALF_WINDOW) if point_mode == 'smooth_binned_median' else 0
        smooth_blend = float(CLEAN_DISCRETE_BIN_SMOOTH_BLEND) if point_mode == 'smooth_binned_median' else 0.0
        return binned_median_relative_phase_points(jd, y, dy, period, bin_width=float(CLEAN_DISCRETE_PHASE_BIN_WIDTH), phase_offset=float(phase_offset), smooth_half_window=smooth_half_window, smooth_blend=smooth_blend, max_error_percentile=float(CLEAN_DISCRETE_MAX_ERROR_PERCENTILE))
    return discrete_relative_phase_points(jd, y, dy, period, bin_width=float(CLEAN_DISCRETE_PHASE_BIN_WIDTH), point_mode=point_mode, phase_offset=float(phase_offset), prefer_low_error=bool(CLEAN_DISCRETE_PREFER_LOW_ERROR), max_error_percentile=float(CLEAN_DISCRETE_MAX_ERROR_PERCENTILE), continuity_selection=bool(CLEAN_DISCRETE_CONTINUITY_SELECTION), continuity_smooth_bins=int(CLEAN_DISCRETE_CONTINUITY_SMOOTH_BINS), hide_local_outliers=bool(CLEAN_DISCRETE_HIDE_LOCAL_OUTLIERS))

def finite_skewness(values):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    if values.size < 3:
        return np.nan
    center = float(np.nanmean(values))
    sigma = float(np.nanstd(values))
    if not np.isfinite(sigma) or sigma <= 0:
        return np.nan
    return float(np.nanmean(((values - center) / sigma) ** 3))

def variable_star_shape_stats(jd, y, dy, period):
    phase, mag, err = clean_discrete_relative_phase_points(jd, y, dy, period)
    mag = np.asarray(mag, dtype=np.float64)
    mag = mag[np.isfinite(mag)]
    if mag.size < 3:
        y = np.asarray(y, dtype=np.float64)
        dy = np.asarray(dy, dtype=np.float64)
        ok = np.isfinite(y) & np.isfinite(dy) & (dy > 0)
        mag = -y[ok]
    if mag.size < 3:
        return (np.nan, np.nan)
    amplitude = float(np.nanmax(mag) - np.nanmin(mag))
    skewness = finite_skewness(mag)
    return (amplitude, skewness)

def welch_stetson_j(mag, err):
    """Single-band Welch-Stetson J variability index from per-epoch (mag, err).
    Same definition used as the stetson_J ML feature in mlcode.ipynb; the sign of
    mag is irrelevant (it uses deviations from the weighted mean)."""
    mag = np.asarray(mag, dtype=np.float64)
    err = np.asarray(err, dtype=np.float64)
    ok = np.isfinite(mag) & np.isfinite(err) & (err > 0)
    mag = mag[ok]; err = err[ok]
    n = int(mag.size)
    if n < 5:
        return np.nan
    wmean = np.sum(mag / err ** 2) / np.sum(1.0 / err ** 2)
    delta = np.sqrt(n / (n - 1.0)) * (mag - wmean) / err
    P = delta ** 2 - 1.0
    return float(np.sum(np.sign(P) * np.sqrt(np.abs(P))) / n)

def phase_binned_points_for_plot(jd, y, dy, period, bin_width=None):
    'Display one observed point per phase bin, selected near the local median trend. These points are a display subset, not a representation of all observed scatter; the exported point table retains the analysed light curve.'
    if bin_width is None:
        bin_width = float(CLEAN_DISCRETE_PHASE_BIN_WIDTH)
    phase_offset = clean_discrete_auto_phase_offset(jd, y, dy, period, float(bin_width))
    return discrete_relative_phase_points(
        jd, y, dy, period, bin_width=float(bin_width),
        point_mode='representative_mediany', phase_offset=float(phase_offset),
        prefer_low_error=False, max_error_percentile=100.0,
        continuity_selection=True, continuity_smooth_bins=3,
        hide_local_outliers=False)

def save_periodogram_phase_plot(out_png, jd, y, dy, period, star_id, comp1_id, comp2_id, mode_name, best_row, experiment_name='standard', adjusted_score=np.nan, removed_points=0, removed_nights=0, raw_relative_jd=None, raw_relative_y=None, raw_relative_dy=None):
    jd = np.asarray(jd, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    dy = np.asarray(dy, dtype=np.float64)
    ok = np.isfinite(jd) & np.isfinite(y) & np.isfinite(dy) & (dy > 0)
    if ok.sum() == 0:
        return
    jd = jd[ok]
    y = y[ok]
    dy = dy[ok]
    pgram_period, pgram_power = build_periodogram_for_plot(jd, y, dy, int(best_row.get('eval_nterms', 1)), center_period=period)
    plot_phase, plot_y, plot_dy = phase_binned_points_for_plot(jd, y, dy, period)
    if plot_phase.size == 0:
        plot_phase = phase_fold(jd, period, 0.0)
        plot_y = -y
        plot_dy = dy
    if DOUBLE_CYCLE_PLOT:
        xp = np.concatenate([plot_phase, plot_phase + 1.0])
        yp = np.concatenate([plot_y, plot_y])
        ep = np.concatenate([plot_dy, plot_dy])
        xlim = (0.0, 2.0)
    else:
        xp = plot_phase
        yp = plot_y
        ep = plot_dy
        xlim = (0.0, 1.0)
    fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.5))
    ax0, ax1 = axes
    ax0.plot(pgram_period, pgram_power, color='black', lw=0.8)
    if SHOW_PERIOD_MARKER_LINE:
        ax0.axvline(period, color='tab:red', lw=1.0, alpha=0.8)
    if pgram_period.size > 0 and np.isfinite(pgram_period).any():
        ax0.set_xlim(float(np.nanmin(pgram_period)), float(np.nanmax(pgram_period)))
    else:
        ax0.set_xlim(PERIOD_MIN_D, PERIOD_MAX_D)
    ax0.set_xlabel('Period (days)')
    ax0.set_ylabel('Lomb-Scargle power')
    ax0.set_title('Periodogram')
    ax1.errorbar(xp, yp, yerr=ep, fmt='o', ms=1.5, lw=0.0, elinewidth=0.7, capsize=0.0, color='black', ecolor='0.6', markerfacecolor='black', markeredgecolor='black', markeredgewidth=0.0, alpha=0.9)
    ax1.set_xlim(*xlim)
    ax1.invert_yaxis()
    ax1.set_xlabel('Phase')
    ax1.set_ylabel('Relative magnitude (comp - var)')
    ax1.set_title('Phase diagram')
    fig.suptitle(f'Star {star_id} | P={period:.6f} d', fontsize=10)
    fig.tight_layout()
    fig.savefig(out_png, dpi=PLOT_DPI)
    plt.close(fig)

def save_combined_phase_grid_plot(out_png, entries, ncols=10, nrows=7):
    if not entries:
        return []
    ncols = max(int(ncols), 1)
    nrows = max(int(nrows), 1)
    slots_per_page = max(ncols * nrows, 1)
    base, ext = os.path.splitext(str(out_png))
    if not ext:
        ext = '.png'
    saved_paths = []
    for page_idx in range(int(np.ceil(len(entries) / float(slots_per_page)))):
        chunk = entries[page_idx * slots_per_page:(page_idx + 1) * slots_per_page]
        fig, axes = plt.subplots(nrows, ncols, figsize=COMBINED_PHASE_GRID_FIGSIZE, squeeze=False)
        axes_flat = axes.ravel()
        for ax in axes_flat:
            ax.set_visible(False)
        for panel_idx, (ax, entry) in enumerate(zip(axes_flat, chunk)):
            jd = np.asarray(entry['jd'], dtype=np.float64)
            y = np.asarray(entry['y'], dtype=np.float64)
            dy = np.asarray(entry['dy'], dtype=np.float64)
            period = float(entry['period'])
            star_id = int(entry['star_id'])
            ok = np.isfinite(jd) & np.isfinite(y) & np.isfinite(dy) & (dy > 0)
            if ok.sum() == 0 or not np.isfinite(period) or period <= 0:
                continue
            jd = jd[ok]
            y = y[ok]
            dy = dy[ok]
            if jd.size == 0:
                continue
            plot_phase, plot_y, plot_dy = phase_binned_points_for_plot(jd, y, dy, period)
            if plot_phase.size == 0:
                plot_phase = phase_fold(jd, period, 0.0)
                plot_y = -y
                plot_dy = dy
            if DOUBLE_CYCLE_PLOT:
                xp = np.concatenate([plot_phase, plot_phase + 1.0])
                yp = np.concatenate([plot_y, plot_y])
                ep = np.concatenate([plot_dy, plot_dy])
                xlim = (0.0, 2.0)
            else:
                xp = plot_phase
                yp = plot_y
                ep = plot_dy
                xlim = (0.0, 1.0)
            ax.set_visible(True)
            ax.errorbar(xp, yp, yerr=ep, fmt='o', ms=0.85, lw=0.0, elinewidth=0.3, capsize=0.0, color='black', ecolor='0.6', markerfacecolor='black', markeredgecolor='black', markeredgewidth=0.0, alpha=0.9)
            ax.set_title(f'{star_id}\nP={period:.4f} d', fontsize=4.2, pad=1.2)
            ax.set_xlabel('Phase', fontsize=3.6, labelpad=0.8)
            ax.set_ylabel('$\\Delta m$ (comp-var)', fontsize=3.6, labelpad=0.8)
            ax.tick_params(direction='in', length=1.3, width=0.45, labelsize=3.0)
            ax.set_xlim(*xlim)
            ax.invert_yaxis()
        fig.tight_layout(rect=(0.01, 0.01, 0.995, 0.995))
        page_path = out_png
        if len(entries) > slots_per_page:
            page_path = f'{base}_page_{page_idx + 1:02d}{ext}'
        fig.savefig(page_path, dpi=COMBINED_PHASE_GRID_DPI)
        plt.close(fig)
        saved_paths.append(page_path)
    return saved_paths

def cell4_wrap_0_2pi(angle):
    if not np.isfinite(angle):
        return np.nan
    return float(np.mod(float(angle), 2.0 * np.pi))

def cell4_empty_fourier_features(period):
    log_period = np.log10(float(period)) if np.isfinite(period) and period > 0 else np.nan
    return {'P1_days': float(period) if np.isfinite(period) else np.nan, 'log10P': float(log_period) if np.isfinite(log_period) else np.nan, 'amplitude_peak_to_peak_mag': np.nan, 'A1_fourier_mag': np.nan, 'A2_fourier_mag': np.nan, 'A3_fourier_mag': np.nan, 'R21': np.nan, 'phi21_rad': np.nan, 'R31': np.nan, 'phi31_rad': np.nan, 'phi1_rad': np.nan, 'phi2_rad': np.nan, 'phi3_rad': np.nan, 'fourier_intercept_mag': np.nan, 'fourier_residual_rms_mag': np.nan, 'n_fourier_points': 0, 'fourier_fit_ok': 0}

def cell4_fit_fourier_features(phase, y, dy, period):
    out = cell4_empty_fourier_features(period)
    phase = np.asarray(phase, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    dy = np.asarray(dy, dtype=np.float64)
    ok = np.isfinite(phase) & np.isfinite(y) & np.isfinite(dy) & (dy > 0)
    out['n_fourier_points'] = int(ok.sum())
    if ok.sum() < 12 or not (np.isfinite(period) and period > 0):
        return out
    phase = np.mod(phase[ok], 1.0)
    y = y[ok]
    dy = dy[ok]
    order = 3
    cols = [np.ones_like(phase)]
    for k in range(1, order + 1):
        angle = 2.0 * np.pi * float(k) * phase
        cols.append(np.cos(angle))
        cols.append(np.sin(angle))
    design = np.column_stack(cols)
    if ok.sum() <= design.shape[1]:
        return out
    finite_err = dy[np.isfinite(dy) & (dy > 0)]
    err_floor = max(1e-06, 0.05 * float(np.nanmedian(finite_err))) if finite_err.size else 1e-06
    sigma = np.maximum(dy, err_floor)
    sqrt_w = 1.0 / sigma
    try:
        coeff, _resid, rank, _singular = np.linalg.lstsq(design * sqrt_w[:, None], y * sqrt_w, rcond=None)
    except Exception:
        return out
    if rank < design.shape[1] or not np.all(np.isfinite(coeff)):
        return out
    amps = {}
    phis = {}
    for k in range(1, order + 1):
        cos_coef = float(coeff[1 + 2 * (k - 1)])
        sin_coef = float(coeff[2 + 2 * (k - 1)])
        amps[k] = float(np.hypot(cos_coef, sin_coef))
        phis[k] = cell4_wrap_0_2pi(math.atan2(-sin_coef, cos_coef))
    model = design @ coeff
    weights = 1.0 / np.maximum(sigma, 1e-12) ** 2
    resid_rms = np.sqrt(np.sum(weights * (y - model) ** 2) / np.sum(weights))
    dense_phase = np.linspace(0.0, 1.0, 1000, endpoint=False)
    dense_cols = [np.ones_like(dense_phase)]
    for k in range(1, order + 1):
        angle = 2.0 * np.pi * float(k) * dense_phase
        dense_cols.append(np.cos(angle))
        dense_cols.append(np.sin(angle))
    dense_model = np.column_stack(dense_cols) @ coeff
    peak_to_peak = float(np.nanmax(dense_model) - np.nanmin(dense_model))
    out.update({'amplitude_peak_to_peak_mag': peak_to_peak, 'A1_fourier_mag': amps[1], 'A2_fourier_mag': amps[2], 'A3_fourier_mag': amps[3], 'R21': float(amps[2] / amps[1]) if amps[1] > 0 else np.nan, 'phi21_rad': cell4_wrap_0_2pi(phis[2] - 2.0 * phis[1]), 'R31': float(amps[3] / amps[1]) if amps[1] > 0 else np.nan, 'phi31_rad': cell4_wrap_0_2pi(phis[3] - 3.0 * phis[1]), 'phi1_rad': phis[1], 'phi2_rad': phis[2], 'phi3_rad': phis[3], 'fourier_intercept_mag': float(coeff[0]), 'fourier_residual_rms_mag': float(resid_rms) if np.isfinite(resid_rms) else np.nan, 'fourier_fit_ok': 1})
    return out

FOURIER_FEATURE_STABILIZE = True
FOURIER_FEATURE_PERIOD_MIN_D = 1.00
FOURIER_FEATURE_PERIOD_MAX_D = 1.85
FOURIER_FEATURE_MAX_AMPLITUDE = 0.125
FOURIER_FEATURE_MIN_COMPLEXITY = 1.0
FOURIER_FEATURE_REL_WIDTH = 0.02
FOURIER_FEATURE_GRID_SIZE = 81
FOURIER_FEATURE_MAX_THETA_ADD = 0.18
FOURIER_FEATURE_MAX_THETA_FACTOR = 1.35
FOURIER_FEATURE_MAX_GAP_ADD = 0.055
FOURIER_FEATURE_MAX_GAP_FACTOR = 1.80
FOURIER_FEATURE_MIN_AMP_FACTOR = 0.40
FOURIER_FEATURE_MIN_COMPLEXITY_GAIN = 0.25

def _fourier_feature_complexity(ft):
    r21 = float(ft.get('R21', np.nan))
    r31 = float(ft.get('R31', np.nan))
    if not (np.isfinite(r21) and np.isfinite(r31)):
        return np.inf
    return float(r21 + r31)

def compute_ml_shape_fourier_features(jd, y, dy, period):
    period = float(period)
    amplitude, skewness = variable_star_shape_stats(jd, y, dy, period)
    stetson_j_value = welch_stetson_j(y, dy)
    base_phase = phase_fold(np.asarray(jd, dtype=np.float64), period, float(np.nanmin(jd)))
    base_ft = cell4_fit_fourier_features(base_phase, -np.asarray(y, dtype=np.float64), np.asarray(dy, dtype=np.float64), period)
    base_complexity = _fourier_feature_complexity(base_ft)
    result = {'amplitude': amplitude, 'skewness': skewness, 'stetson_J': stetson_j_value, 'fourier': base_ft, 'feature_period': period, 'feature_refined': False}
    if not bool(FOURIER_FEATURE_STABILIZE):
        return result
    if not (np.isfinite(period) and float(FOURIER_FEATURE_PERIOD_MIN_D) <= period <= float(FOURIER_FEATURE_PERIOD_MAX_D)):
        return result
    if not (np.isfinite(amplitude) and abs(float(amplitude)) <= float(FOURIER_FEATURE_MAX_AMPLITUDE)):
        return result
    if not (np.isfinite(base_complexity) and base_complexity >= float(FOURIER_FEATURE_MIN_COMPLEXITY)):
        return result
    base_profile = phase_profile_stats(jd, y, dy, period, nbins=NBINS_SCORE, min_bin_pts=MIN_BIN_PTS)
    base_theta = float(base_profile.get('theta', np.inf))
    base_gap = float(phase_gap_metric(jd, period))
    if not (np.isfinite(base_theta) and np.isfinite(base_gap)):
        return result
    pmin = max(float(PERIOD_MIN_D), period * (1.0 - float(FOURIER_FEATURE_REL_WIDTH)))
    pmax = min(float(PERIOD_MAX_D), period * (1.0 + float(FOURIER_FEATURE_REL_WIDTH)))
    if not (np.isfinite(pmin) and np.isfinite(pmax) and pmax > pmin):
        return result
    best = None
    for trial_period in np.linspace(pmin, pmax, int(FOURIER_FEATURE_GRID_SIZE)):
        trial_period = float(trial_period)
        trial_amp, trial_skew = variable_star_shape_stats(jd, y, dy, trial_period)
        if not (np.isfinite(trial_amp) and abs(float(trial_amp)) >= float(FOURIER_FEATURE_MIN_AMP_FACTOR) * abs(float(amplitude))):
            continue
        trial_profile = phase_profile_stats(jd, y, dy, trial_period, nbins=NBINS_SCORE, min_bin_pts=MIN_BIN_PTS)
        trial_theta = float(trial_profile.get('theta', np.inf))
        trial_gap = float(phase_gap_metric(jd, trial_period))
        if not (np.isfinite(trial_theta) and np.isfinite(trial_gap)):
            continue
        theta_limit = max(base_theta + float(FOURIER_FEATURE_MAX_THETA_ADD), base_theta * float(FOURIER_FEATURE_MAX_THETA_FACTOR))
        gap_limit = max(base_gap + float(FOURIER_FEATURE_MAX_GAP_ADD), base_gap * float(FOURIER_FEATURE_MAX_GAP_FACTOR))
        if trial_theta > theta_limit or trial_gap > gap_limit:
            continue
        trial_phase = phase_fold(np.asarray(jd, dtype=np.float64), trial_period, float(np.nanmin(jd)))
        trial_ft = cell4_fit_fourier_features(trial_phase, -np.asarray(y, dtype=np.float64), np.asarray(dy, dtype=np.float64), trial_period)
        trial_complexity = _fourier_feature_complexity(trial_ft)
        if not np.isfinite(trial_complexity):
            continue
        candidate = (trial_complexity, trial_theta, trial_gap, trial_period, trial_amp, trial_skew, trial_ft)
        if best is None or candidate[:3] < best[:3]:
            best = candidate
    if best is None:
        return result
    if float(best[0]) <= (1.0 - float(FOURIER_FEATURE_MIN_COMPLEXITY_GAIN)) * base_complexity:
        result.update({'amplitude': float(best[4]), 'skewness': float(best[5]), 'fourier': best[6], 'feature_period': float(best[3]), 'feature_refined': True})
    return result

UNCERTAINTY_ENABLED = True
UNCERTAINTY_BOOTSTRAP_REPLICATES = 400
UNCERTAINTY_SESSION_GAP_DAYS = 0.35

def _uncertainty_robust_sigma(values):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    if values.size < 3:
        return np.nan
    q16, q84 = np.quantile(values, [0.158655, 0.841345])
    return float(0.5 * (q84 - q16))

def _uncertainty_robust_scatter(values):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    if values.size < 3:
        return np.nan
    center = float(np.median(values))
    return float(1.4826 * np.median(np.abs(values - center)))

def _uncertainty_jackknife_sigma(values):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    n = int(values.size)
    if n < 3:
        return np.nan
    return float(np.sqrt((n - 1.0) / n * np.sum((values - np.mean(values)) ** 2)))

def _uncertainty_finite_max(values):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    return float(np.max(values)) if values.size else np.nan

def _uncertainty_session_groups(jd):
    jd = np.asarray(jd, dtype=np.float64)
    order = np.argsort(jd)
    sorted_jd = jd[order]
    sorted_group = np.zeros(sorted_jd.size, dtype=np.int64)
    if sorted_jd.size > 1:
        sorted_group[1:] = np.cumsum(
            np.diff(sorted_jd) > float(UNCERTAINTY_SESSION_GAP_DAYS)
        )
    group = np.empty_like(sorted_group)
    group[order] = sorted_group
    return group

def _uncertainty_design_matrix(jd, frequency, nterms):
    centered = np.asarray(jd, dtype=np.float64) - float(np.median(jd))
    columns = [np.ones_like(centered)]
    for harmonic in range(1, int(nterms) + 1):
        angle = 2.0 * np.pi * float(harmonic) * float(frequency) * centered
        columns.extend([np.cos(angle), np.sin(angle)])
    return np.column_stack(columns)

def _uncertainty_harmonic_fit(jd, signal, sigma, frequency, nterms):
    design = _uncertainty_design_matrix(jd, frequency, nterms)
    sigma = np.maximum(np.asarray(sigma, dtype=np.float64), 1.0e-9)
    weighted_design = design / sigma[:, None]
    weighted_signal = np.asarray(signal, dtype=np.float64) / sigma
    coefficients, _, _, _ = np.linalg.lstsq(
        weighted_design, weighted_signal, rcond=None
    )
    model = design @ coefficients
    residual = np.asarray(signal, dtype=np.float64) - model
    chi2 = float(np.sum((residual / sigma) ** 2))
    return coefficients, model, residual, chi2

def _uncertainty_local_frequency_fit(
    jd, signal, sigma, center_frequency, nterms, half_width=None
):
    baseline = float(np.ptp(jd))
    if half_width is None:
        half_width = min(0.45 / max(baseline, 1.0e-9), 0.20 * float(center_frequency))
    low = max(float(center_frequency) - float(half_width), 1.0e-8)
    high = float(center_frequency) + float(half_width)

    def objective(frequency):
        return _uncertainty_harmonic_fit(jd, signal, sigma, frequency, nterms)[3]

    result = minimize_scalar(
        objective,
        bounds=(low, high),
        method='bounded',
        options={
            'xatol': max(1.0e-12, float(center_frequency) * 1.0e-11),
            'maxiter': 100,
        },
    )
    frequency = float(result.x)
    coefficients, model, residual, chi2 = _uncertainty_harmonic_fit(
        jd, signal, sigma, frequency, nterms
    )
    return frequency, coefficients, model, residual, chi2

def _uncertainty_linearized_frequency_fit(
    jd, signal, sigma, center_frequency, nterms, half_width
):
    jd = np.asarray(jd, dtype=np.float64)
    signal = np.asarray(signal, dtype=np.float64)
    sigma = np.maximum(np.asarray(sigma, dtype=np.float64), 1.0e-9)
    centered = jd - float(np.median(jd))
    frequency = float(center_frequency)
    coefficients, _, _, _ = _uncertainty_harmonic_fit(
        jd, signal, sigma, frequency, nterms
    )
    design = _uncertainty_design_matrix(jd, frequency, nterms)
    derivative = np.zeros_like(jd)
    for harmonic in range(1, int(nterms) + 1):
        angle = 2.0 * np.pi * float(harmonic) * frequency * centered
        cosine_coefficient = coefficients[2 * harmonic - 1]
        sine_coefficient = coefficients[2 * harmonic]
        derivative += 2.0 * np.pi * float(harmonic) * centered * (
            -cosine_coefficient * np.sin(angle) + sine_coefficient * np.cos(angle)
        )
    augmented = np.column_stack([design, derivative])
    weighted = augmented / sigma[:, None]
    try:
        solution = np.linalg.solve(
            weighted.T @ weighted, weighted.T @ (signal / sigma)
        )
    except np.linalg.LinAlgError:
        solution, _, _, _ = np.linalg.lstsq(weighted, signal / sigma, rcond=None)
    frequency_shift = float(
        np.clip(solution[-1], -float(half_width), float(half_width))
    )
    return max(frequency + frequency_shift, 1.0e-8)

def _estimate_period_amplitude_uncertainties(
    jd,
    signal,
    dy,
    adopted_period,
    adopted_amplitude,
    star_id,
    nterms,
    seed_offset,
    amplitude_statistic,
    amplitude_period_grid,
):
    empty = {'period_sigma_days': np.nan, 'amplitude_sigma_mag': np.nan}
    if not bool(UNCERTAINTY_ENABLED):
        return empty
    jd = np.asarray(jd, dtype=np.float64)
    signal = np.asarray(signal, dtype=np.float64)
    dy = np.asarray(dy, dtype=np.float64)
    good = np.isfinite(jd) & np.isfinite(signal) & np.isfinite(dy) & (dy > 0)
    jd, signal, dy = jd[good], signal[good], dy[good]
    adopted_period = float(adopted_period)
    adopted_amplitude = float(adopted_amplitude)
    nterms = max(1, int(nterms))
    if (
        jd.size < max(20, 2 * nterms + 5)
        or not np.isfinite(adopted_period)
        or adopted_period <= 0
        or not np.isfinite(adopted_amplitude)
        or float(np.ptp(jd)) <= 0
    ):
        return empty

    grid = np.asarray(amplitude_period_grid, dtype=np.float64)
    grid = grid[np.isfinite(grid) & (grid > 0)]
    grid_amplitudes = []
    for trial_period in grid:
        try:
            grid_amplitudes.append(
                float(amplitude_statistic(jd, signal, dy, float(trial_period)))
            )
        except Exception:
            grid_amplitudes.append(np.nan)
    grid_amplitudes = np.asarray(grid_amplitudes, dtype=np.float64)
    usable_grid = np.isfinite(grid_amplitudes)
    if usable_grid.any():
        usable_indices = np.flatnonzero(usable_grid)
        closest = usable_indices[
            int(np.argmin(np.abs(grid_amplitudes[usable_grid] - adopted_amplitude)))
        ]
        amplitude_period_ratio = float(grid[closest] / adopted_period)
    else:
        amplitude_period_ratio = 1.0

    adopted_frequency = 1.0 / adopted_period
    _, _, initial_residual, _ = _uncertainty_harmonic_fit(
        jd, signal, dy, adopted_frequency, nterms
    )
    residual_scatter = _uncertainty_robust_scatter(initial_residual)
    median_error = float(np.median(dy))
    jitter = float(
        np.sqrt(max(0.0, residual_scatter ** 2 - median_error ** 2))
    ) if np.isfinite(residual_scatter) else 0.0
    sigma_effective = np.sqrt(dy ** 2 + jitter ** 2)
    fitted_frequency, coefficients, model, residual, _ = _uncertainty_local_frequency_fit(
        jd, signal, sigma_effective, adopted_frequency, nterms
    )
    baseline = float(np.ptp(jd))
    local_half_width = min(
        0.45 / max(baseline, 1.0e-9), 0.20 * float(fitted_frequency)
    )
    groups = _uncertainty_session_groups(jd)
    unique_groups = np.unique(groups)
    standardized = residual / sigma_effective
    standardized = standardized - float(np.mean(standardized))
    rng = np.random.default_rng(int(seed_offset) + int(star_id))

    night_periods = []
    night_amplitudes = []
    residual_amplitudes = []
    for _ in range(int(UNCERTAINTY_BOOTSTRAP_REPLICATES)):
        signs = rng.choice(np.asarray([-1.0, 1.0]), unique_groups.size)
        simulated_night = model + residual * signs[groups]
        night_frequency = _uncertainty_linearized_frequency_fit(
            jd, simulated_night, sigma_effective, fitted_frequency,
            nterms, local_half_width,
        )
        night_period = 1.0 / night_frequency
        night_periods.append(night_period)
        try:
            night_amplitudes.append(
                float(amplitude_statistic(
                    jd, simulated_night, dy,
                    night_period * amplitude_period_ratio,
                ))
            )
        except Exception:
            night_amplitudes.append(np.nan)

        simulated_residual = model + (
            rng.choice(standardized, jd.size, replace=True) * sigma_effective
        )
        residual_frequency = _uncertainty_linearized_frequency_fit(
            jd, simulated_residual, sigma_effective, fitted_frequency,
            nterms, local_half_width,
        )
        residual_period = 1.0 / residual_frequency
        try:
            residual_amplitudes.append(
                float(amplitude_statistic(
                    jd, simulated_residual, dy,
                    residual_period * amplitude_period_ratio,
                ))
            )
        except Exception:
            residual_amplitudes.append(np.nan)

    leave_one_session_periods = []
    parameter_count = int(coefficients.size)
    for group in unique_groups:
        keep = groups != group
        if int(keep.sum()) <= parameter_count + 3:
            continue
        frequency = _uncertainty_linearized_frequency_fit(
            jd[keep], signal[keep], sigma_effective[keep], fitted_frequency,
            nterms, local_half_width,
        )
        leave_one_session_periods.append(1.0 / frequency)

    period_sigma = _uncertainty_finite_max([
        _uncertainty_robust_sigma(night_periods),
        _uncertainty_jackknife_sigma(leave_one_session_periods),
    ])
    amplitude_sigma = _uncertainty_finite_max([
        _uncertainty_robust_sigma(night_amplitudes),
        _uncertainty_robust_sigma(residual_amplitudes),
    ])
    return {
        'period_sigma_days': float(period_sigma) if np.isfinite(period_sigma) else np.nan,
        'amplitude_sigma_mag': float(amplitude_sigma) if np.isfinite(amplitude_sigma) else np.nan,
    }

def estimate_cell1_period_amplitude_uncertainties(
    jd, y, dy, period, amplitude, star_id, nterms
):
    period = float(period)

    def amplitude_statistic(trial_jd, trial_y, trial_dy, trial_period):
        return variable_star_shape_stats(
            trial_jd, trial_y, trial_dy, trial_period
        )[0]

    amplitude_grid = np.linspace(0.98 * period, 1.02 * period, 81)
    return _estimate_period_amplitude_uncertainties(
        jd, y, dy, period, amplitude, star_id, nterms,
        250600000, amplitude_statistic, amplitude_grid,
    )

def run_standard_pipeline():
    os.makedirs(OUT_DIR, exist_ok=True)
    os.makedirs(PLOT_DIR, exist_ok=True)
    compare_df = parse_comparison_catalog(COMPARE_CSV)
    master_df = load_master_catalog(MASTER_MAP_PATH)
    star_coord_map = build_target_coord_map(master_df, compare_df[COL_STAR_ID].astype(int).tolist())
    phot = build_needed_photometry(master_df, compare_df, jd_path=JD_FILE)
    mode_lookup = build_mode_lookup(compare_df, phot['jd'], phot['corr_mag_mat'], phot['err_mat'], phot['star_to_needed_col'], star_coord_map=star_coord_map, frame_ids=phot['frame_index_global'])
    summary_rows = []
    candidate_rows = []
    point_rows = []
    variable_stat_rows = []
    combined_phase_entries = []
    for i, row in enumerate(compare_df.itertuples(index=False), start=1):
        star_id = int(getattr(row, COL_STAR_ID))
        mode_rows = mode_lookup.get(star_id, [])
        if star_id not in phot['star_to_needed_col']:
            summary = summarize_solution_row(star_id, COMPARE_CSV, best_solution=None)
            summary['failure_reason'] = 'star_missing_from_master_map'
            summary_rows.append(summary)
            continue
        if not mode_rows:
            summary = summarize_solution_row(star_id, COMPARE_CSV, best_solution=None)
            summary['failure_reason'] = 'no_valid_reference_mode'
            summary_rows.append(summary)
            continue
        best_solution, star_candidate_rows = solve_mode_rows(mode_rows)
        summary = summarize_solution_row(star_id, COMPARE_CSV, best_solution, experiment_name='standard', reject_bad_jd=False, reject_bad_nights=False)
        summary_rows.append(summary)
        for cand in star_candidate_rows:
            candidate_rows.append({'star_id': int(star_id), 'experiment_name': 'standard', **cand})
        if best_solution is None:
            continue
        chosen = best_solution['chosen']
        jd = best_solution['jd']
        y = best_solution['y']
        dy = best_solution['dy']
        period = float(chosen['period'])
        _ml_features = compute_ml_shape_fourier_features(jd, y, dy, period)
        amplitude = _ml_features['amplitude']
        skewness = _ml_features['skewness']
        stetson_j_value = _ml_features['stetson_J']
        _ft = _ml_features['fourier']
        _uncertainty = estimate_cell1_period_amplitude_uncertainties(
            jd, y, dy, period, amplitude, star_id,
            int(chosen.get('eval_nterms', 1)),
        )
        period_sigma_days = _uncertainty['period_sigma_days']
        amplitude_sigma_mag = _uncertainty['amplitude_sigma_mag']
        variable_stat_rows.append({
            'star_id': int(star_id),
            'amplitude': float(amplitude) if np.isfinite(amplitude) else np.nan,
            'amplitude_sigma_mag': float(amplitude_sigma_mag) if np.isfinite(amplitude_sigma_mag) else np.nan,
            'skewness': float(skewness) if np.isfinite(skewness) else np.nan,
            'period': float(period),
            'period_sigma_days': float(period_sigma_days) if np.isfinite(period_sigma_days) else np.nan,
            'log10P': float(np.log10(period)) if np.isfinite(period) and period > 0 else np.nan,
            'amplitude_fourier': _ft.get('amplitude_peak_to_peak_mag', np.nan),
            'R21': _ft.get('R21', np.nan),
            'phi21': _ft.get('phi21_rad', np.nan),
            'R31': _ft.get('R31', np.nan),
            'phi31': _ft.get('phi31_rad', np.nan),
            'stetson_J': float(stetson_j_value) if np.isfinite(stetson_j_value) else np.nan,
        })
        combined_phase_entries.append({'star_id': int(star_id), 'jd': np.asarray(jd, dtype=np.float64).copy(), 'y': np.asarray(y, dtype=np.float64).copy(), 'dy': np.asarray(dy, dtype=np.float64).copy(), 'raw_relative_jd': np.asarray(best_solution.get('raw_relative_jd', jd), dtype=np.float64).copy(), 'raw_relative_y': np.asarray(best_solution.get('raw_relative_y', y), dtype=np.float64).copy(), 'raw_relative_dy': np.asarray(best_solution.get('raw_relative_dy', dy), dtype=np.float64).copy(), 'period': float(period)})
        if SAVE_PLOTS:
            out_png = os.path.join(PLOT_DIR, f"starNEW_{star_id}_refmode_{best_solution['reference_mode']}_variant_{best_solution['series_variant']}_P_{period:.6f}.png")
            save_periodogram_phase_plot(out_png, jd, y, dy, period, star_id, best_solution['comp1_star_id'], best_solution['comp2_star_id'], best_solution['reference_mode'], chosen, experiment_name='standard', adjusted_score=float(best_solution.get('adjusted_score', np.nan)), removed_points=int(best_solution['variant_metrics'].get('removed_points', 0)), removed_nights=int(best_solution['variant_metrics'].get('removed_nights', 0)), raw_relative_jd=best_solution.get('raw_relative_jd', jd), raw_relative_y=best_solution.get('raw_relative_y', y), raw_relative_dy=best_solution.get('raw_relative_dy', dy))
        for jj in range(jd.size):
            point_rows.append({'Star_ID_NEW': int(star_id), 'Comp1_NEW': int(best_solution['comp1_star_id']), 'Comp2_NEW': int(best_solution['comp2_star_id']) if pd.notna(best_solution['comp2_star_id']) else np.nan, 'reference_mode': best_solution['reference_mode'], 'series_variant': best_solution['series_variant'], 'JD': float(jd[jj]), 'rel_mag': float(y[jj]), 'rel_err': float(dy[jj]), 'best_period_days': float(period), 'eval_nterms': int(chosen['eval_nterms']), 'candidate_score': float(chosen['score']), 'adjusted_score': float(best_solution.get('adjusted_score', np.nan)), 'theta': float(chosen['theta']), 'model_gain': float(chosen['model_gain']), 'gap': float(chosen['gap']), 'night_drift_ratio': float(best_solution['variant_metrics'].get('night_drift_ratio', np.nan)), 'night_median_amp_ratio': float(best_solution['variant_metrics'].get('night_median_amp_ratio', np.nan)), 'collapse_sigma_ratio': float(best_solution['variant_metrics'].get('collapse_sigma_ratio', np.nan)), 'collapse_amp_ratio': float(best_solution['variant_metrics'].get('collapse_amp_ratio', np.nan)), 'removed_points': int(best_solution['variant_metrics'].get('removed_points', 0)), 'removed_nights': int(best_solution['variant_metrics'].get('removed_nights', 0))})
        print(f"[diag] processed {i}/{len(compare_df)} | star={star_id} | mode={best_solution['reference_mode']} | variant={best_solution['series_variant']} | P={period:.6f} | score={chosen['score']:.3f} | adjusted={best_solution.get('adjusted_score', np.nan):.3f}", flush=True)
    summary_df = pd.DataFrame(summary_rows).sort_values(['star_id']).reset_index(drop=True)
    candidate_df = pd.DataFrame(candidate_rows)
    if not candidate_df.empty:
        candidate_df = candidate_df.sort_values(['star_id', 'reference_mode', 'series_variant', 'candidate_rank']).reset_index(drop=True)
    points_df = pd.DataFrame(point_rows)
    if not points_df.empty:
        points_df = points_df.sort_values(['Star_ID_NEW', 'JD']).reset_index(drop=True)
    variable_stats_df = pd.DataFrame(variable_stat_rows, columns=['star_id', 'amplitude', 'amplitude_sigma_mag', 'skewness', 'period', 'period_sigma_days', 'log10P', 'amplitude_fourier', 'R21', 'phi21', 'R31', 'phi31', 'stetson_J'])
    if not variable_stats_df.empty:
        variable_stats_df = variable_stats_df.sort_values(['star_id']).reset_index(drop=True)
    frame_df = phot['frame_df'].copy()
    summary_df.to_csv(OUT_SUMMARY, index=False)
    candidate_df.to_csv(OUT_TOP_CANDIDATES, index=False)
    points_df.to_csv(OUT_POINTS, index=False)
    variable_stats_df.to_csv(OUT_VARIABLE_STATS, index=False)
    frame_df.to_csv(OUT_FRAME_DIAG, index=False)
    print(f'Saved summary: {OUT_SUMMARY} | rows={len(summary_df)}', flush=True)
    print(f'Saved top candidates: {OUT_TOP_CANDIDATES} | rows={len(candidate_df)}', flush=True)
    print(f'Saved phased points: {OUT_POINTS} | rows={len(points_df)}', flush=True)
    print(f'Saved variable-star stats: {OUT_VARIABLE_STATS} | rows={len(variable_stats_df)}', flush=True)
    print(f'Saved frame diagnostics: {OUT_FRAME_DIAG} | rows={len(frame_df)}', flush=True)
    if SAVE_PLOTS and SAVE_COMBINED_PHASE_GRID and combined_phase_entries:
        combined_grid_paths = save_combined_phase_grid_plot(COMBINED_PHASE_GRID_OUT, combined_phase_entries, ncols=COMBINED_PHASE_GRID_COLS, nrows=COMBINED_PHASE_GRID_ROWS)
        for grid_path in combined_grid_paths:
            print(f'Saved combined phase grid: {grid_path}', flush=True)
    print(f'Done. Plots in: {PLOT_DIR}', flush=True)

def main():
    run_standard_pipeline()
if 'ngc2506' in SELECTED_CLUSTERS:
    main()


## 3. NGC 6709 periods and features

This engine ranks period candidates using binned-fold morphology, sampling diagnostics, Lomb–Scargle support and explicit period-dependent preferences. It uses the discovery CSV’s first comparison star. The local uncertainty calculation is conditional on the selected alias and does not cover the full period-selection ambiguity. Phase figures use a trend-selected subset of real observations.


In [ ]:
import numpy as np
import glob
import os
import gc
import sys
import matplotlib.pyplot as plt

_PYARROW_MASKED_FOR_PANDAS = 'pyarrow' not in sys.modules
sys.modules.setdefault('pyarrow', None)
import pandas as pd
if _PYARROW_MASKED_FOR_PANDAS and sys.modules.get('pyarrow') is None:
    sys.modules.pop('pyarrow', None)
import astropy.units as u
from astropy.coordinates import EarthLocation, SkyCoord
from astropy.time import Time
from astropy.timeseries import LombScargle
from astropy.utils import iers
from scipy.spatial import cKDTree
from scipy.optimize import minimize_scalar

_paths_6709 = CLUSTER_PATHS["ngc6709"]
REF_DIR = str(_paths_6709["rfilter"])
MASTER_MAP = str(_paths_6709["rfilter"] / "ref_stars_xy_radec.dat")
MASTER_REF = str(_paths_6709["rfilter"] / "refimage.ref")
JD_FILE = str(_paths_6709["rfilter"] / "JD_all.dat")
RESULT_DIR = str(_paths_6709["results"])
VARIABLE_CSV = str(_paths_6709["results"] / "cell1_variable_candidates.csv")
OUT_DIR = RESULT_DIR

USE_VARIABLE_CSV       = True
TARGET_LIMIT           = None
PLOT_DIR       = os.path.join(OUT_DIR, 'cell2_phase_plots')
OUT_SUMMARY    = os.path.join(OUT_DIR, 'cell2_period_phase_summary.csv')
OUT_RANKING    = os.path.join(OUT_DIR, 'cell2_phase_quality_ranking.csv')

TIME_BASIS     = TIME_BASIS_SETTING
JD_INPUT_SCALE = JD_INPUT_SCALE_SETTING
OBS_LAT_DEG, OBS_LON_DEG, OBS_HEIGHT_M = 29.3612, 79.6868, 2450.0

MATCH_TOL_PX, SEED_TOL_PX, MIN_SEEDS, MAX_SEEDS, RESID_TOL_PX = 1.2, 3.0, 40, 350, 1.6
MAX_ERR, SEED_ERR_MAX, ZP_CLIP_NSIGMA, ZP_CLIP_MAXITER = 0.3, 0.2, 3.5, 5

MIN_POINTS      = 35
CLIP_NSIGMA, CLIP_MAXITER = 4.5, 4
TIME_BIN_DAYS   = 0.0005
POSTBIN_CLIP_NSIGMA, POSTBIN_CLIP_MAXITER = 5.5, 2

PERIOD_MIN_D, PERIOD_MAX_D = 0.02, 15.0

LS_SEARCH_NTERMS       = (1,)
LS_SAMPLES_PER_PEAK    = 6
TOP_PEAKS_PER_NTERM    = 18
MID_SCAN_MIN_PERIOD_D, MID_SCAN_MAX_PERIOD_D = 0.06, 0.80
MID_SCAN_SAMPLES_PER_PEAK = 10
MID_SCAN_TOP_PEAKS_PER_NTERM = 14
FAMILY_MULTIPLIERS     = (0.25, 1.0 / 3.0, 0.5, 2.0 / 3.0, 1.0, 1.5, 2.0, 3.0)
DAILY_ALIAS_SHIFTS     = (-3, -2, -1, 1, 2, 3)
PEAK_DEDUP_FRAC        = 0.004
LOCAL_REFINE_RESOLUTION_ELEMENTS = 3.0
LOCAL_REFINE_GRID_SIZE = 401
FOLD_REFINE_TOP_N      = 2
FOLD_REFINE_GRID_SIZE  = 15
FOLD_REFINE_REL_WIDTH  = 0.015
PREFILTER_KEEP         = 44
NEUTRAL_GRID_VSHORT_N  = 320
NEUTRAL_GRID_SHORT_N   = 620
NEUTRAL_GRID_LONG_N    = 360
NEUTRAL_GRID_VLONG_N   = 360
NEUTRAL_GRID_LONG_MAX_D = 8.0
NEUTRAL_GRID_KEEP_TOP_N = 320
NEUTRAL_GRID_MIN_POWER = 0.10
MICRO_PERIOD_SOFT_FLOOR_D = 0.030
MICRO_PERIOD_SOFT_PENALTY = 1.20
LONG_WINDOW_ALIAS_PENALTY = 0.12

LONG_PERIOD_PREF          = 2.8

SHORT_PERIOD_PREF         = 0.55

AMP_DOUBLE_RATIO          = 1.35

LONG_ALIAS_MIN_PERIOD_D        = 0.35
LONG_ALIAS_VERY_LONG_D         = 1.00
LONG_ALIAS_MIN_PHASE_QUALITY   = 1.15
LONG_ALIAS_QUALITY_MARGIN      = 0.72
LONG_ALIAS_VLONG_MARGIN        = 1.25
LONG_ALIAS_MIN_DISPLAY_SINE    = 0.46
LONG_ALIAS_MAX_PLOT_GAP        = 0.52
LONG_ALIAS_MIN_AMP_FRAC        = 0.72
LONG_ALIAS_EXTRA_SCORE_TOP     = 5
LONG_ALIAS_MULTIPLIERS         = (1.5, 2.0, 2.5, 3.0, 4.0, 5.0, 6.0, 8.0, 10.0)
LONG_ALIAS_DAILY_ORDERS        = tuple(range(-12, 13))

ULTRASHORT_PROTECT_MAX_PERIOD_D      = 0.10
ULTRASHORT_PROTECT_MIN_PHASE_QUALITY = 2.70
ULTRASHORT_PROTECT_MIN_DISPLAY_SINE  = 0.82
ULTRASHORT_PROTECT_MIN_COVERAGE      = 0.70
ULTRASHORT_PROTECT_MAX_GAP           = 0.18
ULTRASHORT_PROTECT_SCORE_MARGIN      = 1.40
ULTRASHORT_PROTECT_QUALITY_MARGIN    = 0.70

REFERENCE_MODE_SELECTION = 'single_comp1'
ID_COLUMNS    = ('variable_star_id', 'star_id', 'id', 'Star_ID', 'Star_ID_NEW')
COMP1_COLUMNS = ('comp1_star_id', 'nonvariable_star_id', 'comp1', 'Comp1_NEW')
COMP2_COLUMNS = ('comp2_star_id', 'check_star_id', 'comp2', 'Comp2_NEW')

AUTO_COMPARISON_SELECTION = True
AUTO_DIAGNOSTIC_CSVS = (
    os.path.join(RESULT_DIR, 'cell0_all_star_diagnostics.csv'),
)
AUTO_COMP_MAX_CANDIDATES = 24
AUTO_SINGLE_COMP_TRIALS = 1
AUTO_PAIR_COMP_POOL = 12
AUTO_PAIR_TRIALS = 1

ENSEMBLE_COMP_SIZES = (4, 8)
AUTO_COMP_RMS_MAX = 0.080
AUTO_COMP_MAG_DIFF_MAX = 2.35
AUTO_COMP_DIST_MIN_PX, AUTO_COMP_DIST_MAX_PX = 35.0, 1250.0

SHARED_SIGNAL_GUARD = True
SHARED_SIGNAL_MIN_GROUP = 3
SHARED_SIGNAL_CORR_MIN = 0.62
SHARED_SIGNAL_FREQ_RESOLUTION = 1.5
SHARED_SIGNAL_MATCH_TOL_D = 1.5 * TIME_BIN_DAYS

NBINS_SCORE, MIN_BIN_PTS, MIN_BINS_USED = 60, 3, 6
GOOD_THETA      = 0.95
GAP_MAX         = 0.55

DIVERSITY_NBINS = 20
DIVERSITY_REF   = 2.0
DIVERSITY_FLOOR = 0.55

EB_THETA_MARGIN = 0.06
EB_SEC_NSIGMA, EB_SEC_FRAC   = 4.0, 0.15
EB_ASYM_NSIGMA, EB_ASYM_FRAC = 3.0, 0.18

PLOT_DPI               = 220

PHASE_BIN_TARGET_PTS   = 12
PHASE_MIN_BINS         = 20
PHASE_MAX_BINS         = 95
PHASE_BIN_MIN_PTS      = 4
PHASE_BIN_CLIP_NSIGMA  = 3.0

PHASE_OUTLIER_NSIGMA   = 3.3
PHASE_OUTLIER_KEEP_FRAC = 0.72
PHASE_BIN_WIDTH        = 0.0125
PHASE_DISPLAY_SMOOTH_HALF_WINDOW = 2
DOUBLE_CYCLE           = True

DISPLAY_PHASE_BIN_WIDTH        = 0.015

DISPLAY_LONG_PERIOD_MIN_D      = 1.00
DISPLAY_LONG_PHASE_BIN_WIDTH   = 0.005
DISPLAY_VLONG_PHASE_BIN_WIDTH  = 0.0035
DISPLAY_CONTINUITY_SMOOTH_BINS = 4
DISPLAY_CONTINUITY_ERR_WEIGHT  = 0.15

DISPLAY_HIDE_OUTLIERS          = True
DISPLAY_OUTLIER_Z              = 2.3
DISPLAY_OUTLIER_PASSES         = 2
DISPLAY_OUTLIER_MIN_MAG        = 0.012
DISPLAY_MAX_OUTLIER_FRAC       = 0.12

DISPLAY_GAP_AUGMENT_MIN_PERIOD_D = 0.50
DISPLAY_GAP_AUGMENT_TRIGGER      = 0.060
DISPLAY_GAP_AUGMENT_TARGET       = 0.045
DISPLAY_GAP_AUGMENT_MAX_POINTS   = 115
DISPLAY_GAP_AUGMENT_MAX_ADD_FRAC = 0.35
DISPLAY_GAP_AUGMENT_MIN_SEP      = 0.004
LOCAL_PGRAM_REL_WIDTH, LOCAL_PGRAM_MIN_WIDTH = 0.35, 0.12
LOCAL_PGRAM_SAMPLES_PER_PEAK = 90
LOCAL_PGRAM_MIN_POINTS = 600
LOCAL_PGRAM_MAX_POINTS = 6000
SAVE_PLOTS             = True
TARGET_STAR_ID_FILTER  = None

iers.conf.auto_download = False
iers.conf.auto_max_age = None
OBS_LOCATION = EarthLocation.from_geodetic(lon=OBS_LON_DEG * u.deg, lat=OBS_LAT_DEG * u.deg, height=OBS_HEIGHT_M * u.m)

def fast_median(x):
    """Low-overhead median for the small, already-NaN-free arrays in the hot per-bin loops.
    np.median's per-call overhead (nan-check, copy, full sort) dominated the runtime when it
    was called hundreds of thousands of times; np.partition is O(n) and gives the identical
    value. Falls back to np.nanmedian only if NaNs are present."""
    x = np.asarray(x, dtype=np.float64)
    n = x.size
    if n == 0:
        return np.nan
    if n == 1:
        return float(x[0])
    if not np.isfinite(x).all():
        return float(np.nanmedian(x))
    k = n // 2
    part = np.partition(x, k if n % 2 else [k - 1, k])
    return float(part[k]) if n % 2 else 0.5 * (float(part[k - 1]) + float(part[k]))

def mad_sigma(x):
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan
    m = fast_median(x)
    return 1.4826 * fast_median(np.abs(x - m))

def robust_sigma(x):
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan
    s = mad_sigma(x)
    if not np.isfinite(s) or s <= 0:
        s = np.std(x, ddof=1) if x.size > 1 else np.nan
    return float(s) if np.isfinite(s) and s > 0 else np.nan

def sigma_clip_mask(x, nsigma, maxiter):
    x = np.asarray(x, dtype=np.float64)
    mask = np.isfinite(x)
    if mask.sum() < 10:
        return mask
    for _ in range(int(maxiter)):
        med = fast_median(x[mask])
        sig = mad_sigma(x[mask] - med)
        if not np.isfinite(sig) or sig <= 0:
            break
        new = mask & (np.abs(x - med) <= nsigma * sig)
        if new.sum() == mask.sum():
            break
        mask = new
    return mask

def time_bin_weighted(jd, y, dy, bin_days):
    jd, y, dy = (np.asarray(a, dtype=np.float64) for a in (jd, y, dy))
    ok = np.isfinite(jd) & np.isfinite(y) & np.isfinite(dy) & (dy > 0)
    jd, y, dy = jd[ok], y[ok], dy[ok]
    if jd.size == 0:
        return np.array([]), np.array([]), np.array([])
    if not bin_days or bin_days <= 0:
        o = np.argsort(jd)
        return jd[o], y[o], dy[o]
    idx = np.floor((jd - jd.min()) / bin_days).astype(np.int64)
    jb, yb, eb = [], [], []
    for b in np.unique(idx):
        m = idx == b
        w = 1.0 / np.maximum(dy[m], 1e-12) ** 2
        sw = w.sum()
        if not np.isfinite(sw) or sw <= 0:
            continue
        jb.append(float(np.mean(jd[m])))
        yb.append(float(np.sum(w * y[m]) / sw))
        eb.append(float(np.sqrt(1.0 / sw)))
    jb, yb, eb = (np.asarray(a, dtype=np.float64) for a in (jb, yb, eb))
    o = np.argsort(jb)
    return jb[o], yb[o], eb[o]

def typical_night_span(jd):
    jd = np.asarray(jd, dtype=np.float64)
    nights = np.floor(jd - 0.5).astype(np.int64)
    spans = [float(np.ptp(jd[nights == n])) for n in np.unique(nights) if (nights == n).sum() >= 3]
    spans = [s for s in spans if s > 0]
    return float(fast_median(spans)) if spans else 0.05

def read_jd_map(jd_path):
    by_name, by_stem = {}, {}
    with open(jd_path) as f:
        for line in f:
            s = line.split()
            if len(s) < 2:
                continue
            try:
                name, jd = os.path.basename(s[0]), float(s[1])
            except ValueError:
                try:
                    name, jd = os.path.basename(s[1]), float(s[0])
                except (ValueError, IndexError):
                    continue
            by_name[name] = jd
            by_stem[os.path.splitext(name)[0]] = jd
    return by_name, by_stem

def jd_for_ref(ref_path, by_name, by_stem):
    base = os.path.basename(ref_path)
    stem = os.path.splitext(base)[0]
    if base in by_name:
        return by_name[base]
    if stem in by_stem:
        return by_stem[stem]
    return by_name.get(stem + '.fits', np.nan)

def read_ref(ref_path):
    """DAOPHOT .ref: 3 header rows, then id x y mag err ... -> (x, y, mag, err)."""
    try:
        arr = np.loadtxt(ref_path, skiprows=3, usecols=(1, 2, 3, 4), dtype=np.float64)
    except Exception:
        return None
    if arr.size == 0:
        return None
    if arr.ndim == 1:
        arr = arr.reshape(1, -1)
    if arr.shape[1] != 4:
        return None
    x, y, mag, err = arr.T
    ok = np.isfinite(x) & np.isfinite(y) & np.isfinite(mag) & np.isfinite(err)
    return (x[ok], y[ok], mag[ok], err[ok]) if ok.any() else None

def load_master_catalog(path):
    df = pd.read_csv(path, sep=r'\s+', header=None, names=['star_id', 'x', 'y', 'ra', 'dec'], engine='python')
    df['star_id'] = df['star_id'].astype(int)
    df['x'] = df['x'].astype(float)
    df['y'] = df['y'].astype(float)
    df['ra'] = df['ra'].astype(str)
    df['dec'] = df['dec'].astype(str)
    return df.sort_values('star_id').reset_index(drop=True)

def parse_coord(ra, dec):
    for units in ((u.hourangle, u.deg), (u.deg, u.deg)):
        try:
            return SkyCoord(str(ra).strip(), str(dec).strip(), unit=units, frame='icrs')
        except Exception:
            continue
    return None

def to_hjd(jd, coord):
    jd = np.asarray(jd, dtype=np.float64)
    if TIME_BASIS == 'jd' or coord is None:
        return jd.copy()
    t = Time(jd, format='jd', scale=JD_INPUT_SCALE, location=OBS_LOCATION)
    return (t.utc + t.light_travel_time(coord, kind='heliocentric')).jd

def solve_affine(src, dst):
    if src.shape[0] < 3:
        return None
    M = np.column_stack([src[:, 0], src[:, 1], np.ones(src.shape[0])])
    try:
        cx = np.linalg.lstsq(M, dst[:, 0], rcond=None)[0]
        cy = np.linalg.lstsq(M, dst[:, 1], rcond=None)[0]
    except Exception:
        return None
    return np.vstack([cx, cy])

def apply_affine(A, xy):
    M = np.column_stack([xy[:, 0], xy[:, 1], np.ones(xy.shape[0])])
    return np.column_stack([M @ A[0], M @ A[1]])

def fit_affine(seed_xy, frame_xy, tree):
    if seed_xy.shape[0] < MIN_SEEDS or frame_xy.shape[0] < MIN_SEEDS:
        return None
    d, j = tree.query(seed_xy, k=1, workers=-1)
    ok = np.isfinite(d) & (d <= SEED_TOL_PX)
    if ok.sum() < MIN_SEEDS:
        return None
    A = solve_affine(seed_xy[ok], frame_xy[j[ok]])
    for _ in range(2):
        if A is None:
            return None
        d1, j1 = tree.query(apply_affine(A, seed_xy), k=1, workers=-1)
        ok1 = np.isfinite(d1) & (d1 <= RESID_TOL_PX)
        if ok1.sum() < MIN_SEEDS:
            break
        A = solve_affine(seed_xy[ok1], frame_xy[j1[ok1]])
    return A

def unique_matches(master_xy, frame_xy, tree, A):
    d, idx = tree.query(apply_affine(A, master_xy), k=1, workers=-1)
    ok = np.isfinite(d) & (d <= MATCH_TOL_PX)
    if not ok.any():
        return np.array([], dtype=int), np.array([], dtype=int)
    m_idx, f_idx, dd = np.where(ok)[0], idx[ok], d[ok]
    used, keep = set(), []
    for oi in np.argsort(dd):
        fj = int(f_idx[oi])
        if fj in used:
            continue
        used.add(fj)
        keep.append(oi)
    keep = np.asarray(keep, dtype=int)
    return m_idx[keep], f_idx[keep].astype(int)

def build_photometry(master_df, star_ids):
    """Ensemble zero-pointed magnitude matrix for the requested star ids."""
    master_xy = master_df[['x', 'y']].to_numpy(np.float64)
    id_to_idx = {int(s): i for i, s in enumerate(master_df['star_id'])}

    ref = read_ref(MASTER_REF)
    if ref is None:
        raise RuntimeError('could not read refimage.ref')
    rx, ry, rmag, rerr = ref
    tree_ref = cKDTree(np.column_stack([rx, ry]))
    d, j = tree_ref.query(master_xy, k=1, workers=-1)
    seen = np.isfinite(d) & (d <= 1.0)
    mag_seed = np.where(seen, rmag[j], np.nan)
    err_seed = np.where(seen, rerr[j], np.nan)
    seed_idx = np.where(np.isfinite(mag_seed) & np.isfinite(err_seed) & (err_seed <= SEED_ERR_MAX))[0]
    if seed_idx.size > MAX_SEEDS:
        seed_idx = seed_idx[np.argsort(mag_seed[seed_idx])[:MAX_SEEDS]]
    if seed_idx.size < MIN_SEEDS:
        raise RuntimeError('not enough zero-point seed stars')
    seed_xy = master_xy[seed_idx]

    need_ids = [int(s) for s in star_ids if int(s) in id_to_idx]
    missing = sorted(set(int(s) for s in star_ids) - set(need_ids))
    need_master_idx = np.array([id_to_idx[s] for s in need_ids], dtype=int)
    need_xy = master_xy[need_master_idx]
    star_to_col = {s: i for i, s in enumerate(need_ids)}

    by_name, by_stem = read_jd_map(JD_FILE)
    ref_files, jds = [], []
    for p in sorted(glob.glob(os.path.join(REF_DIR, '*.ref'))):
        if os.path.basename(p) == os.path.basename(MASTER_REF):
            continue
        jd = jd_for_ref(p, by_name, by_stem)
        if np.isfinite(jd):
            ref_files.append(p)
            jds.append(float(jd))
    n_f, n_s = len(ref_files), len(need_ids)
    corr = np.full((n_f, n_s), np.nan, dtype=np.float32)
    emat = np.full((n_f, n_s), np.nan, dtype=np.float32)
    print(f'  stars needed: {n_s} (missing from map: {len(missing)}) | frames: {n_f} | seeds: {seed_idx.size}', flush=True)
    n_ok = 0
    for fi, (p, jd) in enumerate(zip(ref_files, jds)):
        data = read_ref(p)
        if data is None:
            continue
        fx, fy, fmag, ferr = data
        frame_xy = np.column_stack([fx, fy])
        tree = cKDTree(frame_xy)
        A = fit_affine(seed_xy, frame_xy, tree)
        if A is None:
            continue
        sm, sf = unique_matches(seed_xy, frame_xy, tree, A)
        if sm.size == 0:
            continue
        smg = seed_idx[sm]
        good = (np.isfinite(fmag[sf]) & (ferr[sf] <= MAX_ERR) & np.isfinite(mag_seed[smg]) & (err_seed[smg] <= SEED_ERR_MAX))
        zpv = fmag[sf[good]] - mag_seed[smg[good]]
        keep = sigma_clip_mask(zpv, ZP_CLIP_NSIGMA, ZP_CLIP_MAXITER) if zpv.size else np.array([], dtype=bool)
        if keep.sum() < 10:
            continue
        zp = float(fast_median(zpv[keep]))
        nm, nf = unique_matches(need_xy, frame_xy, tree, A)
        if nm.size:
            g = np.isfinite(fmag[nf]) & (ferr[nf] <= MAX_ERR)
            corr[fi, nm[g]] = (fmag[nf[g]] - zp).astype(np.float32)
            emat[fi, nm[g]] = ferr[nf[g]].astype(np.float32)
        n_ok += 1
        if (fi + 1) % 800 == 0 or fi + 1 == n_f:
            print(f'    processed {fi + 1}/{n_f} frames ({n_ok} usable)', flush=True)
    return {'jd': np.asarray(jds, dtype=np.float64), 'corr': corr, 'emat': emat,
            'star_to_col': star_to_col, 'missing': missing}

def phase_fold(jd, period, t0):
    return ((jd - t0) / period) % 1.0

def phase_gap_metric(jd, period):
    jd = np.asarray(jd, dtype=np.float64)
    jd = jd[np.isfinite(jd)]
    if jd.size < 3 or not np.isfinite(period) or period <= 0:
        return np.inf
    ph = np.sort(phase_fold(jd, period, jd.min()))
    diffs = np.diff(ph)
    return float(max(diffs.max(), ph[0] + 1.0 - ph[-1]))

def day_alias_distance(period):
    if not np.isfinite(period) or period <= 0:
        return np.inf
    f = 1.0 / period
    k = np.arange(1, 9, dtype=np.float64)
    targets = np.unique(np.concatenate([k, 1.0 / k]))
    return float(np.min(np.abs(f - targets)))

def fold_night_diversity(jd, period, nbins=DIVERSITY_NBINS):
    """Median number of DISTINCT nights among occupied phase bins. High for the true
    (sub-day) period, ~1 for a long window alias where one night fills each bin."""
    jd = np.asarray(jd, dtype=np.float64)
    jd = jd[np.isfinite(jd)]
    if jd.size < 6 or not np.isfinite(period) or period <= 0:
        return 1.0
    nights = np.floor(jd - 0.5).astype(np.int64)
    ph = phase_fold(jd, period, jd.min())
    inds = np.clip((ph * int(nbins)).astype(int), 0, int(nbins) - 1)
    counts = [np.unique(nights[inds == b]).size for b in range(int(nbins)) if np.any(inds == b)]
    return float(fast_median(counts)) if counts else 1.0

def diversity_factor(nd):
    if not np.isfinite(nd):
        return 1.0
    frac = float(np.clip((nd - 1.0) / max(DIVERSITY_REF - 1.0, 1e-6), 0.0, 1.0))
    return float(DIVERSITY_FLOOR + (1.0 - DIVERSITY_FLOOR) * frac)

def fast_fold_beauty(jd, y, period, nbins=30):
    'Compute a binned-shape proxy used to shortlist period candidates before full scoring.'
    jd = np.asarray(jd, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    if jd.size < 10 or not np.isfinite(period) or period <= 0:
        return 0.0
    ph = ((jd - jd.min()) / period) % 1.0
    inds = np.clip((ph * nbins).astype(np.int64), 0, nbins - 1)
    cnts = np.bincount(inds, minlength=nbins).astype(np.float64)
    sums = np.bincount(inds, weights=y, minlength=nbins)
    occ = cnts >= 2
    if int(occ.sum()) < int(MIN_BINS_USED):
        return 0.0
    means = (sums / np.maximum(cnts, 1.0))[occ]
    centers = ((np.arange(nbins) + 0.5) / nbins)[occ]
    m = means - np.mean(means)
    amp = float(np.quantile(m, 0.9) - np.quantile(m, 0.1))
    if not np.isfinite(amp) or amp <= 0:
        return 0.0
    powers = []
    for k in (1.0, 2.0, 3.0):
        ang = 2.0 * np.pi * k * centers
        c = float(np.mean(m * np.cos(ang))); s = float(np.mean(m * np.sin(ang)))
        powers.append(c * c + s * s)
    sine_frac = powers[0] / max(float(np.sum(powers)), 1e-12)
    sem = float(np.std(y)) / np.sqrt(max(float(np.median(cnts[occ])), 1.0))
    snr = amp / max(sem, 1e-8)
    return float(sine_frac * float(np.clip(snr / 4.0, 0.0, 1.6)))

def phase_profile(jd, y, dy, period, nbins=NBINS_SCORE, min_pts=MIN_BIN_PTS):
    """theta = median within-bin scatter / raw scatter (phase dispersion). Harmonic-safe."""
    jd, y, dy = (np.asarray(a, dtype=np.float64) for a in (jd, y, dy))
    ok = np.isfinite(jd) & np.isfinite(y) & np.isfinite(dy) & (dy > 0)
    jd, y, dy = jd[ok], y[ok], dy[ok]
    out = {'theta': np.inf, 'amp': np.nan, 'bins_used': 0, 'phase': np.array([])}
    if jd.size < 10 or not np.isfinite(period) or period <= 0:
        return out
    raw = robust_sigma(y)
    if not np.isfinite(raw) or raw <= 0:
        return out
    ph = phase_fold(jd, period, jd.min())
    out['phase'] = ph
    edges = np.linspace(0.0, 1.0, nbins + 1)
    inds = np.clip(np.digitize(ph, edges) - 1, 0, nbins - 1)
    sigs, meds = [], []
    for b in range(nbins):
        m = inds == b
        if m.sum() < min_pts:
            continue
        s = mad_sigma(y[m] - fast_median(y[m]))
        if np.isfinite(s):
            sigs.append(s)
            meds.append(fast_median(y[m]))
    if len(sigs) < MIN_BINS_USED:
        return out
    out['theta'] = float(fast_median(sigs) / max(raw, 1e-8))
    out['amp'] = float(np.quantile(meds, 0.9) - np.quantile(meds, 0.1)) if len(meds) >= 5 else float(np.ptp(meds))
    out['bins_used'] = len(sigs)
    return out

def sine_shape_metrics(jd, y, dy, period, width=PHASE_BIN_WIDTH):
    """Unsupervised first-harmonic coherence + roughness of the folded, binned curve."""
    jd, y, dy = (np.asarray(a, dtype=np.float64) for a in (jd, y, dy))
    ok = np.isfinite(jd) & np.isfinite(y) & np.isfinite(dy) & (dy > 0)
    if ok.sum() < 10 or not np.isfinite(period) or period <= 0:
        return {'sine_coherence': 0.0, 'curve_roughness': np.inf}
    phase = phase_fold(jd[ok], period, 0.0)
    rel = -y[ok]
    edges = phase_bin_edges(width)
    _, med, cnt = bin_medians(phase, rel, edges)
    med = circular_rolling_median_observed(med, 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    good = np.isfinite(med) & (cnt > 0)
    if good.sum() < 8:
        return {'sine_coherence': 0.0, 'curve_roughness': np.inf}
    ph = centers[good]
    yy = med[good] - np.nanmedian(med[good])
    amp = float(np.nanquantile(yy, 0.9) - np.nanquantile(yy, 0.1))
    if not np.isfinite(amp) or amp <= 0:
        return {'sine_coherence': 0.0, 'curve_roughness': np.inf}
    w = np.sqrt(np.maximum(cnt[good].astype(float), 1.0))
    powers = []
    for k in range(1, 5):
        ang = 2.0 * np.pi * float(k) * ph
        c = np.sum(w * yy * np.cos(ang)) / max(np.sum(w), 1e-12)
        s = np.sum(w * yy * np.sin(ang)) / max(np.sum(w), 1e-12)
        powers.append(float(c * c + s * s))
    total = float(np.sum(powers))
    sine_coh = float(powers[0] / max(total, 1e-12)) if total > 0 else 0.0
    order = np.argsort(ph)
    yy_ord = yy[order] / max(amp, 1e-8)
    second = np.roll(yy_ord, -1) - 2.0 * yy_ord + np.roll(yy_ord, 1)
    rough = float(np.nanpercentile(np.abs(second), 75)) if second.size else np.inf
    return {'sine_coherence': float(np.clip(sine_coh, 0.0, 1.0)), 'curve_roughness': rough}

def ls_peaks(jd, y, dy, pmin, pmax, samples_per_peak=None, topn_per_nterm=None,
             nterms_values=None, source_label='ls_peak'):
    """Independent weighted-LS peak seeds. LS proposes periods; fold quality chooses."""
    jd, y, dy = (np.asarray(a, dtype=np.float64) for a in (jd, y, dy))
    ok = np.isfinite(jd) & np.isfinite(y) & np.isfinite(dy) & (dy > 0)
    jd, y, dy = jd[ok], y[ok], dy[ok]
    if jd.size < 10:
        return [], {}, {}, 0.0, 1.0 / pmax, 1.0 / pmin
    fmin, fmax = 1.0 / float(pmax), 1.0 / float(pmin)
    baseline = float(np.ptp(jd))
    if not np.isfinite(baseline) or baseline <= 0:
        return [], {}, {}, 0.0, fmin, fmax
    candidates, ls_map, best_power = [], {}, {}
    if nterms_values is None:
        nterms_values = LS_SEARCH_NTERMS
    for nterms in nterms_values:
        nterms = int(nterms)
        try:
            ls = LombScargle(jd, y, dy=dy, center_data=True, nterms=nterms)
            freq, power = ls.autopower(minimum_frequency=fmin, maximum_frequency=fmax,
                                       samples_per_peak=int(samples_per_peak or LS_SAMPLES_PER_PEAK))
        except Exception:
            continue
        if power.size == 0 or not np.isfinite(power).any():
            continue
        ls_map[nterms] = ls
        best_power[nterms] = float(np.nanmax(power))
        kept, rank = [], 0
        for i in np.argsort(power)[::-1]:
            if not np.isfinite(freq[i]) or freq[i] <= 0 or not np.isfinite(power[i]):
                continue
            P = 1.0 / float(freq[i])
            if not (pmin <= P <= pmax):
                continue
            if any(abs(P - q) / q <= PEAK_DEDUP_FRAC for q in kept):
                continue
            rank += 1
            kept.append(P)
            candidates.append({'base_period': float(P), 'base_power': float(power[i]),
                               'base_rank': int(rank), 'search_nterms': int(nterms),
                               'candidate_source': str(source_label),
                               'harmonic_multiplier': 1.0})
            if len(kept) >= int(topn_per_nterm or TOP_PEAKS_PER_NTERM):
                break
    return candidates, ls_map, best_power, baseline, fmin, fmax

def refine_period(ls, period, fmin, fmax, baseline):
    if ls is None or not np.isfinite(period) or period <= 0:
        return float(period), np.nan
    f0 = 1.0 / float(period)
    if baseline is None or not np.isfinite(baseline) or baseline <= 0:
        half_width = 0.003 * f0
    else:
        half_width = max(float(LOCAL_REFINE_RESOLUTION_ELEMENTS) / max(float(baseline), 1e-8), 0.004 * f0)
    grid = np.linspace(max(float(fmin), f0 - half_width), min(float(fmax), f0 + half_width),
                       int(LOCAL_REFINE_GRID_SIZE))
    if grid.size == 0:
        try:
            return float(period), float(ls.power(f0))
        except Exception:
            return float(period), np.nan
    try:
        p = np.asarray(ls.power(grid), dtype=np.float64)
    except Exception:
        p = np.array([], dtype=np.float64)
    if p.size == 0 or not np.isfinite(p).any():
        return float(period), np.nan
    best_freq = float(grid[int(np.nanargmax(p))])
    return float(1.0 / best_freq), float(np.nanmax(p))

def dedup_candidate_rows(rows, rel_tol=None):
    if rel_tol is None:
        rel_tol = PEAK_DEDUP_FRAC
    log_step = np.log1p(max(float(rel_tol), 1e-8))

    def rank_key(r):
        score = float(r.get('fold_score', np.nan))
        quality = float(r.get('phase_quality', np.nan))
        theta = float(r.get('theta_raw', np.inf))
        base_power = float(r.get('base_power', np.nan))
        has_score = np.isfinite(score)
        has_quality = np.isfinite(quality)
        return (0 if has_score else 1, -score if has_score else 0.0,
                -quality if has_quality else 0.0,
                -base_power if np.isfinite(base_power) else 0.0, theta,
                float(r.get('base_rank', 999)), float(r['period']))

    out = []
    seen_bins = {}
    for row in sorted(rows, key=rank_key):
        P = float(row['period'])
        if not np.isfinite(P) or P <= 0:
            continue
        key = int(np.floor(np.log(P) / log_step))
        duplicate = False
        for kk in (key - 1, key, key + 1):
            for oldP in seen_bins.get(kk, ()):
                if abs(P - float(oldP)) / float(oldP) <= float(rel_tol):
                    duplicate = True
                    break
            if duplicate:
                break
        if duplicate:
            continue
        out.append(row)
        seen_bins.setdefault(key, []).append(P)
    return out

def build_period_candidates(jd, y, dy):
    base, ls_map, best_power, baseline, fmin, fmax = ls_peaks(
        jd, y, dy, PERIOD_MIN_D, PERIOD_MAX_D, source_label='broad_ls_peak')
    mid_base, mid_ls_map, mid_best_power, mid_baseline, _, _ = ls_peaks(
        jd, y, dy, MID_SCAN_MIN_PERIOD_D, MID_SCAN_MAX_PERIOD_D,
        samples_per_peak=MID_SCAN_SAMPLES_PER_PEAK,
        topn_per_nterm=MID_SCAN_TOP_PEAKS_PER_NTERM,
        nterms_values=LS_SEARCH_NTERMS,
        source_label='subday_ls_peak')
    if mid_base:
        base = list(base) + list(mid_base)
    if not ls_map and mid_ls_map:
        ls_map = mid_ls_map
        best_power = mid_best_power
    if (not np.isfinite(baseline) or baseline <= 0) and np.isfinite(mid_baseline):
        baseline = mid_baseline
    if not base or not ls_map:
        return [], ls_map, best_power, baseline
    rows = []
    for seed in base:
        nterms = int(seed.get('search_nterms', 1))
        ls = ls_map.get(nterms)
        refined_period, refined_power = refine_period(ls, seed['base_period'], fmin, fmax, baseline)
        if not np.isfinite(refined_period) or refined_period <= 0:
            refined_period = float(seed['base_period'])
        base_freqs = [(1.0 / refined_period, 0, 'ls_refined')]
        for shift in DAILY_ALIAS_SHIFTS:
            fa = (1.0 / refined_period) + float(shift)
            if fa > 0:
                base_freqs.append((fa, int(shift), 'daily_alias'))
        for freq, alias_shift, source in base_freqs:
            alias_period = 1.0 / float(freq)
            for mult in FAMILY_MULTIPLIERS:
                P = alias_period * float(mult)
                if not (PERIOD_MIN_D <= P <= PERIOD_MAX_D):
                    continue
                rows.append({'period': float(P), 'base_period': float(refined_period),
                             'base_power': float(refined_power) if np.isfinite(refined_power) else float(seed.get('base_power', np.nan)),
                             'base_rank': int(seed.get('base_rank', 999)),
                             'search_nterms': nterms, 'candidate_source': source,
                             'alias_shift': int(alias_shift), 'harmonic_multiplier': float(mult)})

    neutral_periods = np.concatenate([
        np.linspace(max(PERIOD_MIN_D, 0.030), 0.10, int(NEUTRAL_GRID_VSHORT_N)),
        np.linspace(0.10, 0.35, int(NEUTRAL_GRID_SHORT_N)),
        np.linspace(0.35, 1.20, int(NEUTRAL_GRID_LONG_N)),
        np.linspace(1.20, min(PERIOD_MAX_D, NEUTRAL_GRID_LONG_MAX_D), int(NEUTRAL_GRID_VLONG_N)),
    ])
    neutral_scored = []
    for P in neutral_periods:
        P = float(P)
        if not (PERIOD_MIN_D <= P <= PERIOD_MAX_D):
            continue
        pnorm = ls_power_norm_at(P, ls_map, best_power)
        if pnorm >= float(NEUTRAL_GRID_MIN_POWER):
            neutral_scored.append((float(pnorm), P))
    neutral_scored.sort(reverse=True)
    for rank, (pnorm, P) in enumerate(neutral_scored[:int(NEUTRAL_GRID_KEEP_TOP_N)], start=1):
        rows.append({'period': float(P), 'base_period': float(P), 'base_power': float(pnorm),
                     'base_rank': 900 + int(rank), 'search_nterms': 1,
                     'candidate_source': 'neutral_power_grid',
                     'alias_shift': 0, 'harmonic_multiplier': 1.0})
    return dedup_candidate_rows(rows), ls_map, best_power, baseline

def ls_power_norm_at(period, ls_map, best_power):
    if not ls_map or not np.isfinite(period) or period <= 0:
        return 0.0
    freq = 1.0 / float(period)
    vals = []
    for nterms, ls in ls_map.items():
        denom = float(best_power.get(int(nterms), np.nan)) if isinstance(best_power, dict) else np.nan
        if not np.isfinite(denom) or denom <= 0:
            continue
        try:
            pwr = float(ls.power(freq))
        except Exception:
            continue
        if np.isfinite(pwr):
            vals.append(pwr / max(denom, 1e-12))
    return float(np.clip(max(vals), 0.0, 2.0)) if vals else 0.0

def displayed_fold_metrics(jd, y, dy, period, width=PHASE_BIN_WIDTH):
    'Compute metrics from the binned observed-data fold used for ranking. Saved phase figures can use a different subset of observations.'
    jj, yy0, dd0 = (np.asarray(a, dtype=np.float64) for a in (jd, y, dy))
    okv = np.isfinite(jj) & np.isfinite(yy0) & np.isfinite(dd0) & (dd0 > 0)
    n_valid = int(okv.sum())

    if n_valid > 0 and np.isfinite(period) and period > 0:
        ph_e = np.mod(phase_fold(jj[okv], period, 0.0), 1.0)
        expected_bins = max(choose_plot_nbins(ph_e, n_valid), 1)
    else:
        expected_bins = max(int(PHASE_MIN_BINS), 1)
    try:
        px, py, pe = binned_phase_points(jd, y, dy, period, width=width)
    except NameError:
        return {'plot_points': 0, 'plot_coverage': 0.0, 'plot_gap': 1.0, 'plot_jump': np.inf,
                'display_sine': 0.0, 'display_roughness': np.inf,
                'display_spikiness': np.inf, 'display_maxjump': np.inf,
                'display_snr': 0.0}
    px, py, pe = (np.asarray(a, dtype=np.float64) for a in (px, py, pe))
    ok = np.isfinite(px) & np.isfinite(py)
    if pe.shape == py.shape:
        pe = pe[ok]
    else:
        pe = np.full(int(ok.sum()), np.nan, dtype=np.float64)
    px, py = px[ok], py[ok]
    if px.size < 4:
        return {'plot_points': int(px.size), 'plot_coverage': float(px.size / expected_bins),
                'plot_gap': 1.0, 'plot_jump': np.inf,
                'display_sine': 0.0, 'display_roughness': np.inf,
                'display_spikiness': np.inf, 'display_maxjump': np.inf,
                'display_snr': 0.0}
    o = np.argsort(px)
    px, py = px[o], py[o]
    pe = pe[o] if pe.size == px.size else pe
    gaps = np.diff(px)
    plot_gap = max(float(np.max(gaps)) if gaps.size else 1.0, float(px[0] + 1.0 - px[-1]))
    y0 = py - np.nanmedian(py)
    amp = float(np.nanquantile(y0, 0.90) - np.nanquantile(y0, 0.10))
    if not np.isfinite(amp) or amp <= 0:
        amp = 1e-6
    smooth = circular_rolling_median_observed(y0, int(PHASE_DISPLAY_SMOOTH_HALF_WINDOW)) if px.size >= 6 else y0
    good = np.isfinite(smooth)
    if good.sum() >= 4:
        yy = smooth[good]
        ph = px[good]
    else:
        yy, ph = y0, px
    jumps = np.abs(np.diff(np.concatenate([yy, [yy[0]]]))) if yy.size else np.array([], dtype=np.float64)
    plot_jump = float(np.nanmedian(jumps) / max(amp, 1e-8)) if jumps.size else np.inf
    jump_p95 = float(np.nanpercentile(jumps, 95)) if jumps.size else np.inf
    jump_max = float(np.nanmax(jumps)) if jumps.size else np.inf
    jump_med = float(np.nanmedian(jumps)) if jumps.size else np.inf
    display_spike = float(jump_p95 / max(jump_med, 1e-8)) if np.isfinite(jump_p95) and np.isfinite(jump_med) else np.inf
    display_maxjump = float(jump_max / max(amp, 1e-8)) if np.isfinite(jump_max) else np.inf
    second = np.roll(yy, -1) - 2.0 * yy + np.roll(yy, 1) if yy.size >= 4 else np.array([], dtype=np.float64)
    display_rough = float(np.nanpercentile(np.abs(second), 75) / max(amp, 1e-8)) if second.size else np.inf
    powers = []
    for k in range(1, 5):
        ang = 2.0 * np.pi * float(k) * ph
        c = np.nanmean(yy * np.cos(ang))
        s = np.nanmean(yy * np.sin(ang))
        powers.append(float(c * c + s * s) if np.isfinite(c) and np.isfinite(s) else 0.0)
    total = float(np.sum(powers))
    display_sine = float(powers[0] / max(total, 1e-12)) if total > 0 else 0.0
    egood = pe[np.isfinite(pe) & (pe > 0)] if pe.size else np.array([], dtype=np.float64)
    display_snr = float(amp / max(float(np.nanmedian(egood)), 1e-8)) if egood.size else 0.0
    return {'plot_points': int(px.size), 'plot_coverage': float(np.clip(px.size / expected_bins, 0.0, 1.0)),
            'plot_gap': float(plot_gap), 'plot_jump': float(plot_jump),
            'display_sine': float(np.clip(display_sine, 0.0, 1.0)),
            'display_roughness': float(display_rough),
            'display_spikiness': float(display_spike),
            'display_maxjump': float(display_maxjump), 'display_snr': float(display_snr)}

def phase_quality_score(disp):
    'Combine the binned-fold morphology and sampling diagnostics used in candidate ranking.'
    coverage = float(np.clip(float(disp.get('plot_coverage', 0.0)), 0.0, 1.0))
    gap = float(disp.get('plot_gap', 1.0))
    jump = float(disp.get('plot_jump', np.inf))
    sine = float(np.clip(float(disp.get('display_sine', 0.0)), 0.0, 1.0))
    rough = float(disp.get('display_roughness', np.inf))
    spike = float(disp.get('display_spikiness', np.inf))
    maxjump = float(disp.get('display_maxjump', np.inf))
    snr = float(np.clip(float(disp.get('display_snr', 0.0)) / 4.0, 0.0, 1.5))
    gap_reward = 1.0 - float(np.clip(gap, 0.0, 1.0))
    jump_reward = 1.0 / (1.0 + 4.0 * max(jump, 0.0)) if np.isfinite(jump) else 0.0
    rough_pen = min(max(rough, 0.0), 2.0) if np.isfinite(rough) else 2.0
    spike_pen = min(max((spike - 2.7) / 1.25, 0.0), 3.0) if np.isfinite(spike) else 3.0
    maxjump_pen = min(max((maxjump - 0.16) / 0.14, 0.0), 3.0) if np.isfinite(maxjump) else 3.0

    cov_pen = float(np.clip((0.42 - coverage) / 0.20, 0.0, 1.0))
    big_gap_pen = float(np.clip((gap - 0.24) / 0.14, 0.0, 1.0))

    sig = float(np.clip((float(disp.get('display_snr', 0.0)) - 0.8) / 2.4, 0.0, 1.0))
    sine_eff = sine * (0.62 + 0.38 * sig)
    return float(2.05 * sine_eff + 1.10 * jump_reward + 0.90 * gap_reward
                 + 0.50 * coverage + 0.22 * snr - 0.65 * rough_pen
                 - 0.62 * spike_pen - 0.95 * maxjump_pen
                 - 0.70 * cov_pen - 0.60 * big_gap_pen)

def score_period(jd, y, dy, period, ls_map=None, best_power=None, baseline=None, candidate=None):
    'Score period candidates using folded morphology, sampling coverage, Lomb-Scargle support and the configured period-dependent preferences.'
    prof = phase_profile(jd, y, dy, period)
    if not np.isfinite(prof['theta']):
        return None
    raw_sigma = robust_sigma(y)
    gap = phase_gap_metric(jd, period)
    if np.isfinite(gap) and gap > GAP_MAX:
        return None
    shape = sine_shape_metrics(jd, y, dy, period)
    disp = displayed_fold_metrics(jd, y, dy, period)
    amp = float(prof['amp']) if np.isfinite(prof['amp']) else np.nan
    amp_ratio = float(amp / max(raw_sigma, 1e-8)) if np.isfinite(amp) and np.isfinite(raw_sigma) and raw_sigma > 0 else 0.0
    alias_dist = day_alias_distance(period)
    alias_pen = 0.35 if (np.isfinite(alias_dist) and alias_dist < 0.012) else (0.12 if (np.isfinite(alias_dist) and alias_dist < 0.025) else 0.0)
    cycles = float(baseline / period) if baseline is not None and np.isfinite(baseline) and period > 0 else np.nan
    cycle_pen = 0.18 if (np.isfinite(cycles) and cycles < 1.5) else (0.08 if (np.isfinite(cycles) and cycles < 2.0) else 0.0)
    mult = float(candidate.get('harmonic_multiplier', 1.0)) if isinstance(candidate, dict) else 1.0
    mult_pen = 0.02 * abs(np.log2(max(mult, 1e-8)))
    power_norm = ls_power_norm_at(period, ls_map or {}, best_power or {})
    theta_raw = float(prof['theta'])
    sine_coh = float(shape.get('sine_coherence', 0.0))
    rough = float(shape.get('curve_roughness', np.inf))
    nd = fold_night_diversity(jd, period)
    dfac = diversity_factor(nd)
    plot_gap = float(disp.get('plot_gap', 1.0))
    plot_jump = float(disp.get('plot_jump', np.inf))
    display_sine = float(disp.get('display_sine', 0.0))
    display_rough = float(disp.get('display_roughness', np.inf))
    display_spike = float(disp.get('display_spikiness', np.inf))
    display_maxjump = float(disp.get('display_maxjump', np.inf))
    display_snr = float(disp.get('display_snr', 0.0))
    plot_coverage = float(disp.get('plot_coverage', 0.0))
    phase_quality = phase_quality_score(disp)
    micro_pen = 0.0
    if float(period) < float(MICRO_PERIOD_SOFT_FLOOR_D):
        span = max(float(MICRO_PERIOD_SOFT_FLOOR_D) - float(PERIOD_MIN_D), 1e-8)
        frac = float(np.clip((float(MICRO_PERIOD_SOFT_FLOOR_D) - float(period)) / span, 0.0, 1.0))
        micro_pen = float(MICRO_PERIOD_SOFT_PENALTY) * frac * (1.10 - 0.45 * display_sine)
    long_alias_pen = 0.0
    if float(period) >= 1.0 and nd < 1.35:
        long_alias_pen = float(LONG_WINDOW_ALIAS_PENALTY) * float(np.clip((float(period) - 1.0) / 2.0, 0.0, 1.0))

    div_weight = 0.82 + 0.18 * dfac
    fold_reward = (3.05 * phase_quality + 0.35 * sine_coh) * div_weight

    good = float(np.clip(display_sine, 0.0, 1.0)) * float(np.clip(plot_coverage, 0.0, 1.0))
    long_pref = 0.0
    if float(period) >= 0.9 and float(LONG_PERIOD_PREF) > 0:
        long_pref = float(LONG_PERIOD_PREF) * good * float(np.clip(np.log10(float(period) / 0.9) / np.log10(8.0), 0.0, 1.0))
    short_pref = 0.0
    if float(period) <= 0.12 and float(SHORT_PERIOD_PREF) > 0:
        short_pref = float(SHORT_PERIOD_PREF) * good * float(np.clip((0.12 - float(period)) / 0.08, 0.0, 1.0))
    score = (fold_reward
             + 0.08 * power_norm
             + 0.22 * float(np.clip(amp_ratio, 0.0, 2.5))
             - 0.80 * min(float(gap), 1.0)
             - 0.38 * min(max(rough, 0.0), 2.0)
             + long_pref + short_pref
             - alias_pen - cycle_pen - mult_pen - micro_pen - long_alias_pen)
    return {'period': float(period), 'theta': float(theta_raw + alias_pen), 'theta_raw': theta_raw,
            'amp': amp, 'amp_ratio': float(amp_ratio), 'gap': float(gap),
            'alias_dist': float(alias_dist), 'power_norm': float(power_norm),
            'fold_score': float(score), 'phase_quality': float(phase_quality),
            'sine_coherence': float(sine_coh), 'curve_roughness': float(rough),
            'night_diversity': float(nd), 'plot_points': int(disp.get('plot_points', 0)),
            'plot_coverage': float(plot_coverage), 'plot_gap': float(plot_gap),
            'plot_jump': float(plot_jump), 'display_sine': float(display_sine),
            'display_roughness': float(display_rough),
            'display_spikiness': float(display_spike),
            'display_maxjump': float(display_maxjump), 'display_snr': float(display_snr),
            'micro_penalty': float(micro_pen), 'long_alias_penalty': float(long_alias_pen),
            'bins_used': int(prof['bins_used'])}

def folded_profile(jd, y, dy, period, nbins=40, min_pts=2):
    """Binned (comp-var) fold profile plus realistic within-bin noise (for EB test)."""
    jd, y, dy = (np.asarray(a, dtype=np.float64) for a in (jd, y, dy))
    ok = np.isfinite(jd) & np.isfinite(y) & np.isfinite(dy) & (dy > 0)
    jd, y = jd[ok], y[ok]
    if jd.size < 10 or not np.isfinite(period) or period <= 0:
        return None
    rel = -y
    ph = phase_fold(jd, period, jd.min())
    edges = np.linspace(0.0, 1.0, int(nbins) + 1)
    inds = np.clip(np.digitize(ph, edges) - 1, 0, int(nbins) - 1)
    prof = np.full(int(nbins), np.nan, dtype=np.float64)
    scat = np.full(int(nbins), np.nan, dtype=np.float64)
    for b in range(int(nbins)):
        m = inds == b
        if m.sum() < int(min_pts):
            continue
        prof[b] = fast_median(rel[m])
        scat[b] = mad_sigma(rel[m] - prof[b])
    fin = np.isfinite(prof)
    if fin.sum() < 8:
        return None
    noise = np.nanmedian(scat[np.isfinite(scat) & (scat > 0)])
    if not np.isfinite(noise) or noise <= 0:
        noise = robust_sigma(rel)
    level = float(np.nanmedian(np.sort(prof[fin])[-max(3, int(nbins) // 8):]))
    return {'prof': prof, 'noise': float(noise), 'level': level,
            'amp': float(np.nanmax(prof) - np.nanmin(prof))}

def eclipse_depths(prof_dict):
    """Primary/secondary depths below the bright level, robust to phase offset."""
    pr = prof_dict['prof']
    n = pr.size
    level = prof_dict['level']
    if np.isfinite(pr).sum() < 8:
        return 0.0, 0.0
    pr = np.roll(pr, n // 4 - int(np.nanargmin(pr)))
    prim = level - np.nanmin(pr)
    opp = pr[n // 2:]
    sec = level - np.nanmin(opp) if np.isfinite(opp).any() else 0.0
    return float(prim), float(max(sec, 0.0))

def fold_refine_grid(jd, period, baseline):
    if not np.isfinite(period) or period <= 0:
        return np.array([], dtype=np.float64)
    if baseline is None or not np.isfinite(baseline) or baseline <= 0:
        half = float(FOLD_REFINE_REL_WIDTH) * float(period)
    else:
        half = max(float(FOLD_REFINE_REL_WIDTH) * float(period),
                   float(LOCAL_REFINE_RESOLUTION_ELEMENTS) * float(period) ** 2 / max(float(baseline), 1e-8))
    lo = max(float(PERIOD_MIN_D), float(period) - half)
    hi = min(float(PERIOD_MAX_D), float(period) + half)
    if hi <= lo:
        return np.array([float(period)], dtype=np.float64)
    return np.linspace(lo, hi, int(FOLD_REFINE_GRID_SIZE))

POLISH_PERIOD       = True
POLISH_RES_ELEMENTS = 4.0
POLISH_GRID         = 121

def polish_period(jd, y, dy, period, baseline=None):
    'Refine the selected period with a local Lomb-Scargle frequency scan. The caller retains the refinement only when the phase-quality guard passes.'
    jd, y, dy = (np.asarray(a, dtype=np.float64) for a in (jd, y, dy))
    ok = np.isfinite(jd) & np.isfinite(y) & np.isfinite(dy) & (dy > 0)
    if ok.sum() < 12 or not np.isfinite(period) or period <= 0:
        return float(period)
    jj, yy, dd = jd[ok], -y[ok], dy[ok]
    if baseline is None or not np.isfinite(baseline) or baseline <= 0:
        baseline = float(jj.max() - jj.min())
    if not np.isfinite(baseline) or baseline <= 0:
        return float(period)
    f0 = 1.0 / float(period)
    df = float(POLISH_RES_ELEMENTS) / float(baseline)
    fs = np.linspace(max(f0 - df, 1.0 / float(PERIOD_MAX_D)), f0 + df, int(POLISH_GRID))
    try:
        pw = LombScargle(jj, yy, dd).power(fs)
    except Exception:
        return float(period)
    fbest = float(fs[int(np.nanargmax(pw))])
    if fbest <= 0:
        return float(period)
    Pb = 1.0 / fbest
    return float(Pb) if (PERIOD_MIN_D <= Pb <= PERIOD_MAX_D) else float(period)

FOURIER_N_HARM         = 4

def _fourier_design(phase, n_harm):
    cols = [np.ones_like(phase)]
    for k in range(1, int(n_harm) + 1):
        ang = 2.0 * np.pi * float(k) * phase
        cols.append(np.cos(ang)); cols.append(np.sin(ang))
    return np.column_stack(cols)

def finite_skewness(values):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    if values.size < 3:
        return np.nan
    center = float(np.nanmean(values))
    sigma = float(np.nanstd(values))
    if not np.isfinite(sigma) or sigma <= 0:
        return np.nan
    return float(np.nanmean(((values - center) / sigma) ** 3))

def welch_stetson_j(mag, err):
    """Single-band Welch-Stetson J variability index from per-epoch (mag, err).
    Same definition used as the stetson_J ML feature in mlcode.ipynb; the sign of
    mag is irrelevant (it uses deviations from the weighted mean)."""
    mag = np.asarray(mag, dtype=np.float64)
    err = np.asarray(err, dtype=np.float64)
    ok = np.isfinite(mag) & np.isfinite(err) & (err > 0)
    mag = mag[ok]; err = err[ok]
    n = int(mag.size)
    if n < 5:
        return np.nan
    wmean = np.sum(mag / err ** 2) / np.sum(1.0 / err ** 2)
    delta = np.sqrt(n / (n - 1.0)) * (mag - wmean) / err
    P = delta ** 2 - 1.0
    return float(np.sum(np.sign(P) * np.sqrt(np.abs(P))) / n)

def fourier_fit(jd, y, dy, period, n_harm=FOURIER_N_HARM):
    """Weighted least-squares truncated Fourier fit of the (comp-var) signal at `period`.
    Returns the weighted residual RMS (period goodness) plus the classic Fourier features
    (A1..A3, R21, phi21, R31, phi31, peak-to-peak). NOT used to render the plot."""
    jd, y, dy = (np.asarray(a, dtype=np.float64) for a in (jd, y, dy))
    ok = np.isfinite(jd) & np.isfinite(y) & np.isfinite(dy) & (dy > 0)
    out = {'fourier_fit_ok': 0, 'resid_rms': np.inf, 'amplitude_fourier': np.nan,
           'A1': np.nan, 'A2': np.nan, 'A3': np.nan, 'R21': np.nan, 'phi21': np.nan,
           'R31': np.nan, 'phi31': np.nan, 'n_fourier_points': int(ok.sum())}
    nh = int(n_harm)
    if int(ok.sum()) < 2 * nh + 2 or not np.isfinite(period) or period <= 0:
        return out
    jj, yy, dd = jd[ok], -y[ok], dy[ok]
    ph = np.mod(jj / float(period), 1.0)
    A = _fourier_design(ph, nh)
    sw = 1.0 / np.maximum(dd, 1e-9)
    try:
        coeff, _, _, _ = np.linalg.lstsq(A * sw[:, None], yy * sw, rcond=None)
    except Exception:
        return out
    resid = yy - A @ coeff
    w = sw ** 2
    out['resid_rms'] = float(np.sqrt(np.average(resid ** 2, weights=w)))
    amps = [0.0]; phis = [0.0]
    for k in range(1, nh + 1):
        a, b = float(coeff[2 * k - 1]), float(coeff[2 * k])
        amps.append(float(np.hypot(a, b))); phis.append(float(np.arctan2(b, a)))
    fine = np.linspace(0.0, 1.0, 256)
    mod = _fourier_design(fine, nh) @ coeff
    out['amplitude_fourier'] = float(np.max(mod) - np.min(mod))
    out['A1'], out['A2'] = amps[1], amps[2] if nh >= 2 else np.nan
    out['A3'] = amps[3] if nh >= 3 else np.nan
    out['R21'] = float(amps[2] / amps[1]) if (nh >= 2 and amps[1] > 0) else np.nan
    out['R31'] = float(amps[3] / amps[1]) if (nh >= 3 and amps[1] > 0) else np.nan
    out['phi21'] = float((phis[2] - 2.0 * phis[1]) % (2.0 * np.pi)) if nh >= 2 else np.nan
    out['phi31'] = float((phis[3] - 3.0 * phis[1]) % (2.0 * np.pi)) if nh >= 3 else np.nan
    out['fourier_fit_ok'] = 1
    return out

UNCERTAINTY_ENABLED = True
UNCERTAINTY_BOOTSTRAP_REPLICATES = 400
UNCERTAINTY_SESSION_GAP_DAYS = 0.35

def _uncertainty_robust_sigma(values):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    if values.size < 3:
        return np.nan
    q16, q84 = np.quantile(values, [0.158655, 0.841345])
    return float(0.5 * (q84 - q16))

def _uncertainty_robust_scatter(values):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    if values.size < 3:
        return np.nan
    center = float(np.median(values))
    return float(1.4826 * np.median(np.abs(values - center)))

def _uncertainty_jackknife_sigma(values):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    n = int(values.size)
    if n < 3:
        return np.nan
    return float(np.sqrt((n - 1.0) / n * np.sum((values - np.mean(values)) ** 2)))

def _uncertainty_finite_max(values):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    return float(np.max(values)) if values.size else np.nan

def _uncertainty_session_groups(jd):
    jd = np.asarray(jd, dtype=np.float64)
    order = np.argsort(jd)
    sorted_jd = jd[order]
    sorted_group = np.zeros(sorted_jd.size, dtype=np.int64)
    if sorted_jd.size > 1:
        sorted_group[1:] = np.cumsum(
            np.diff(sorted_jd) > float(UNCERTAINTY_SESSION_GAP_DAYS)
        )
    group = np.empty_like(sorted_group)
    group[order] = sorted_group
    return group

def _uncertainty_design_matrix(jd, frequency, nterms):
    centered = np.asarray(jd, dtype=np.float64) - float(np.median(jd))
    columns = [np.ones_like(centered)]
    for harmonic in range(1, int(nterms) + 1):
        angle = 2.0 * np.pi * float(harmonic) * float(frequency) * centered
        columns.extend([np.cos(angle), np.sin(angle)])
    return np.column_stack(columns)

def _uncertainty_harmonic_fit(jd, signal, sigma, frequency, nterms):
    design = _uncertainty_design_matrix(jd, frequency, nterms)
    sigma = np.maximum(np.asarray(sigma, dtype=np.float64), 1.0e-9)
    weighted_design = design / sigma[:, None]
    weighted_signal = np.asarray(signal, dtype=np.float64) / sigma
    coefficients, _, _, _ = np.linalg.lstsq(
        weighted_design, weighted_signal, rcond=None
    )
    model = design @ coefficients
    residual = np.asarray(signal, dtype=np.float64) - model
    chi2 = float(np.sum((residual / sigma) ** 2))
    return coefficients, model, residual, chi2

def _uncertainty_local_frequency_fit(
    jd, signal, sigma, center_frequency, nterms, half_width=None
):
    baseline = float(np.ptp(jd))
    if half_width is None:
        half_width = min(0.45 / max(baseline, 1.0e-9), 0.20 * float(center_frequency))
    low = max(float(center_frequency) - float(half_width), 1.0e-8)
    high = float(center_frequency) + float(half_width)

    def objective(frequency):
        return _uncertainty_harmonic_fit(jd, signal, sigma, frequency, nterms)[3]

    result = minimize_scalar(
        objective,
        bounds=(low, high),
        method='bounded',
        options={
            'xatol': max(1.0e-12, float(center_frequency) * 1.0e-11),
            'maxiter': 100,
        },
    )
    frequency = float(result.x)
    coefficients, model, residual, chi2 = _uncertainty_harmonic_fit(
        jd, signal, sigma, frequency, nterms
    )
    return frequency, coefficients, model, residual, chi2

def _uncertainty_linearized_frequency_fit(
    jd, signal, sigma, center_frequency, nterms, half_width
):
    jd = np.asarray(jd, dtype=np.float64)
    signal = np.asarray(signal, dtype=np.float64)
    sigma = np.maximum(np.asarray(sigma, dtype=np.float64), 1.0e-9)
    centered = jd - float(np.median(jd))
    frequency = float(center_frequency)
    coefficients, _, _, _ = _uncertainty_harmonic_fit(
        jd, signal, sigma, frequency, nterms
    )
    design = _uncertainty_design_matrix(jd, frequency, nterms)
    derivative = np.zeros_like(jd)
    for harmonic in range(1, int(nterms) + 1):
        angle = 2.0 * np.pi * float(harmonic) * frequency * centered
        cosine_coefficient = coefficients[2 * harmonic - 1]
        sine_coefficient = coefficients[2 * harmonic]
        derivative += 2.0 * np.pi * float(harmonic) * centered * (
            -cosine_coefficient * np.sin(angle) + sine_coefficient * np.cos(angle)
        )
    augmented = np.column_stack([design, derivative])
    weighted = augmented / sigma[:, None]
    try:
        solution = np.linalg.solve(
            weighted.T @ weighted, weighted.T @ (signal / sigma)
        )
    except np.linalg.LinAlgError:
        solution, _, _, _ = np.linalg.lstsq(weighted, signal / sigma, rcond=None)
    frequency_shift = float(
        np.clip(solution[-1], -float(half_width), float(half_width))
    )
    return max(frequency + frequency_shift, 1.0e-8)

def _estimate_period_amplitude_uncertainties(
    jd,
    signal,
    dy,
    adopted_period,
    adopted_amplitude,
    star_id,
    nterms,
    seed_offset,
    amplitude_statistic,
    amplitude_period_grid,
):
    empty = {'period_sigma_days': np.nan, 'amplitude_sigma_mag': np.nan}
    if not bool(UNCERTAINTY_ENABLED):
        return empty
    jd = np.asarray(jd, dtype=np.float64)
    signal = np.asarray(signal, dtype=np.float64)
    dy = np.asarray(dy, dtype=np.float64)
    good = np.isfinite(jd) & np.isfinite(signal) & np.isfinite(dy) & (dy > 0)
    jd, signal, dy = jd[good], signal[good], dy[good]
    adopted_period = float(adopted_period)
    adopted_amplitude = float(adopted_amplitude)
    nterms = max(1, int(nterms))
    if (
        jd.size < max(20, 2 * nterms + 5)
        or not np.isfinite(adopted_period)
        or adopted_period <= 0
        or not np.isfinite(adopted_amplitude)
        or float(np.ptp(jd)) <= 0
    ):
        return empty

    grid = np.asarray(amplitude_period_grid, dtype=np.float64)
    grid = grid[np.isfinite(grid) & (grid > 0)]
    grid_amplitudes = []
    for trial_period in grid:
        try:
            grid_amplitudes.append(
                float(amplitude_statistic(jd, signal, dy, float(trial_period)))
            )
        except Exception:
            grid_amplitudes.append(np.nan)
    grid_amplitudes = np.asarray(grid_amplitudes, dtype=np.float64)
    usable_grid = np.isfinite(grid_amplitudes)
    if usable_grid.any():
        usable_indices = np.flatnonzero(usable_grid)
        closest = usable_indices[
            int(np.argmin(np.abs(grid_amplitudes[usable_grid] - adopted_amplitude)))
        ]
        amplitude_period_ratio = float(grid[closest] / adopted_period)
    else:
        amplitude_period_ratio = 1.0

    adopted_frequency = 1.0 / adopted_period
    _, _, initial_residual, _ = _uncertainty_harmonic_fit(
        jd, signal, dy, adopted_frequency, nterms
    )
    residual_scatter = _uncertainty_robust_scatter(initial_residual)
    median_error = float(np.median(dy))
    jitter = float(
        np.sqrt(max(0.0, residual_scatter ** 2 - median_error ** 2))
    ) if np.isfinite(residual_scatter) else 0.0
    sigma_effective = np.sqrt(dy ** 2 + jitter ** 2)
    fitted_frequency, coefficients, model, residual, _ = _uncertainty_local_frequency_fit(
        jd, signal, sigma_effective, adopted_frequency, nterms
    )
    baseline = float(np.ptp(jd))
    local_half_width = min(
        0.45 / max(baseline, 1.0e-9), 0.20 * float(fitted_frequency)
    )
    groups = _uncertainty_session_groups(jd)
    unique_groups = np.unique(groups)
    standardized = residual / sigma_effective
    standardized = standardized - float(np.mean(standardized))
    rng = np.random.default_rng(int(seed_offset) + int(star_id))

    night_periods = []
    night_amplitudes = []
    residual_amplitudes = []
    for _ in range(int(UNCERTAINTY_BOOTSTRAP_REPLICATES)):
        signs = rng.choice(np.asarray([-1.0, 1.0]), unique_groups.size)
        simulated_night = model + residual * signs[groups]
        night_frequency = _uncertainty_linearized_frequency_fit(
            jd, simulated_night, sigma_effective, fitted_frequency,
            nterms, local_half_width,
        )
        night_period = 1.0 / night_frequency
        night_periods.append(night_period)
        try:
            night_amplitudes.append(
                float(amplitude_statistic(
                    jd, simulated_night, dy,
                    night_period * amplitude_period_ratio,
                ))
            )
        except Exception:
            night_amplitudes.append(np.nan)

        simulated_residual = model + (
            rng.choice(standardized, jd.size, replace=True) * sigma_effective
        )
        residual_frequency = _uncertainty_linearized_frequency_fit(
            jd, simulated_residual, sigma_effective, fitted_frequency,
            nterms, local_half_width,
        )
        residual_period = 1.0 / residual_frequency
        try:
            residual_amplitudes.append(
                float(amplitude_statistic(
                    jd, simulated_residual, dy,
                    residual_period * amplitude_period_ratio,
                ))
            )
        except Exception:
            residual_amplitudes.append(np.nan)

    leave_one_session_periods = []
    parameter_count = int(coefficients.size)
    for group in unique_groups:
        keep = groups != group
        if int(keep.sum()) <= parameter_count + 3:
            continue
        frequency = _uncertainty_linearized_frequency_fit(
            jd[keep], signal[keep], sigma_effective[keep], fitted_frequency,
            nterms, local_half_width,
        )
        leave_one_session_periods.append(1.0 / frequency)

    period_sigma = _uncertainty_finite_max([
        _uncertainty_robust_sigma(night_periods),
        _uncertainty_jackknife_sigma(leave_one_session_periods),
    ])
    amplitude_sigma = _uncertainty_finite_max([
        _uncertainty_robust_sigma(night_amplitudes),
        _uncertainty_robust_sigma(residual_amplitudes),
    ])
    return {
        'period_sigma_days': float(period_sigma) if np.isfinite(period_sigma) else np.nan,
        'amplitude_sigma_mag': float(amplitude_sigma) if np.isfinite(amplitude_sigma) else np.nan,
    }

def estimate_cell2_period_amplitude_uncertainties(
    jd, y, dy, period, amplitude, star_id
):
    period = float(period)

    def amplitude_statistic(trial_jd, trial_y, trial_dy, trial_period):
        profile = phase_profile(trial_jd, trial_y, trial_dy, trial_period)
        return float(profile.get('amp', np.nan)) if profile is not None else np.nan

    amplitude_grid = np.linspace(
        max(period - 0.5001e-6, 1.0e-8), period + 0.5001e-6, 101
    )
    return _estimate_period_amplitude_uncertainties(
        jd, y, dy, period, amplitude, star_id, 1,
        670900000, amplitude_statistic, amplitude_grid,
    )

def choose_period(jd, y, dy):
    candidates, ls_map, best_power, baseline = build_period_candidates(jd, y, dy)
    if not candidates:
        return None

    if len(candidates) > int(PREFILTER_KEEP):
        lite = []
        for cand in candidates:
            P = float(cand['period'])
            beauty = fast_fold_beauty(jd, y, P)
            pn = ls_power_norm_at(P, ls_map, best_power)

            gp = float(np.clip(beauty, 0.0, 1.0))
            ext = 0.0
            if P <= 0.12:
                ext += 0.35 * float(SHORT_PERIOD_PREF) * gp * float(np.clip((0.12 - P) / 0.08, 0.0, 1.0))
            if P >= float(LONG_ALIAS_MIN_PERIOD_D):
                long_span = np.log10(float(PERIOD_MAX_D) / float(LONG_ALIAS_MIN_PERIOD_D))
                long_pos = float(np.clip(np.log10(P / float(LONG_ALIAS_MIN_PERIOD_D)) / max(long_span, 1e-8), 0.0, 1.0))
                ext += 0.75 * float(LONG_PERIOD_PREF) * gp * long_pos
            lite.append((1.8 * beauty + 0.5 * pn + ext, cand))
        lite.sort(key=lambda r: -r[0])
        candidates = [c for _s, c in lite[:int(PREFILTER_KEEP)]]

    if ls_map:
        snapped = []
        for cand in candidates:
            c = dict(cand)
            if str(c.get('candidate_source', '')).startswith('neutral_power_grid'):
                nt = int(c.get('search_nterms', 1))
                ls = ls_map.get(nt, next(iter(ls_map.values())))
                Ps, pwr = refine_period(ls, float(c['period']), 1.0 / PERIOD_MAX_D,
                                        1.0 / PERIOD_MIN_D, baseline)
                if np.isfinite(Ps) and PERIOD_MIN_D <= Ps <= PERIOD_MAX_D:
                    c['period'] = float(Ps)
                    c['base_power'] = float(pwr) if np.isfinite(pwr) else c.get('base_power', np.nan)
                    c['candidate_source'] = str(c.get('candidate_source', '')) + '+ls_snap'
            snapped.append(c)
        candidates = dedup_candidate_rows(snapped, rel_tol=PEAK_DEDUP_FRAC / 4.0)
    rows = []
    for cand in candidates:
        s = score_period(jd, y, dy, cand['period'], ls_map=ls_map, best_power=best_power,
                         baseline=baseline, candidate=cand)
        if s is not None:
            rows.append({**cand, **s})
    if not rows:
        return None
    rows.sort(key=lambda r: (-float(r['fold_score']), -float(r.get('phase_quality', 0.0)), float(r['theta_raw'])))

    refine_n = max(int(FOLD_REFINE_TOP_N), 0)
    phase_rows = sorted(rows, key=lambda r: (-float(r.get('phase_quality', 0.0)),
                                             float(r.get('display_maxjump', np.inf)),
                                             float(r.get('display_spikiness', np.inf)),
                                             -float(r.get('display_sine', 0.0)),
                                             float(r['theta_raw'])))
    refine_pool = []
    for pool in (rows[:refine_n], phase_rows[:refine_n]):
        for row in pool:
            if not any(abs(float(row['period']) - float(old['period'])) / float(old['period']) <= PEAK_DEDUP_FRAC / 4.0 for old in refine_pool):
                refine_pool.append(row)
    refined = []
    for base_row in refine_pool:
        for P in fold_refine_grid(jd, float(base_row['period']), baseline):
            if any(abs(P - float(old['period'])) / float(old['period']) <= PEAK_DEDUP_FRAC / 4.0 for old in refined):
                continue
            cand = {**base_row, 'period': float(P), 'candidate_source': str(base_row.get('candidate_source', '')) + '+fold_refine'}
            s = score_period(jd, y, dy, float(P), ls_map=ls_map, best_power=best_power,
                             baseline=baseline, candidate=cand)
            if s is not None:
                refined.append({**cand, **s})
    if refined:
        rows.extend(refined)
        rows = dedup_candidate_rows(rows, rel_tol=PEAK_DEDUP_FRAC / 4.0)
        rows.sort(key=lambda r: (-float(r['fold_score']), -float(r.get('phase_quality', 0.0)), float(r['theta_raw'])))

    best = rows[0]
    protected_short = protected_ultrashort_candidate(rows, best)
    if protected_short is not None:
        return protected_short
    P_before = float(best['period'])
    best = prefer_fundamental(jd, y, dy, best, ls_map=ls_map, best_power=best_power, baseline=baseline)
    best = prefer_long_coherent_alias(jd, y, dy, best, scored_rows=rows,
                                      ls_map=ls_map, best_power=best_power, baseline=baseline)

    if abs(float(best['period']) - P_before) <= 1e-9:
        best = maybe_double_for_eb(jd, y, dy, best, ls_map=ls_map, best_power=best_power, baseline=baseline)

    return best

def finalize_period(jd, y, dy, period, baseline=None, ref_quality=None):
    'Apply local period refinement to the final selected series, retaining it only when its binned phase quality does not decrease.'
    if not POLISH_PERIOD:
        return float(period), None
    Pp = polish_period(jd, y, dy, float(period), baseline)
    if not (np.isfinite(Pp) and Pp > 0 and abs(Pp - float(period)) > 1e-9):
        return float(period), None
    sp = score_period(jd, y, dy, float(Pp))
    if sp is None:
        return float(period), None
    if ref_quality is not None and float(sp.get('phase_quality', -9e9)) < float(ref_quality) - 1e-6:
        return float(period), None
    return float(Pp), sp

def is_excellent_ultrashort(row):
    """True only for a visibly clean <0.1 d folded curve; no labels or models."""
    if not isinstance(row, dict):
        return False
    P = float(row.get('period', np.nan))
    if not (np.isfinite(P) and PERIOD_MIN_D <= P < float(ULTRASHORT_PROTECT_MAX_PERIOD_D)):
        return False
    q = float(row.get('phase_quality', -np.inf))
    ds = float(row.get('display_sine', 0.0))
    cov = float(row.get('plot_coverage', 0.0))
    gap = float(row.get('plot_gap', 1.0))
    return (np.isfinite(q) and q >= float(ULTRASHORT_PROTECT_MIN_PHASE_QUALITY)
            and ds >= float(ULTRASHORT_PROTECT_MIN_DISPLAY_SINE)
            and cov >= float(ULTRASHORT_PROTECT_MIN_COVERAGE)
            and np.isfinite(gap) and gap <= float(ULTRASHORT_PROTECT_MAX_GAP))

def protected_ultrashort_candidate(rows, best):
    'Apply the configured preference for a high-quality short-period candidate when it is competitive with the nominal winner.'
    shorts = [dict(r) for r in rows if is_excellent_ultrashort(r)]
    if not shorts:
        return None
    best_score = float(best.get('fold_score', -np.inf)) if isinstance(best, dict) else -np.inf
    best_q = float(best.get('phase_quality', -np.inf)) if isinstance(best, dict) else -np.inf
    shorts.sort(key=lambda r: (-float(r.get('phase_quality', -np.inf)),
                               -float(r.get('fold_score', -np.inf)),
                               float(r.get('period', np.inf))))
    short = shorts[0]
    short_score = float(short.get('fold_score', -np.inf))
    short_q = float(short.get('phase_quality', -np.inf))
    if (short_score >= best_score - float(ULTRASHORT_PROTECT_SCORE_MARGIN)
            or short_q >= best_q - float(ULTRASHORT_PROTECT_QUALITY_MARGIN)):
        short['candidate_source'] = str(short.get('candidate_source', '')) + '+ultrashort_protect'
        short['fold_score'] = max(short_score, best_score + 0.02 if np.isfinite(best_score) else short_score)
        short['ultrashort_protected'] = True
        return short
    return None

def prefer_fundamental(jd, y, dy, best, ls_map=None, best_power=None, baseline=None):
    'Test periods near twice the selected value using amplitude and folded-shape criteria.'
    base = dict(best)
    if is_excellent_ultrashort(base):
        return base
    P0 = float(base['period'])
    amp0 = float(base.get('amp', np.nan))
    if not (np.isfinite(P0) and P0 > 0 and np.isfinite(amp0) and amp0 > 0):
        return base
    if not (PERIOD_MIN_D <= 2.0 * P0 <= PERIOD_MAX_D):
        return base
    lo, hi = 1.95 * P0, min(2.05 * P0, PERIOD_MAX_D)
    qualified = []
    for P in np.linspace(lo, hi, 13):
        s = score_period(jd, y, dy, float(P), ls_map=ls_map, best_power=best_power,
                         baseline=baseline, candidate={**base, 'period': float(P), 'harmonic_multiplier': 2.0})
        if s is None:
            continue
        a = float(s.get('amp', np.nan))
        ds = float(s.get('display_sine', 0.0))
        pq = float(s.get('phase_quality', -1e9))

        if np.isfinite(a) and a / amp0 >= float(AMP_DOUBLE_RATIO) and ds >= 0.45 and pq > 0.0:
            qualified.append((s, float(P)))
    if not qualified:
        return base

    bestlong, bestP = max(qualified, key=lambda c: float(c[0].get('phase_quality', 0.0)))
    cand = {**base, 'period': float(bestP),
            'harmonic_multiplier': float(base.get('harmonic_multiplier', 1.0)) * 2.0,
            'candidate_source': str(base.get('candidate_source', '')) + '+fundamental_x2'}
    out = {**cand, **bestlong, 'dealiased': True}

    out['fold_score'] = max(float(bestlong.get('fold_score', -1e9)), float(base.get('fold_score', -1e9)))
    return out

def prefer_long_coherent_alias(jd, y, dy, best, scored_rows=None, ls_map=None, best_power=None, baseline=None):
    'Test longer aliases and harmonics using the configured morphology, amplitude and period-dependent ranking terms.'
    base = dict(best)
    if is_excellent_ultrashort(base):
        return base
    P0 = float(base.get('period', np.nan))
    if not (np.isfinite(P0) and P0 > 0):
        return base
    q0 = float(base.get('phase_quality', -np.inf))
    score0 = float(base.get('fold_score', -np.inf))
    amp0 = float(base.get('amp', np.nan))
    if not np.isfinite(q0):
        q0 = -np.inf

    rows = []
    seen = []

    def near_seen(P):
        return any(abs(float(P) - float(q)) / float(q) <= PEAK_DEDUP_FRAC for q in seen if q > 0)

    def add_scored(row):
        P = float(row.get('period', np.nan))
        if not (np.isfinite(P) and LONG_ALIAS_MIN_PERIOD_D <= P <= PERIOD_MAX_D and P > 1.22 * P0):
            return
        if near_seen(P):
            return
        rows.append(dict(row))
        seen.append(P)

    for row in scored_rows or []:
        add_scored(row)

    generated = []
    for mult in LONG_ALIAS_MULTIPLIERS:
        P = float(P0) * float(mult)
        if LONG_ALIAS_MIN_PERIOD_D <= P <= PERIOD_MAX_D and P > 1.22 * P0 and not near_seen(P):
            generated.append(P)
    f0 = 1.0 / float(P0)
    for order in LONG_ALIAS_DAILY_ORDERS:
        if int(order) == 0:
            continue
        f = f0 + float(order)
        if f <= 0:
            continue
        P = 1.0 / f
        if LONG_ALIAS_MIN_PERIOD_D <= P <= PERIOD_MAX_D and P > 1.22 * P0 and not near_seen(P):
            generated.append(float(P))

    generated = sorted(set(round(float(P), 10) for P in generated))
    if generated:
        lite = []
        for P in generated:
            beauty = fast_fold_beauty(jd, y, float(P))
            pn = ls_power_norm_at(float(P), ls_map or {}, best_power or {})
            long_span = np.log10(float(PERIOD_MAX_D) / float(LONG_ALIAS_MIN_PERIOD_D))
            long_pos = float(np.clip(np.log10(float(P) / float(LONG_ALIAS_MIN_PERIOD_D)) / max(long_span, 1e-8), 0.0, 1.0))
            lite.append((1.8 * beauty + 0.45 * pn + 0.65 * long_pos, float(P)))
        lite.sort(key=lambda r: -r[0])
        for _lite_score, P in lite[:int(LONG_ALIAS_EXTRA_SCORE_TOP)]:
            if near_seen(P):
                continue
            cand = {**base, 'period': float(P),
                    'candidate_source': str(base.get('candidate_source', '')) + '+long_alias_probe'}
            s = score_period(jd, y, dy, float(P), ls_map=ls_map, best_power=best_power,
                             baseline=baseline, candidate=cand)
            if s is not None:
                add_scored({**cand, **s})

    qualified = []
    for row in rows:
        P = float(row.get('period', np.nan))
        q = float(row.get('phase_quality', -np.inf))
        ds = float(row.get('display_sine', 0.0))
        gap = float(row.get('plot_gap', 1.0))
        amp = float(row.get('amp', np.nan))
        if not (np.isfinite(P) and np.isfinite(q)):
            continue
        margin = float(LONG_ALIAS_VLONG_MARGIN if P >= LONG_ALIAS_VERY_LONG_D else LONG_ALIAS_QUALITY_MARGIN)
        min_q = 0.90 if P >= LONG_ALIAS_VERY_LONG_D else float(LONG_ALIAS_MIN_PHASE_QUALITY)
        if q < min_q:
            continue
        if np.isfinite(q0) and q < q0 - margin:
            continue
        if ds < float(LONG_ALIAS_MIN_DISPLAY_SINE):
            continue
        if np.isfinite(gap) and gap > float(LONG_ALIAS_MAX_PLOT_GAP):
            continue
        if np.isfinite(amp0) and amp0 > 0 and np.isfinite(amp) and amp < float(LONG_ALIAS_MIN_AMP_FRAC) * amp0:
            continue
        long_span = np.log10(float(PERIOD_MAX_D) / float(LONG_ALIAS_MIN_PERIOD_D))
        long_pos = float(np.clip(np.log10(P / float(LONG_ALIAS_MIN_PERIOD_D)) / max(long_span, 1e-8), 0.0, 1.0))
        amp_frac = float(np.clip(amp / max(amp0, 1e-8), 0.0, 2.5)) if np.isfinite(amp) and np.isfinite(amp0) and amp0 > 0 else 1.0
        rank = (q + 0.35 * ds + 0.28 * amp_frac + 0.90 * long_pos
                + (0.75 if P >= LONG_ALIAS_VERY_LONG_D else 0.0)
                - 0.20 * min(max(gap, 0.0), 1.0))
        qualified.append((rank, row))

    if not qualified:
        return base
    _rank, chosen = max(qualified, key=lambda r: r[0])
    out = {**base, **chosen}
    out['candidate_source'] = str(chosen.get('candidate_source', base.get('candidate_source', ''))) + '+long_coherent_alias'
    if np.isfinite(score0):
        out['fold_score'] = max(float(chosen.get('fold_score', -1e9)),
                                score0 + (0.05 if float(out['period']) >= LONG_ALIAS_VERY_LONG_D else -0.03))
    out['dealiased_long'] = True
    return out

def maybe_double_for_eb(jd, y, dy, best, ls_map=None, best_power=None, baseline=None):
    """Only double when 2*P shows a genuine UNEQUAL secondary eclipse and stays clean."""
    P2 = float(best['period']) * 2.0
    best = dict(best); best['eb'] = False
    if not (PERIOD_MIN_D <= P2 <= PERIOD_MAX_D):
        return best
    prof = folded_profile(jd, y, dy, P2, nbins=40, min_pts=2)
    if prof is None:
        return best
    prim, sec = eclipse_depths(prof)
    noise, amp = prof['noise'], prof['amp']
    if not (amp > 0 and np.isfinite(noise) and noise > 0):
        return best
    s2 = score_period(jd, y, dy, P2, ls_map=ls_map, best_power=best_power, baseline=baseline,
                      candidate={**best, 'harmonic_multiplier': 2.0, 'candidate_source': 'eb_double'})
    if s2 is None:
        return best
    real_secondary = sec > max(EB_SEC_NSIGMA * noise, EB_SEC_FRAC * prim)
    unequal = (prim - sec) > max(EB_ASYM_NSIGMA * noise, EB_ASYM_FRAC * prim)
    if real_secondary and unequal and s2['theta_raw'] <= best['theta_raw'] + EB_THETA_MARGIN and s2['fold_score'] >= best['fold_score'] - 0.45:
        out = {**best, **s2}
        out['eb'] = True
        out['candidate_source'] = str(best.get('candidate_source', '')) + '+eb_double'
        return out
    return best

def clean_series(jd, y, dy):
    jd, y, dy = (np.asarray(a, dtype=np.float64) for a in (jd, y, dy))
    ok = np.isfinite(jd) & np.isfinite(y) & np.isfinite(dy) & (dy > 0)
    jd, y, dy = jd[ok], y[ok], dy[ok]
    if jd.size < MIN_POINTS:
        return None
    keep = sigma_clip_mask(y, CLIP_NSIGMA, CLIP_MAXITER)
    jd, y, dy = jd[keep], y[keep], dy[keep]
    if jd.size < MIN_POINTS:
        return None
    return jd, y, dy

def finish_variant(jd, y, dy):
    jb, yb, eb = time_bin_weighted(jd, y, dy, TIME_BIN_DAYS)
    if jb.size < MIN_POINTS:
        return None
    keep = sigma_clip_mask(yb, POSTBIN_CLIP_NSIGMA, POSTBIN_CLIP_MAXITER)
    jb, yb, eb = jb[keep], yb[keep], eb[keep]
    if jb.size < MIN_POINTS:
        return None
    o = np.argsort(jb)
    return jb[o], yb[o], eb[o]

def build_mode_series(hjd, corr, emat, star_to_col, var_id, comp1_id, comp2_id, auto_comp_ids=None):
    """Differential light curves vs comparison star(s): (variable - comparison).

    In production single_comp1 mode, only the CSV's first comparison star is used.
    Comp2 is retained only as table metadata/check information. The function returns
    immediately after the comp1 differential series and never constructs or ranks
    comp2, a pair, an ensemble, or an automatic alternative.
    """
    vc = star_to_col.get(int(var_id))
    if vc is None:
        return []
    vmag, verr = corr[:, vc].astype(np.float64), emat[:, vc].astype(np.float64)

    def clean_id_list(*items):
        out = []
        for item in items:
            if item is None:
                continue
            if isinstance(item, (list, tuple, np.ndarray, pd.Series)):
                vals = item
            else:
                vals = [item]
            for v in vals:
                if pd.isna(v):
                    continue
                try:
                    sid = int(float(v))
                except Exception:
                    continue
                if sid != int(var_id) and sid in star_to_col and sid not in out:
                    out.append(sid)
        return out

    provided = clean_id_list(comp1_id, comp2_id)
    candidate_ids = (clean_id_list(comp1_id) if REFERENCE_MODE_SELECTION == 'single_comp1'
                     else clean_id_list(comp1_id, comp2_id, auto_comp_ids or []))
    if not candidate_ids:
        return []

    def single_series(cid):
        cc = star_to_col.get(int(cid))
        if cc is None:
            return None
        cmag, cerr = corr[:, cc].astype(np.float64), emat[:, cc].astype(np.float64)
        ok = (np.isfinite(hjd) & np.isfinite(vmag) & np.isfinite(verr) & np.isfinite(cmag) & np.isfinite(cerr)
              & (verr > 0) & (cerr > 0) & (verr <= MAX_ERR) & (cerr <= MAX_ERR))
        if ok.sum() < MIN_POINTS:
            return None
        return hjd[ok], vmag[ok] - cmag[ok], np.sqrt(verr[ok] ** 2 + cerr[ok] ** 2), int(ok.sum())

    out, valid = [], []
    for cid in candidate_ids:
        s = single_series(cid)
        if s is None:
            continue
        valid.append((cid, star_to_col[int(cid)], s[3]))

    for cid, _cc, _n in valid[:int(AUTO_SINGLE_COMP_TRIALS)]:
        s = single_series(cid)
        if s is None:
            continue
        name = 'single_comp1' if provided and cid == provided[0] else ('single_comp2' if len(provided) > 1 and cid == provided[1] else f'auto_single_{cid}')
        out.append((name, s[0], s[1], s[2]))

    if REFERENCE_MODE_SELECTION in ('single_comp1', 'best_single'):
        return out

    def pair_series(cid1, cid2):
        c1, c2 = star_to_col.get(int(cid1)), star_to_col.get(int(cid2))
        if c1 is None or c2 is None:
            return None
        c1m, c1e = corr[:, c1].astype(np.float64), emat[:, c1].astype(np.float64)
        c2m, c2e = corr[:, c2].astype(np.float64), emat[:, c2].astype(np.float64)
        ok = (np.isfinite(hjd) & np.isfinite(vmag) & np.isfinite(verr) & np.isfinite(c1m) & np.isfinite(c2m)
              & np.isfinite(c1e) & np.isfinite(c2e) & (verr > 0) & (c1e > 0) & (c2e > 0)
              & (verr <= MAX_ERR) & (c1e <= MAX_ERR) & (c2e <= MAX_ERR))
        if ok.sum() < MIN_POINTS:
            return None
        y = vmag[ok] - 0.5 * (c1m[ok] + c2m[ok])
        dy = np.sqrt(verr[ok] ** 2 + 0.25 * c1e[ok] ** 2 + 0.25 * c2e[ok] ** 2)
        return hjd[ok], y, dy, int(ok.sum())

    pair_rows = []
    pool = valid[:int(AUTO_PAIR_COMP_POOL)]
    for i in range(len(pool)):
        cid1, c1, _n1 = pool[i]
        c1m, c1e = corr[:, c1].astype(np.float64), emat[:, c1].astype(np.float64)
        for j in range(i + 1, len(pool)):
            cid2, c2, _n2 = pool[j]
            c2m, c2e = corr[:, c2].astype(np.float64), emat[:, c2].astype(np.float64)
            ok = (np.isfinite(c1m) & np.isfinite(c2m) & np.isfinite(c1e) & np.isfinite(c2e)
                  & (c1e > 0) & (c2e > 0) & (c1e <= MAX_ERR) & (c2e <= MAX_ERR))
            if ok.sum() < max(MIN_POINTS, 80):
                continue
            diff = c1m[ok] - c2m[ok]
            sig = mad_sigma(diff - np.nanmedian(diff))
            if not np.isfinite(sig):
                continue

            pair_rows.append((float(sig) - 1e-6 * int(ok.sum()), int(cid1), int(cid2)))
    pair_rows.sort(key=lambda r: r[0])

    seen_pairs = set()
    if len(provided) >= 2:
        pair_rows.insert(0, (-np.inf, int(provided[0]), int(provided[1])))
    for _score, cid1, cid2 in pair_rows:
        key = tuple(sorted((int(cid1), int(cid2))))
        if key in seen_pairs:
            continue
        ps = pair_series(cid1, cid2)
        if ps is None:
            continue
        seen_pairs.add(key)
        name = 'pair_mean' if len(provided) >= 2 and key == tuple(sorted((int(provided[0]), int(provided[1])))) else f'auto_pair_{cid1}_{cid2}'
        out.append((name, ps[0], ps[1], ps[2]))
        if len(seen_pairs) >= int(AUTO_PAIR_TRIALS):
            break

    def comp_stability(col):
        c = corr[:, col].astype(np.float64)
        c = c[np.isfinite(c)]
        return float(mad_sigma(c)) if c.size >= MIN_POINTS else np.inf

    ranked = sorted(valid, key=lambda t: comp_stability(t[1]))
    quiet_ids = [cid for cid, _c, _n in ranked]

    def ensemble_series(cids):
        cols = [star_to_col[int(c)] for c in cids if star_to_col.get(int(c)) is not None]
        if len(cols) < 2:
            return None
        C = corr[:, cols].astype(np.float64)
        E = emat[:, cols].astype(np.float64)
        good = np.isfinite(C) & np.isfinite(E) & (E > 0) & (E <= MAX_ERR)
        Cc = np.where(good, C, np.nan)
        with np.errstate(invalid='ignore'):
            med = np.nanmedian(Cc, axis=0)
        Cc = Cc - med[None, :]
        W = np.where(good, 1.0 / np.maximum(E, 1e-12) ** 2, 0.0)
        sw = W.sum(axis=1)
        ncomp = good.sum(axis=1)
        with np.errstate(invalid='ignore', divide='ignore'):
            ref = np.where(sw > 0, np.nansum(W * Cc, axis=1) / np.where(sw > 0, sw, 1.0), np.nan)
            ref_err = np.where(sw > 0, np.sqrt(1.0 / np.where(sw > 0, sw, 1.0)), np.nan)
        need_k = max(2, int(np.ceil(0.5 * len(cols))))
        ok = (np.isfinite(hjd) & np.isfinite(vmag) & np.isfinite(verr) & (verr > 0) & (verr <= MAX_ERR)
              & np.isfinite(ref) & np.isfinite(ref_err) & (ncomp >= need_k))
        if int(ok.sum()) < MIN_POINTS:
            return None
        return hjd[ok], vmag[ok] - ref[ok], np.sqrt(verr[ok] ** 2 + ref_err[ok] ** 2), int(ok.sum())

    for K in ENSEMBLE_COMP_SIZES:
        kk = min(int(K), len(quiet_ids))
        if kk >= 2:
            es_ = ensemble_series(quiet_ids[:kk])
            if es_ is not None and not any(m[0] == f'ensemble{kk}' for m in out):
                out.append((f'ensemble{kk}', es_[0], es_[1], es_[2]))

    if REFERENCE_MODE_SELECTION == 'pair_first' and out:
        pair = [m for m in out if m[0].startswith(('pair', 'auto_pair'))]
        return pair[:1] if pair else out[:1]
    return out

def analyze_star(jd, y, dy):
    'Search configured period candidates using cleaned and time-binned differential photometry.'
    base = clean_series(jd, y, dy)
    if base is None:
        return None
    jc, yc, ec = base
    night_span = typical_night_span(jc)
    raw = finish_variant(jc, yc, ec)
    if raw is None:
        return None
    raw_sel = choose_period(*raw)
    if raw_sel is None:
        return None
    best = {'variant': 'raw', 'jd': raw[0], 'y': raw[1], 'dy': raw[2],
            'night_span_cycles': float(night_span / max(float(raw_sel['period']), 1e-9)), **raw_sel}
    return best

def classify(period, sel, jd, y, dy):
    """Lightweight type HINT only -- never used for any period/fold decision.
    EB := period-doubled to orbital P, OR a deep dip in an otherwise flat cycle,
    OR two distinct unequal minima; otherwise split by period (gamma Dor 0.3-3 d,
    short-period pulsator < 0.3 d, longer otherwise)."""
    if sel.get('eb'):
        return 'EB'
    prof = folded_profile(jd, y, dy, period)
    if prof is None:
        return 'SHORT-P?' if period < 0.3 else ('GDOR?' if period <= 3.0 else 'LONG-P?')
    prim, sec = eclipse_depths(prof)
    pr, noise, amp, level = prof['prof'], prof['noise'], prof['amp'], prof['level']
    fin = np.isfinite(pr)
    flat_frac = float(np.mean((level - pr[fin]) < max(2.0 * noise, 0.12 * amp))) if amp > 0 else 1.0
    eclipse_like = prim > max(5.0 * noise, 0.30 * amp) and flat_frac > 0.55
    two_unequal = sec > max(4.0 * noise, 0.15 * prim) and (prim - sec) > max(3.0 * noise, 0.18 * prim)
    if eclipse_like or two_unequal:
        return 'EB'
    if period < 0.3:
        return 'SHORT-P?'
    if period <= 3.0:
        return 'GDOR?'
    return 'LONG-P?'

def phase_bin_edges(width):
    nbin = max(int(np.ceil(1.0 / max(width, 1e-4))), 1)
    return np.linspace(0.0, 1.0, nbin + 1)

def phase_bin_indices(phase, edges):
    out = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        if hi >= 1.0:
            out.append(np.where((phase >= lo) & (phase <= 1.0))[0])
        else:
            out.append(np.where((phase >= lo) & (phase < hi))[0])
    return out

def circular_rolling_median_observed(values, half):
    values = np.asarray(values, dtype=np.float64)
    n = values.size
    if n == 0 or half <= 0:
        return values.copy()
    out = np.full(n, np.nan)
    for i in range(n):
        loc = values[[(i + j) % n for j in range(-half, half + 1)]]
        loc = loc[np.isfinite(loc)]
        if loc.size:
            out[i] = fast_median(loc)
    return out

def bin_medians(phase, vals, edges):
    bins = phase_bin_indices(phase, edges)
    med = np.full(len(bins), np.nan)
    cnt = np.zeros(len(bins), dtype=int)
    for i, idx in enumerate(bins):
        cnt[i] = idx.size
        if idx.size:
            med[i] = fast_median(vals[idx])
    return bins, med, cnt

def auto_phase_offset(jd, y, period, width):
    """Cosmetic horizontal shift only: put the largest wrap/gap in the folded
    baseline at phase 0 so the unavoidable cycle-boundary discontinuity is hidden.
    Never changes the period."""
    ok = np.isfinite(jd) & np.isfinite(y)
    if ok.sum() == 0 or not np.isfinite(period) or period <= 0:
        return 0.0
    phase = phase_fold(jd[ok], period, 0.0)
    edges = phase_bin_edges(width)
    _, med, cnt = bin_medians(phase, -y[ok], edges)
    if np.isfinite(med).sum() < 4:
        return 0.0
    base = circular_rolling_median_observed(med, 2)
    scores = []
    for i in range(base.size):
        j = (i + 1) % base.size
        if not (np.isfinite(base[i]) and np.isfinite(base[j])):
            scores.append(np.inf)
            continue
        sparse_bonus = 0.004 if cnt[i] < 2 or cnt[j] < 2 else 0.0
        scores.append(abs(base[i] - base[j]) + sparse_bonus)
    scores = np.asarray(scores)
    if not np.isfinite(scores).any():
        return 0.0
    edge_idx = int(np.nanargmax(scores)) + 1
    boundary = float(edges[edge_idx]) if edge_idx < edges.size else 1.0
    return float(-boundary % 1.0)

def choose_plot_nbins(phase, n_points):
    """Number of EQUAL-COUNT phase bins: ~PHASE_BIN_TARGET_PTS real obs per bin, clamped to
    [PHASE_MIN_BINS, PHASE_MAX_BINS]. (phase arg kept for signature compatibility.)"""
    if not np.isfinite(n_points) or n_points <= 0:
        return int(PHASE_MIN_BINS)
    return int(np.clip(round(float(n_points) / max(float(PHASE_BIN_TARGET_PTS), 1.0)),
                       int(PHASE_MIN_BINS), int(PHASE_MAX_BINS)))

def binned_phase_points(jd, y, dy, period, width=PHASE_BIN_WIDTH):
    'Return equal-count phase-bin weighted means, estimated errors and optional cross-bin outlier rejection for candidate scoring.'
    jd, y, dy = (np.asarray(a, dtype=np.float64) for a in (jd, y, dy))
    ok = np.isfinite(jd) & np.isfinite(y) & np.isfinite(dy) & (dy > 0)
    if ok.sum() == 0 or not np.isfinite(period) or period <= 0:
        return np.array([]), np.array([]), np.array([])
    jd, y, dy = jd[ok], y[ok], dy[ok]
    offset = auto_phase_offset(jd, y, period, width)
    phase = np.mod(phase_fold(jd, period, 0.0) + offset, 1.0)
    rel = -y

    n = rel.size
    nbins = choose_plot_nbins(phase, n)
    order = np.argsort(phase)
    ph_s, rel_s, dy_s = phase[order], rel[order], dy[order]
    bounds = np.linspace(0, n, nbins + 1).astype(int)
    min_pts = max(3, int(PHASE_BIN_MIN_PTS))
    xs, ys, ns, scats, forms = [], [], [], [], []
    for b in range(nbins):
        lo, hi = int(bounds[b]), int(bounds[b + 1])
        if hi - lo < min_pts:
            continue
        vals, errs, phs = rel_s[lo:hi], dy_s[lo:hi], ph_s[lo:hi]

        cen = float(fast_median(vals))
        sig = mad_sigma(vals - cen)
        floor = max(float(fast_median(errs)), 1e-6)
        scale = max(float(sig) if np.isfinite(sig) else 0.0, floor)
        keep = np.abs(vals - cen) <= float(PHASE_BIN_CLIP_NSIGMA) * scale
        if int(keep.sum()) >= max(min_pts, int(0.6 * (hi - lo))):
            vals, errs, phs = vals[keep], errs[keep], phs[keep]
        nbk = int(vals.size)
        w = 1.0 / np.maximum(errs, 1e-12) ** 2
        sw = float(np.sum(w))
        ymean = float(np.sum(w * vals) / sw) if sw > 0 else float(np.mean(vals))
        scat = mad_sigma(vals - ymean) if nbk > 1 else np.nan
        xs.append(float(np.mean(phs)))
        ys.append(ymean); ns.append(nbk)
        scats.append(float(scat) if np.isfinite(scat) else np.nan)
        forms.append(float(fast_median(errs)))
    if not xs:
        return np.array([]), np.array([]), np.array([])
    xs, ys, ns = np.asarray(xs), np.asarray(ys), np.asarray(ns, dtype=np.float64)
    scats, forms = np.asarray(scats), np.asarray(forms)

    gform = float(np.nanmedian(forms[np.isfinite(forms) & (forms > 0)])) if np.isfinite(forms).any() else 1e-3
    per_bin = np.where(np.isfinite(scats) & (scats > 0), scats, gform)
    formal_floor = gform / np.sqrt(np.maximum(ns, 1.0))
    es = np.maximum(per_bin / np.sqrt(np.maximum(ns, 1.0)), formal_floor)
    med_es = float(np.nanmedian(es[np.isfinite(es) & (es > 0)])) if np.isfinite(es).any() else np.nan
    if np.isfinite(med_es) and med_es > 0:
        es = np.minimum(es, 3.5 * med_es)
    o = np.argsort(xs)
    xs, ys, es = xs[o], ys[o], es[o]

    if ys.size >= 8:
        local = circular_rolling_median_observed(ys, 2)
        resid = ys - local
        s = mad_sigma(resid[np.isfinite(resid)])
        if np.isfinite(s) and s > 0:
            keep = np.isfinite(resid) & (np.abs(resid) <= float(PHASE_OUTLIER_NSIGMA) * s)
            if int(keep.sum()) >= max(8, int(float(PHASE_OUTLIER_KEEP_FRAC) * ys.size)):
                xs, ys, es = xs[keep], ys[keep], es[keep]

    if ys.size:
        ys = ys - fast_median(ys)
    return xs, ys, es

def local_periodogram(jd, y, dy, period):
    half = max(LOCAL_PGRAM_MIN_WIDTH, LOCAL_PGRAM_REL_WIDTH * period)
    pmin = max(PERIOD_MIN_D, period - half)
    pmax = min(PERIOD_MAX_D, period + half)
    if pmax <= pmin:
        pmin, pmax = PERIOD_MIN_D, PERIOD_MAX_D
    jd, y, dy = (np.asarray(a, dtype=np.float64) for a in (jd, y, dy))
    ok0 = np.isfinite(jd) & np.isfinite(y) & np.isfinite(dy) & (dy > 0)
    if int(ok0.sum()) < 10:
        return np.array([], dtype=np.float64), np.array([], dtype=np.float64)
    jd, y, dy = jd[ok0], y[ok0], dy[ok0]
    fmin, fmax = 1.0 / pmax, 1.0 / pmin
    baseline = float(np.ptp(jd))
    raw_n = int(np.ceil(max(fmax - fmin, 0.0) * max(baseline, 1.0) * float(LOCAL_PGRAM_SAMPLES_PER_PEAK)))
    nfreq = int(np.clip(raw_n, int(LOCAL_PGRAM_MIN_POINTS), int(LOCAL_PGRAM_MAX_POINTS)))
    freq = np.linspace(fmin, fmax, nfreq)
    ls = LombScargle(jd, y, dy=dy, center_data=True, nterms=1)
    power = ls.power(freq)
    period_arr = np.divide(1.0, freq, out=np.full_like(freq, np.nan, dtype=np.float64), where=freq > 0)
    power = np.asarray(power, dtype=np.float64)
    ok = np.isfinite(period_arr) & np.isfinite(power) & (power >= 0)
    if not ok.any():
        return np.array([], dtype=np.float64), np.array([], dtype=np.float64)
    period_arr, power = period_arr[ok], power[ok]
    order = np.argsort(period_arr)
    return period_arr[order], power[order]

def augment_long_display_gaps(phase, rel, dy, xs, ys, es, period, display_width):
    """Add the fewest extra REAL observations needed to split large displayed gaps.

    This is plot-only and never changes the period or any measured value. It is used
    only for P>=0.5 d, only when the current approved display has a visible phase gap,
    and only when an actual observation lies inside that gap. Existing continuous
    plots are returned byte-for-byte in spirit: no extra points are introduced.
    """
    if float(period) < float(DISPLAY_GAP_AUGMENT_MIN_PERIOD_D):
        return xs, ys, es
    xs = np.asarray(xs, dtype=np.float64)
    ys = np.asarray(ys, dtype=np.float64)
    es = np.asarray(es, dtype=np.float64)
    phase = np.asarray(phase, dtype=np.float64)
    rel = np.asarray(rel, dtype=np.float64)
    dy = np.asarray(dy, dtype=np.float64)
    ok = np.isfinite(phase) & np.isfinite(rel) & np.isfinite(dy) & (dy > 0)
    if xs.size < 6 or int(ok.sum()) <= xs.size:
        return xs, ys, es
    phase, rel, dy = phase[ok], rel[ok], dy[ok]

    def cdist(a, b):
        return np.abs((np.asarray(a, dtype=np.float64) - float(b) + 0.5) % 1.0 - 0.5)

    def gap_rows(xarr):
        if xarr.size < 2:
            return []
        sx = np.sort(xarr[np.isfinite(xarr)])
        if sx.size < 2:
            return []
        rows = []
        for i in range(sx.size):
            lo = float(sx[i])
            hi = float(sx[(i + 1) % sx.size])
            gap = (hi - lo) % 1.0
            rows.append((float(gap), lo, hi))
        rows.sort(key=lambda r: -r[0])
        return rows

    gaps = gap_rows(xs)
    if not gaps or float(gaps[0][0]) <= float(DISPLAY_GAP_AUGMENT_TRIGGER):
        return xs, ys, es

    max_points = min(int(DISPLAY_GAP_AUGMENT_MAX_POINTS),
                     int(np.ceil(xs.size * (1.0 + float(DISPLAY_GAP_AUGMENT_MAX_ADD_FRAC)))))
    max_add = max(0, int(max_points) - int(xs.size))
    if max_add <= 0:
        return xs, ys, es

    min_sep = max(float(DISPLAY_GAP_AUGMENT_MIN_SEP), 0.55 * float(display_width))
    err_med = float(fast_median(dy)) if dy.size else 1.0
    if not np.isfinite(err_med) or err_med <= 0:
        err_med = 1.0
    global_scale = mad_sigma(ys - fast_median(ys)) if ys.size else np.nan
    if not np.isfinite(global_scale) or global_scale <= 0:
        global_scale = 0.0025

    for _ in range(max_add):
        added = False
        for gap, lo, hi in gap_rows(xs):
            if float(gap) <= float(DISPLAY_GAP_AUGMENT_TARGET):
                return xs, ys, es
            if lo < hi:
                inside = (phase > lo) & (phase < hi)
            else:
                inside = (phase > lo) | (phase < hi)
            if not np.any(inside):
                continue
            dist_existing = np.min(np.abs((phase[:, None] - xs[None, :] + 0.5) % 1.0 - 0.5), axis=1)
            cand = np.where(inside & (dist_existing >= min_sep))[0]
            if cand.size == 0:
                continue
            mid = (lo + 0.5 * float(gap)) % 1.0
            near_existing = np.argsort(cdist(xs, mid))[:min(6, xs.size)]
            local_level = fast_median(ys[near_existing]) if near_existing.size else fast_median(ys)
            local_scale = mad_sigma(ys[near_existing] - local_level) if near_existing.size >= 3 else np.nan
            scale = max(float(local_scale) if np.isfinite(local_scale) else 0.0,
                        0.45 * float(global_scale), 0.0025)
            cost = (cdist(phase[cand], mid) / max(float(gap), 1e-6)
                    + 0.30 * np.abs(rel[cand] - local_level) / scale
                    + 0.08 * dy[cand] / err_med)
            pick = int(cand[int(np.nanargmin(cost))])
            xs = np.append(xs, float(phase[pick]))
            ys = np.append(ys, float(rel[pick]))
            es = np.append(es, float(dy[pick]))
            order = np.argsort(xs)
            xs, ys, es = xs[order], ys[order], es[order]
            added = True
            break
        if not added:
            break
    return xs, ys, es

def display_phase_points(jd, y, dy, period, width=DISPLAY_PHASE_BIN_WIDTH):
    'Choose one real observation per fixed phase bin near a rolling-median trend, optionally remove local outliers and add real observations in displayed gaps. This display subset suppresses scatter and should be interpreted alongside the analysed light curve.'
    jd, y, dy = (np.asarray(a, dtype=np.float64) for a in (jd, y, dy))
    ok = np.isfinite(jd) & np.isfinite(y) & np.isfinite(dy) & (dy > 0)
    if ok.sum() == 0 or not np.isfinite(period) or period <= 0:
        return np.array([]), np.array([]), np.array([])
    jd, y, dy = jd[ok], y[ok], dy[ok]
    display_width = float(width)
    if float(period) >= float(DISPLAY_LONG_PERIOD_MIN_D):
        display_width = float(DISPLAY_VLONG_PHASE_BIN_WIDTH if float(period) >= 4.0 else DISPLAY_LONG_PHASE_BIN_WIDTH)
    offset = auto_phase_offset(jd, y, period, display_width)
    phase = np.mod(phase_fold(jd, period, 0.0) + offset, 1.0)
    rel = -y
    edges = phase_bin_edges(display_width)
    bins = phase_bin_indices(phase, edges)
    nb = len(bins)
    bin_med = np.full(nb, np.nan)
    for b, idx in enumerate(bins):
        if idx.size:
            bin_med[b] = fast_median(rel[idx])
    baseline = circular_rolling_median_observed(bin_med, int(DISPLAY_CONTINUITY_SMOOTH_BINS))
    err_med = float(fast_median(dy)) if dy.size else 1.0
    if not np.isfinite(err_med) or err_med <= 0:
        err_med = 1.0
    base_scale = mad_sigma(bin_med - baseline)
    if not np.isfinite(base_scale) or base_scale <= 0:
        base_scale = 0.0025
    xs, ys, es = [], [], []
    for b, idx in enumerate(bins):
        if idx.size == 0:
            continue
        if np.isfinite(baseline[b]):
            local_med = fast_median(rel[idx])
            local_scale = mad_sigma(rel[idx] - local_med)
            if not np.isfinite(local_scale) or local_scale <= 0:
                local_scale = base_scale

            cost = (np.abs(rel[idx] - baseline[b]) / max(float(local_scale), 0.0025)
                    + float(DISPLAY_CONTINUITY_ERR_WEIGHT) * dy[idx] / err_med)
            chosen = idx[int(np.nanargmin(cost))]
        else:
            center = 0.5 * (float(edges[b]) + float(edges[b + 1]))
            chosen = idx[int(np.argmin(np.abs(phase[idx] - center)))]
        xs.append(float(phase[chosen])); ys.append(float(rel[chosen])); es.append(float(dy[chosen]))
    if not xs:
        return np.array([]), np.array([]), np.array([])
    xs, ys, es = np.asarray(xs), np.asarray(ys), np.asarray(es)
    o = np.argsort(xs)
    xs, ys, es = xs[o], ys[o], es[o]

    if DISPLAY_HIDE_OUTLIERS and xs.size >= 8:

        min_keep = max(8, int(np.ceil((1.0 - float(DISPLAY_MAX_OUTLIER_FRAC)) * xs.size)))
        for _pass in range(int(DISPLAY_OUTLIER_PASSES)):
            if xs.size <= min_keep:
                break
            trend = circular_rolling_median_observed(ys, int(DISPLAY_CONTINUITY_SMOOTH_BINS))
            resid = ys - trend
            good = np.isfinite(resid)
            sig = mad_sigma(resid[good])
            if not (np.isfinite(sig) and sig > 0):
                break
            limit = max(float(DISPLAY_OUTLIER_MIN_MAG), float(DISPLAY_OUTLIER_Z) * sig)
            keep = (~good) | (np.abs(resid) <= limit)
            if keep.all() or int(keep.sum()) < min_keep:
                break
            xs, ys, es = xs[keep], ys[keep], es[keep]
    xs, ys, es = augment_long_display_gaps(phase, rel, dy, xs, ys, es, period, display_width)
    if ys.size:
        ys = ys - fast_median(ys)
    return xs, ys, es

def save_phase_plot(out_png, jd, y, dy, period, star_id):
    jd, y, dy = (np.asarray(a, dtype=np.float64) for a in (jd, y, dy))
    ok = np.isfinite(jd) & np.isfinite(y) & np.isfinite(dy) & (dy > 0)
    if ok.sum() == 0:
        return
    jd, y, dy = jd[ok], y[ok], dy[ok]
    pg_p, pg_pow = local_periodogram(jd, y, dy, period)
    px, py, pe = display_phase_points(jd, y, dy, period)
    if px.size == 0:
        px, py, pe = phase_fold(jd, period, jd.min()), -y, dy
    if DOUBLE_CYCLE:
        xp = np.concatenate([px, px + 1.0]); yp = np.concatenate([py, py]); ep = np.concatenate([pe, pe]); xlim = (0.0, 2.0)
    else:
        xp, yp, ep, xlim = px, py, pe, (0.0, 1.0)
    fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(12.0, 4.5))
    ax0.plot(pg_p, pg_pow, color='black', lw=0.8)
    if pg_p.size and np.isfinite(pg_p).any():
        ax0.set_xlim(float(np.nanmin(pg_p)), float(np.nanmax(pg_p)))
    ax0.set_xlabel('Period (days)')
    ax0.set_ylabel('Lomb-Scargle power')
    ax0.set_title('Periodogram')
    ax1.errorbar(xp, yp, yerr=ep, fmt='o', ms=1.5, lw=0.0, elinewidth=0.7, capsize=0.0,
                 color='black', ecolor='0.6', markerfacecolor='black', markeredgecolor='black',
                 markeredgewidth=0.0, alpha=0.9)
    ax1.set_xlim(*xlim)
    ax1.invert_yaxis()
    ax1.set_xlabel('Phase')
    ax1.set_ylabel('Relative magnitude (comp - var)')
    ax1.set_title('Phase diagram')
    fig.suptitle(f'Star {star_id} | P={period:.6f} d', fontsize=10)
    fig.tight_layout()
    fig.savefig(out_png, dpi=PLOT_DPI)
    plt.close(fig)
    del fig, ax0, ax1, pg_p, pg_pow, px, py, pe, xp, yp, ep

def finite_int_or_nan(v):
    try:
        if pd.isna(v):
            return np.nan
        s = str(v).strip()
        if s == '':
            return np.nan
        return int(float(s))
    except Exception:
        return np.nan

def _pick_first_column(df, candidates):
    return next((c for c in candidates if c in df.columns), None)

def load_auto_diagnostics(paths=AUTO_DIAGNOSTIC_CSVS):
    """Load per-star x/y/mag/rms/n diagnostics for unsupervised comparison search."""
    for path in paths:
        if not path or not os.path.exists(path):
            continue
        try:
            df = pd.read_csv(path)
        except Exception:
            continue
        idc = _pick_first_column(df, ('star_id', 'master_index', 'id'))
        xc = _pick_first_column(df, ('x', 'X', 'variable_x'))
        yc = _pick_first_column(df, ('y', 'Y', 'variable_y'))
        magc = _pick_first_column(df, ('ref_mag', 'mag_ref', 'median_mag', 'mean_mag', 'variable_ref_mag'))
        nc = _pick_first_column(df, ('n_obs', 'n_frames', 'N', 'variable_n_obs'))
        rmsc = _pick_first_column(df, ('star_rms', 'rms_mag', 'rms', 'std_mag', 'star_sd', 'sigma_mad_mag'))
        if not all([idc, xc, yc, magc, nc, rmsc]):
            continue
        out = pd.DataFrame({
            'star_id': pd.to_numeric(df[idc], errors='coerce'),
            'x': pd.to_numeric(df[xc], errors='coerce'),
            'y': pd.to_numeric(df[yc], errors='coerce'),
            'ref_mag': pd.to_numeric(df[magc], errors='coerce'),
            'n_obs': pd.to_numeric(df[nc], errors='coerce'),
            'star_rms': pd.to_numeric(df[rmsc], errors='coerce'),
        }).dropna()
        if out.empty:
            continue
        out['star_id'] = out['star_id'].astype(int)
        return out.drop_duplicates('star_id').set_index('star_id', drop=False)
    return pd.DataFrame(columns=['star_id', 'x', 'y', 'ref_mag', 'n_obs', 'star_rms']).set_index('star_id', drop=False)

def select_auto_comparisons(var_id, diag_df, excluded_ids):
    if diag_df is None or diag_df.empty or int(var_id) not in diag_df.index:
        return []
    t = diag_df.loc[int(var_id)]
    rows = []
    for r in diag_df.itertuples(index=False):
        sid = int(r.star_id)
        if sid == int(var_id) or sid in excluded_ids:
            continue
        vals = (float(r.x), float(r.y), float(r.ref_mag), float(r.n_obs), float(r.star_rms))
        if not all(np.isfinite(v) for v in vals):
            continue
        if float(r.n_obs) < max(float(MIN_POINTS) * 4.0, min(2200.0, 0.45 * float(t.n_obs))):
            continue
        if float(r.star_rms) > float(AUTO_COMP_RMS_MAX):
            continue
        magdiff = abs(float(r.ref_mag) - float(t.ref_mag))
        if magdiff > float(AUTO_COMP_MAG_DIFF_MAX):
            continue
        dist = float(np.hypot(float(r.x) - float(t.x), float(r.y) - float(t.y)))
        if not (float(AUTO_COMP_DIST_MIN_PX) <= dist <= float(AUTO_COMP_DIST_MAX_PX)):
            continue
        score = (22.0 * float(r.star_rms) + 0.018 * magdiff + 0.00005 * dist
                 - 0.000003 * min(float(r.n_obs), 3000.0))
        rows.append((score, sid))
    rows.sort(key=lambda x: x[0])
    return [sid for _score, sid in rows[:int(AUTO_COMP_MAX_CANDIDATES)]]

def build_auto_comparison_map(targets, diag_df):
    if REFERENCE_MODE_SELECTION == 'single_comp1':
        return {int(t[0]): [] for t in targets}
    if not bool(AUTO_COMPARISON_SELECTION):
        return {int(t[0]): [] for t in targets}
    var_ids = {int(t[0]) for t in targets}
    out = {}
    for var, comp1, comp2 in targets:

        if REFERENCE_MODE_SELECTION == 'best_single' and (pd.notna(comp1) or pd.notna(comp2)):
            out[int(var)] = []
            continue
        excluded = set(var_ids)

        out[int(var)] = select_auto_comparisons(int(var), diag_df, excluded)
    return out

def periods_share_resolution(P1, jd1, P2, jd2):
    """Same frequency to within the resolving power of these two observing baselines."""
    if not (np.isfinite(P1) and np.isfinite(P2) and P1 > 0 and P2 > 0):
        return False
    b1 = float(np.nanmax(jd1) - np.nanmin(jd1)) if np.size(jd1) else np.nan
    b2 = float(np.nanmax(jd2) - np.nanmin(jd2)) if np.size(jd2) else np.nan
    if not (np.isfinite(b1) and np.isfinite(b2) and b1 > 0 and b2 > 0):
        return False
    df_res = max(1.0 / b1, 1.0 / b2)
    return abs(1.0 / float(P1) - 1.0 / float(P2)) <= float(SHARED_SIGNAL_FREQ_RESOLUTION) * df_res

def differential_time_correlation(jd1, y1, jd2, y2, tol_d=SHARED_SIGNAL_MATCH_TOL_D):
    """Robust Pearson correlation of genuine differential observations at matching times.

    This works entirely in the time domain; it does not fold, fit, interpolate, or resample.
    Each datum is paired only with the nearest real observation inside `tol_d`."""
    t1, a, t2, b = (np.asarray(v, dtype=np.float64) for v in (jd1, y1, jd2, y2))
    g1 = np.isfinite(t1) & np.isfinite(a); g2 = np.isfinite(t2) & np.isfinite(b)
    t1, a, t2, b = t1[g1], a[g1], t2[g2], b[g2]
    if min(t1.size, t2.size) < 100:
        return np.nan, 0
    o1, o2 = np.argsort(t1), np.argsort(t2)
    t1, a, t2, b = t1[o1], a[o1], t2[o2], b[o2]
    hi = np.searchsorted(t2, t1, side='left')
    jh = np.clip(hi, 0, t2.size - 1); jl = np.clip(hi - 1, 0, t2.size - 1)
    use_lo = np.abs(t1 - t2[jl]) <= np.abs(t1 - t2[jh])
    jj = np.where(use_lo, jl, jh)
    keep = np.abs(t1 - t2[jj]) <= float(tol_d)
    a, b = a[keep], b[jj[keep]]
    if a.size < 100:
        return np.nan, int(a.size)
    a, b = a - fast_median(a), b - fast_median(b)
    sa, sb = mad_sigma(a), mad_sigma(b)
    if not (np.isfinite(sa) and np.isfinite(sb) and sa > 0 and sb > 0):
        return np.nan, int(a.size)
    clean = (np.abs(a) <= 6.0 * sa) & (np.abs(b) <= 6.0 * sb)
    if int(clean.sum()) < 100:
        return np.nan, int(clean.sum())
    return float(np.corrcoef(a[clean] / sa, b[clean] / sb)[0, 1]), int(clean.sum())

def read_targets(csv_path):
    """(variable_id, comp1_id, comp2_id-or-nan) per row, from the NGC2506-format CSV."""
    df = pd.read_csv(csv_path)
    pick = lambda cands: next((c for c in cands if c in df.columns), None)
    idc, c1c, c2c = pick(ID_COLUMNS) or df.columns[0], pick(COMP1_COLUMNS), pick(COMP2_COLUMNS)
    id_s = df[idc].astype(float)
    c1_s = df[c1c] if c1c else pd.Series([np.nan] * len(df))
    c2_s = df[c2c] if c2c else pd.Series([np.nan] * len(df))
    targets = [(int(float(id_s.iloc[i])), finite_int_or_nan(c1_s.iloc[i]), finite_int_or_nan(c2_s.iloc[i])) for i in range(len(df))]
    return targets, (idc, c1c, c2c)

def run_cell2():
    os.makedirs(OUT_DIR, exist_ok=True)
    os.makedirs(PLOT_DIR, exist_ok=True)
    master_df = load_master_catalog(MASTER_MAP)
    targets, seen_var, src_bits = [], set(), []

    if USE_VARIABLE_CSV and os.path.exists(VARIABLE_CSV):
        csv_targets, (idc, _c1, _c2) = read_targets(VARIABLE_CSV)
        if TARGET_STAR_ID_FILTER:
            wanted = {int(v) for v in TARGET_STAR_ID_FILTER}
            csv_targets = [t for t in csv_targets if t[0] in wanted]
        if TARGET_LIMIT:
            csv_targets = csv_targets[:int(TARGET_LIMIT)]
        for t in csv_targets:
            if int(t[0]) not in seen_var:
                targets.append(t); seen_var.add(int(t[0]))
        src_bits.append(f'{len(csv_targets)} from {os.path.basename(VARIABLE_CSV)} (id={idc})')
    id_source = ' + '.join(src_bits) if src_bits else '(no targets)'
    print(f'Cell 2 | {len(targets)} variables: {id_source} | modes={REFERENCE_MODE_SELECTION}', flush=True)
    auto_diag = load_auto_diagnostics()
    auto_comp_map = build_auto_comparison_map(targets, auto_diag)
    if bool(AUTO_COMPARISON_SELECTION):
        n_auto = sum(len(v) for v in auto_comp_map.values())
        print(f'  auto comparison search: {n_auto} local stable candidates across {len(targets)} targets', flush=True)
    need = set()
    for var, c1, c2 in targets:
        need.add(int(var))
        if pd.notna(c1):
            need.add(int(c1))
        if pd.notna(c2):
            need.add(int(c2))
        for cid in auto_comp_map.get(int(var), []):
            need.add(int(cid))
    var_ids = {t[0] for t in targets}
    coord_map = {int(r.star_id): parse_coord(r.ra, r.dec)
                 for r in master_df.itertuples(index=False) if int(r.star_id) in var_ids}
    print('Building differential photometry (one pass over frames)...', flush=True)
    phot = build_photometry(master_df, sorted(need))
    jd_all, corr, emat, star_to_col = phot['jd'], phot['corr'], phot['emat'], phot['star_to_col']
    rows = []
    solution_bank = []
    n_plot = 0
    for k, (var, comp1, comp2) in enumerate(targets, start=1):
        c1n = comp1 if pd.notna(comp1) else np.nan
        c2n = comp2 if pd.notna(comp2) else np.nan
        if star_to_col.get(int(var)) is None:
            rows.append({'star_id': var, 'status': 'var_not_in_master_map', 'comp1': c1n, 'comp2': c2n})
            continue
        hjd = to_hjd(jd_all, coord_map.get(int(var)))
        modes = build_mode_series(hjd, corr, emat, star_to_col, var, comp1, comp2, auto_comp_map.get(int(var), []))
        if not modes:
            rows.append({'star_id': var, 'status': 'no_valid_comparison', 'comp1': c1n, 'comp2': c2n})
            continue
        results = []
        for mname, mjd, my, mdy in modes:
            r = analyze_star(mjd, my, mdy)
            if r is not None:
                r['reference_mode'] = mname
                results.append(r)
        if not results:
            rows.append({'star_id': var, 'status': 'no_period', 'comp1': c1n, 'comp2': c2n})
            continue
        res = max(results, key=lambda r: (float(r.get('fold_score', -np.inf)),
                                           float(r.get('phase_quality', -np.inf)),
                                           -float(r.get('theta_raw', r.get('theta', np.inf)))))

        base_fin = float(np.nanmax(res['jd']) - np.nanmin(res['jd']))
        Pp, sp = finalize_period(res['jd'], res['y'], res['dy'], float(res['period']), base_fin,
                                 ref_quality=res.get('phase_quality'))
        if sp is not None:
            res = {**res, **sp, 'period': float(Pp),
                   'candidate_source': str(res.get('candidate_source', '')) + '+polish'}
        P = res['period']
        theta = float(res.get('theta_raw', res['theta']))
        stype = classify(P, res, res['jd'], res['y'], res['dy'])

        ft = fourier_fit(res['jd'], res['y'], res['dy'], P)
        _uncertainty = estimate_cell2_period_amplitude_uncertainties(
            res['jd'], res['y'], res['dy'], P, res['amp'], var
        )
        period_sigma_days = _uncertainty['period_sigma_days']
        amplitude_sigma_mag = _uncertainty['amplitude_sigma_mag']
        stetson_j_value = welch_stetson_j(res['y'], res['dy'])
        skewness_value = finite_skewness(-np.asarray(res['y'], dtype=np.float64))
        log10P = float(np.log10(P)) if (np.isfinite(P) and P > 0) else np.nan

        shared_hits, shared_corr_max = [], np.nan
        if bool(SHARED_SIGNAL_GUARD):
            for old in solution_bank:
                if not periods_share_resolution(P, res['jd'], old['period'], old['jd']):
                    continue
                rho, nmatch = differential_time_correlation(res['jd'], res['y'], old['jd'], old['y'])
                if np.isfinite(rho) and rho >= float(SHARED_SIGNAL_CORR_MIN):
                    shared_hits.append((old, float(rho), int(nmatch)))
            if shared_hits:
                shared_corr_max = max(v[1] for v in shared_hits)
        inherited_group = any(h[0]['row'].get('status') == 'shared_signal_rejected' for h in shared_hits)
        reject_shared = bool(shared_hits) and (len(shared_hits) + 1 >= int(SHARED_SIGNAL_MIN_GROUP) or inherited_group)
        row_status = 'shared_signal_rejected' if reject_shared else 'ok'
        shared_ids = sorted({int(h[0]['star_id']) for h in shared_hits})
        if reject_shared:

            group_ids = sorted(set(shared_ids + [int(var)]))
            for old, rho, nmatch in shared_hits:
                old['row']['status'] = 'shared_signal_rejected'
                prior = {int(v) for v in str(old['row'].get('shared_signal_with', '')).split(';') if str(v).strip()}
                old['row']['shared_signal_with'] = ';'.join(map(str, sorted(prior | (set(group_ids) - {int(old['star_id'])}))))
                old_rho = float(old['row'].get('shared_signal_corr_max', np.nan))
                old['row']['shared_signal_corr_max'] = round(max(old_rho if np.isfinite(old_rho) else -np.inf, rho), 4)
                for old_png in glob.glob(os.path.join(PLOT_DIR, f'star_{old["star_id"]}_P_*.png')):
                    try:
                        os.remove(old_png); n_plot = max(0, n_plot - 1)
                    except OSError:
                        pass

        if SAVE_PLOTS and row_status == 'ok':
            for old_png in glob.glob(os.path.join(PLOT_DIR, f'star_{var}_P_*.png')):
                try:
                    os.remove(old_png)
                except OSError:
                    pass
            out_png = os.path.join(PLOT_DIR, f'star_{var}_P_{P:.6f}_{res["reference_mode"]}_{res["variant"]}.png')
            save_phase_plot(out_png, res['jd'], res['y'], res['dy'], P, var)
            n_plot += 1
            if n_plot % 20 == 0:
                plt.close('all')
                gc.collect()
        row = {'star_id': var,
                     'status': row_status, 'period_days': round(P, 6),
                     'period_sigma_days': float(period_sigma_days) if np.isfinite(period_sigma_days) else np.nan,
                     'amplitude': round(float(res['amp']), 5) if np.isfinite(res['amp']) else np.nan,
                     'amplitude_sigma_mag': float(amplitude_sigma_mag) if np.isfinite(amplitude_sigma_mag) else np.nan,
                     'theta': round(theta, 4), 'gap': round(float(res['gap']), 4),
                     'fold_score': round(float(res.get('fold_score', np.nan)), 4),
                     'phase_quality': round(float(res.get('phase_quality', np.nan)), 4),
                     'sine_coherence': round(float(res.get('sine_coherence', np.nan)), 4),
                     'display_sine': round(float(res.get('display_sine', np.nan)), 4),
                     'display_roughness': round(float(res.get('display_roughness', np.nan)), 4),
                     'display_spikiness': round(float(res.get('display_spikiness', np.nan)), 4),
                     'display_maxjump': round(float(res.get('display_maxjump', np.nan)), 4),
                     'plot_points': int(res.get('plot_points', 0)),
                     'plot_coverage': round(float(res.get('plot_coverage', np.nan)), 4),
                     'plot_gap': round(float(res.get('plot_gap', np.nan)), 4),
                     'plot_jump': round(float(res.get('plot_jump', np.nan)), 4),
                     'amp_ratio': round(float(res.get('amp_ratio', np.nan)), 4),
                     'power_norm': round(float(res.get('power_norm', np.nan)), 4),
                     'micro_penalty': round(float(res.get('micro_penalty', np.nan)), 4),
                     'long_alias_penalty': round(float(res.get('long_alias_penalty', np.nan)), 4),
                     'night_span_cycles': round(float(res.get('night_span_cycles', np.nan)), 4),
                     'n_obs': int(res['jd'].size), 'reference_mode': res['reference_mode'],
                     'comp1': c1n, 'comp2': c2n, 'variant': res['variant'],
                     'candidate_source': res.get('candidate_source', ''),
                     'eb_orbital': bool(res.get('eb', False)), 'suspected_type': stype,
                     'clear_fold': bool(theta < GOOD_THETA),
                     'log10P': round(log10P, 5) if np.isfinite(log10P) else np.nan,
                     'amplitude_fourier': round(float(ft['amplitude_fourier']), 5) if np.isfinite(ft['amplitude_fourier']) else np.nan,
                     'A1_fourier': round(float(ft['A1']), 5) if np.isfinite(ft['A1']) else np.nan,
                     'R21': round(float(ft['R21']), 4) if np.isfinite(ft['R21']) else np.nan,
                     'phi21': round(float(ft['phi21']), 4) if np.isfinite(ft['phi21']) else np.nan,
                     'R31': round(float(ft['R31']), 4) if np.isfinite(ft['R31']) else np.nan,
                     'phi31': round(float(ft['phi31']), 4) if np.isfinite(ft['phi31']) else np.nan,
                     'fourier_resid_rms': round(float(ft['resid_rms']), 5) if np.isfinite(ft['resid_rms']) else np.nan,
                     'fourier_fit_ok': int(ft['fourier_fit_ok']), 'stetson_J': float(stetson_j_value) if np.isfinite(stetson_j_value) else np.nan,
                     'skewness': float(skewness_value) if np.isfinite(skewness_value) else np.nan,
                     'shared_signal_corr_max': round(float(shared_corr_max), 4) if np.isfinite(shared_corr_max) else np.nan,
                     'shared_signal_with': ';'.join(map(str, shared_ids))}
        rows.append(row)
        solution_bank.append({'star_id': int(var), 'period': float(P), 'jd': res['jd'],
                              'y': res['y'], 'row': row})
        if k % 25 == 0 or k == len(targets):
            print(f'  [{k}/{len(targets)}] star {var}: P={P:.5f} d  theta={theta:.3f}  '
                  f'score={float(res.get("fold_score", np.nan)):.2f} q={float(res.get("phase_quality", np.nan)):.2f} '
                  f'display_sine={float(res.get("display_sine", np.nan)):.2f}  '
                  f'{res["reference_mode"]}/{res["variant"]}  {stype}', flush=True)
        if k % 25 == 0:
            gc.collect()
    summary = pd.DataFrame(rows).sort_values('star_id').reset_index(drop=True)
    summary.to_csv(OUT_SUMMARY, index=False)
    ok_n = int((summary.get('status') == 'ok').sum()) if 'status' in summary else 0
    shared_n = int((summary.get('status') == 'shared_signal_rejected').sum()) if 'status' in summary else 0
    if ok_n and 'phase_quality' in summary:
        ranking = summary.loc[summary['status'] == 'ok'].copy()
        ranking = ranking.sort_values(['phase_quality', 'fold_score', 'display_sine'],
                                      ascending=[False, False, False]).reset_index(drop=True)
        ranking.to_csv(OUT_RANKING, index=False)
    else:
        pd.DataFrame(rows).to_csv(OUT_RANKING, index=False)
    if ok_n:
        pp = pd.to_numeric(summary.loc[summary['status'] == 'ok', 'period_days'], errors='coerce').dropna()
        print(f'  period distribution: <0.1d {np.mean(pp < 0.1):.0%} | '
              f'0.1-0.35d {np.mean((pp >= 0.1) & (pp <= 0.35)):.0%} | '
              f'>0.35d {np.mean(pp > 0.35):.0%} | >=1d {np.mean(pp >= 1):.0%}')
    print(f'\nDone. {ok_n} stars solved, {shared_n} shared-signal candidates rejected, '
          f'{n_plot} phase plots in {PLOT_DIR}')
    print(f'Summary: {OUT_SUMMARY}')
    print(f'Phase-quality ranking: {OUT_RANKING}')
    return summary

if 'ngc6709' in SELECTED_CLUSTERS:
    run_cell2()


## 4. Gaia DR3 cross-match

For each selected cluster, match the reference catalogue to the nearest Gaia DR3 source within the configured angular radius. Preserve source IDs as integers/strings. Parallax-derived absolute magnitudes are direct estimates for positive parallaxes; the later CMD uses an adopted common cluster distance and reddening.


In [ ]:
CLUSTER_CONFIG = {
    key: {
        "name": values["label"],
        "radec_file": values["rfilter"] / "ref_stars_xy_radec.dat",
        "out_csv": values["results"] / f"{key}_gaia_crossmatch.csv",
    }
    for key, values in CLUSTER_PATHS.items()
}

GAIA_TABLE          = "gaiadr3.gaia_source"
MATCH_RADIUS_ARCSEC = 1.0
CONE_MARGIN_ARCMIN  = 2.0
GAIA_GMAG_LIMIT     = None
CACHE_GAIA_CONE     = USE_GAIA_CACHE
CEPHEID_WESENHEIT_COLOR_COEFF = 1.90

import os
import json
import hashlib
import time
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u
from astroquery.gaia import Gaia
from astroquery.utils.tap.core import TapPlus

gaia_mirror = TapPlus(url="https://gaia.ari.uni-heidelberg.de/tap")
gaia_services = [("ESA", Gaia), ("ARI mirror", gaia_mirror)]

def gaia_query_with_retry(adql, async_query=False, attempts_per_service=2):
    """Run against ESA first, then the public ARI Gaia TAP mirror if needed."""
    global gaia_services
    last_error = None
    for service_name, service in list(gaia_services):
        for attempt in range(1, attempts_per_service + 1):
            try:
                job = (service.launch_job_async(adql, dump_to_file=False) if async_query
                       else service.launch_job(adql, dump_to_file=False))
                if service_name != "ESA":
                    print(f"  Gaia query succeeded through {service_name}")
                gaia_services = [(service_name, service)] + [item for item in gaia_services if item[0] != service_name]
                return job.get_results().to_pandas()
            except Exception as exc:
                last_error = exc
                print(f"  {service_name} Gaia query attempt {attempt}/{attempts_per_service} failed: {exc}")
                if attempt < attempts_per_service:
                    time.sleep(3 * attempt)
        print(f"  switching from {service_name} to the next Gaia TAP service")
    raise RuntimeError(f"All Gaia TAP services failed: {last_error}") from last_error

for cluster_key in SELECTED_CLUSTERS:
    cfg = CLUSTER_CONFIG[cluster_key]
    print(f"\nGaia cross-match: {cfg['name']}")

    ids, xs, ys, ra_s, dec_s = [], [], [], [], []
    with open(cfg["radec_file"]) as fh:
        for line in fh:
            p = line.split()
            if len(p) < 5:
                continue
            try:
                sid, x, y = int(p[0]), float(p[1]), float(p[2])
            except ValueError:
                continue
            ids.append(sid); xs.append(x); ys.append(y); ra_s.append(p[3]); dec_s.append(p[4])
    star = SkyCoord(ra_s, dec_s, unit=(u.hourangle, u.deg))
    center = SkyCoord(np.median(star.ra), np.median(star.dec))
    cone_deg = (center.separation(star).max() + CONE_MARGIN_ARCMIN * u.arcmin).to(u.deg).value
    print(f"  {len(ids)} stars | field centre ({center.ra.deg:.5f}, {center.dec.deg:.5f}) | cone {cone_deg*60:.1f}'")

    cache_identity = {
        "catalogue_sha256": hashlib.sha256(Path(cfg["radec_file"]).read_bytes()).hexdigest(),
        "gaia_table": GAIA_TABLE,
        "match_radius_arcsec": MATCH_RADIUS_ARCSEC,
        "cone_margin_arcmin": CONE_MARGIN_ARCMIN,
        "gmag_limit": GAIA_GMAG_LIMIT,
        "columns": ["source_id", "ra", "dec", "pmra", "pmdec", "pmra_error", "pmdec_error", "parallax", "parallax_error", "phot_g_mean_mag", "bp_rp", "ag_gspphot", "ebpminrp_gspphot"],
    }
    cache_key = hashlib.sha256(json.dumps(cache_identity, sort_keys=True).encode()).hexdigest()[:20]
    cache = CACHE_ROOT / f"{cluster_key}_gaia_cone_{cache_key}.csv"
    gaia_columns = [
        "source_id", "ra", "dec", "pmra", "pmdec", "pmra_error",
        "pmdec_error", "parallax", "parallax_error", "phot_g_mean_mag",
        "bp_rp", "ag_gspphot", "ebpminrp_gspphot",
    ]
    gdf = None
    if CACHE_GAIA_CONE and os.path.exists(cache):
        cached = pd.read_csv(cache, dtype={"source_id": "string"})
        missing_cache_columns = [column for column in gaia_columns if column not in cached.columns]
        if missing_cache_columns:
            print(f"  cached Gaia cone is missing {missing_cache_columns}; refreshing the cone")
        else:
            gdf = cached
            print(f"  loaded cached Gaia cone: {len(gdf)} sources <- {cache}")
    if gdf is None:
        cols = ", ".join(gaia_columns)
        maglim = f" AND phot_g_mean_mag < {GAIA_GMAG_LIMIT}" if GAIA_GMAG_LIMIT is not None else ""
        adql = (f"SELECT {cols} FROM {GAIA_TABLE} "
                f"WHERE 1=CONTAINS(POINT('ICRS', ra, dec), "
                f"CIRCLE('ICRS', {center.ra.deg}, {center.dec.deg}, {cone_deg})){maglim}")
        gdf = gaia_query_with_retry(adql, async_query=True)
        print(f"  Gaia DR3 returned {len(gdf)} sources")
        if CACHE_GAIA_CONE:
            os.makedirs(os.path.dirname(cache), exist_ok=True)
            gdf.to_csv(cache, index=False)
            print(f"  cached cone -> {cache}")

    gaia = SkyCoord(gdf["ra"].to_numpy() * u.deg, gdf["dec"].to_numpy() * u.deg)
    idx, sep2d, _ = star.match_to_catalog_sky(gaia)
    matched = sep2d.arcsec <= MATCH_RADIUS_ARCSEC
    g = gdf.iloc[idx].reset_index(drop=True)

    plx  = g["parallax"].to_numpy(float)
    gmag = g["phot_g_mean_mag"].to_numpy(float)
    with np.errstate(invalid="ignore", divide="ignore"):
        abs_g = np.where(plx > 0, gmag + 5.0 * np.log10(plx) - 10.0, np.nan)
    bp_rp = g["bp_rp"].to_numpy(float)
    cepheid_wesenheit_g = abs_g - CEPHEID_WESENHEIT_COLOR_COEFF * bp_rp

    src = pd.array(g["source_id"].to_numpy(), dtype="Int64")
    src[~matched] = pd.NA

    out = pd.DataFrame({
        "star_id":         ids,
        "x":               xs,
        "y":               ys,
        "ra_deg":          np.round(star.ra.deg, 7),
        "dec_deg":         np.round(star.dec.deg, 7),
        "ra_sexagesimal":  ra_s,
        "dec_sexagesimal": dec_s,
        "gaia_source_id":  src,
        "pmra":            g["pmra"].to_numpy(float),
        "pmra_error":      g["pmra_error"].to_numpy(float),
        "pmdec":           g["pmdec"].to_numpy(float),
        "pmdec_error":     g["pmdec_error"].to_numpy(float),
        "parallax":        plx,
        "parallax_error":  g["parallax_error"].to_numpy(float),
        "bp_rp":           bp_rp,
        "ag_gspphot":      g["ag_gspphot"].to_numpy(float),
        "ebpminrp_gspphot": g["ebpminrp_gspphot"].to_numpy(float),
        "gmag":            gmag,
        "abs_gmag":        abs_g,
        "cepheid_wesenheit_g": cepheid_wesenheit_g,
        "gaia_sep_arcsec": np.round(sep2d.arcsec, 4),
    })

    gaia_value_cols = ["pmra", "pmra_error", "pmdec", "pmdec_error", "parallax",
                       "parallax_error", "bp_rp", "ag_gspphot", "ebpminrp_gspphot",
                       "gmag", "abs_gmag", "cepheid_wesenheit_g", "gaia_sep_arcsec"]
    out.loc[~matched, gaia_value_cols] = np.nan

    os.makedirs(os.path.dirname(cfg["out_csv"]), exist_ok=True)
    out.to_csv(cfg["out_csv"], index=False)

    print(f"  matched {int(matched.sum())}/{len(ids)} stars "
          f"(within {MATCH_RADIUS_ARCSEC}\", median sep {np.nanmedian(sep2d.arcsec[matched]):.3f}\")")
    print(f"  with 5-param astrometry (finite parallax): {int(np.isfinite(out['parallax']).sum())}")
    print(f"  saved -> {cfg['out_csv']}")
    out.head(8)


## 5. Astrometric membership

Use fixed cluster proper-motion centres, a fitted core-plus-wing radial proper-motion model, a heteroscedastic cluster-parallax likelihood and a field-parallax KDE. Export PM-only probability, joint astrometric probability and the separately gated membership score for all stars. Secure and borderline labels follow the configured thresholds and gates.


In [ ]:
TABLE_DIR = TABLE_ROOT
FIGURE_DIR = FIGURE_ROOT

from pathlib import Path
import math

import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.ticker import AutoMinorLocator
import numpy as np
import pandas as pd
from scipy.optimize import brentq, minimize
from scipy.special import gammaln, logsumexp
from scipy.stats import chi2

PNG_DPI = 600

SECURE_THRESHOLD = 0.60
BORDERLINE_THRESHOLD = 0.48
PARALLAX_ERROR_MAX_MAS = 0.20
PARALLAX_FIELD_KDE_BANDWIDTH_MAS = 0.02

MEMBERSHIP_CONFIG = {'ngc2506': {'label': 'NGC 2506',
             'pm_centre': (-2.571, 3.912),
             'parallax_window': (0.05, 0.5),
             'parallax_gate': (0.27848368418079367, 0.05017850971214709),
             'radial_initial': (12.5, 18400.0, 0.091, 1100.0, 0.184)},
 'ngc6709': {'label': 'NGC 6709',
             'pm_centre': (1.414, -3.532),
             'parallax_window': (0.7, 1.25),
             'parallax_gate': (0.9210768759289425, 0.044759241663590965),
             'radial_initial': (27.4, 1600.0, 0.101, 192.0, 0.248)}}
for key, config in MEMBERSHIP_CONFIG.items():
    config["crossmatch"] = CLUSTER_PATHS[key]["results"] / f"{key}_gaia_crossmatch.csv"

PLOT_STYLE = {
    "font.family": "serif",
    "mathtext.fontset": "stix",
    "font.size": 7.4,
    "axes.labelsize": 8.0,
    "xtick.labelsize": 6.6,
    "ytick.labelsize": 6.6,
    "axes.linewidth": 0.65,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
}

def _poisson_nll(expected, observed):
    expected = np.clip(np.asarray(expected, float), 1.0e-12, None)
    observed = np.asarray(observed, float)
    return float(np.sum(expected - observed * np.log(expected) + gammaln(observed + 1.0)))

def _poisson_deviance(expected, observed):
    expected = np.clip(np.asarray(expected, float), 1.0e-12, None)
    observed = np.asarray(observed, float)
    positive = observed > 0
    terms = np.array(expected, copy=True)
    terms[positive] = (
        observed[positive] * np.log(observed[positive] / expected[positive])
        - (observed[positive] - expected[positive])
    )
    return float(2.0 * np.sum(terms))

def _fit_annular_corewing(radius, initial, bin_width=0.05, maximum_radius=1.50):
    """Poisson fit of two Gaussian excesses plus a constant PM field.

    The wider Gaussian is an empirical wing, not a separate physical cluster.
    The parametrisation s_wing=s_core+delta_s prevents label swapping.
    """
    edges = np.arange(0.0, maximum_radius + 0.5 * bin_width, bin_width)
    counts, _ = np.histogram(radius, bins=edges)
    inner, outer = edges[:-1], edges[1:]
    area = np.pi * (outer**2 - inner**2)
    midpoint = 0.5 * (inner + outer)
    density = counts / area
    error = np.sqrt(np.maximum(counts, 1.0)) / area

    field0, amp10, sigma10, amp20, sigma20 = initial
    delta0 = max(sigma20 - sigma10, 0.01)
    starts = []
    for core_scale in (0.85, 1.0, 1.15):
        for wing_scale in (0.85, 1.0, 1.20):
            starts.append(np.log([
                field0,
                amp10,
                sigma10 * core_scale,
                amp20,
                max(sigma20 * wing_scale - sigma10 * core_scale, 0.01),
            ]))

    bounds = [
        (np.log(0.01), np.log(1.0e4)),
        (np.log(10.0), np.log(1.0e7)),
        (np.log(0.01), np.log(0.50)),
        (np.log(0.1), np.log(1.0e6)),
        (np.log(0.002), np.log(1.0)),
    ]

    def unpack(log_parameters):
        field, amp1, sigma1, amp2, delta_sigma = np.exp(log_parameters)
        return field, amp1, sigma1, amp2, sigma1 + delta_sigma

    def expected(log_parameters):
        field, amp1, sigma1, amp2, sigma2 = unpack(log_parameters)
        core_counts = 2.0 * np.pi * amp1 * sigma1**2 * (
            np.exp(-inner**2 / (2.0 * sigma1**2))
            - np.exp(-outer**2 / (2.0 * sigma1**2))
        )
        wing_counts = 2.0 * np.pi * amp2 * sigma2**2 * (
            np.exp(-inner**2 / (2.0 * sigma2**2))
            - np.exp(-outer**2 / (2.0 * sigma2**2))
        )
        return field * area + core_counts + wing_counts

    candidates = []
    for start in starts:
        fit = minimize(
            lambda p: _poisson_nll(expected(p), counts),
            start,
            method="L-BFGS-B",
            bounds=bounds,
            options={"maxiter": 20000, "ftol": 1.0e-14, "gtol": 1.0e-9},
        )
        if np.isfinite(fit.fun):
            candidates.append(fit)
    if not candidates:
        raise RuntimeError("Core+wing annular-density fit produced no finite solution")
    fit = min(candidates, key=lambda item: item.fun)
    if not fit.success:
        raise RuntimeError(f"Core+wing annular-density fit failed: {fit.message}")

    field, amp1, sigma1, amp2, sigma2 = unpack(fit.x)

    def cluster_minus_field(value):
        cluster = (
            amp1 * np.exp(-value**2 / (2.0 * sigma1**2))
            + amp2 * np.exp(-value**2 / (2.0 * sigma2**2))
        )
        return float(cluster - field)

    cutoff = brentq(cluster_minus_field, 0.0, maximum_radius)
    fitted_core = amp1 * np.exp(-midpoint**2 / (2.0 * sigma1**2))
    fitted_wing = amp2 * np.exp(-midpoint**2 / (2.0 * sigma2**2))
    fitted_total = field + fitted_core + fitted_wing
    expected_counts = expected(fit.x)
    deviance = _poisson_deviance(expected_counts, counts)
    dof = len(counts) - 5

    profile = pd.DataFrame({
        "r_inner_masyr": inner,
        "r_outer_masyr": outer,
        "r_mid_masyr": midpoint,
        "star_count": counts,
        "annular_area_masyr2": area,
        "surface_density_stars_per_masyr2": density,
        "poisson_error_stars_per_masyr2": error,
        "fitted_field_density_stars_per_masyr2": field,
        "fitted_core_density_stars_per_masyr2": fitted_core,
        "fitted_wing_density_stars_per_masyr2": fitted_wing,
        "fitted_total_density_stars_per_masyr2": fitted_total,
    })
    parameters = {
        "field_density": float(field),
        "core_amplitude": float(amp1),
        "core_sigma": float(sigma1),
        "wing_amplitude": float(amp2),
        "wing_sigma": float(sigma2),
        "cutoff": float(cutoff),
        "poisson_deviance": deviance,
        "degrees_of_freedom": int(dof),
        "goodness_of_fit_p_value": float(chi2.sf(deviance, dof)),
        "aic": float(2 * 5 + 2 * fit.fun),
        "bic": float(5 * np.log(len(counts)) + 2 * fit.fun),
    }
    return parameters, profile

def _pm_corewing_probability(radius, parameters):
    cluster = (
        parameters["core_amplitude"]
        * np.exp(-radius**2 / (2.0 * parameters["core_sigma"]**2))
        + parameters["wing_amplitude"]
        * np.exp(-radius**2 / (2.0 * parameters["wing_sigma"]**2))
    )
    return cluster / (cluster + parameters["field_density"])

def _fit_heteroscedastic_cluster_parallax(parallax, error, pm_weight, window):
    low, high = window
    keep = (
        np.isfinite(parallax)
        & np.isfinite(error)
        & (error > 0.0)
        & (error < PARALLAX_ERROR_MAX_MAS)
        & (parallax >= low)
        & (parallax <= high)
        & np.isfinite(pm_weight)
    )
    y, e, w = parallax[keep], error[keep], pm_weight[keep]

    def objective(theta):
        mean = theta[0]
        intrinsic_sigma = np.exp(theta[1])
        variance = intrinsic_sigma**2 + e**2
        return 0.5 * np.sum(
            w * (np.log(2.0 * np.pi * variance) + (y - mean)**2 / variance)
        )

    fit = minimize(
        objective,
        [np.average(y, weights=w), np.log(0.03)],
        method="L-BFGS-B",
        bounds=[(low, high), (np.log(1.0e-5), np.log(1.0))],
        options={"maxiter": 20000, "ftol": 1.0e-14, "gtol": 1.0e-10},
    )
    if not fit.success:
        raise RuntimeError(f"Heteroscedastic parallax fit failed: {fit.message}")
    return float(fit.x[0]), float(np.exp(fit.x[1])), int(keep.sum()), float(w.sum())

def _field_kde_log_density(target, target_error, field, field_error, bandwidth):
    result = np.full(len(target), np.nan)
    valid = np.isfinite(target) & np.isfinite(target_error) & (target_error > 0.0)
    valid_index = np.flatnonzero(valid)
    for start in range(0, len(valid_index), 256):
        index = valid_index[start:start + 256]
        variance = (
            bandwidth**2
            + target_error[index, None]**2
            + field_error[None, :]**2
        )
        log_kernel = -0.5 * (
            (target[index, None] - field[None, :])**2 / variance
            + np.log(2.0 * np.pi * variance)
        )
        result[index] = logsumexp(log_kernel, axis=1) - np.log(len(field))
    return result

def _style_axis(axis):
    axis.tick_params(which="major", length=3.0, width=0.65)
    axis.tick_params(which="minor", length=1.7, width=0.5)
    axis.xaxis.set_minor_locator(AutoMinorLocator())
    axis.yaxis.set_minor_locator(AutoMinorLocator())

def _save_figure(figure, path_without_suffix):
    path_without_suffix.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(path_without_suffix.with_suffix(".pdf"), bbox_inches="tight")
    figure.savefig(
        path_without_suffix.with_suffix(".png"),
        dpi=PNG_DPI,
        bbox_inches="tight",
    )
    plt.close(figure)

def _plot_membership(prefix, config, result, profile):
    figure_directory = FIGURE_DIR / prefix
    centre = np.asarray(config["pm_centre"], float)
    cutoff = float(result["pm_cutoff_radius_masyr"].iloc[0])

    finite = result[["pmra", "pmdec"]].notna().all(axis=1)
    pm = result.loc[finite, ["pmra", "pmdec"]].to_numpy(float)
    fig, axis = plt.subplots(figsize=(3.45, 3.25))
    axis.scatter(pm[:, 0], pm[:, 1], s=1.0, color="black", linewidths=0, rasterized=True)
    axis.add_patch(Circle(centre, cutoff, fill=False, edgecolor="#0057ff", linewidth=1.0))
    xlow, xhigh = np.percentile(pm[:, 0], [1, 99])
    ylow, yhigh = np.percentile(pm[:, 1], [1, 99])
    axis.set_xlim(xlow - 0.04 * (xhigh - xlow), xhigh + 0.04 * (xhigh - xlow))
    axis.set_ylim(ylow - 0.04 * (yhigh - ylow), yhigh + 0.04 * (yhigh - ylow))
    axis.set_xlabel(r"$\mu_{\alpha *}$ (mas yr$^{-1}$)")
    axis.set_ylabel(r"$\mu_{\delta}$ (mas yr$^{-1}$)")
    axis.set_aspect("equal", adjustable="box")
    _style_axis(axis)
    fig.tight_layout(pad=0.35)
    _save_figure(fig, figure_directory / f"{prefix}_vpd_membership_zoomout")

    fig, axis = plt.subplots(figsize=(3.35, 2.55))
    axis.errorbar(
        profile["r_mid_masyr"],
        profile["surface_density_stars_per_masyr2"],
        yerr=profile["poisson_error_stars_per_masyr2"],
        fmt="s", ms=2.2, color="black", ecolor="black",
        elinewidth=0.55, capsize=1.2,
    )
    axis.plot(
        profile["r_mid_masyr"],
        profile["fitted_total_density_stars_per_masyr2"],
        color="#2166ac", linewidth=1.15, label="core + wing + field",
    )
    axis.plot(
        profile["r_mid_masyr"],
        profile["fitted_field_density_stars_per_masyr2"],
        color="0.35", linewidth=0.75, linestyle="--", label="field",
    )
    ymax = 1.08 * float(
        profile.loc[
            profile["r_mid_masyr"] <= 1.06,
            "surface_density_stars_per_masyr2",
        ].max()
    )
    axis.vlines(cutoff, 0.0, 0.10 * ymax, color="#b2182b",
                linestyle=(0, (2, 2)), linewidth=0.9)
    axis.set_xlim(0.0, 1.06)
    axis.set_ylim(-0.04 * ymax, ymax)
    axis.set_xlabel(r"Proper-motion radial distance (mas yr$^{-1}$)")
    axis.set_ylabel(r"$\Sigma_{\mu}$ [stars (mas yr$^{-1}$)$^{-2}$]")
    _style_axis(axis)
    fig.tight_layout(pad=0.35)
    _save_figure(fig, figure_directory / f"{prefix}_pm_radial_density")

    for column, label, stem in (
        ("p_mu", r"Proper-motion probability, $P_{\mu}$", "proper_motion_probability"),
        ("p_ast", r"Joint astrometric probability, $P_{\rm ast}$", "joint_astrometric_probability"),
    ):
        fig, axis = plt.subplots(figsize=(3.35, 2.55))
        axis.hist(result[column].dropna(), bins=np.linspace(0, 1, 21),
                  histtype="step", color="black", linewidth=0.85)
        axis.axvline(SECURE_THRESHOLD, color="#b2182b",
                     linestyle=(0, (3, 2)), linewidth=0.85)
        axis.set_xlim(0, 1)
        axis.set_xlabel(label)
        axis.set_ylabel("Number of stars")
        _style_axis(axis)
        fig.tight_layout(pad=0.35)
        _save_figure(fig, figure_directory / f"{prefix}_{stem}")

    mean = float(result["parallax_fit_mean_mas"].iloc[0])
    sigma = float(result["parallax_fit_sigma_mas"].iloc[0])
    low, high = config["parallax_window"]
    sample = result.loc[
        result["in_pm_circle"]
        & result["parallax_error_lt_0p2_mas"]
        & result["parallax"].between(low, high),
        "parallax",
    ].dropna().to_numpy(float)
    edges = np.arange(low, high + 1.0e-12, 0.02)
    if edges[-1] < high - 1.0e-12:
        edges = np.r_[edges, high]
    counts, _ = np.histogram(sample, bins=edges)
    ymax = max(float(counts.max()) * 1.10, 1.0)
    fig, axis = plt.subplots(figsize=(3.35, 2.55))
    axis.stairs(counts, edges, color="black", linewidth=0.85)
    for value in (mean - 3.0 * sigma, mean + 3.0 * sigma):
        axis.vlines(value, 0.0, 0.10 * ymax, color="#b2182b",
                    linestyle=(0, (2, 2)), linewidth=0.85)
    axis.set_xlim(low, high)
    axis.set_ylim(0.0, ymax)
    axis.set_xlabel("Parallax (mas)")
    axis.set_ylabel("Number of stars")
    _style_axis(axis)
    fig.tight_layout(pad=0.35)
    _save_figure(fig, figure_directory / f"{prefix}_parallax_histogram")

def run_membership(prefix, config):
    frame = pd.read_csv(config["crossmatch"], dtype={"gaia_source_id": "string"})
    numeric = ["pmra", "pmra_error", "pmdec", "pmdec_error", "parallax", "parallax_error"]
    for column in numeric:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")

    pm_valid = frame[["pmra", "pmdec"]].notna().all(axis=1).to_numpy()
    centre = np.asarray(config["pm_centre"], float)
    radius_all = np.full(len(frame), np.nan)
    radius_all[pm_valid] = np.hypot(
        frame.loc[pm_valid, "pmra"].to_numpy(float) - centre[0],
        frame.loc[pm_valid, "pmdec"].to_numpy(float) - centre[1],
    )

    radial_parameters, profile = _fit_annular_corewing(
        radius_all[pm_valid], config["radial_initial"]
    )
    p_mu = np.full(len(frame), np.nan)
    p_mu[pm_valid] = _pm_corewing_probability(
        radius_all[pm_valid], radial_parameters
    )
    in_circle = pm_valid & (radius_all <= radial_parameters["cutoff"])

    parallax = frame["parallax"].to_numpy(float)
    parallax_error = frame["parallax_error"].to_numpy(float)
    parallax_mean_het, parallax_sigma_int, parallax_fit_n, parallax_fit_weight = (
        _fit_heteroscedastic_cluster_parallax(
            parallax, parallax_error, p_mu, config["parallax_window"]
        )
    )

    field_sample = (
        pm_valid
        & (radius_all > radial_parameters["cutoff"])
        & np.isfinite(parallax)
        & np.isfinite(parallax_error)
        & (parallax_error > 0.0)
        & (parallax_error < PARALLAX_ERROR_MAX_MAS)
    )
    field_parallax = parallax[field_sample]
    field_parallax_error = parallax_error[field_sample]
    if len(field_parallax) < 20:
        raise RuntimeError(f"Too few PM-field stars for {config['label']} parallax KDE")

    parallax_cluster_log_density = np.full(len(frame), np.nan)
    parallax_valid = (
        np.isfinite(parallax)
        & np.isfinite(parallax_error)
        & (parallax_error > 0.0)
    )
    variance = parallax_sigma_int**2 + parallax_error[parallax_valid]**2
    parallax_cluster_log_density[parallax_valid] = -0.5 * (
        (parallax[parallax_valid] - parallax_mean_het)**2 / variance
        + np.log(2.0 * np.pi * variance)
    )
    parallax_field_log_density = _field_kde_log_density(
        parallax,
        parallax_error,
        field_parallax,
        field_parallax_error,
        PARALLAX_FIELD_KDE_BANDWIDTH_MAS,
    )

    p_ast = np.full(len(frame), np.nan)
    joint_valid = (
        np.isfinite(p_mu)
        & np.isfinite(parallax_cluster_log_density)
        & np.isfinite(parallax_field_log_density)
    )
    q = np.clip(p_mu[joint_valid], 1.0e-300, 1.0 - 1.0e-15)
    log_odds = (
        np.log(q) - np.log1p(-q)
        + parallax_cluster_log_density[joint_valid]
        - parallax_field_log_density[joint_valid]
    )
    p_ast[joint_valid] = 1.0 / (
        1.0 + np.exp(-np.clip(log_odds, -700.0, 700.0))
    )

    parallax_gate_mean, parallax_gate_sigma = config["parallax_gate"]
    in_parallax = (
        np.isfinite(parallax)
        & (np.abs(parallax - parallax_gate_mean) <= 3.0 * parallax_gate_sigma)
    )
    good_parallax_error = (
        np.isfinite(parallax_error)
        & (parallax_error > 0.0)
        & (parallax_error < PARALLAX_ERROR_MAX_MAS)
    )
    astrometric_gate = in_circle & in_parallax & good_parallax_error
    gated_score = np.where(astrometric_gate & np.isfinite(p_ast), p_ast, 0.0)
    secure = gated_score >= SECURE_THRESHOLD
    borderline = (
        (gated_score >= BORDERLINE_THRESHOLD)
        & (gated_score < SECURE_THRESHOLD)
    )

    result = frame.copy()
    result.insert(0, "cluster", config["label"])
    result["pm_centre_ra_masyr_literature"] = centre[0]
    result["pm_centre_dec_masyr_literature"] = centre[1]
    result["pm_radial_distance_masyr"] = radius_all
    result["pm_cutoff_radius_masyr"] = radial_parameters["cutoff"]
    result["in_pm_circle"] = in_circle

    result["p_mu"] = p_mu
    result["p_ast"] = p_ast
    result["gated_astrometric_membership_score"] = gated_score
    result["parallax_fit_mean_mas"] = parallax_gate_mean
    result["parallax_fit_sigma_mas"] = parallax_gate_sigma
    result["parallax_3sigma_low_mas"] = parallax_gate_mean - 3.0 * parallax_gate_sigma
    result["parallax_3sigma_high_mas"] = parallax_gate_mean + 3.0 * parallax_gate_sigma
    result["parallax_error_lt_0p2_mas"] = good_parallax_error
    result["in_parallax_3sigma_window"] = in_parallax
    result["secure_member"] = secure
    result["borderline_member"] = borderline
    result["membership_status"] = np.select(
        [secure, borderline],
        ["Secure member", "Borderline"],
        default="Field/non-secure",
    )
    result["parallax_cluster_mean_heteroscedastic_mas"] = parallax_mean_het
    result["parallax_cluster_intrinsic_sigma_mas"] = parallax_sigma_int
    result["parallax_field_kde_bandwidth_mas"] = PARALLAX_FIELD_KDE_BANDWIDTH_MAS
    result["parallax_cluster_log_density"] = parallax_cluster_log_density
    result["parallax_field_log_density"] = parallax_field_log_density

    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    result.to_csv(TABLE_DIR / f"{prefix}_membership_all_stars.csv", index=False)
    profile.to_csv(TABLE_DIR / f"{prefix}_proper_motion_radial_density.csv", index=False)

    parameters = {
        "cluster": config["label"],
        "pm_centre_ra_masyr": centre[0],
        "pm_centre_dec_masyr": centre[1],
        "pm_cutoff_radius_masyr": radial_parameters["cutoff"],
        "annular_field_density": radial_parameters["field_density"],
        "annular_core_amplitude": radial_parameters["core_amplitude"],
        "annular_core_sigma_masyr": radial_parameters["core_sigma"],
        "annular_wing_amplitude": radial_parameters["wing_amplitude"],
        "annular_wing_sigma_masyr": radial_parameters["wing_sigma"],
        "pm_poisson_deviance": radial_parameters["poisson_deviance"],
        "pm_degrees_of_freedom": radial_parameters["degrees_of_freedom"],
        "pm_goodness_of_fit_p_value": radial_parameters["goodness_of_fit_p_value"],
        "parallax_cluster_mean_heteroscedastic_mas": parallax_mean_het,
        "parallax_cluster_intrinsic_sigma_mas": parallax_sigma_int,
        "parallax_cluster_fit_n": parallax_fit_n,
        "parallax_cluster_fit_weight_sum": parallax_fit_weight,
        "parallax_field_kde_n": len(field_parallax),
        "parallax_field_kde_bandwidth_mas": PARALLAX_FIELD_KDE_BANDWIDTH_MAS,
        "parallax_gate_mean_mas": parallax_gate_mean,
        "parallax_gate_sigma_mas": parallax_gate_sigma,
        "n_secure": int(secure.sum()),
        "n_borderline": int(borderline.sum()),
    }
    _plot_membership(prefix, config, result, profile)
    print(
        f"{config['label']}: R_mu={radial_parameters['cutoff']:.6f} mas/yr, "
        f"p_mu is PM-only, p_ast is joint, "
        f"secure={secure.sum()}, borderline={borderline.sum()}"
    )
    return parameters

with plt.rc_context(PLOT_STYLE):
    membership_parameters = [
        run_membership(prefix, config)
        for prefix, config in MEMBERSHIP_CONFIG.items() if prefix in SELECTED_CLUSTERS
    ]
pd.DataFrame(membership_parameters).to_csv(
    TABLE_DIR / "membership_parameters.csv", index=False
)


## 6. Colour–magnitude diagrams

Apply the adopted cluster reddening and true distance modulus to all stars: `(BP-RP)_0 = BP-RP - 1.28 E(B-V)` and `M_G,0 = G - 2.45 E(B-V) - (m-M)_0`. The common-distance values compare the cluster sequence; they are not individual distances for field stars.


In [ ]:
TABLE_DIR = TABLE_ROOT
FIGURE_DIR = FIGURE_ROOT

from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
import numpy as np
import pandas as pd

PNG_DPI = 600

CMD_CONFIG = {
    "ngc2506": {"label": "NGC 2506", "ebv": 0.080, "distance_modulus": 12.458},
    "ngc6709": {"label": "NGC 6709", "ebv": 0.290, "distance_modulus": 10.134},
}

PLOT_STYLE = {
    "font.family": "serif",
    "mathtext.fontset": "stix",
    "font.size": 7.4,
    "axes.labelsize": 8.0,
    "xtick.labelsize": 6.6,
    "ytick.labelsize": 6.6,
    "axes.linewidth": 0.65,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
}

def _save_figure(fig, path_without_suffix):
    path_without_suffix.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path_without_suffix.with_suffix(".pdf"), bbox_inches="tight", pad_inches=0.035)
    fig.savefig(path_without_suffix.with_suffix(".png"), dpi=PNG_DPI,
                bbox_inches="tight", pad_inches=0.035)
    plt.close(fig)

def run_cmd(prefix, config):
    frame = pd.read_csv(
        TABLE_DIR / f"{prefix}_membership_all_stars.csv",
        dtype={"gaia_source_id": "string"},
    )
    ebv = float(config["ebv"])
    distance_modulus = float(config["distance_modulus"])
    a_g = 2.45 * ebv
    e_bp_rp = 1.28 * ebv

    frame["E_BV_adopted_mag"] = ebv
    frame["A_G_adopted_mag"] = a_g
    frame["E_BP_RP_adopted_mag"] = e_bp_rp
    frame["bp_rp_0_mag"] = pd.to_numeric(frame["bp_rp"], errors="coerce") - e_bp_rp
    frame["g_0_mag"] = pd.to_numeric(frame["gmag"], errors="coerce") - a_g
    frame["M_G_0_common_distance_mag"] = frame["g_0_mag"] - distance_modulus
    frame["distance_modulus_adopted_mag"] = distance_modulus

    all_path = TABLE_DIR / f"{prefix}_cmd_all_stars.csv"
    frame.to_csv(all_path, index=False)
    finite = frame[["bp_rp_0_mag", "M_G_0_common_distance_mag"]].notna().all(axis=1)
    members = finite & (frame["secure_member"] | frame["borderline_member"])
    field = finite & ~members
    fig, axis = plt.subplots(figsize=(3.35, 3.15))
    axis.scatter(
        frame.loc[field, "bp_rp_0_mag"], frame.loc[field, "M_G_0_common_distance_mag"],
        s=2.0, color="#a6a6a6", alpha=0.45, linewidths=0, rasterized=True,
        label="Field/non-secure",
    )
    axis.scatter(
        frame.loc[members, "bp_rp_0_mag"], frame.loc[members, "M_G_0_common_distance_mag"],
        s=4.5, color="#2166ac", linewidths=0, rasterized=True,
        label="Secure + borderline",
    )
    axis.set_xlabel(r"$(BP-RP)_0$ (mag)")
    axis.set_ylabel(r"$M_{G,0}$ (mag)")
    axis.invert_yaxis()
    axis.tick_params(direction="in", which="both", top=True, right=True)
    axis.xaxis.set_minor_locator(AutoMinorLocator(2))
    axis.yaxis.set_minor_locator(AutoMinorLocator(2))
    axis.legend(frameon=False, loc="best", markerscale=1.8)
    fig.tight_layout(pad=0.35)
    _save_figure(fig, FIGURE_DIR / prefix / f"{prefix}_cmd_membership")
    print(
        f"{config['label']}: E(B-V)={ebv:.3f}, (m-M)_0={distance_modulus:.3f}, "
        f"members shown={int(members.sum())}"
    )

with plt.rc_context(PLOT_STYLE):
    for cluster_prefix, cluster_config in CMD_CONFIG.items():
        if cluster_prefix not in SELECTED_CLUSTERS:
            continue
        run_cmd(cluster_prefix, cluster_config)
